# Phase 2: Ground Truth & Temporal Snapshots



In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

# Existing project directories
GC1_ROOT = Path('/content/drive/MyDrive/GC1')

ENTITY_OUTPUT = GC1_ROOT / 'entity_resolution_output'

print("GC1 directory:", GC1_ROOT)
print("Phase 1 outputs:", ENTITY_OUTPUT)

print("\nDirectory exists:", ENTITY_OUTPUT.exists())

Mounted at /content/drive
GC1 directory: /content/drive/MyDrive/GC1
Phase 1 outputs: /content/drive/MyDrive/GC1/entity_resolution_output

Directory exists: True


In [ ]:
import json
import zipfile
import pandas as pd

# Search inside the existing Phase 1 output directory
archives = sorted(
    ENTITY_OUTPUT.rglob("temporal_kg_v2_output.zip")
)

folders = sorted([
    p for p in ENTITY_OUTPUT.rglob("temporal_kg_v2_output")
    if p.is_dir() and (p / "nodes.jsonl").exists()
])

print("ZIP archives found:")
for p in archives:
    print(p)

print("\nExtracted folders found:")
for p in folders:
    print(p)

ZIP archives found:
/content/drive/MyDrive/GC1/entity_resolution_output/temporal_kg_v2_output.zip

Extracted folders found:
/content/drive/MyDrive/GC1/entity_resolution_output/temporal_kg_v2_output


In [ ]:
if folders:
    if len(folders) > 1:
        raise ValueError(
            "Multiple extracted KG folders found. "
            "Please select the correct final export."
        )

    KG_ROOT = folders[0]

elif archives:
    if len(archives) > 1:
        raise ValueError(
            "Multiple KG ZIP archives found. "
            "Please select the correct final archive."
        )

    # Extract into Colab's temporary storage,
    # not into a new Google Drive folder.
    KG_ROOT = Path("/content/temporal_kg_v2_output")

    with zipfile.ZipFile(archives[0]) as z:
        z.extractall(KG_ROOT)

    # Handle ZIP files containing an additional root folder
    if not (KG_ROOT / "nodes.jsonl").exists():
        matches = list(KG_ROOT.rglob("nodes.jsonl"))

        if len(matches) != 1:
            raise FileNotFoundError(
                "Could not uniquely locate nodes.jsonl"
            )

        KG_ROOT = matches[0].parent

else:
    raise FileNotFoundError(
        "No final Temporal KG export found inside "
        "entity_resolution_output"
    )

print("Selected KG directory:", KG_ROOT)
print("\nFiles available:")

for p in sorted(KG_ROOT.iterdir()):
    print(p.name)

Selected KG directory: /content/drive/MyDrive/GC1/entity_resolution_output/temporal_kg_v2_output

Files available:
README.md
edges
links.jsonl
manifest.json
neo4j
nodes
nodes.jsonl
sample
validation


In [ ]:
def load_jsonl(path):
    records = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))

    return records


nodes = load_jsonl(KG_ROOT / "nodes.jsonl")
links = load_jsonl(KG_ROOT / "links.jsonl")

nodes_df = pd.DataFrame(nodes)
links_df = pd.DataFrame(links)

print("Total nodes:", len(nodes_df))
print("Total relationships:", len(links_df))

print("\nNode sample:")
display(nodes_df.head())

print("\nRelationship sample:")
display(links_df.head())

Total nodes: 206634
Total relationships: 600623

Node sample:


,canonical_id,entity_type,instance_id,label,name,payload_json,project_key,uid,created_at,observed_at,...,temporal_status,chunk_id,source_id,source_timestamp,source_type,evidence_text,event_time,valid_from,valid_to,relation_type
0,COMPONENT::AEROGEAR-I01::admin-server::73ed893...,COMPONENT,AEROGEAR-I01,CanonicalEntity,admin server,"{""aliases"": [""admin server""], ""canonical_id"": ...",AEROGEAR,C_COMPONENT::AEROGEAR-I01::admin-server::73ed8...,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,COMPONENT::AEROGEAR-I01::admin-ui-module::3a3e...,COMPONENT,AEROGEAR-I01,CanonicalEntity,admin-ui module,"{""aliases"": [""admin-ui module""], ""canonical_id...",AEROGEAR,C_COMPONENT::AEROGEAR-I01::admin-ui-module::3a...,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,COMPONENT::AEROGEAR-I01::admin-ui::b280ae5b97f6,COMPONENT,AEROGEAR-I01,CanonicalEntity,admin UI,"{""aliases"": [""admin UI""], ""canonical_id"": ""COM...",AEROGEAR,C_COMPONENT::AEROGEAR-I01::admin-ui::b280ae5b97f6,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,COMPONENT::AEROGEAR-I01::build-farm-components...,COMPONENT,AEROGEAR-I01,CanonicalEntity,Build farm components,"{""aliases"": [""Build farm components""], ""canoni...",AEROGEAR,C_COMPONENT::AEROGEAR-I01::build-farm-componen...,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,COMPONENT::AEROGEAR-I01::data-sync::6c66d7790a93,COMPONENT,AEROGEAR-I01,CanonicalEntity,data-sync,"{""aliases"": [""data-sync""], ""canonical_id"": ""CO...",AEROGEAR,C_COMPONENT::AEROGEAR-I01::data-sync::6c66d779...,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Relationship sample:


,from_uid,instance_id,to_uid,type,uid,role
0,R_f20150ac2dd405a99ad2c161,JBEAP-I01,E_f20150ac2dd405a99ad2c161,HAS_EVIDENCE,L_d141189203a421a7ce0deda2,NaN
1,R_f20150ac2dd405a99ad2c161,JBEAP-I01,C_PROJECT::JBEAP-I01::jbeap::70b147b02096,MENTIONS,L_e9469702309b81898623372c,NaN
2,R_f20150ac2dd405a99ad2c161,JBEAP-I01,C_ISSUE::JBEAP-I01::jbeap-7698::fcc8a4360aa0,MENTIONS,L_248652fe83c4e5a537ece317,NaN
3,R_f20150ac2dd405a99ad2c161,JBEAP-I01,C_PERSON::JBEAP-I01::author-displayname-668abc...,MENTIONS,L_a4175c254566c6fbe67884cf,NaN
4,R_f20150ac2dd405a99ad2c161,JBEAP-I01,C_PERSON::JBEAP-I01::author-displayname-4cba0b...,MENTIONS,L_3a575a8099e484c7abebf58c,NaN


In [ ]:
# Check duplicate node and relationship IDs
duplicate_nodes = nodes_df["uid"].duplicated().sum()
duplicate_links = links_df["uid"].duplicated().sum()

# Check missing identifiers
missing_node_ids = nodes_df["uid"].isna().sum()
missing_link_ids = links_df["uid"].isna().sum()

# Check whether relationship endpoints exist
node_ids = set(nodes_df["uid"].dropna())

invalid_source = (
    ~links_df["from_uid"].isin(node_ids)
).sum()

invalid_target = (
    ~links_df["to_uid"].isin(node_ids)
).sum()

print("Duplicate node IDs:", duplicate_nodes)
print("Duplicate relationship IDs:", duplicate_links)
print("Missing node IDs:", missing_node_ids)
print("Missing relationship IDs:", missing_link_ids)
print("Invalid source references:", invalid_source)
print("Invalid target references:", invalid_target)

Duplicate node IDs: 0
Duplicate relationship IDs: 0
Missing node IDs: 0
Missing relationship IDs: 0
Invalid source references: 0
Invalid target references: 0


In [ ]:
node_instances = nodes_df.set_index("uid")["instance_id"]

links_check = links_df.copy()

links_check["source_instance"] = (
    links_check["from_uid"].map(node_instances)
)

links_check["target_instance"] = (
    links_check["to_uid"].map(node_instances)
)

cross_instance_links = links_check[
    (links_check["source_instance"] !=
     links_check["target_instance"]) |
    (links_check["instance_id"] !=
     links_check["source_instance"])
]

print(
    "Cross-instance or inconsistent relationships:",
    len(cross_instance_links)
)

Cross-instance or inconsistent relationships: 0


In [ ]:
temporal_fields = [
    "observed_at",
    "event_time",
    "valid_from",
    "valid_to",
    "created_at",
    "source_timestamp"
]

results = []

for field in temporal_fields:
    if field in nodes_df.columns:
        values = nodes_df[field]

        populated = (
            values.notna() &
            values.astype(str).str.strip().ne("")
        ).sum()

        results.append({
            "field": field,
            "populated": int(populated),
            "missing": int(len(nodes_df) - populated),
            "coverage_pct": round(
                100 * populated / max(len(nodes_df), 1), 2
            )
        })

temporal_summary = pd.DataFrame(results)

display(temporal_summary)

,field,populated,missing,coverage_pct
0,observed_at,93290,113344,45.15
1,event_time,86219,120415,41.73
2,valid_from,86219,120415,41.73
3,valid_to,267,206367,0.13
4,created_at,5926,200708,2.87
5,source_timestamp,4805,201829,2.33


In [ ]:
if "observed_at" in nodes_df.columns:

    coverage_by_type = (
        nodes_df.assign(
            has_observed_at=lambda df:
                df["observed_at"].notna() &
                df["observed_at"].astype(str).str.strip().ne("")
        )
        .groupby("label")["has_observed_at"]
        .agg(["count", "sum", "mean"])
        .reset_index()
    )

    coverage_by_type["mean"] *= 100

    display(coverage_by_type)

,label,count,sum,mean
0,ActionItem,207,207,100.000000
1,Blocker,443,167,37.697517
2,CanonicalEntity,40325,5926,14.695598
3,Decision,414,414,100.000000
4,Dependency,7709,237,3.074329
5,Event,82577,82396,99.780811
6,Evidence,5827,67,1.149820
7,Fact,63305,3809,6.016902
8,SourceRecord,5827,67,1.149820


In [ ]:
validation_dir = KG_ROOT / "validation"

integrity_path = validation_dir / "integrity_report.json"
coverage_path = validation_dir / "temporal_coverage.csv"
missing_path = validation_dir / "missing_observation_times.jsonl"

if integrity_path.exists():
    with open(integrity_path, encoding="utf-8") as f:
        integrity_report = json.load(f)

    print("Phase 1 integrity report:")
    print(json.dumps(integrity_report, indent=2))

if coverage_path.exists():
    print("\nTemporal coverage report:")
    display(pd.read_csv(coverage_path).head(20))

if missing_path.exists():
    missing_records = load_jsonl(missing_path)

    print("\nRecords with missing observation times:")
    print(len(missing_records))

    display(pd.DataFrame(missing_records).head(10))

Phase 1 integrity report:
{
  "node_count": 206634,
  "edge_count": 600623,
  "node_types": {
    "CanonicalEntity": 40325,
    "SourceRecord": 5827,
    "Evidence": 5827,
    "Dependency": 7709,
    "Fact": 63305,
    "Event": 82577,
    "Decision": 414,
    "ActionItem": 207,
    "Blocker": 443
  },
  "edge_types": {
    "HAS_EVIDENCE": 5827,
    "MENTIONS": 61315,
    "HAS_DEPENDENCY": 7709,
    "SUPPORTED_BY": 154655,
    "REFERS_TO": 97561,
    "FROM_ENTITY": 63305,
    "TO_ENTITY": 63305,
    "ASSERTS": 63305,
    "HAS_EVENT": 82577,
    "HAS_DECISION": 414,
    "HAS_ACTIONITEM": 207,
    "HAS_BLOCKER": 443
  },
  "jira_tickets_mapped": 9000,
  "original_documents_found": 97,
  "jira_mapping_issues": 0,
  "document_issues": 0,
  "unknown_temporal_observation": 67425
}

Temporal coverage report:


,label,total,observed_at,event_time,temporal_unknown
0,ActionItem,207,207,0,0
1,Blocker,443,167,276,276
2,CanonicalEntity,40325,5926,0,0
3,Decision,414,414,0,0
4,Dependency,7709,237,205,7472
5,Event,82577,82396,82372,181
6,Evidence,5827,67,0,5760
7,Fact,63305,3809,3366,59496
8,SourceRecord,5827,67,0,5760



Records with missing observation times:
67425


,instance_id,label,reason,source_id,source_type,uid
0,JBEAP-I01,Dependency,unavailable,JBEAP-7698,JIRA,T_698632360ce18e192e4cbaf7
1,JBEAP-I01,Dependency,unavailable,JBEAP-7698,JIRA,T_9574a8eadc238c10fbd7c46a
2,JBEAP-I01,Fact,unavailable,JBEAP-7698,JIRA,F_b430e75cd339a161d240845d
3,JBEAP-I01,Fact,unavailable,JBEAP-7698,JIRA,F_eb2a7e71206547d9adbeb9a7
4,JBEAP-I01,Fact,unavailable,JBEAP-7698,JIRA,F_fba221f59dca5cdaf2a5317b
5,JBEAP-I01,Fact,unavailable,JBEAP-7698,JIRA,F_f31cf960620f4e84e6644eda
6,JBEAP-I01,Fact,unavailable,JBEAP-7698,JIRA,F_c27cbfef54694035c89172ea
7,JBEAP-I01,Fact,unavailable,JBEAP-7698,JIRA,F_04e3cbfb00848693c35d465c
8,JBEAP-I01,Fact,unavailable,JBEAP-7698,JIRA,F_cec8f3625f9f267520746504
9,JBEAP-I01,Fact,unavailable,JBEAP-7698,JIRA,F_c569c5f787149ca5ceaed18a


In [ ]:
validation_summary = {
    "total_nodes": len(nodes_df),
    "total_relationships": len(links_df),
    "project_instances": int(
        nodes_df["instance_id"].nunique()
    ),
    "duplicate_node_ids": int(duplicate_nodes),
    "duplicate_relationship_ids": int(duplicate_links),
    "missing_node_ids": int(missing_node_ids),
    "missing_relationship_ids": int(missing_link_ids),
    "invalid_source_references": int(invalid_source),
    "invalid_target_references": int(invalid_target),
    "cross_instance_links": int(len(cross_instance_links)),
    "temporal_coverage": temporal_summary.to_dict(
        orient="records"
    )
}

output_path = GC1_ROOT /"Temporal_snapshots"/ "step1_validation_summary.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(validation_summary, f, indent=2)

print("Validation summary saved to:", output_path)

Validation summary saved to: /content/drive/MyDrive/GC1/Temporal_snapshots/step1_validation_summary.json


In [ ]:
import json

events = nodes_df[
    (nodes_df["label"] == "Event") &
    (nodes_df["temporal_status"] == "EXACT")
]

print("Total usable timestamped events:", len(events))

for _, row in events.head(5).iterrows():
    print("\n" + "=" * 60)
    print("Event UID:", row["uid"])
    print("Project:", row["instance_id"])
    print("Observed at:", row["observed_at"])
    print("Source:", row["source_id"])

    payload = row["payload_json"]

    if isinstance(payload, str):
        try:
            payload = json.loads(payload)
        except json.JSONDecodeError:
            pass

    print("Payload:")
    print(json.dumps(payload, indent=2, default=str)[:2500])

Total usable timestamped events: 82396

Event UID: T_78880e35c6d61ac7fd1cb6c0
Project: JBEAP-I01
Observed at: 2016-07-29T13:23:29+00:00
Source: JBEAP-5471
Payload:
{
  "actor": "<<|author_displayName|2fb24984-a4ec-4b4b-b178-989451ed427c|>>",
  "attributes": {
    "custom_field_fallback": false,
    "field_id": null,
    "field_type": "custom",
    "jira_field": "CDW pm_ack",
    "source_structure": "full_changelog"
  },
  "entity": "ISSUE::JBEAP-I01::jbeap-5471::a39b18f1d399",
  "event_time": "2016-07-29T13:23:29.000+0000",
  "event_type": "PM_ACK_CHANGE",
  "extraction_method": "structured",
  "from_value": null,
  "to_value": "?"
}

Event UID: T_139d055c4525c8627deec04c
Project: JBEAP-I01
Observed at: 2016-07-29T13:23:29+00:00
Source: JBEAP-5471
Payload:
{
  "actor": "<<|author_displayName|2fb24984-a4ec-4b4b-b178-989451ed427c|>>",
  "attributes": {
    "custom_field_fallback": false,
    "field_id": null,
    "field_type": "custom",
    "jira_field": "CDW release",
    "source_struct

In [ ]:
import json
import pandas as pd

events = nodes_df[
    nodes_df["label"] == "Event"
].copy()

def parse_payload(value):
    if isinstance(value, dict):
        return value
    if isinstance(value, str):
        try:
            return json.loads(value)
        except (json.JSONDecodeError, TypeError):
            return {}
    return {}

payloads = events["payload_json"].apply(parse_payload)

events["event_type"] = payloads.apply(
    lambda x: x.get("event_type")
)

events["jira_field"] = payloads.apply(
    lambda x: x.get("attributes", {}).get("jira_field")
    if isinstance(x.get("attributes"), dict)
    else None
)

events["from_value"] = payloads.apply(
    lambda x: x.get("from_value")
)

events["to_value"] = payloads.apply(
    lambda x: x.get("to_value")
)

# 1. Event types across all projects
print("EVENT TYPE DISTRIBUTION")
display(
    events["event_type"]
    .value_counts(dropna=False)
    .reset_index(name="count")
    .rename(columns={"event_type": "event_type"})
)

# 2. Jira fields represented in event history
print("\nJIRA FIELD DISTRIBUTION")
display(
    events["jira_field"]
    .value_counts(dropna=False)
    .head(40)
    .reset_index(name="count")
)

# 3. Timestamp availability by event type
print("\nTEMPORAL COVERAGE BY EVENT TYPE")

events["has_observed_at"] = (
    events["observed_at"].notna()
    & events["observed_at"].astype(str).str.strip().ne("")
)

coverage = (
    events.groupby("event_type", dropna=False)
    .agg(
        total=("uid", "count"),
        timestamped=("has_observed_at", "sum")
    )
    .reset_index()
)

coverage["coverage_pct"] = (
    coverage["timestamped"] / coverage["total"] * 100
).round(2)

display(
    coverage.sort_values("total", ascending=False)
)

# 4. Event distribution across projects
print("\nEVENT COVERAGE BY PROJECT")

display(
    events.groupby("instance_id")
    .agg(
        total_events=("uid", "count"),
        timestamped_events=("has_observed_at", "sum"),
        unique_event_types=("event_type", "nunique")
    )
    .reset_index()
)

EVENT TYPE DISTRIBUTION


,event_type,count
0,CUSTOM_FIELD_CHANGE,14138
1,STATUS_CHANGE,12228
2,LINK_CHANGE,11516
3,REMOTE_LINK_CHANGE,7560
4,VERSION_CHANGE,5892
...,...,...
71,PLATFORM_CHANGE,2
72,START_DATE_CHANGE,2
73,HELPDESK_REFERENCE_CHANGE,1
74,INFO_REQUEST_OWNER_CHANGE,1



JIRA FIELD DISTRIBUTION


,jira_field,count
0,status,12228
1,Link,9831
2,UIS,9718
3,RemoteIssueLink,7560
4,Workflow,5192
5,Attachment,4138
6,Fix Version,3298
7,Rank,3089
8,resolution,2885
9,labels,2792



TEMPORAL COVERAGE BY EVENT TYPE


,event_type,total,timestamped,coverage_pct
14,CUSTOM_FIELD_CHANGE,14138,14138,100.0
63,STATUS_CHANGE,12228,12228,100.0
35,LINK_CHANGE,11516,11516,100.0
53,REMOTE_LINK_CHANGE,7560,7560,100.0
73,VERSION_CHANGE,5892,5892,100.0
...,...,...,...,...
22,DUE_DATE_CHANGE,2,2,100.0
68,TEAM_IMPACT_CHANGE,2,2,100.0
8,CALL_TO_ACTION_CHANGE,1,1,100.0
31,INFO_REQUEST_OWNER_CHANGE,1,1,100.0



EVENT COVERAGE BY PROJECT


,instance_id,total_events,timestamped_events,unique_event_types
0,AEROGEAR-I01,1601,1601,34
1,AMBARI-I01,1853,1825,23
2,BXMSDOC-I01,1127,1127,28
3,CONFSERVER-I01,6648,6648,26
4,DAOS-I01,3295,3295,29
5,DOCS-I01,1336,1320,32
6,ENTESB-I01,3864,3836,38
7,EVG-I01,3257,3257,25
8,FLEX-I01,1178,1156,21
9,HBASE-I01,1392,1392,18


In [ ]:
# Project timeline coverage from timestamped events

dated_events = events.copy()

dated_events["observed_dt"] = pd.to_datetime(
    dated_events["observed_at"],
    errors="coerce",
    utc=True
)

dated_events = dated_events.dropna(
    subset=["observed_dt"]
)

project_timeline = (
    dated_events.groupby("instance_id")
    .agg(
        first_observation=("observed_dt", "min"),
        last_observation=("observed_dt", "max"),
        event_count=("uid", "count")
    )
    .reset_index()
)

project_timeline["duration_days"] = (
    project_timeline["last_observation"]
    - project_timeline["first_observation"]
).dt.days

project_timeline["approx_weeks"] = (
    project_timeline["duration_days"] / 7
).round(1)

display(project_timeline)

print("\nTimeline summary:")
display(
    project_timeline[
        ["duration_days", "approx_weeks", "event_count"]
    ].describe()
)

,instance_id,first_observation,last_observation,event_count,duration_days,approx_weeks
0,AEROGEAR-I01,2018-01-17 05:30:01+00:00,2018-09-25 09:29:15+00:00,1601,251,35.9
1,AMBARI-I01,2016-03-03 00:46:21+00:00,2016-11-09 22:26:15+00:00,1825,251,35.9
2,BXMSDOC-I01,2020-03-20 04:45:40+00:00,2020-11-26 03:06:05+00:00,1127,250,35.7
3,CONFSERVER-I01,2016-03-17 00:01:19+00:00,2016-11-23 14:23:28+00:00,6648,251,35.9
4,DAOS-I01,2021-01-06 09:40:54+00:00,2021-09-14 23:40:04+00:00,3295,251,35.9
5,DOCS-I01,2016-07-05 19:07:09+00:00,2017-03-13 01:42:03+00:00,1320,250,35.7
6,ENTESB-I01,2020-04-08 07:32:26+00:00,2020-12-14 10:17:29+00:00,3836,250,35.7
7,EVG-I01,2020-05-27 14:34:54+00:00,2021-02-02 23:07:55+00:00,3257,251,35.9
8,FLEX-I01,2013-09-02 04:32:52+00:00,2014-05-06 13:02:31+00:00,1156,246,35.1
9,HBASE-I01,2016-07-01 02:40:06+00:00,2017-03-09 15:00:42+00:00,1392,251,35.9



Timeline summary:


,duration_days,approx_weeks,event_count
count,30.000000,30.000000,30.000000
mean,248.966667,35.583333,2746.533333
std,5.833810,0.847342,2161.734899
min,220.000000,31.400000,403.000000
25%,249.000000,35.600000,1168.500000
50%,250.500000,35.800000,1713.000000
75%,251.000000,35.900000,3700.750000
max,254.000000,36.300000,9087.000000


### Source data preparation

Load source Jira records, graph events and checkpoint metadata. Original outputs are retained.

In [ ]:
import pandas as pd

# Candidate event categories to inspect
risk_keywords = {
    "Schedule": [
        "STATUS", "DUE_DATE", "VERSION",
        "RELEASE", "SPRINT", "RESOLUTION"
    ],
    "Scope": [
        "DESCRIPTION", "SUMMARY", "STORY_POINT",
        "EPIC", "ISSUE_TYPE", "PRIORITY"
    ],
    "Dependency": [
        "LINK", "BLOCK", "DEPEND"
    ],
    "Decision": [
        "DECISION", "APPROVAL", "ACK",
        "REVIEW", "WORKFLOW"
    ]
}

event_types = (
    events["event_type"]
    .dropna()
    .astype(str)
    .unique()
)

for risk, keywords in risk_keywords.items():

    matching_types = sorted([
        event_type
        for event_type in event_types
        if any(
            keyword in event_type.upper()
            for keyword in keywords
        )
    ])

    print(f"\n{'=' * 55}")
    print(f"{risk.upper()} RISK — CANDIDATE EVENT TYPES")
    print(f"{'=' * 55}")

    for event_type in matching_types:
        count = (
            events["event_type"] == event_type
        ).sum()

        print(f"{event_type}: {count}")


SCHEDULE RISK — CANDIDATE EVENT TYPES
DOCS_QA_STATUS_CHANGE: 2
DUE_DATE_CHANGE: 2
EPIC_STATUS_CHANGE: 20
RELEASE_CHANGE: 330
RELEASE_NOTE_STATUS_CHANGE: 9
RELEASE_NOTE_TEXT_CHANGE: 32
RESOLUTION_CHANGE: 2885
SPRINT_CHANGE: 1221
STATUS_CHANGE: 12228
TARGET_RELEASE_CHANGE: 328
VERSION_CHANGE: 5892

SCOPE RISK — CANDIDATE EVENT TYPES
DESCRIPTION_CHANGE: 1425
EPIC_CHILD_CHANGE: 410
EPIC_LINK_CHANGE: 367
EPIC_NAME_CHANGE: 38
EPIC_STATUS_CHANGE: 20
ISSUE_TYPE_CHANGE: 190
PRIORITY_CHANGE: 364
STORY_POINT_CHANGE: 355
SUMMARY_CHANGE: 694

DEPENDENCY RISK — CANDIDATE EVENT TYPES
BLOCKER_FIELD_CHANGE: 5
EPIC_LINK_CHANGE: 367
LINK_CHANGE: 11516
REMOTE_LINK_CHANGE: 7560

DECISION RISK — CANDIDATE EVENT TYPES
DECISION_MADE: 319
DEVELOPMENT_ACK_CHANGE: 279
DOCS_ACK_CHANGE: 131
PM_ACK_CHANGE: 257
QA_ACK_CHANGE: 205
REVIEWER_CHANGE: 3
REVIEW_DATE_CHANGE: 23
WORKFLOW_CHANGE: 5192


In [ ]:
status_events = events[
    events["event_type"] == "STATUS_CHANGE"
].copy()

status_transitions = (
    status_events.groupby(
        ["from_value", "to_value"],
        dropna=False
    )
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

print("Total status changes:", len(status_events))

print("\nMost common status transitions:")
display(status_transitions.head(30))

Total status changes: 12228

Most common status transitions:


,from_value,to_value,count
355,Reopened,Reopened,911
248,Open,In Progress,555
259,Open,Open,522
364,Resolved,Closed,502
376,Resolved,Reopened,422
356,Reopened,Resolved,355
260,Open,Patch Available,325
282,Patch Available,Open,297
265,Open,Resolved,270
131,In Progress,In Code Review,229


In [ ]:
link_events = events[
    events["event_type"].isin([
        "LINK_CHANGE",
        "REMOTE_LINK_CHANGE"
    ])
].copy()

print("Total link events:", len(link_events))

display(
    link_events[
        [
            "instance_id",
            "source_id",
            "event_type",
            "from_value",
            "to_value",
            "observed_at"
        ]
    ].head(30)
)

Total link events: 19076


,instance_id,source_id,event_type,from_value,to_value,observed_at
40362,JBEAP-I01,JBEAP-5471,LINK_CHANGE,None,This issue follows up on JBEAP-4679,2016-07-29T13:23:57+00:00
40365,JBEAP-I01,JBEAP-5471,LINK_CHANGE,None,This issue incorporates JBEAP-4852,2016-07-29T13:28:37+00:00
40366,JBEAP-I01,JBEAP-5471,LINK_CHANGE,None,This issue incorporates JBEAP-4407,2016-07-29T13:28:38+00:00
40367,JBEAP-I01,JBEAP-5471,LINK_CHANGE,None,This issue incorporates JBEAP-3100,2016-07-29T13:28:38+00:00
40368,JBEAP-I01,JBEAP-5471,LINK_CHANGE,None,This issue incorporates JBEAP-4772,2016-07-29T13:28:38+00:00
40369,JBEAP-I01,JBEAP-5471,LINK_CHANGE,None,This issue incorporates JBEAP-3918,2016-07-29T13:28:38+00:00
40370,JBEAP-I01,JBEAP-5471,LINK_CHANGE,None,This issue incorporates JBEAP-3038,2016-07-29T13:31:25+00:00
40371,JBEAP-I01,JBEAP-5471,LINK_CHANGE,None,This issue incorporates JBEAP-3624,2016-07-29T13:31:26+00:00
40372,JBEAP-I01,JBEAP-5471,LINK_CHANGE,None,This issue incorporates JBEAP-1289,2016-07-29T13:31:26+00:00
40424,JBEAP-I01,JBEAP-5464,LINK_CHANGE,None,This issue is blocked by JBEAP-5465,2016-07-29T07:55:38+00:00


In [ ]:
decision_nodes = nodes_df[
    nodes_df["label"] == "Decision"
].copy()

print("Total Decision nodes:", len(decision_nodes))

print("\nDecision coverage by project:")
display(
    decision_nodes.groupby("instance_id")
    .size()
    .reset_index(name="decision_count")
)

print("\nSample Decision payloads:")

for _, row in decision_nodes.head(5).iterrows():

    print("\n" + "=" * 60)
    print("Project:", row["instance_id"])
    print("Observed at:", row["observed_at"])
    print("Source:", row["source_id"])
    print("Payload:", str(row["payload_json"])[:1500])

Total Decision nodes: 414

Decision coverage by project:


,instance_id,decision_count
0,AEROGEAR-I01,11
1,AMBARI-I01,10
2,BXMSDOC-I01,10
3,CONFSERVER-I01,13
4,DAOS-I01,8
5,DOCS-I01,19
6,ENTESB-I01,10
7,EVG-I01,15
8,FLEX-I01,11
9,HBASE-I01,15



Sample Decision payloads:

Project: JBEAP-I01
Observed at: 2015-11-29T00:00:00+00:00
Source: JBEAP-I01-M001
Payload: {"decision_id": "JBEAP-I01-M001_DECISION_001", "decision_maker": "Product Manager", "decision_state": "MADE", "decision_target": null, "decision_text": "Agreed, it stays \"pending QA confirmation\". Now the journal exhaustion issue, 1987. The endless AMQ142030 warnings.", "decision_time": "2015-11-29T00:00:00+00:00", "evidence_text": "Agreed, it stays \"pending QA confirmation\". Now the journal exhaustion issue, 1987. The endless AMQ142030 warnings.", "extraction_method": "rule", "reversed_decision": null, "speaker_timestamp": "04:40"}

Project: JBEAP-I01
Observed at: 2015-11-29T00:00:00+00:00
Source: JBEAP-I01-M001
Payload: {"decision_id": "JBEAP-I01-M001_DECISION_002", "decision_maker": "Messaging Developer", "decision_state": "TENTATIVE", "decision_target": null, "decision_text": "There's some disagreement on this one. My first reaction was that it isn't really a bu

In [ ]:
scope_keywords = [
    "DESCRIPTION",
    "SUMMARY",
    "STORY_POINT",
    "EPIC",
    "ISSUE_TYPE"
]

scope_events = events[
    events["event_type"].fillna("").str.upper().apply(
        lambda x: any(
            keyword in x for keyword in scope_keywords
        )
    )
].copy()

print("Candidate scope-related events:", len(scope_events))

display(
    scope_events[
        [
            "instance_id",
            "source_id",
            "event_type",
            "from_value",
            "to_value",
            "observed_at"
        ]
    ].head(30)
)

Candidate scope-related events: 3499


,instance_id,source_id,event_type,from_value,to_value,observed_at
40528,JBEAP-I01,JBEAP-5248,SUMMARY_CHANGE,Excluded contexts not specific to a host shoul...,(7.0.z) Excluded contexts not specific to a ho...,2016-07-29T00:22:33+00:00
40555,JBEAP-I01,JBEAP-5161,DESCRIPTION_CHANGE,This is a tracker JIRA fr bean validation rela...,This is a tracker JIRA for bean validation rel...,2016-07-07T02:19:54+00:00
40619,JBEAP-I01,JBEAP-4962,DESCRIPTION_CHANGE,EAP7 Management CLI Guide issues tracker\n\nht...,EAP7 Management CLI Guide issues tracker\n\nht...,2016-06-14T09:27:56+00:00
40723,JBEAP-I01,JBEAP-4742,SUMMARY_CHANGE,Duplicate messages in replicated HA topology w...,(7.0.z) Duplicate messages in replicated HA to...,2016-07-28T13:08:19+00:00
40744,JBEAP-I01,JBEAP-4617,SUMMARY_CHANGE,Docs for configuraing EAP on Azure,Docs for configuring EAP on Azure,2016-05-18T19:42:04+00:00
40791,JBEAP-I01,JBEAP-4530,DESCRIPTION_CHANGE,When defining new broadcast-group in Configura...,When defining new broadcast-group in Configura...,2016-05-05T06:02:16+00:00
40792,JBEAP-I01,JBEAP-4530,SUMMARY_CHANGE,Broadcast group is possible to create with jus...,Broadcast/Discovery group is possible to creat...,2016-05-05T06:03:50+00:00
40819,JBEAP-I01,JBEAP-4519,DESCRIPTION_CHANGE,Tracker bug for jboss-ec2-eap for EAP 7.0.0,Tracker bug for jboss-ec2-eap for EAP 7.0.1,2016-05-04T11:54:01+00:00
40917,JBEAP-I01,JBEAP-4453,SUMMARY_CHANGE,Document for RN,Document JBEAP-3998 for RN,2016-06-21T07:50:14+00:00
40944,JBEAP-I01,JBEAP-4439,DESCRIPTION_CHANGE,I have configured the SSL realm as follows:\n\...,I have configured the SSL realm as follows:\n\...,2016-04-29T12:40:22+00:00


In [ ]:
from pathlib import Path

source_root = GC1_ROOT

candidate_files = []

for path in source_root.rglob("*"):
    if not path.is_file():
        continue

    if path.suffix.lower() in {
        ".json", ".jsonl", ".csv", ".xlsx"
    }:
        name = path.name.lower()

        if any(term in name for term in [
            "jira", "issue", "prd",
            "requirement", "scenario",
            "ground_truth", "risk"
        ]):
            candidate_files.append({
                "filename": path.name,
                "path": str(path),
                "size_mb": round(
                    path.stat().st_size / (1024 ** 2), 2
                )
            })

source_files_df = pd.DataFrame(candidate_files)

print("Candidate source files:", len(source_files_df))

if not source_files_df.empty:
    display(
        source_files_df.sort_values("path")
    )

Candidate source files: 175


,filename,path,size_mb
111,jira_extracted.jsonl,/content/drive/MyDrive/GC1/GC1_Entity_Extracti...,4.25
112,prd_extracted.jsonl,/content/drive/MyDrive/GC1/GC1_Entity_Extracti...,0.07
109,jira_extracted.jsonl,/content/drive/MyDrive/GC1/GC1_Entity_Extracti...,2.80
110,prd_extracted.jsonl,/content/drive/MyDrive/GC1/GC1_Entity_Extracti...,0.05
107,jira_extracted.jsonl,/content/drive/MyDrive/GC1/GC1_Entity_Extracti...,1.99
...,...,...,...
6,phase2_partial_risk_labels.csv,/content/drive/MyDrive/GC1/Temporal_snapshots/...,0.04
8,phase2_three_risk_labels.csv,/content/drive/MyDrive/GC1/Temporal_snapshots/...,0.04
5,schedule_risk_labels.csv,/content/drive/MyDrive/GC1/Temporal_snapshots/...,0.07
7,scope_risk_labels.csv,/content/drive/MyDrive/GC1/Temporal_snapshots/...,0.07


In [ ]:
import json
from pathlib import Path
import pandas as pd

# Locate original Jira files
jira_root = GC1_ROOT / "GC1_Jira_Dataset"

jira_files = sorted(jira_root.rglob("jira_tickets.jsonl"))
raw_files = sorted(jira_root.rglob("raw_issues.jsonl"))

print("Jira ticket files:", len(jira_files))
print("Raw issue files:", len(raw_files))

def read_first_record(path):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                return json.loads(line)
    return {}

# Inspect one representative file of each type
for files, label in [
    (jira_files, "JIRA TICKETS"),
    (raw_files, "RAW ISSUES")
]:
    if not files:
        continue

    sample = read_first_record(files[0])

    print(f"\n{'=' * 60}")
    print(label)
    print("File:", files[0])
    print("Top-level fields:")
    print(list(sample.keys()))

    # Inspect nested fields
    for key, value in sample.items():
        if isinstance(value, dict):
            print(f"\nNested fields under '{key}':")
            print(list(value.keys())[:80])

Jira ticket files: 30
Raw issue files: 30

JIRA TICKETS
File: /content/drive/MyDrive/GC1/GC1_Jira_Dataset/01_JBEAP/jira_tickets.jsonl
Top-level fields:
['source_type', 'source_repository', 'project_id', 'project_key', 'project_name', 'issue_id', 'issue_key', 'issue_type', 'summary', 'description', 'related_requirement', 'epic_feature', 'status', 'status_history', 'priority', 'assignee', 'reporter', 'creator', 'team', 'created_at', 'updated_at', 'due_at', 'resolved_at', 'blocked_at', 'unblocked_at', 'resolution', 'blocks', 'blocked_by', 'depends_on', 'other_issue_links', 'comments', 'labels', 'components', 'fix_versions', 'affected_versions', 'original_issue_links', 'full_changelog']

RAW ISSUES
File: /content/drive/MyDrive/GC1/GC1_Jira_Dataset/01_JBEAP/raw_issues.jsonl
Top-level fields:
['_id', 'expand', 'id', 'self', 'key', 'fields', 'changelog']

Nested fields under 'fields':
['customfield_12316242', 'customfield_12317298', 'fixVersions', 'resolution', 'customfield_12312442', 'custom

In [ ]:
from collections import Counter

field_counts = Counter()
total_records = 0

for path in jira_files:

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            record = json.loads(line)
            total_records += 1

            # Check both top-level and nested Jira fields
            fields = record.get("fields", {})

            if not isinstance(fields, dict):
                fields = {}

            available = set(record.keys()) | set(fields.keys())

            for key in available:
                field_counts[key] += 1

field_coverage = pd.DataFrame([
    {
        "field": field,
        "populated_records": count,
        "coverage_pct": round(
            100 * count / max(total_records, 1), 2
        )
    }
    for field, count in field_counts.items()
])

print("Total Jira records:", total_records)

display(
    field_coverage.sort_values(
        "coverage_pct",
        ascending=False
    ).head(80)
)

Total Jira records: 9000


,field,populated_records,coverage_pct
0,reporter,9000,100.0
1,summary,9000,100.0
2,depends_on,9000,100.0
3,epic_feature,9000,100.0
4,labels,9000,100.0
5,components,9000,100.0
6,fix_versions,9000,100.0
7,project_name,9000,100.0
8,blocked_by,9000,100.0
9,other_issue_links,9000,100.0


In [ ]:
import json
import pandas as pd
from collections import Counter

important_fields = [
    "created_at",
    "updated_at",
    "due_at",
    "resolved_at",
    "blocked_at",
    "unblocked_at",
    "status_history",
    "full_changelog",
    "blocks",
    "blocked_by",
    "depends_on",
    "related_requirement",
    "epic_feature",
    "description",
    "resolution",
    "fix_versions"
]

def has_value(value):
    if value is None:
        return False

    if isinstance(value, str):
        return bool(value.strip())

    if isinstance(value, (list, dict, tuple, set)):
        return len(value) > 0

    return True


field_counts = Counter()
total_records = 0

for path in jira_files:

    with open(path, "r", encoding="utf-8") as f:

        for line in f:
            if not line.strip():
                continue

            record = json.loads(line)
            total_records += 1

            for field in important_fields:
                if has_value(record.get(field)):
                    field_counts[field] += 1


completeness_df = pd.DataFrame([
    {
        "field": field,
        "non_empty_records": field_counts[field],
        "missing_or_empty": total_records - field_counts[field],
        "coverage_pct": round(
            field_counts[field] / total_records * 100, 2
        )
    }
    for field in important_fields
])

display(
    completeness_df.sort_values(
        "coverage_pct",
        ascending=False
    )
)

,field,non_empty_records,missing_or_empty,coverage_pct
0,created_at,9000,0,100.00
1,updated_at,9000,0,100.00
7,full_changelog,8996,4,99.96
6,status_history,8774,226,97.49
13,description,8456,544,93.96
3,resolved_at,7455,1545,82.83
14,resolution,7455,1545,82.83
15,fix_versions,5778,3222,64.20
10,depends_on,1768,7232,19.64
9,blocked_by,1011,7989,11.23


In [ ]:
from pathlib import Path
import pandas as pd

# Existing GC1 directory
GC1_ROOT = Path("/content/drive/MyDrive/GC1")

# Search for PRD and meeting-related files
keywords = [
    "prd",
    "requirement",
    "meeting",
    "transcript",
    "teams"
]

extensions = {
    ".json", ".jsonl", ".csv",
    ".txt", ".md", ".docx", ".pdf"
}

source_files = []

for path in GC1_ROOT.rglob("*"):

    if not path.is_file():
        continue

    if path.suffix.lower() not in extensions:
        continue

    name = path.name.lower()

    if any(keyword in name for keyword in keywords):

        source_files.append({
            "filename": path.name,
            "path": str(path),
            "size_mb": round(
                path.stat().st_size / (1024 ** 2), 3
            )
        })

source_files_df = pd.DataFrame(source_files)

print("Total candidate files:", len(source_files_df))

if not source_files_df.empty:
    display(
        source_files_df.sort_values("path").head(100)
    )

Total candidate files: 167


,filename,path,size_mb
164,meetings_extracted.jsonl,/content/drive/MyDrive/GC1/GC1_Entity_Extracti...,0.141
163,prd_extracted.jsonl,/content/drive/MyDrive/GC1/GC1_Entity_Extracti...,0.067
162,meetings_extracted.jsonl,/content/drive/MyDrive/GC1/GC1_Entity_Extracti...,0.104
161,prd_extracted.jsonl,/content/drive/MyDrive/GC1/GC1_Entity_Extracti...,0.048
160,meetings_extracted.jsonl,/content/drive/MyDrive/GC1/GC1_Entity_Extracti...,0.137
...,...,...,...
71,ENTESB-I01-PRD-V1.meta.json,/content/drive/MyDrive/GC1/GC1_Generated_PRD_M...,0.002
70,ENTESB-I01-PRD-V1.prompt.txt,/content/drive/MyDrive/GC1/GC1_Generated_PRD_M...,0.537
69,ENTESB-I01-PRD-V1.txt,/content/drive/MyDrive/GC1/GC1_Generated_PRD_M...,0.016
68,SPR-I01-PRD-V1.meta.json,/content/drive/MyDrive/GC1/GC1_Generated_PRD_M...,0.001


In [ ]:
import json

prd_files = sorted(
    GC1_ROOT.rglob("prd_extracted.jsonl")
)

print("PRD extraction files:", len(prd_files))

def read_jsonl_sample(path, n=3):
    records = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            records.append(json.loads(line))

            if len(records) >= n:
                break

    return records

if prd_files:

    sample_prds = read_jsonl_sample(prd_files[0], n=3)

    print("Sample file:", prd_files[0])

    for i, record in enumerate(sample_prds, 1):

        print(f"\n{'=' * 60}")
        print(f"PRD RECORD {i}")
        print("=" * 60)

        print("Top-level fields:")
        print(list(record.keys()))

        print("\nSample content:")
        print(
            json.dumps(
                record,
                indent=2,
                default=str
            )[:3500]
        )
else:
    print("No prd_extracted.jsonl files found.")

PRD extraction files: 30
Sample file: /content/drive/MyDrive/GC1/GC1_Entity_Extraction/01_JBEAP/JBEAP-I01/prd_extracted.jsonl

PRD RECORD 1
Top-level fields:
['repository_project', 'project_key', 'instance_id', 'source', 'entities', 'relationships', 'events', 'decisions', 'action_items', 'blockers', 'dependencies', 'temporal', 'attributes', 'provenance']

Sample content:
{
  "repository_project": "01_JBEAP",
  "project_key": "JBEAP",
  "instance_id": "JBEAP-I01",
  "source": {
    "source_type": "PRD",
    "source_id": "JBEAP-I01-PRD-V1",
    "source_timestamp": "2015-11-29"
  },
  "entities": [
    {
      "local_id": "PRD_JBEAP-I01-PRD-V1_E001",
      "type": "PROJECT",
      "name": "JBEAP",
      "normalized_name": null,
      "attributes": {
        "project_name": "JBoss Enterprise Application Platform",
        "instance_id": "JBEAP-I01"
      },
      "extraction_method": "structured",
      "confidence": 1.0
    },
    {
      "local_id": "PRD_JBEAP-I01-PRD-V1_E002",
      "ty

In [ ]:
meeting_files = sorted([
    p for p in GC1_ROOT.rglob("*.jsonl")
    if any(term in p.name.lower() for term in [
        "meeting",
        "teams",
        "transcript"
    ])
])

print("Meeting-related JSONL files:", len(meeting_files))

for path in meeting_files[:40]:
    print(path)

Meeting-related JSONL files: 30
/content/drive/MyDrive/GC1/GC1_Entity_Extraction/01_JBEAP/JBEAP-I01/meetings_extracted.jsonl
/content/drive/MyDrive/GC1/GC1_Entity_Extraction/02_JBIDE/JBIDE-I01/meetings_extracted.jsonl
/content/drive/MyDrive/GC1/GC1_Entity_Extraction/03_SAK/SAK-I01/meetings_extracted.jsonl
/content/drive/MyDrive/GC1/GC1_Entity_Extraction/04_JBDS/JBDS-I01/meetings_extracted.jsonl
/content/drive/MyDrive/GC1/GC1_Entity_Extraction/05_MNG/MNG-I01/meetings_extracted.jsonl
/content/drive/MyDrive/GC1/GC1_Entity_Extraction/06_SPARK/SPARK-I01/meetings_extracted.jsonl
/content/drive/MyDrive/GC1/GC1_Entity_Extraction/07_ENTESB/ENTESB-I01/meetings_extracted.jsonl
/content/drive/MyDrive/GC1/GC1_Entity_Extraction/08_SPR/SPR-I01/meetings_extracted.jsonl
/content/drive/MyDrive/GC1/GC1_Entity_Extraction/09_AMBARI/AMBARI-I01/meetings_extracted.jsonl
/content/drive/MyDrive/GC1/GC1_Entity_Extraction/10_MC/MC-I01/meetings_extracted.jsonl
/content/drive/MyDrive/GC1/GC1_Entity_Extraction/11_OS

In [ ]:
if meeting_files:

    sample_meetings = read_jsonl_sample(
        meeting_files[0],
        n=3
    )

    print("Sample file:", meeting_files[0])

    for i, record in enumerate(sample_meetings, 1):

        print(f"\n{'=' * 60}")
        print(f"MEETING RECORD {i}")
        print("=" * 60)

        print("Top-level fields:")
        print(list(record.keys()))

        print("\nSample content:")
        print(
            json.dumps(
                record,
                indent=2,
                default=str
            )[:4000]
        )
else:
    print("No meeting JSONL files found.")

Sample file: /content/drive/MyDrive/GC1/GC1_Entity_Extraction/01_JBEAP/JBEAP-I01/meetings_extracted.jsonl

MEETING RECORD 1
Top-level fields:
['repository_project', 'project_key', 'instance_id', 'source', 'entities', 'relationships', 'events', 'decisions', 'action_items', 'blockers', 'dependencies', 'temporal', 'attributes', 'provenance']

Sample content:
{
  "repository_project": "01_JBEAP",
  "project_key": "JBEAP",
  "instance_id": "JBEAP-I01",
  "source": {
    "source_type": "MEETING",
    "source_id": "JBEAP-I01-M001",
    "source_timestamp": "2015-11-29T00:00:00+00:00"
  },
  "entities": [
    {
      "local_id": "MEETING_JBEAP-I01-M001_E001",
      "type": "PROJECT",
      "name": "JBEAP",
      "normalized_name": null,
      "attributes": {
        "project_name": "JBoss Enterprise Application Platform",
        "instance_id": "JBEAP-I01"
      },
      "extraction_method": "structured",
      "confidence": 1.0
    },
    {
      "local_id": "MEETING_JBEAP-I01-M001_E002",
    

In [ ]:
from collections import Counter
import pandas as pd
import json

sections = [
    "entities",
    "relationships",
    "events",
    "decisions",
    "action_items",
    "blockers",
    "dependencies"
]

summary = []
examples = {}

for category, files in [
    ("PRD", prd_files),
    ("Meeting", meeting_files)
]:
    for path in files:
        with open(path, encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue

                record = json.loads(line)
                project = record.get("instance_id")

                for section in sections:
                    items = record.get(section) or []

                    if not isinstance(items, list):
                        continue

                    summary.append({
                        "category": category,
                        "project": project,
                        "section": section,
                        "count": len(items)
                    })

                    key = (category, section)

                    if items and key not in examples:
                        examples[key] = items[0]

summary_df = pd.DataFrame(summary)

print("EXTRACTION CONTENT SUMMARY")

display(
    summary_df.groupby(
        ["category", "section"]
    )["count"]
    .agg(["sum", "mean", "min", "max"])
    .reset_index()
)

print("\nSAMPLE STRUCTURED RECORDS")

for key, value in examples.items():
    print("\n" + "=" * 60)
    print(key[0], "-", key[1])
    print(json.dumps(value, indent=2, default=str)[:2000])

EXTRACTION CONTENT SUMMARY


,category,section,sum,mean,min,max
0,Meeting,action_items,207,3.450000,0,15
1,Meeting,blockers,167,2.783333,0,10
2,Meeting,decisions,414,6.900000,3,12
3,Meeting,dependencies,205,3.416667,0,8
4,Meeting,entities,2953,49.216667,29,84
5,Meeting,events,319,5.316667,1,11
6,Meeting,relationships,3366,56.100000,30,87
7,PRD,action_items,0,0.000000,0,0
8,PRD,blockers,0,0.000000,0,0
9,PRD,decisions,0,0.000000,0,0



SAMPLE STRUCTURED RECORDS

PRD - entities
{
  "local_id": "PRD_JBEAP-I01-PRD-V1_E001",
  "type": "PROJECT",
  "name": "JBEAP",
  "normalized_name": null,
  "attributes": {
    "project_name": "JBoss Enterprise Application Platform",
    "instance_id": "JBEAP-I01"
  },
  "extraction_method": "structured",
  "confidence": 1.0
}

PRD - relationships
{
  "source_entity": "PRD_JBEAP-I01-PRD-V1_E001",
  "type": "HAS_REQUIREMENT",
  "target_entity": "PRD_JBEAP-I01-PRD-V1_E002",
  "timestamp": "2015-11-29",
  "attributes": {},
  "extraction_method": "structured",
  "confidence": 1.0
}

PRD - events
{
  "event_type": "REQUIREMENT_CREATED",
  "entity": "PRD_JBEAP-I01-PRD-V1_E002",
  "event_time": "2015-11-29",
  "actor": null,
  "from_value": null,
  "to_value": "REQ-001",
  "attributes": {
    "prd_version": "1"
  },
  "extraction_method": "structured"
}

PRD - dependencies
{
  "dependency_id": "JBEAP-I01-PRD-V1_DEP_001",
  "text": "Upstream WildFly and WildFly Core fixes, merged only with an 

## Dependency Risk — Ground-Truth Labelling

Reconstruct blocking history, inspect evidence, exclude suspected bulk additions, and create future 14-day labels.

In [ ]:
import json
import pandas as pd

jira_records = []

for path in jira_files:
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                jira_records.append(json.loads(line))

jira_df = pd.DataFrame(jira_records)

print("Total Jira tickets:", len(jira_df))
print("Projects:", jira_df["project_key"].nunique())

display(
    jira_df[
        [
            "project_key",
            "issue_key",
            "status",
            "created_at",
            "resolved_at",
            "blocked_at",
            "unblocked_at"
        ]
    ].head()
)

Total Jira tickets: 9000
Projects: 30


,project_key,issue_key,status,created_at,resolved_at,blocked_at,unblocked_at
0,JBEAP,JBEAP-22995,Pull Request Sent,2021-12-21T14:39:41.000+0000,None,None,None
1,JBEAP,JBEAP-22981,Closed,2021-12-17T05:27:13.000+0000,2021-12-23T11:41:14.000+0000,None,None
2,JBEAP,JBEAP-22948,Verified,2021-12-14T11:47:45.000+0000,2021-12-14T15:05:09.000+0000,None,None
3,JBEAP,JBEAP-22945,Pull Request Sent,2021-12-13T15:16:58.000+0000,None,None,None
4,JBEAP,JBEAP-22917,Verified,2021-12-10T06:17:20.000+0000,2021-12-17T11:51:07.000+0000,None,None


In [ ]:
dependency_records = []

for _, issue in jira_df.iterrows():

    issue_key = issue["issue_key"]
    project_key = issue["project_key"]

    for field in ["blocks", "blocked_by"]:

        links = issue.get(field) or []

        if not isinstance(links, list):
            continue

        for link in links:

            if not isinstance(link, dict):
                continue

            linked_key = link.get("issue_key")

            if not linked_key:
                continue

            if field == "blocks":
                upstream = issue_key
                downstream = linked_key
            else:
                upstream = linked_key
                downstream = issue_key

            dependency_records.append({
                "project_key": project_key,
                "upstream_issue": upstream,
                "downstream_issue": downstream,
                "source_issue": issue_key,
                "relationship_field": field
            })

dependency_df = pd.DataFrame(
    dependency_records,
    columns=[
        "project_key",
        "upstream_issue",
        "downstream_issue",
        "source_issue",
        "relationship_field"
    ]
)

print("Extracted blocking records:", len(dependency_df))

display(dependency_df.head(20))

Extracted blocking records: 4685


,project_key,upstream_issue,downstream_issue,source_issue,relationship_field
0,JBEAP,JBEAP-22610,JBEAP-22380,JBEAP-22380,blocked_by
1,JBEAP,JBEAP-22470,JBEAP-22380,JBEAP-22380,blocked_by
2,JBEAP,JBEAP-22041,JBEAP-22380,JBEAP-22380,blocked_by
3,JBEAP,JBEAP-22515,JBEAP-22254,JBEAP-22254,blocked_by
4,JBEAP,JBEAP-22517,JBEAP-22254,JBEAP-22254,blocked_by
5,JBEAP,JBEAP-22242,JBEAP-21970,JBEAP-21970,blocked_by
6,JBEAP,JBEAP-21765,JBEAP-21035,JBEAP-21035,blocked_by
7,JBEAP,JBEAP-21081,JBEAP-20899,JBEAP-20899,blocked_by
8,JBEAP,JBEAP-21011,JBEAP-20899,JBEAP-20899,blocked_by
9,JBEAP,JBEAP-20900,JBEAP-20899,JBEAP-20899,blocked_by


In [ ]:
dependency_pairs = (
    dependency_df
    .drop_duplicates(
        subset=[
            "upstream_issue",
            "downstream_issue"
        ]
    )
    .reset_index(drop=True)
)

print("Unique blocking pairs:", len(dependency_pairs))

display(dependency_pairs.head(20))

Unique blocking pairs: 4115


,project_key,upstream_issue,downstream_issue,source_issue,relationship_field
0,JBEAP,JBEAP-22610,JBEAP-22380,JBEAP-22380,blocked_by
1,JBEAP,JBEAP-22470,JBEAP-22380,JBEAP-22380,blocked_by
2,JBEAP,JBEAP-22041,JBEAP-22380,JBEAP-22380,blocked_by
3,JBEAP,JBEAP-22515,JBEAP-22254,JBEAP-22254,blocked_by
4,JBEAP,JBEAP-22517,JBEAP-22254,JBEAP-22254,blocked_by
5,JBEAP,JBEAP-22242,JBEAP-21970,JBEAP-21970,blocked_by
6,JBEAP,JBEAP-21765,JBEAP-21035,JBEAP-21035,blocked_by
7,JBEAP,JBEAP-21081,JBEAP-20899,JBEAP-20899,blocked_by
8,JBEAP,JBEAP-21011,JBEAP-20899,JBEAP-20899,blocked_by
9,JBEAP,JBEAP-20900,JBEAP-20899,JBEAP-20899,blocked_by


In [ ]:
available_issues = set(jira_df["issue_key"])

dependency_pairs["upstream_available"] = (
    dependency_pairs["upstream_issue"].isin(available_issues)
)

dependency_pairs["downstream_available"] = (
    dependency_pairs["downstream_issue"].isin(available_issues)
)

print("Both endpoints available:",
      (
          dependency_pairs["upstream_available"] &
          dependency_pairs["downstream_available"]
      ).sum())

print("Missing upstream records:",
      (~dependency_pairs["upstream_available"]).sum())

print("Missing downstream records:",
      (~dependency_pairs["downstream_available"]).sum())

Both endpoints available: 570
Missing upstream records: 2361
Missing downstream records: 1184


### 2.1 Historical dependency event reconstruction

In [ ]:
import re

link_history = events[
    events["event_type"] == "LINK_CHANGE"
].copy()

def parse_blocking_link(text, source_issue):
    if not isinstance(text, str):
        return None

    text = text.strip()

    match = re.search(
        r"\b([A-Z][A-Z0-9]+-\d+)\b",
        text
    )

    if not match:
        return None

    linked_issue = match.group(1)

    lower = text.lower()

    if "is blocked by" in lower:
        return {
            "upstream_issue": linked_issue,
            "downstream_issue": source_issue
        }

    if "blocks" in lower:
        return {
            "upstream_issue": source_issue,
            "downstream_issue": linked_issue
        }

    return None


historical_links = []

for _, row in link_history.iterrows():

    for value_field, action in [
        ("to_value", "ADD"),
        ("from_value", "REMOVE")
    ]:

        parsed = parse_blocking_link(
            row.get(value_field),
            row["source_id"]
        )

        if parsed is None:
            continue

        historical_links.append({
            "instance_id": row["instance_id"],
            "event_uid": row["uid"],
            "observed_at": row["observed_at"],
            "action": action,
            **parsed
        })

historical_links_df = pd.DataFrame(
    historical_links,
    columns=[
        "instance_id",
        "event_uid",
        "observed_at",
        "action",
        "upstream_issue",
        "downstream_issue"
    ]
)

print(
    "Historical blocking-link changes:",
    len(historical_links_df)
)

display(historical_links_df.head(20))

Historical blocking-link changes: 4563


,instance_id,event_uid,observed_at,action,upstream_issue,downstream_issue
0,JBEAP-I01,T_60b9a9e53ee877d3b82c4598,2016-07-29T07:55:38+00:00,ADD,JBEAP-5465,JBEAP-5464
1,JBEAP-I01,T_7d01fe67dc9d47a055a621f8,2016-07-29T08:16:19+00:00,ADD,JBEAP-5466,JBEAP-5464
2,JBEAP-I01,T_e27ceb1862f05fc0731a3e53,2016-07-29T08:19:03+00:00,ADD,JBEAP-5467,JBEAP-5464
3,JBEAP-I01,T_89d9e7ead6092c093c9eee2f,2016-07-19T12:03:29+00:00,ADD,WFCORE-1658,JBEAP-5275
4,JBEAP-I01,T_a6bf3d537bdeb6c5aca7ae34,2016-07-22T15:11:16+00:00,ADD,JBEAP-5310,JBEAP-5275
5,JBEAP-I01,T_3113cdda5a969b4fbd70d733,2016-07-19T01:46:33+00:00,ADD,MODCLUSTER-509,JBEAP-5248
6,JBEAP-I01,T_4cdeadb15baf47d2d3b850c8,2016-07-07T02:18:36+00:00,ADD,JBEAP-5146,JBEAP-5161
7,JBEAP-I01,T_3fa6120d153765a7849735e7,2016-07-07T02:19:30+00:00,ADD,JBEAP-5143,JBEAP-5161
8,JBEAP-I01,T_e63d0e0c23b395bed488b622,2016-07-07T02:19:30+00:00,ADD,JBEAP-5095,JBEAP-5161
9,JBEAP-I01,T_e9d92b82c8f607b762c4f2e4,2016-07-07T02:19:31+00:00,ADD,JBEAP-5096,JBEAP-5161


In [ ]:
historical_links_df["observed_dt"] = pd.to_datetime(
    historical_links_df["observed_at"],
    utc=True,
    errors="coerce"
)

print("Missing timestamps:",
      historical_links_df["observed_dt"].isna().sum())

print("\nActions:")
display(
    historical_links_df["action"].value_counts()
)

print("\nBlocking-link events by project:")
display(
    historical_links_df.groupby("instance_id")
    .size()
    .reset_index(name="link_events")
)

Missing timestamps: 0

Actions:


,count
action,
ADD,2692
REMOVE,1871



Blocking-link events by project:


,instance_id,link_events
0,AEROGEAR-I01,49
1,AMBARI-I01,13
2,BXMSDOC-I01,98
3,CONFSERVER-I01,2
4,DAOS-I01,197
5,ENTESB-I01,1125
6,FLEX-I01,6
7,HBASE-I01,29
8,INFRA-I01,17
9,JBDS-I01,25


In [ ]:
import pandas as pd

history = historical_links_df.copy()

history["observed_dt"] = pd.to_datetime(
    history["observed_at"],
    utc=True,
    errors="coerce"
)

history = history.dropna(subset=["observed_dt"])

history = history.sort_values(
    [
        "instance_id",
        "upstream_issue",
        "downstream_issue",
        "observed_dt",
        "action",
        "event_uid"
    ]
).reset_index(drop=True)

print("Historical link actions:", len(history))

display(history.head(10))

Historical link actions: 4563


,instance_id,event_uid,observed_at,action,upstream_issue,downstream_issue,observed_dt
0,AEROGEAR-I01,T_62d9e53fc092cf712ba348bf,2018-03-01T09:46:57+00:00,ADD,AEROGEAR-2210,AEROGEAR-2218,2018-03-01 09:46:57+00:00
1,AEROGEAR-I01,T_e49b1923fb0c5ef13292af3e,2018-03-01T09:46:57+00:00,ADD,AEROGEAR-2210,AEROGEAR-2218,2018-03-01 09:46:57+00:00
2,AEROGEAR-I01,T_008617171e2bba5c69d039a8,2018-03-05T07:30:44+00:00,ADD,AEROGEAR-2243,AEROGEAR-2242,2018-03-05 07:30:44+00:00
3,AEROGEAR-I01,T_36fe574bd0cf69a23db63895,2018-03-05T07:30:43+00:00,ADD,AEROGEAR-2244,AEROGEAR-2242,2018-03-05 07:30:43+00:00
4,AEROGEAR-I01,T_945abcfa921ed8ff4baf345b,2018-03-05T07:30:43+00:00,ADD,AEROGEAR-2244,AEROGEAR-2242,2018-03-05 07:30:43+00:00
5,AEROGEAR-I01,T_43e8d8780eb1370115794c18,2018-03-05T07:30:21+00:00,ADD,AEROGEAR-2244,AEROGEAR-2243,2018-03-05 07:30:21+00:00
6,AEROGEAR-I01,T_397ec8597f2000ddb67c0c9c,2018-03-06T04:23:32+00:00,ADD,AEROGEAR-2248,AEROGEAR-2263,2018-03-06 04:23:32+00:00
7,AEROGEAR-I01,T_56ee013fa0c5c004a887b0df,2018-03-06T04:23:32+00:00,ADD,AEROGEAR-2248,AEROGEAR-2263,2018-03-06 04:23:32+00:00
8,AEROGEAR-I01,T_23afb9522f5c47289b99f7a7,2018-03-06T04:23:33+00:00,ADD,AEROGEAR-2249,AEROGEAR-2263,2018-03-06 04:23:33+00:00
9,AEROGEAR-I01,T_e3ac2d397bec7d6d78145465,2018-03-06T04:23:33+00:00,ADD,AEROGEAR-2249,AEROGEAR-2263,2018-03-06 04:23:33+00:00


In [ ]:
intervals = []
anomalies = []

group_cols = [
    "instance_id",
    "upstream_issue",
    "downstream_issue"
]

for keys, group in history.groupby(group_cols):

    instance_id, upstream, downstream = keys

    active_since = None
    add_event_uid = None

    for _, row in group.iterrows():

        action = row["action"]
        timestamp = row["observed_dt"]

        if action == "ADD":

            if active_since is None:
                active_since = timestamp
                add_event_uid = row["event_uid"]

            else:
                anomalies.append({
                    "instance_id": instance_id,
                    "upstream_issue": upstream,
                    "downstream_issue": downstream,
                    "event_uid": row["event_uid"],
                    "reason": "ADD_WHILE_ACTIVE"
                })

        elif action == "REMOVE":

            if active_since is not None:

                intervals.append({
                    "instance_id": instance_id,
                    "upstream_issue": upstream,
                    "downstream_issue": downstream,
                    "active_from": active_since,
                    "active_to": timestamp,
                    "add_event_uid": add_event_uid,
                    "remove_event_uid": row["event_uid"]
                })

                active_since = None
                add_event_uid = None

            else:
                anomalies.append({
                    "instance_id": instance_id,
                    "upstream_issue": upstream,
                    "downstream_issue": downstream,
                    "event_uid": row["event_uid"],
                    "reason": "REMOVE_WITHOUT_ADD"
                })

    # Relationship still active at the end of observed history
    if active_since is not None:

        intervals.append({
            "instance_id": instance_id,
            "upstream_issue": upstream,
            "downstream_issue": downstream,
            "active_from": active_since,
            "active_to": pd.NaT,
            "add_event_uid": add_event_uid,
            "remove_event_uid": None
        })

intervals_df = pd.DataFrame(intervals)
anomalies_df = pd.DataFrame(anomalies)

print("Reconstructed intervals:", len(intervals_df))
print("Anomalous actions:", len(anomalies_df))

if not intervals_df.empty:
    display(intervals_df.head(20))

if not anomalies_df.empty:
    display(anomalies_df["reason"].value_counts())

Reconstructed intervals: 2465
Anomalous actions: 1197


,instance_id,upstream_issue,downstream_issue,active_from,active_to,add_event_uid,remove_event_uid
0,AEROGEAR-I01,AEROGEAR-2210,AEROGEAR-2218,2018-03-01 09:46:57+00:00,NaT,T_62d9e53fc092cf712ba348bf,None
1,AEROGEAR-I01,AEROGEAR-2243,AEROGEAR-2242,2018-03-05 07:30:44+00:00,NaT,T_008617171e2bba5c69d039a8,None
2,AEROGEAR-I01,AEROGEAR-2244,AEROGEAR-2242,2018-03-05 07:30:43+00:00,NaT,T_36fe574bd0cf69a23db63895,None
3,AEROGEAR-I01,AEROGEAR-2244,AEROGEAR-2243,2018-03-05 07:30:21+00:00,NaT,T_43e8d8780eb1370115794c18,None
4,AEROGEAR-I01,AEROGEAR-2248,AEROGEAR-2263,2018-03-06 04:23:32+00:00,NaT,T_397ec8597f2000ddb67c0c9c,None
5,AEROGEAR-I01,AEROGEAR-2249,AEROGEAR-2263,2018-03-06 04:23:33+00:00,NaT,T_23afb9522f5c47289b99f7a7,None
6,AEROGEAR-I01,AEROGEAR-2250,AEROGEAR-2248,2018-03-05 10:50:41+00:00,NaT,T_0477e50f9ea3446798d732bb,None
7,AEROGEAR-I01,AEROGEAR-2250,AEROGEAR-2249,2018-03-05 10:50:56+00:00,NaT,T_11e719c87e5093c8f005ad77,None
8,AEROGEAR-I01,AEROGEAR-2250,AEROGEAR-2263,2018-03-06 04:23:32+00:00,NaT,T_98aca45816a244e8941f958d,None
9,AEROGEAR-I01,AEROGEAR-2266,AEROGEAR-2218,2018-03-07 03:23:43+00:00,NaT,T_91c202b163b627817ca23d0d,None


,count
reason,
REMOVE_WITHOUT_ADD,970
ADD_WHILE_ACTIVE,227


In [ ]:
if not intervals_df.empty:

    intervals_df["active_from"] = pd.to_datetime(
        intervals_df["active_from"],
        utc=True
    )

    intervals_df["active_to"] = pd.to_datetime(
        intervals_df["active_to"],
        utc=True
    )

    closed_intervals = intervals_df.dropna(
        subset=["active_to"]
    ).copy()

    closed_intervals["duration_days"] = (
        closed_intervals["active_to"]
        - closed_intervals["active_from"]
    ).dt.total_seconds() / 86400

    print("Closed intervals:", len(closed_intervals))

    print(
        "Open intervals:",
        intervals_df["active_to"].isna().sum()
    )

    print(
        "Invalid negative durations:",
        (closed_intervals["duration_days"] < 0).sum()
    )

    display(
        closed_intervals["duration_days"].describe()
    )

Closed intervals: 901
Open intervals: 1564
Invalid negative durations: 0


,duration_days
count,901.000000
mean,4.636728
std,22.297321
min,0.000000
25%,0.000000
50%,0.000000
75%,0.000000
max,182.030799


In [ ]:
import pandas as pd

history = historical_links_df.copy()

history["observed_dt"] = pd.to_datetime(
    history["observed_at"],
    utc=True,
    errors="coerce"
)

key_cols = [
    "instance_id",
    "upstream_issue",
    "downstream_issue",
    "observed_dt"
]

# 1. Exact duplicate actions
duplicate_mask = history.duplicated(
    subset=key_cols + ["action"],
    keep=False
)

duplicate_actions = history[duplicate_mask]

print(
    "Rows involved in duplicate actions:",
    len(duplicate_actions)
)

print(
    "Duplicate action rows removable:",
    history.duplicated(
        subset=key_cols + ["action"]
    ).sum()
)

# 2. Same timestamp with both ADD and REMOVE
simultaneous = (
    history.groupby(key_cols)["action"]
    .agg(lambda x: set(x))
    .reset_index()
)

simultaneous = simultaneous[
    simultaneous["action"].apply(
        lambda x: {"ADD", "REMOVE"}.issubset(x)
    )
]

print(
    "\nPairs with simultaneous ADD and REMOVE:",
    len(simultaneous)
)

# 3. Inspect examples
print("\nDuplicate action examples:")
display(
    duplicate_actions[
        [
            "instance_id",
            "upstream_issue",
            "downstream_issue",
            "observed_dt",
            "action",
            "event_uid"
        ]
    ].head(20)
)

print("\nSimultaneous action examples:")
display(simultaneous.head(20))

Rows involved in duplicate actions: 752
Duplicate action rows removable: 376

Pairs with simultaneous ADD and REMOVE: 808

Duplicate action examples:


,instance_id,upstream_issue,downstream_issue,observed_dt,action,event_uid
41,JBEAP-I01,JBEAP-2433,JBEAP-4120,2016-04-07 07:45:37+00:00,ADD,T_d745335b369fe136a680b7a0
61,JBEAP-I01,JBEAP-3862,JBEAP-2153,2016-03-17 09:44:30+00:00,ADD,T_f1dd1ba2b17fee50a63d1049
67,JBEAP-I01,JBEAP-2532,JBEAP-3636,2016-03-02 05:47:10+00:00,ADD,T_5b1154ec9959c845323caa1c
73,JBEAP-I01,JBEAP-3032,JBEAP-3636,2016-03-02 06:45:14+00:00,ADD,T_e55597c382b81b56ecbe7d4a
77,JBEAP-I01,JBEAP-2532,JBEAP-3636,2016-03-09 05:50:43+00:00,REMOVE,T_9426e67e61692996696eee42
82,JBEAP-I01,JBEAP-3032,JBEAP-3636,2016-03-09 05:51:00+00:00,REMOVE,T_3a443e3c8bf801e38ffaae1d
93,JBEAP-I01,JBEAP-3636,JBEAP-3032,2016-03-09 05:53:42+00:00,ADD,T_9090ad229a7830ed4770b62d
96,JBEAP-I01,JBEAP-3636,JBEAP-2532,2016-03-09 05:53:42+00:00,ADD,T_0b8878fbbc2bb7cc96c5a70b
112,JBEAP-I01,JBEAP-3577,JBEAP-2315,2016-03-14 05:58:21+00:00,ADD,T_2d8cd853651eff5933da9e62
113,JBEAP-I01,JBEAP-3577,JBEAP-2946,2016-03-21 06:18:32+00:00,ADD,T_f0fa7a025b22c3e50d309815



Simultaneous action examples:


,instance_id,upstream_issue,downstream_issue,observed_dt,action
53,BXMSDOC-I01,BXMSDOC-1119,BXMSDOC-407,2020-09-14 05:41:51+00:00,"{ADD, REMOVE}"
60,BXMSDOC-I01,BXMSDOC-1429,BXMSDOC-1317,2020-09-14 05:41:47+00:00,"{ADD, REMOVE}"
62,BXMSDOC-I01,BXMSDOC-1430,BXMSDOC-1317,2020-09-14 05:41:47+00:00,"{ADD, REMOVE}"
64,BXMSDOC-I01,BXMSDOC-1431,BXMSDOC-1317,2020-09-14 05:41:47+00:00,"{ADD, REMOVE}"
68,BXMSDOC-I01,BXMSDOC-1432,BXMSDOC-1317,2020-09-14 05:41:47+00:00,"{ADD, REMOVE}"
70,BXMSDOC-I01,BXMSDOC-1432,BXMSDOC-1431,2020-09-14 05:41:47+00:00,"{ADD, REMOVE}"
72,BXMSDOC-I01,BXMSDOC-1433,BAPL-662,2020-09-14 05:42:32+00:00,"{ADD, REMOVE}"
75,BXMSDOC-I01,BXMSDOC-1613,BXMSDOC-1612,2020-09-14 05:41:47+00:00,"{ADD, REMOVE}"
78,BXMSDOC-I01,BXMSDOC-3307,AF-1664,2020-09-14 05:43:38+00:00,"{ADD, REMOVE}"
80,BXMSDOC-I01,BXMSDOC-3449,BAPL-1003,2020-09-14 05:42:22+00:00,"{ADD, REMOVE}"


In [ ]:
# Get event IDs involved in simultaneous actions
simultaneous_keys = simultaneous[key_cols]

simultaneous_events = history.merge(
    simultaneous_keys,
    on=key_cols,
    how="inner"
)

sample_event_ids = (
    simultaneous_events["event_uid"]
    .drop_duplicates()
    .head(5)
    .tolist()
)

sample_nodes = nodes_df[
    nodes_df["uid"].isin(sample_event_ids)
]

for _, row in sample_nodes.iterrows():

    print("\n" + "=" * 60)
    print("Event UID:", row["uid"])
    print("Observed at:", row["observed_at"])
    print("Source:", row["source_id"])
    print("Payload:")
    print(str(row["payload_json"])[:2500])


Event UID: T_fb54090d4b0eea5b567b9b72
Observed at: 2020-09-14T05:09:50+00:00
Source: JBIDE-26599
Payload:
{"actor": "<<|author_displayName|082e60dc-fe91-45a3-adae-7927ef9e1288|>>", "attributes": {"custom_field_fallback": false, "field_id": null, "field_type": "jira", "jira_field": "Link", "source_structure": "full_changelog"}, "entity": "ISSUE::JBIDE-I01::jbide-26599::a3ae241ca59b", "event_time": "2020-09-14T05:09:50.000+0000", "event_type": "LINK_CHANGE", "extraction_method": "structured", "from_value": "This issue blocks-legacy RHDEVDOCS-816", "to_value": null}

Event UID: T_1e3c40e7c8fc0ca79507eece
Observed at: 2020-09-14T05:09:50+00:00
Source: JBIDE-26599
Payload:
{"actor": "<<|author_displayName|082e60dc-fe91-45a3-adae-7927ef9e1288|>>", "attributes": {"custom_field_fallback": false, "field_id": null, "field_type": "jira", "jira_field": "Link", "source_structure": "full_changelog"}, "entity": "ISSUE::JBIDE-I01::jbide-26599::a3ae241ca59b", "event_time": "2020-09-14T05:09:50.000+000

In [ ]:
import re
import pandas as pd

LINK_PATTERN = re.compile(
    r"^This issue (is blocked by|blocks)(-legacy)?\s+"
    r"([A-Z][A-Z0-9]+-\d+)$",
    re.IGNORECASE
)

def parse_link(value, source_issue):
    if not isinstance(value, str):
        return None

    match = LINK_PATTERN.match(value.strip())

    if not match:
        return None

    relation = match.group(1).lower()
    legacy = bool(match.group(2))
    linked_issue = match.group(3).upper()

    if relation == "blocks":
        upstream = source_issue
        downstream = linked_issue
    else:
        upstream = linked_issue
        downstream = source_issue

    return {
        "upstream_issue": upstream,
        "downstream_issue": downstream,
        "legacy": legacy
    }


link_history = events[
    events["event_type"] == "LINK_CHANGE"
].copy()

corrected_records = []

for _, row in link_history.iterrows():

    old_link = parse_link(
        row.get("from_value"),
        row["source_id"]
    )

    new_link = parse_link(
        row.get("to_value"),
        row["source_id"]
    )

    base = {
        "instance_id": row["instance_id"],
        "event_uid": row["uid"],
        "observed_at": row["observed_at"]
    }

    if old_link and new_link:

        same_pair = (
            old_link["upstream_issue"] ==
            new_link["upstream_issue"]
            and
            old_link["downstream_issue"] ==
            new_link["downstream_issue"]
        )

        if same_pair:
            action = (
                "MIGRATION"
                if old_link["legacy"] != new_link["legacy"]
                else "NO_STATE_CHANGE"
            )

            corrected_records.append({
                **base,
                "action": action,
                **new_link
            })

        else:
            corrected_records.append({
                **base,
                "action": "REMOVE",
                **old_link
            })

            corrected_records.append({
                **base,
                "action": "ADD",
                **new_link
            })

    elif new_link:
        corrected_records.append({
            **base,
            "action": "ADD",
            **new_link
        })

    elif old_link:
        corrected_records.append({
            **base,
            "action": "REMOVE",
            **old_link
        })


corrected_links_df = pd.DataFrame(corrected_records)

corrected_links_df["observed_dt"] = pd.to_datetime(
    corrected_links_df["observed_at"],
    utc=True,
    errors="coerce"
)

print("Corrected link records:", len(corrected_links_df))

display(
    corrected_links_df["action"].value_counts()
)

Corrected link records: 4563


,count
action,
ADD,2692
REMOVE,1871


In [ ]:
key_cols = [
    "instance_id",
    "upstream_issue",
    "downstream_issue",
    "observed_dt",
    "action"
]

corrected_links_df = (
    corrected_links_df
    .sort_values("event_uid")
    .drop_duplicates(subset=key_cols)
    .reset_index(drop=True)
)

print("Records after deduplication:", len(corrected_links_df))

display(
    corrected_links_df["action"].value_counts()
)

Records after deduplication: 4187


,count
action,
ADD,2467
REMOVE,1720


In [ ]:
# Attach original changelog values to corrected link records

original_values = events[
    ["uid", "from_value", "to_value", "source_id"]
].drop_duplicates(subset=["uid"])

links_with_text = corrected_links_df.merge(
    original_values,
    left_on="event_uid",
    right_on="uid",
    how="left"
)

print("Records with original text:", len(links_with_text))

display(
    links_with_text[
        [
            "instance_id",
            "upstream_issue",
            "downstream_issue",
            "observed_dt",
            "action",
            "from_value",
            "to_value"
        ]
    ].head(10)
)

Records with original text: 4187


,instance_id,upstream_issue,downstream_issue,observed_dt,action,from_value,to_value
0,JBIDE-I01,JBIDE-24013,JBIDE-23317,2020-09-13 09:01:12+00:00,REMOVE,This issue is blocked by-legacy JBIDE-24013,None
1,JBIDE-I01,JBDS-2539,JBIDE-13618,2020-09-13 09:44:38+00:00,ADD,None,This issue is blocked by-legacy JBDS-2539
2,JBIDE-I01,JBIDE-15639,JBIDE-15649,2020-09-13 09:05:27+00:00,ADD,None,This issue blocks-legacy JBIDE-15649
3,JBIDE-I01,JBIDE-20333,JBIDE-19335,2020-09-13 09:02:33+00:00,ADD,None,This issue is blocked by-legacy JBIDE-20333
4,JBIDE-I01,FUSETOOLS-2420,JBIDE-24243,2020-09-13 09:00:57+00:00,REMOVE,This issue is blocked by-legacy FUSETOOLS-2420,None
5,ENTESB-I01,ENTESB-6976,ENTESB-6989,2020-09-14 05:38:32+00:00,REMOVE,This issue blocks-legacy ENTESB-6989,None
6,JBEAP-I01,JBEAP-3001,JBEAP-3408,2016-02-15 10:19:18+00:00,ADD,None,This issue is blocked by JBEAP-3001
7,AEROGEAR-I01,AEROGEAR-2243,AEROGEAR-2242,2018-03-05 07:30:44+00:00,ADD,None,This issue is blocked by AEROGEAR-2243
8,ENTESB-I01,ENTESB-6976,ENTESB-6979,2020-09-14 05:38:33+00:00,ADD,None,This issue is blocked by ENTESB-6976
9,JBIDE-I01,JBIDE-8554,JBIDE-8624,2020-09-14 05:29:13+00:00,ADD,None,This issue is blocked by JBIDE-8554


In [ ]:
key_cols = [
    "instance_id",
    "upstream_issue",
    "downstream_issue",
    "observed_dt"
]

def is_legacy_migration(group):

    if not {"ADD", "REMOVE"}.issubset(
        set(group["action"])
    ):
        return False

    removes = group[group["action"] == "REMOVE"]
    adds = group[group["action"] == "ADD"]

    legacy_remove = removes["from_value"].fillna(
        ""
    ).str.contains(
        r"-legacy\b",
        case=False,
        regex=True
    ).any()

    standard_add = (
        adds["to_value"]
        .fillna("")
        .str.contains(
            r"^This issue (blocks|is blocked by)\s+"
            r"[A-Z][A-Z0-9]+-\d+$",
            case=False,
            regex=True
        )
        .any()
    )

    return legacy_remove and standard_add


migration_groups = []

for keys, group in links_with_text.groupby(key_cols):

    if is_legacy_migration(group):
        migration_groups.append({
            **dict(zip(key_cols, keys)),
            "event_uids": group["event_uid"].tolist()
        })

migration_df = pd.DataFrame(migration_groups)

print("Confirmed legacy migration groups:", len(migration_df))

display(migration_df.head(10))

/tmp/ipykernel_1144/1558336693.py:29: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(


Confirmed legacy migration groups: 808


,instance_id,upstream_issue,downstream_issue,observed_dt,event_uids
0,BXMSDOC-I01,BXMSDOC-1119,BXMSDOC-407,2020-09-14 05:41:51+00:00,"[T_6aad7890e10bafd68d472906, T_ae91c5df906c390..."
1,BXMSDOC-I01,BXMSDOC-1429,BXMSDOC-1317,2020-09-14 05:41:47+00:00,"[T_3347d0641ef035f0821b112b, T_ba022c1c0f1348b..."
2,BXMSDOC-I01,BXMSDOC-1430,BXMSDOC-1317,2020-09-14 05:41:47+00:00,"[T_6d8d31c0763047db36b72a93, T_b61cec9756490f6..."
3,BXMSDOC-I01,BXMSDOC-1431,BXMSDOC-1317,2020-09-14 05:41:47+00:00,"[T_291d4e690c59566d338d106d, T_a9f1bb27d52c135..."
4,BXMSDOC-I01,BXMSDOC-1432,BXMSDOC-1317,2020-09-14 05:41:47+00:00,"[T_6b4095413c69abf2c0e51abb, T_caf5eba322adea8..."
5,BXMSDOC-I01,BXMSDOC-1432,BXMSDOC-1431,2020-09-14 05:41:47+00:00,"[T_11c2131d4e548da03fc1c5da, T_51a94678595579b..."
6,BXMSDOC-I01,BXMSDOC-1433,BAPL-662,2020-09-14 05:42:32+00:00,"[T_11f21f7a9d27bd472ed6e57e, T_d7fd5fac2bd61ca..."
7,BXMSDOC-I01,BXMSDOC-1613,BXMSDOC-1612,2020-09-14 05:41:47+00:00,"[T_6e4d044d919f21a1d1098d35, T_df95e3c5d397cc8..."
8,BXMSDOC-I01,BXMSDOC-3307,AF-1664,2020-09-14 05:43:38+00:00,"[T_588a97045f930ba053ddc437, T_825763ae32c45fb..."
9,BXMSDOC-I01,BXMSDOC-3449,BAPL-1003,2020-09-14 05:42:22+00:00,"[T_4db022b5442d9555b83cb1d7, T_da3309eb6d96acb..."


In [ ]:
import pandas as pd

key_cols = [
    "instance_id",
    "upstream_issue",
    "downstream_issue",
    "observed_dt"
]

# Work from the deduplicated records with original text
clean_links = links_with_text.copy()

migration_records = []
remove_indices = []

for _, migration in migration_df.iterrows():

    mask = pd.Series(True, index=clean_links.index)

    for col in key_cols:
        mask &= clean_links[col].eq(migration[col])

    group = clean_links[mask]

    legacy_remove = group[
        (group["action"] == "REMOVE") &
        (group["from_value"].fillna("")
         .str.contains(r"-legacy\b", case=False))
    ]

    standard_add = group[
        (group["action"] == "ADD") &
        (~group["to_value"].fillna("")
          .str.contains(r"-legacy\b", case=False))
    ]

    if legacy_remove.empty or standard_add.empty:
        continue

    old = legacy_remove.iloc[0]
    new = standard_add.iloc[0]

    remove_indices.extend([old.name, new.name])

    migration_records.append({
        "instance_id": old["instance_id"],
        "upstream_issue": old["upstream_issue"],
        "downstream_issue": old["downstream_issue"],
        "observed_dt": old["observed_dt"],
        "action": "MIGRATION",
        "event_uid": new["event_uid"],
        "source_event_uids": [
            old["event_uid"],
            new["event_uid"]
        ]
    })

# Remove paired ADD/REMOVE records
clean_links = clean_links.drop(
    index=set(remove_indices)
)

# Add consolidated migrations
clean_links = pd.concat(
    [
        clean_links,
        pd.DataFrame(migration_records)
    ],
    ignore_index=True
)

clean_links = clean_links.sort_values(
    key_cols + ["action"]
).reset_index(drop=True)

print("Consolidated migrations:", len(migration_records))
print("Remaining link actions:", len(clean_links))

display(clean_links["action"].value_counts())

Consolidated migrations: 808
Remaining link actions: 3379


,count
action,
ADD,1659
REMOVE,912
MIGRATION,808


### 2.2 Corrected dependency intervals

In [ ]:
interval_records = []
interval_anomalies = []

pair_cols = [
    "instance_id",
    "upstream_issue",
    "downstream_issue"
]

for keys, group in clean_links.groupby(pair_cols):

    instance_id, upstream, downstream = keys

    group = group.sort_values(
        ["observed_dt", "action"]
    )

    active = False
    start_time = None
    start_known = False
    start_event = None

    for _, row in group.iterrows():

        action = row["action"]
        timestamp = row["observed_dt"]

        if action == "ADD":

            if not active:
                active = True
                start_time = timestamp
                start_known = True
                start_event = row["event_uid"]

            else:
                interval_anomalies.append({
                    "instance_id": instance_id,
                    "upstream_issue": upstream,
                    "downstream_issue": downstream,
                    "reason": "ADD_WHILE_ACTIVE",
                    "event_uid": row["event_uid"]
                })

        elif action == "MIGRATION":

            if not active:
                active = True
                start_time = timestamp
                start_known = False
                start_event = row["event_uid"]

            # If already active, migration does not
            # change the blocking state.

        elif action == "REMOVE":

            if active:

                interval_records.append({
                    "instance_id": instance_id,
                    "upstream_issue": upstream,
                    "downstream_issue": downstream,
                    "active_from": (
                        start_time if start_known else pd.NaT
                    ),
                    "first_confirmed_at": start_time,
                    "active_to": timestamp,
                    "start_known": start_known,
                    "start_event_uid": start_event,
                    "end_event_uid": row["event_uid"]
                })

                active = False
                start_time = None
                start_known = False
                start_event = None

            else:
                interval_anomalies.append({
                    "instance_id": instance_id,
                    "upstream_issue": upstream,
                    "downstream_issue": downstream,
                    "reason": "REMOVE_WITHOUT_KNOWN_START",
                    "event_uid": row["event_uid"]
                })

    if active:
        interval_records.append({
            "instance_id": instance_id,
            "upstream_issue": upstream,
            "downstream_issue": downstream,
            "active_from": (
                start_time if start_known else pd.NaT
            ),
            "first_confirmed_at": start_time,
            "active_to": pd.NaT,
            "start_known": start_known,
            "start_event_uid": start_event,
            "end_event_uid": None
        })

corrected_intervals_df = pd.DataFrame(interval_records)
interval_anomalies_df = pd.DataFrame(interval_anomalies)

print("Corrected intervals:", len(corrected_intervals_df))
print("Remaining anomalies:", len(interval_anomalies_df))

if not corrected_intervals_df.empty:
    display(corrected_intervals_df.head(15))

if not interval_anomalies_df.empty:
    display(interval_anomalies_df["reason"].value_counts())

Corrected intervals: 2465
Remaining anomalies: 820


,instance_id,upstream_issue,downstream_issue,active_from,first_confirmed_at,active_to,start_known,start_event_uid,end_event_uid
0,AEROGEAR-I01,AEROGEAR-2210,AEROGEAR-2218,2018-03-01 09:46:57+00:00,2018-03-01 09:46:57+00:00,NaT,True,T_62d9e53fc092cf712ba348bf,None
1,AEROGEAR-I01,AEROGEAR-2243,AEROGEAR-2242,2018-03-05 07:30:44+00:00,2018-03-05 07:30:44+00:00,NaT,True,T_008617171e2bba5c69d039a8,None
2,AEROGEAR-I01,AEROGEAR-2244,AEROGEAR-2242,2018-03-05 07:30:43+00:00,2018-03-05 07:30:43+00:00,NaT,True,T_36fe574bd0cf69a23db63895,None
3,AEROGEAR-I01,AEROGEAR-2244,AEROGEAR-2243,2018-03-05 07:30:21+00:00,2018-03-05 07:30:21+00:00,NaT,True,T_43e8d8780eb1370115794c18,None
4,AEROGEAR-I01,AEROGEAR-2248,AEROGEAR-2263,2018-03-06 04:23:32+00:00,2018-03-06 04:23:32+00:00,NaT,True,T_397ec8597f2000ddb67c0c9c,None
5,AEROGEAR-I01,AEROGEAR-2249,AEROGEAR-2263,2018-03-06 04:23:33+00:00,2018-03-06 04:23:33+00:00,NaT,True,T_23afb9522f5c47289b99f7a7,None
6,AEROGEAR-I01,AEROGEAR-2250,AEROGEAR-2248,2018-03-05 10:50:41+00:00,2018-03-05 10:50:41+00:00,NaT,True,T_0477e50f9ea3446798d732bb,None
7,AEROGEAR-I01,AEROGEAR-2250,AEROGEAR-2249,2018-03-05 10:50:56+00:00,2018-03-05 10:50:56+00:00,NaT,True,T_11e719c87e5093c8f005ad77,None
8,AEROGEAR-I01,AEROGEAR-2250,AEROGEAR-2263,2018-03-06 04:23:32+00:00,2018-03-06 04:23:32+00:00,NaT,True,T_98aca45816a244e8941f958d,None
9,AEROGEAR-I01,AEROGEAR-2266,AEROGEAR-2218,2018-03-07 03:23:43+00:00,2018-03-07 03:23:43+00:00,NaT,True,T_91c202b163b627817ca23d0d,None


,count
reason,
REMOVE_WITHOUT_KNOWN_START,818
ADD_WHILE_ACTIVE,2


In [ ]:
if not corrected_intervals_df.empty:

    intervals = corrected_intervals_df.copy()

    for col in [
        "active_from",
        "first_confirmed_at",
        "active_to"
    ]:
        intervals[col] = pd.to_datetime(
            intervals[col],
            utc=True,
            errors="coerce"
        )

    known_start = intervals[
        intervals["start_known"]
    ].copy()

    closed_known = known_start.dropna(
        subset=["active_to"]
    ).copy()

    closed_known["duration_days"] = (
        closed_known["active_to"] -
        closed_known["active_from"]
    ).dt.total_seconds() / 86400

    print("Total intervals:", len(intervals))
    print("Known-start intervals:", len(known_start))
    print(
        "Unknown-start intervals:",
        (~intervals["start_known"]).sum()
    )
    print(
        "Open intervals:",
        intervals["active_to"].isna().sum()
    )
    print(
        "Negative durations:",
        (closed_known["duration_days"] < 0).sum()
    )
    print(
        "Zero-duration intervals:",
        (closed_known["duration_days"] == 0).sum()
    )

    print("\nKnown-start closed interval durations:")
    display(closed_known["duration_days"].describe())

Total intervals: 2465
Known-start intervals: 1657
Unknown-start intervals: 808
Open intervals: 2371
Negative durations: 0
Zero-duration intervals: 0

Known-start closed interval durations:


,duration_days
count,93.000000
mean,44.921416
std,55.084564
min,0.000150
25%,1.718160
50%,6.967801
75%,101.258299
max,182.030799


In [ ]:
import pandas as pd

status_events = events[
    events["event_type"] == "STATUS_CHANGE"
].copy()

status_events["event_dt"] = pd.to_datetime(
    status_events["observed_at"],
    utc=True,
    errors="coerce"
)

status_events["from_status"] = (
    status_events["from_value"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

status_events["to_status"] = (
    status_events["to_value"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

def classify_disruption(row):

    old = row["from_status"]
    new = row["to_status"]

    if old in {"resolved", "closed"} and new in {
        "reopened", "open", "in progress"
    }:
        return "REOPENED_WORK"

    if old == "in progress" and new == "open":
        return "WORKFLOW_REGRESSION"

    if new in {"blocked", "on hold", "waiting for dependency"}:
        return "BLOCKED_STATUS"

    return None


status_events["disruption_type"] = (
    status_events.apply(classify_disruption, axis=1)
)

disruptions_df = status_events[
    status_events["disruption_type"].notna()
    & status_events["event_dt"].notna()
].copy()

print("Candidate downstream disruptions:", len(disruptions_df))

display(
    disruptions_df["disruption_type"].value_counts()
)

display(
    disruptions_df[
        [
            "instance_id",
            "source_id",
            "event_dt",
            "from_value",
            "to_value",
            "disruption_type"
        ]
    ].head(20)
)

Candidate downstream disruptions: 1062


,count
disruption_type,
REOPENED_WORK,804
WORKFLOW_REGRESSION,143
BLOCKED_STATUS,115


,instance_id,source_id,event_dt,from_value,to_value,disruption_type
40875,JBEAP-I01,JBEAP-4518,2016-07-20 10:26:26+00:00,Closed,Reopened,REOPENED_WORK
40994,JBEAP-I01,JBEAP-4347,2016-05-01 22:00:30+00:00,Resolved,Reopened,REOPENED_WORK
40999,JBEAP-I01,JBEAP-4347,2016-05-03 02:25:51+00:00,Resolved,Reopened,REOPENED_WORK
41078,JBEAP-I01,JBEAP-4162,2016-05-03 04:26:45+00:00,Resolved,Reopened,REOPENED_WORK
41255,JBEAP-I01,JBEAP-3969,2016-05-31 12:02:09+00:00,Resolved,Reopened,REOPENED_WORK
41420,JBEAP-I01,JBEAP-3825,2016-08-01 14:26:33+00:00,Resolved,Reopened,REOPENED_WORK
41702,JBEAP-I01,JBEAP-3459,2016-03-07 08:09:50+00:00,Closed,Reopened,REOPENED_WORK
42347,JBEAP-I01,JBEAP-3154,2016-02-23 09:34:12+00:00,Closed,Reopened,REOPENED_WORK
42351,JBEAP-I01,JBEAP-3154,2016-02-25 06:26:39+00:00,Closed,Reopened,REOPENED_WORK
42405,JBEAP-I01,JBEAP-3134,2016-02-24 20:29:53+00:00,Resolved,Reopened,REOPENED_WORK


In [ ]:
intervals = corrected_intervals_df.copy()

intervals["first_confirmed_at"] = pd.to_datetime(
    intervals["first_confirmed_at"],
    utc=True,
    errors="coerce"
)

intervals["active_to"] = pd.to_datetime(
    intervals["active_to"],
    utc=True,
    errors="coerce"
)

# Only retain intervals with a confirmed observation
intervals = intervals.dropna(
    subset=["first_confirmed_at"]
)

# Join downstream issue history with blocking intervals
candidate_matches = disruptions_df.merge(
    intervals,
    left_on=["instance_id", "source_id"],
    right_on=["instance_id", "downstream_issue"],
    how="inner"
)

# Require disruption after first confirmed dependency observation
valid_time = (
    candidate_matches["event_dt"]
    >= candidate_matches["first_confirmed_at"]
)

# Require disruption before removal, where observed
valid_end = (
    candidate_matches["active_to"].isna()
    |
    (
        candidate_matches["event_dt"]
        < candidate_matches["active_to"]
    )
)

dependency_disruptions_df = candidate_matches[
    valid_time & valid_end
].copy()

print(
    "Temporally associated dependency disruptions:",
    len(dependency_disruptions_df)
)

display(
    dependency_disruptions_df[
        [
            "instance_id",
            "upstream_issue",
            "downstream_issue",
            "first_confirmed_at",
            "event_dt",
            "disruption_type"
        ]
    ].head(20)
)

Temporally associated dependency disruptions: 160


,instance_id,upstream_issue,downstream_issue,first_confirmed_at,event_dt,disruption_type
0,JBEAP-I01,JBEAP-4646,JBEAP-3969,2016-05-20 12:36:08+00:00,2016-05-31 12:02:09+00:00,REOPENED_WORK
5,JBEAP-I01,WFCORE-1356,JBEAP-3154,2016-02-03 07:37:37+00:00,2016-02-23 09:34:12+00:00,REOPENED_WORK
6,JBEAP-I01,WFCORE-1356,JBEAP-3154,2016-02-03 07:37:37+00:00,2016-02-25 06:26:39+00:00,REOPENED_WORK
7,JBEAP-I01,JCSP-20,JBEAP-3134,2016-02-05 06:59:38+00:00,2016-02-24 20:29:53+00:00,REOPENED_WORK
18,JBEAP-I01,JBEAP-2499,JBEAP-2946,2016-01-26 03:54:24+00:00,2016-04-18 08:48:22+00:00,REOPENED_WORK
19,JBEAP-I01,JBEAP-3577,JBEAP-2946,2016-03-21 06:18:32+00:00,2016-04-18 08:48:22+00:00,REOPENED_WORK
20,JBEAP-I01,JBEAP-4223,JBEAP-2946,2016-04-15 08:36:22+00:00,2016-04-18 08:48:22+00:00,REOPENED_WORK
22,JBEAP-I01,WFLY-6086,JBEAP-2739,2016-01-28 09:51:31+00:00,2016-05-12 02:33:18+00:00,REOPENED_WORK
30,JBEAP-I01,JBEAP-2597,JBEAP-2698,2016-01-12 06:16:18+00:00,2016-02-24 14:44:08+00:00,REOPENED_WORK
36,JBEAP-I01,JBEAP-1269,JBEAP-2574,2016-01-06 06:31:41+00:00,2016-01-12 08:31:37+00:00,REOPENED_WORK


In [ ]:
available_issue_keys = set(jira_df["issue_key"])

dependency_disruptions_df["upstream_available"] = (
    dependency_disruptions_df["upstream_issue"]
    .isin(available_issue_keys)
)

print("Total associated disruptions:",
      len(dependency_disruptions_df))

print("Upstream issue available:",
      dependency_disruptions_df["upstream_available"].sum())

print("Upstream issue unavailable:",
      (~dependency_disruptions_df["upstream_available"]).sum())

print("\nDisruptions by project:")
display(
    dependency_disruptions_df.groupby("instance_id")
    .size()
    .reset_index(name="candidate_count")
    .sort_values("candidate_count", ascending=False)
)

Total associated disruptions: 160
Upstream issue available: 43
Upstream issue unavailable: 117

Disruptions by project:


,instance_id,candidate_count
2,DAOS-I01,72
5,JBEAP-I01,41
6,JBIDE-I01,26
4,JBDS-I01,12
8,SPARK-I01,4
3,HBASE-I01,2
0,AEROGEAR-I01,1
1,AMBARI-I01,1
7,MC-I01,1


In [ ]:
# Select an available upstream issue
available_candidates = dependency_disruptions_df[
    dependency_disruptions_df["upstream_available"]
]

if available_candidates.empty:
    print("No available upstream issues found.")

else:
    sample_upstream = available_candidates.iloc[0]["upstream_issue"]

    issue_record = jira_df[
        jira_df["issue_key"] == sample_upstream
    ].iloc[0]

    print("Upstream issue:", sample_upstream)
    print("Created at:", issue_record["created_at"])
    print("Current status:", issue_record["status"])

    print("\nStatus history:")
    print(
        json.dumps(
            issue_record["status_history"][:5],
            indent=2,
            default=str
        )
    )

Upstream issue: JBEAP-2499
Created at: 2015-12-21T09:04:56.000+0000
Current status: Closed

Status history:
[
  {
    "changed_at": "2016-01-12T08:38:24.000+0000",
    "changed_by": "<<|author_displayName|6fbb59d8-967c-4205-a3e4-186a1a70842a|>>",
    "from_status": "New",
    "to_status": "Closed"
  },
  {
    "changed_at": "2016-01-12T08:38:31.000+0000",
    "changed_by": "<<|author_displayName|6fbb59d8-967c-4205-a3e4-186a1a70842a|>>",
    "from_status": "Closed",
    "to_status": "Reopened"
  },
  {
    "changed_at": "2016-01-13T06:55:00.000+0000",
    "changed_by": "<<|author_displayName|2fb24984-a4ec-4b4b-b178-989451ed427c|>>",
    "from_status": "Reopened",
    "to_status": "New"
  },
  {
    "changed_at": "2016-01-25T06:30:48.000+0000",
    "changed_by": "<<|author_displayName|6fbb59d8-967c-4205-a3e4-186a1a70842a|>>",
    "from_status": "New",
    "to_status": "Closed"
  },
  {
    "changed_at": "2016-01-25T06:30:53.000+0000",
    "changed_by": "<<|author_displayName|6fbb59d8-967

In [ ]:
import pandas as pd

# Define statuses that generally indicate completed work
TERMINAL_STATUSES = {
    "closed",
    "resolved",
    "done",
    "verified",
    "completed"
}

def normalize_status(value):
    if value is None:
        return None

    return str(value).strip().lower()


def status_at_time(issue_record, cutoff):

    created = pd.to_datetime(
        issue_record.get("created_at"),
        utc=True,
        errors="coerce"
    )

    if pd.isna(created) or cutoff < created:
        return None

    history = issue_record.get("status_history") or []

    transitions = []

    for item in history:

        timestamp = pd.to_datetime(
            item.get("changed_at"),
            utc=True,
            errors="coerce"
        )

        if pd.notna(timestamp):
            transitions.append({
                "time": timestamp,
                "from": normalize_status(
                    item.get("from_status")
                ),
                "to": normalize_status(
                    item.get("to_status")
                )
            })

    transitions.sort(key=lambda x: x["time"])

    # If no status history exists, historical status is unknown.
    if not transitions:
        return None

    # Initial status before the first transition
    current_status = transitions[0]["from"]

    # Apply only transitions observed by the cutoff
    for transition in transitions:

        if transition["time"] > cutoff:
            break

        current_status = transition["to"]

    return current_status

In [ ]:
# Create lookup for Jira issue records
issue_lookup = {
    row["issue_key"]: row
    for _, row in jira_df.iterrows()
}

verification_records = []

for _, row in dependency_disruptions_df.iterrows():

    upstream_key = row["upstream_issue"]
    disruption_time = row["event_dt"]

    issue_record = issue_lookup.get(upstream_key)

    if issue_record is None:
        upstream_status = None
        verification = "UPSTREAM_UNAVAILABLE"

    else:
        upstream_status = status_at_time(
            issue_record,
            disruption_time
        )

        if upstream_status is None:
            verification = "STATUS_UNKNOWN"

        elif upstream_status in TERMINAL_STATUSES:
            verification = "UPSTREAM_TERMINAL"

        else:
            verification = "UPSTREAM_UNRESOLVED"

    verification_records.append({
        "event_uid": row["uid"],
        "instance_id": row["instance_id"],
        "upstream_issue": upstream_key,
        "downstream_issue": row["downstream_issue"],
        "disruption_time": disruption_time,
        "disruption_type": row["disruption_type"],
        "upstream_status": upstream_status,
        "verification_status": verification,
        "blocking_start_known": row["start_known"],
        "blocking_event_uid": row["start_event_uid"]
    })

verification_df = pd.DataFrame(verification_records)

print("Total candidate associations:", len(verification_df))

display(
    verification_df["verification_status"]
    .value_counts()
)

display(verification_df.head(15))

Total candidate associations: 160


,count
verification_status,
UPSTREAM_UNAVAILABLE,117
UPSTREAM_UNRESOLVED,31
UPSTREAM_TERMINAL,12


,event_uid,instance_id,upstream_issue,downstream_issue,disruption_time,disruption_type,upstream_status,verification_status,blocking_start_known,blocking_event_uid
0,T_c61b17b353e492806f08b590,JBEAP-I01,JBEAP-4646,JBEAP-3969,2016-05-31 12:02:09+00:00,REOPENED_WORK,None,UPSTREAM_UNAVAILABLE,True,T_097cba8729cf966a92a2e495
1,T_645c755ca868ae7196cd7ed3,JBEAP-I01,WFCORE-1356,JBEAP-3154,2016-02-23 09:34:12+00:00,REOPENED_WORK,None,UPSTREAM_UNAVAILABLE,True,T_9a1010efba88ef5c001a9cbf
2,T_e203021d0ec6ca3296aa71fe,JBEAP-I01,WFCORE-1356,JBEAP-3154,2016-02-25 06:26:39+00:00,REOPENED_WORK,None,UPSTREAM_UNAVAILABLE,True,T_9a1010efba88ef5c001a9cbf
3,T_32f78097263374023fe78be0,JBEAP-I01,JCSP-20,JBEAP-3134,2016-02-24 20:29:53+00:00,REOPENED_WORK,None,UPSTREAM_UNAVAILABLE,True,T_1b04ed55b4b4c9fb464787af
4,T_3b17d687d8f06fa67bc3c990,JBEAP-I01,JBEAP-2499,JBEAP-2946,2016-04-18 08:48:22+00:00,REOPENED_WORK,verified,UPSTREAM_TERMINAL,True,T_1194320055e1e46bb40898fb
5,T_3b17d687d8f06fa67bc3c990,JBEAP-I01,JBEAP-3577,JBEAP-2946,2016-04-18 08:48:22+00:00,REOPENED_WORK,verified,UPSTREAM_TERMINAL,True,T_7839781042cb8e7eff535516
6,T_3b17d687d8f06fa67bc3c990,JBEAP-I01,JBEAP-4223,JBEAP-2946,2016-04-18 08:48:22+00:00,REOPENED_WORK,None,UPSTREAM_UNAVAILABLE,True,T_3f5cc79c63db168ae2f049b7
7,T_e853fdcf28383600d8f8853c,JBEAP-I01,WFLY-6086,JBEAP-2739,2016-05-12 02:33:18+00:00,REOPENED_WORK,None,UPSTREAM_UNAVAILABLE,True,T_5da66366d29f7e7085344fec
8,T_1ab7b2746c3a346ffbd370ee,JBEAP-I01,JBEAP-2597,JBEAP-2698,2016-02-24 14:44:08+00:00,REOPENED_WORK,None,UPSTREAM_UNAVAILABLE,True,T_745b99e9e594c3c185a5a7cb
9,T_a5f7e1c33dc58c5a7132c3de,JBEAP-I01,JBEAP-1269,JBEAP-2574,2016-01-12 08:31:37+00:00,REOPENED_WORK,None,UPSTREAM_UNAVAILABLE,True,T_651a37c2c4309ef53a134981


In [ ]:
def summarize_verification(group):

    statuses = set(group["verification_status"])

    if "UPSTREAM_UNRESOLVED" in statuses:
        return "TEMPORALLY_SUPPORTED"

    if statuses == {"UPSTREAM_TERMINAL"}:
        return "NO_UNRESOLVED_UPSTREAM_FOUND"

    return "INSUFFICIENT_UPSTREAM_EVIDENCE"


outcome_summary_df = (
    verification_df.groupby("event_uid")
    .apply(
        summarize_verification,
        include_groups=False
    )
    .reset_index(name="outcome_status")
)

print("Unique downstream outcomes:",
      len(outcome_summary_df))

display(
    outcome_summary_df["outcome_status"]
    .value_counts()
)

Unique downstream outcomes: 91


,count
outcome_status,
INSUFFICIENT_UPSTREAM_EVIDENCE,58
TEMPORALLY_SUPPORTED,29
NO_UNRESOLVED_UPSTREAM_FOUND,4


In [ ]:
# Section 2.4.5 — Consolidate Dependency Risk candidates

import pandas as pd
import json

# Preserve the downstream event information
event_details = (
    dependency_disruptions_df[
        [
            "uid",
            "instance_id",
            "source_id",
            "event_dt",
            "disruption_type"
        ]
    ]
    .drop_duplicates(subset=["uid"])
    .rename(columns={
        "uid": "event_uid",
        "source_id": "downstream_issue",
        "event_dt": "outcome_time"
    })
)

# Group supporting upstream dependencies
supporting_links = (
    verification_df.groupby("event_uid")
    .agg(
        upstream_issue_ids=(
            "upstream_issue",
            lambda x: sorted(set(x))
        ),
        upstream_statuses=(
            "upstream_status",
            lambda x: sorted(set(
                str(v) for v in x if pd.notna(v)
            ))
        ),
        blocking_evidence_ids=(
            "blocking_event_uid",
            lambda x: sorted(set(
                str(v) for v in x if pd.notna(v)
            ))
        )
    )
    .reset_index()
)

# Merge event details, supporting links and verification
dependency_candidates = (
    event_details
    .merge(supporting_links, on="event_uid", how="left")
    .merge(outcome_summary_df, on="event_uid", how="left")
)

dependency_candidates["risk_type"] = "DEPENDENCY"

dependency_candidates["label_basis"] = (
    "TEMPORAL_ASSOCIATION_PROXY"
)

dependency_candidates["evidence_review_status"] = (
    "PENDING"
)

print("Total unique candidates:", len(dependency_candidates))

display(
    dependency_candidates["outcome_status"]
    .value_counts()
)

display(dependency_candidates.head(10))

Total unique candidates: 91


,count
outcome_status,
INSUFFICIENT_UPSTREAM_EVIDENCE,58
TEMPORALLY_SUPPORTED,29
NO_UNRESOLVED_UPSTREAM_FOUND,4


,event_uid,instance_id,downstream_issue,outcome_time,disruption_type,upstream_issue_ids,upstream_statuses,blocking_evidence_ids,outcome_status,risk_type,label_basis,evidence_review_status
0,T_c61b17b353e492806f08b590,JBEAP-I01,JBEAP-3969,2016-05-31 12:02:09+00:00,REOPENED_WORK,[JBEAP-4646],[],[T_097cba8729cf966a92a2e495],INSUFFICIENT_UPSTREAM_EVIDENCE,DEPENDENCY,TEMPORAL_ASSOCIATION_PROXY,PENDING
1,T_645c755ca868ae7196cd7ed3,JBEAP-I01,JBEAP-3154,2016-02-23 09:34:12+00:00,REOPENED_WORK,[WFCORE-1356],[],[T_9a1010efba88ef5c001a9cbf],INSUFFICIENT_UPSTREAM_EVIDENCE,DEPENDENCY,TEMPORAL_ASSOCIATION_PROXY,PENDING
2,T_e203021d0ec6ca3296aa71fe,JBEAP-I01,JBEAP-3154,2016-02-25 06:26:39+00:00,REOPENED_WORK,[WFCORE-1356],[],[T_9a1010efba88ef5c001a9cbf],INSUFFICIENT_UPSTREAM_EVIDENCE,DEPENDENCY,TEMPORAL_ASSOCIATION_PROXY,PENDING
3,T_32f78097263374023fe78be0,JBEAP-I01,JBEAP-3134,2016-02-24 20:29:53+00:00,REOPENED_WORK,[JCSP-20],[],[T_1b04ed55b4b4c9fb464787af],INSUFFICIENT_UPSTREAM_EVIDENCE,DEPENDENCY,TEMPORAL_ASSOCIATION_PROXY,PENDING
4,T_3b17d687d8f06fa67bc3c990,JBEAP-I01,JBEAP-2946,2016-04-18 08:48:22+00:00,REOPENED_WORK,"[JBEAP-2499, JBEAP-3577, JBEAP-4223]",[verified],"[T_1194320055e1e46bb40898fb, T_3f5cc79c63db168...",INSUFFICIENT_UPSTREAM_EVIDENCE,DEPENDENCY,TEMPORAL_ASSOCIATION_PROXY,PENDING
5,T_e853fdcf28383600d8f8853c,JBEAP-I01,JBEAP-2739,2016-05-12 02:33:18+00:00,REOPENED_WORK,[WFLY-6086],[],[T_5da66366d29f7e7085344fec],INSUFFICIENT_UPSTREAM_EVIDENCE,DEPENDENCY,TEMPORAL_ASSOCIATION_PROXY,PENDING
6,T_1ab7b2746c3a346ffbd370ee,JBEAP-I01,JBEAP-2698,2016-02-24 14:44:08+00:00,REOPENED_WORK,[JBEAP-2597],[],[T_745b99e9e594c3c185a5a7cb],INSUFFICIENT_UPSTREAM_EVIDENCE,DEPENDENCY,TEMPORAL_ASSOCIATION_PROXY,PENDING
7,T_a5f7e1c33dc58c5a7132c3de,JBEAP-I01,JBEAP-2574,2016-01-12 08:31:37+00:00,REOPENED_WORK,"[JBEAP-1269, JBEAP-1273, JBEAP-1276, JBEAP-144...",[resolved],"[T_0991e634f05df74bfe698441, T_0a7bdb5fb67988e...",INSUFFICIENT_UPSTREAM_EVIDENCE,DEPENDENCY,TEMPORAL_ASSOCIATION_PROXY,PENDING
8,T_7ad42d7799a9b2bdcd0d137f,JBEAP-I01,JBEAP-2544,2016-01-12 07:26:52+00:00,REOPENED_WORK,[JBEAP-2499],[new],[T_8031f54b085d301563e59fe8],TEMPORALLY_SUPPORTED,DEPENDENCY,TEMPORAL_ASSOCIATION_PROXY,PENDING
9,T_3fa7a9ef832fd67b02be4d60,JBEAP-I01,JBEAP-2544,2016-01-19 02:20:43+00:00,REOPENED_WORK,[JBEAP-2499],[new],[T_8031f54b085d301563e59fe8],TEMPORALLY_SUPPORTED,DEPENDENCY,TEMPORAL_ASSOCIATION_PROXY,PENDING


In [ ]:
supported_candidates = dependency_candidates[
    dependency_candidates["outcome_status"]
    == "TEMPORALLY_SUPPORTED"
].copy()

print(
    "Temporally supported outcomes:",
    len(supported_candidates)
)

print(
    "Projects with supported outcomes:",
    supported_candidates["instance_id"].nunique()
)

display(
    supported_candidates.groupby("instance_id")
    .size()
    .reset_index(name="supported_outcomes")
    .sort_values(
        "supported_outcomes",
        ascending=False
    )
)

Temporally supported outcomes: 29
Projects with supported outcomes: 5


,instance_id,supported_outcomes
0,DAOS-I01,13
2,JBEAP-I01,11
4,SPARK-I01,3
1,JBDS-I01,1
3,JBIDE-I01,1


In [ ]:
output_path = (
    GC1_ROOT /"Temporal_snapshots"/ "dependency_outcome_candidates.jsonl"
)

with open(output_path, "w", encoding="utf-8") as f:
    for record in dependency_candidates.to_dict(
        orient="records"
    ):
        record["outcome_time"] = (
            record["outcome_time"].isoformat()
            if pd.notna(record["outcome_time"])
            else None
        )

        f.write(
            json.dumps(record, default=str) + "\n"
        )

print("Saved:", output_path)

Saved: /content/drive/MyDrive/GC1/Temporal_snapshots/dependency_outcome_candidates.jsonl


In [ ]:
import pandas as pd
import json

# Retain associations where upstream was unresolved
supported_links = verification_df[
    verification_df["verification_status"]
    == "UPSTREAM_UNRESOLVED"
].copy()

# Retain the corresponding unique outcomes
supported_outcomes = dependency_candidates[
    dependency_candidates["outcome_status"]
    == "TEMPORALLY_SUPPORTED"
].copy()

print("Supported dependency associations:", len(supported_links))
print("Unique supported outcomes:", len(supported_outcomes))

display(
    supported_links[
        [
            "event_uid",
            "instance_id",
            "upstream_issue",
            "downstream_issue",
            "upstream_status",
            "disruption_type"
        ]
    ].head(10)
)

Supported dependency associations: 31
Unique supported outcomes: 29


,event_uid,instance_id,upstream_issue,downstream_issue,upstream_status,disruption_type
24,T_7ad42d7799a9b2bdcd0d137f,JBEAP-I01,JBEAP-2499,JBEAP-2544,new,REOPENED_WORK
25,T_3fa7a9ef832fd67b02be4d60,JBEAP-I01,JBEAP-2499,JBEAP-2544,new,REOPENED_WORK
26,T_26bce03fe181b51967caede5,JBEAP-I01,JBEAP-2499,JBEAP-2544,new,REOPENED_WORK
27,T_a2985d18e7d2ac3922214930,JBEAP-I01,JBEAP-2499,JBEAP-2544,new,REOPENED_WORK
28,T_747b915ce87c8c5f17c850e4,JBEAP-I01,JBEAP-2499,JBEAP-2100,new,REOPENED_WORK
29,T_8ba13d520d628c4bf8d2d996,JBEAP-I01,JBEAP-2499,JBEAP-2100,new,REOPENED_WORK
30,T_020d268afc763c4d946f55ff,JBEAP-I01,JBEAP-2499,JBEAP-1707,new,REOPENED_WORK
31,T_179c66df36e43af5e34d2a8a,JBEAP-I01,JBEAP-2499,JBEAP-1707,reopened,REOPENED_WORK
35,T_c78c514d885da65279246103,JBEAP-I01,JBEAP-2443,JBEAP-1515,ready for qa,REOPENED_WORK
36,T_269866c37032727f1a2aa256,JBEAP-I01,JBEAP-2443,JBEAP-1515,ready for qa,REOPENED_WORK


In [ ]:
import re

issue_lookup = {
    row["issue_key"]: row
    for _, row in jira_df.iterrows()
}

def extract_text(value):
    if value is None:
        return ""

    if isinstance(value, str):
        return value

    if isinstance(value, list):
        return "\n".join(
            extract_text(item) for item in value
        )

    if isinstance(value, dict):
        # Jira comments may use different structures
        if "body" in value:
            return extract_text(value["body"])

        return json.dumps(value, default=str)

    return str(value)


def get_issue_evidence(issue_key):
    record = issue_lookup.get(issue_key)

    if record is None:
        return {
            "description": "",
            "comments": ""
        }

    return {
        "description": extract_text(
            record.get("description")
        ),
        "comments": extract_text(
            record.get("comments")
        )
    }


evidence_records = []

for _, row in supported_links.iterrows():

    upstream = row["upstream_issue"]
    downstream = row["downstream_issue"]

    upstream_evidence = get_issue_evidence(upstream)
    downstream_evidence = get_issue_evidence(downstream)

    evidence_records.append({
        "event_uid": row["event_uid"],
        "instance_id": row["instance_id"],
        "upstream_issue": upstream,
        "downstream_issue": downstream,
        "disruption_time": row["disruption_time"],
        "disruption_type": row["disruption_type"],
        "upstream_status": row["upstream_status"],
        "upstream_description":
            upstream_evidence["description"],
        "upstream_comments":
            upstream_evidence["comments"],
        "downstream_description":
            downstream_evidence["description"],
        "downstream_comments":
            downstream_evidence["comments"]
    })

evidence_review_df = pd.DataFrame(evidence_records)

print("Evidence review records:", len(evidence_review_df))

display(
    evidence_review_df[
        [
            "upstream_issue",
            "downstream_issue",
            "disruption_type",
            "upstream_status"
        ]
    ].head(10)
)

Evidence review records: 31


,upstream_issue,downstream_issue,disruption_type,upstream_status
0,JBEAP-2499,JBEAP-2544,REOPENED_WORK,new
1,JBEAP-2499,JBEAP-2544,REOPENED_WORK,new
2,JBEAP-2499,JBEAP-2544,REOPENED_WORK,new
3,JBEAP-2499,JBEAP-2544,REOPENED_WORK,new
4,JBEAP-2499,JBEAP-2100,REOPENED_WORK,new
5,JBEAP-2499,JBEAP-2100,REOPENED_WORK,new
6,JBEAP-2499,JBEAP-1707,REOPENED_WORK,new
7,JBEAP-2499,JBEAP-1707,REOPENED_WORK,reopened
8,JBEAP-2443,JBEAP-1515,REOPENED_WORK,ready for qa
9,JBEAP-2443,JBEAP-1515,REOPENED_WORK,ready for qa


In [ ]:
def find_issue_mentions(text, issue_key):
    if not isinstance(text, str):
        return []

    pattern = re.compile(
        rf"(?<![A-Z0-9-]){re.escape(issue_key)}(?![A-Z0-9-])",
        re.IGNORECASE
    )

    matches = []

    for match in pattern.finditer(text):
        start = max(0, match.start() - 180)
        end = min(len(text), match.end() + 180)

        matches.append(
            text[start:end].replace("\n", " ")
        )

    return matches[:10]


evidence_review_df["direct_mentions"] = (
    evidence_review_df.apply(
        lambda row: find_issue_mentions(
            row["downstream_description"]
            + "\n"
            + row["downstream_comments"],
            row["upstream_issue"]
        ),
        axis=1
    )
)

evidence_review_df["has_direct_mention"] = (
    evidence_review_df["direct_mentions"].apply(
        lambda x: len(x) > 0
    )
)

print("Associations with direct upstream references:",
      evidence_review_df["has_direct_mention"].sum())

print("Associations without direct references:",
      (~evidence_review_df["has_direct_mention"]).sum())

display(
    evidence_review_df[
        [
            "upstream_issue",
            "downstream_issue",
            "has_direct_mention",
            "direct_mentions"
        ]
    ].head(15)
)

Associations with direct upstream references: 10
Associations without direct references: 21


,upstream_issue,downstream_issue,has_direct_mention,direct_mentions
0,JBEAP-2499,JBEAP-2544,False,[]
1,JBEAP-2499,JBEAP-2544,False,[]
2,JBEAP-2499,JBEAP-2544,False,[]
3,JBEAP-2499,JBEAP-2544,False,[]
4,JBEAP-2499,JBEAP-2100,False,[]
5,JBEAP-2499,JBEAP-2100,False,[]
6,JBEAP-2499,JBEAP-1707,False,[]
7,JBEAP-2499,JBEAP-1707,False,[]
8,JBEAP-2443,JBEAP-1515,False,[]
9,JBEAP-2443,JBEAP-1515,False,[]


In [ ]:
review_table = evidence_review_df[
    [
        "event_uid",
        "instance_id",
        "upstream_issue",
        "downstream_issue",
        "disruption_time",
        "disruption_type",
        "upstream_status",
        "has_direct_mention",
        "direct_mentions"
    ]
].copy()

review_table["review_decision"] = "PENDING"
review_table["review_notes"] = ""
review_table["evidence_source_ids"] = ""

# Preserve all supporting associations.
# One downstream event may have multiple upstream issues.

output_path = GC1_ROOT / "Temporal_snapshots"/"dependency_evidence_review.csv"

review_table.to_csv(
    output_path,
    index=False
)

print("Saved evidence review table:", output_path)

Saved evidence review table: /content/drive/MyDrive/GC1/Temporal_snapshots/dependency_evidence_review.csv


In [ ]:
import json

# Select a supported downstream issue with comments
sample_issue = None

for _, row in supported_links.iterrows():

    issue_key = row["downstream_issue"]
    record = issue_lookup.get(issue_key)

    if record is not None and record.get("comments"):
        sample_issue = issue_key
        break

if sample_issue is None:
    print("No downstream issue with comments found.")

else:
    record = issue_lookup[sample_issue]

    print("Issue:", sample_issue)
    print("Comments type:", type(record["comments"]))

    print("\nFirst two comments:")
    print(
        json.dumps(
            record["comments"][:2],
            indent=2,
            default=str
        )[:5000]
    )

Issue: JBEAP-2544
Comments type: <class 'list'>

First two comments:
[
  {
    "comment_id": "13162032",
    "author": "<<|author_displayName|a5cbea5c-44f7-45eb-9fe9-f58be11616c1|>>",
    "body": "Setting as verified with EAP 7.0.0.ER5. Thanks guys for the fix!",
    "created_at": "2016-02-12T02:52:14.000+0000",
    "updated_at": "2016-02-12T02:52:14.000+0000"
  },
  {
    "comment_id": "13264949",
    "author": "<<|author_displayName|4cba0b88-c353-4847-8bea-9c1f5ddb9f88|>>",
    "body": "Verified and delivered in EAP 7.0.0.GA, closing.",
    "created_at": "2016-07-13T12:41:18.000+0000",
    "updated_at": "2016-07-13T12:41:18.000+0000"
  }
]


In [ ]:
import pandas as pd

def get_timestamped_comments(issue_key):

    record = issue_lookup.get(issue_key)

    if record is None:
        return []

    comments = record.get("comments") or []
    results = []

    for comment in comments:

        if not isinstance(comment, dict):
            continue

        created = pd.to_datetime(
            comment.get("created_at"),
            utc=True,
            errors="coerce"
        )

        if pd.isna(created):
            continue

        results.append({
            "issue_key": issue_key,
            "comment_id": comment.get("comment_id"),
            "comment_time": created,
            "comment_updated_at": pd.to_datetime(
                comment.get("updated_at"),
                utc=True,
                errors="coerce"
            ),
            "body": comment.get("body") or ""
        })

    return results

In [ ]:
evidence_window_records = []

for _, row in supported_links.iterrows():

    upstream = row["upstream_issue"]
    downstream = row["downstream_issue"]

    disruption_time = pd.to_datetime(
        row["disruption_time"],
        utc=True
    )

    window_start = disruption_time - pd.Timedelta(days=14)
    window_end = disruption_time + pd.Timedelta(days=7)

    for role, issue_key in [
        ("UPSTREAM", upstream),
        ("DOWNSTREAM", downstream)
    ]:

        comments = get_timestamped_comments(issue_key)

        for comment in comments:

            comment_time = comment["comment_time"]

            if not (
                window_start <= comment_time <= window_end
            ):
                continue

            # Exclude comments edited after the review window:
            # the original wording may no longer be recoverable.
            updated = comment["comment_updated_at"]

            if pd.notna(updated) and updated > window_end:
                continue

            evidence_window_records.append({
                "event_uid": row["event_uid"],
                "instance_id": row["instance_id"],
                "upstream_issue": upstream,
                "downstream_issue": downstream,
                "disruption_time": disruption_time,
                "comment_role": role,
                **comment
            })

evidence_window_df = pd.DataFrame(
    evidence_window_records,
    columns=[
        "event_uid", "instance_id",
        "upstream_issue", "downstream_issue",
        "disruption_time", "comment_role",
        "issue_key", "comment_id",
        "comment_time", "comment_updated_at", "body"
    ]
)

print("Comments in review windows:", len(evidence_window_df))

print(
    "Unique disruptions with comments:",
    evidence_window_df["event_uid"].nunique()
)

display(evidence_window_df.head(10))

Comments in review windows: 224
Unique disruptions with comments: 24


,event_uid,instance_id,upstream_issue,downstream_issue,disruption_time,comment_role,issue_key,comment_id,comment_time,comment_updated_at,body
0,T_3fa7a9ef832fd67b02be4d60,JBEAP-I01,JBEAP-2499,JBEAP-2544,2016-01-19 02:20:43+00:00,UPSTREAM,JBEAP-2499,13152965,2016-01-25 06:31:07+00:00,2016-01-25 06:31:07+00:00,now available in nexus and at https://github.c...
1,T_26bce03fe181b51967caede5,JBEAP-I01,JBEAP-2499,JBEAP-2544,2016-01-19 02:23:10+00:00,UPSTREAM,JBEAP-2499,13152965,2016-01-25 06:31:07+00:00,2016-01-25 06:31:07+00:00,now available in nexus and at https://github.c...
2,T_a2985d18e7d2ac3922214930,JBEAP-I01,JBEAP-2499,JBEAP-2544,2016-01-19 02:23:30+00:00,UPSTREAM,JBEAP-2499,13152965,2016-01-25 06:31:07+00:00,2016-01-25 06:31:07+00:00,now available in nexus and at https://github.c...
3,T_747b915ce87c8c5f17c850e4,JBEAP-I01,JBEAP-2499,JBEAP-2100,2016-01-19 02:33:50+00:00,UPSTREAM,JBEAP-2499,13152965,2016-01-25 06:31:07+00:00,2016-01-25 06:31:07+00:00,now available in nexus and at https://github.c...
4,T_747b915ce87c8c5f17c850e4,JBEAP-I01,JBEAP-2499,JBEAP-2100,2016-01-19 02:33:50+00:00,DOWNSTREAM,JBEAP-2100,13146084,2016-01-08 03:15:00+00:00,2016-01-08 03:15:00+00:00,"Hi, any update on this? This issue was reporte..."
5,T_747b915ce87c8c5f17c850e4,JBEAP-I01,JBEAP-2499,JBEAP-2100,2016-01-19 02:33:50+00:00,DOWNSTREAM,JBEAP-2100,13146166,2016-01-08 05:36:54+00:00,2016-01-08 05:36:54+00:00,not as yet
6,T_747b915ce87c8c5f17c850e4,JBEAP-I01,JBEAP-2499,JBEAP-2100,2016-01-19 02:33:50+00:00,DOWNSTREAM,JBEAP-2100,13147382,2016-01-12 05:53:50+00:00,2016-01-12 05:53:50+00:00,I'll increase priority of this jira. Customer ...
7,T_747b915ce87c8c5f17c850e4,JBEAP-I01,JBEAP-2499,JBEAP-2100,2016-01-19 02:33:50+00:00,DOWNSTREAM,JBEAP-2100,13147519,2016-01-12 08:35:10+00:00,2016-01-12 08:35:10+00:00,This could be the same issue as [~clebert] fix...
8,T_747b915ce87c8c5f17c850e4,JBEAP-I01,JBEAP-2499,JBEAP-2100,2016-01-19 02:33:50+00:00,DOWNSTREAM,JBEAP-2100,13147588,2016-01-12 09:31:54+00:00,2016-01-12 09:32:15+00:00,Yes I think so
9,T_747b915ce87c8c5f17c850e4,JBEAP-I01,JBEAP-2499,JBEAP-2100,2016-01-19 02:33:50+00:00,DOWNSTREAM,JBEAP-2100,13147640,2016-01-12 10:34:54+00:00,2016-01-12 10:34:54+00:00,I've tried test ClusterTestCase#clusterTestWit...


In [ ]:
import re

dependency_terms = [
    "blocked",
    "blocking",
    "dependency",
    "depends on",
    "waiting for",
    "cannot proceed",
    "held up",
    "reopened",
    "regression",
    "failed",
    "delayed"
]

def classify_comment_candidate(row):

    text = str(row["body"]).lower()

    other_issue = (
        row["upstream_issue"]
        if row["comment_role"] == "DOWNSTREAM"
        else row["downstream_issue"]
    )

    has_issue_reference = bool(
        re.search(
            rf"(?<![A-Z0-9-]){re.escape(other_issue)}"
            rf"(?![A-Z0-9-])",
            str(row["body"]),
            re.IGNORECASE
        )
    )

    matched_terms = [
        term for term in dependency_terms
        if term in text
    ]

    return pd.Series({
        "has_issue_reference": has_issue_reference,
        "matched_terms": matched_terms,
        "has_dependency_language": len(matched_terms) > 0
    })


if not evidence_window_df.empty:

    classified = evidence_window_df.apply(
        classify_comment_candidate,
        axis=1
    )

    evidence_window_df = pd.concat(
        [
            evidence_window_df.reset_index(drop=True),
            classified.reset_index(drop=True)
        ],
        axis=1
    )

    evidence_window_df["review_priority"] = (
        evidence_window_df["has_issue_reference"].astype(int)
        +
        evidence_window_df["has_dependency_language"].astype(int)
    )

    print("Comments referencing the linked issue:",
          evidence_window_df["has_issue_reference"].sum())

    print("Comments containing dependency language:",
          evidence_window_df["has_dependency_language"].sum())

    print("Comments satisfying both:",
          (evidence_window_df["review_priority"] == 2).sum())

    display(
        evidence_window_df[
            [
                "event_uid",
                "issue_key",
                "comment_time",
                "comment_role",
                "has_issue_reference",
                "matched_terms",
                "body"
            ]
        ]
        .sort_values(
            "has_issue_reference",
            ascending=False
        )
        .head(20)
    )

else:
    print("No comments found in the review windows.")

Comments referencing the linked issue: 8
Comments containing dependency language: 38
Comments satisfying both: 6


,event_uid,issue_key,comment_time,comment_role,has_issue_reference,matched_terms,body
153,T_d6a83a7ecd4f74fb2a63d561,DAOS-6743,2021-02-25 15:12:33+00:00,DOWNSTREAM,True,[blocked],Soak is currently blocked by DAOS-6808. The ...
205,T_09694fd8a0ab7038e8123486,DAOS-6600,2021-02-22 14:05:49+00:00,DOWNSTREAM,True,[],exclude/reintegrate was not run this weekend d...
182,T_73b1868c7e9658dca1264c21,DAOS-6600,2021-02-22 14:05:49+00:00,DOWNSTREAM,True,[],exclude/reintegrate was not run this weekend d...
212,T_659d8724f9ff8b7644d9066a,DAOS-4837,2021-01-15 01:00:18+00:00,DOWNSTREAM,True,[failed],Start testing on daos container redundant fact...
218,T_5c8facaa5312b00dea3049fe,DAOS-4836,2021-01-15 01:01:24+00:00,DOWNSTREAM,True,[failed],tart testing on daos container redundant facto...
123,T_e2740c38cce24c0127ca385a,DAOS-6743,2021-02-25 15:12:33+00:00,DOWNSTREAM,True,[blocked],Soak is currently blocked by DAOS-6808. The ...
73,T_7bd6cffd2d2d7528b58ca9a2,DAOS-8242,2021-08-24 13:02:49.461000+00:00,DOWNSTREAM,True,[failed],*Soak harasser – offline exclude/reintegrate w...
93,T_b442f54be47cce2d5c9557a1,DAOS-7449,2021-08-06 11:17:39+00:00,DOWNSTREAM,True,[blocked],[~accountid:60357e914623c60069b22199] [~accoun...
149,T_d6a83a7ecd4f74fb2a63d561,DAOS-6743,2021-02-24 03:52:01+00:00,DOWNSTREAM,False,[blocked],I think we might be re-using the same ticket f...
150,T_d6a83a7ecd4f74fb2a63d561,DAOS-6743,2021-02-24 03:59:10+00:00,DOWNSTREAM,False,[],So theoretically this one shouldn't be marked ...


In [ ]:
if not evidence_window_df.empty:

    evidence_summary = (
        evidence_window_df.groupby("event_uid")
        .agg(
            comment_count=("comment_id", "count"),
            direct_reference_count=(
                "has_issue_reference", "sum"
            ),
            dependency_language_count=(
                "has_dependency_language", "sum"
            ),
            high_priority_count=(
                "review_priority",
                lambda x: (x == 2).sum()
            )
        )
        .reset_index()
    )

    all_supported_events = supported_outcomes[
        ["event_uid", "instance_id"]
    ].drop_duplicates()

    evidence_summary = all_supported_events.merge(
        evidence_summary,
        on="event_uid",
        how="left"
    )

    count_cols = [
        "comment_count",
        "direct_reference_count",
        "dependency_language_count",
        "high_priority_count"
    ]

    evidence_summary[count_cols] = (
        evidence_summary[count_cols].fillna(0).astype(int)
    )

    print("Total supported disruptions:",
          len(evidence_summary))

    print(
        "Disruptions with any nearby comments:",
        (evidence_summary["comment_count"] > 0).sum()
    )

    print(
        "Disruptions with high-priority comments:",
        (evidence_summary["high_priority_count"] > 0).sum()
    )

    display(evidence_summary)

else:
    print("No timestamped comment evidence available.")

Total supported disruptions: 29
Disruptions with any nearby comments: 24
Disruptions with high-priority comments: 6


,event_uid,instance_id,comment_count,direct_reference_count,dependency_language_count,high_priority_count
0,T_7ad42d7799a9b2bdcd0d137f,JBEAP-I01,0,0,0,0
1,T_3fa7a9ef832fd67b02be4d60,JBEAP-I01,1,0,0,0
2,T_26bce03fe181b51967caede5,JBEAP-I01,1,0,0,0
3,T_a2985d18e7d2ac3922214930,JBEAP-I01,1,0,0,0
4,T_747b915ce87c8c5f17c850e4,JBEAP-I01,10,0,0,0
5,T_8ba13d520d628c4bf8d2d996,JBEAP-I01,10,0,0,0
6,T_020d268afc763c4d946f55ff,JBEAP-I01,3,0,1,0
7,T_179c66df36e43af5e34d2a8a,JBEAP-I01,3,0,1,0
8,T_c78c514d885da65279246103,JBEAP-I01,4,0,1,0
9,T_269866c37032727f1a2aa256,JBEAP-I01,4,0,1,0


In [ ]:
high_priority = evidence_window_df[
    evidence_window_df["review_priority"] == 2
].copy()

high_priority = high_priority.sort_values(
    ["event_uid", "comment_time"]
)

for _, row in high_priority.iterrows():

    print("\n" + "=" * 75)

    print("Disruption Event:", row["event_uid"])
    print("Project:", row["instance_id"])

    print("Upstream:", row["upstream_issue"])
    print("Downstream:", row["downstream_issue"])

    print("Disruption time:", row["disruption_time"])
    print("Comment time:", row["comment_time"])
    print("Comment ID:", row["comment_id"])
    print("Comment source:", row["comment_role"])

    print("\nMatched terms:", row["matched_terms"])

    print("\nCOMMENT:")
    print(row["body"])

    print("=" * 75)


Disruption Event: T_5c8facaa5312b00dea3049fe
Project: DAOS-I01
Upstream: DAOS-6510
Downstream: DAOS-4836
Disruption time: 2021-01-20 19:26:00+00:00
Comment time: 2021-01-15 01:01:24+00:00
Comment ID: 34146
Comment source: DOWNSTREAM

Matched terms: ['failed']

COMMENT:
tart testing on daos container redundant factor.

Test failed due to DAOS-6510: Unable to query container after rebuild.

Disruption Event: T_659d8724f9ff8b7644d9066a
Project: DAOS-I01
Upstream: DAOS-6510
Downstream: DAOS-4837
Disruption time: 2021-01-20 19:26:22+00:00
Comment time: 2021-01-15 01:00:18+00:00
Comment ID: 34145
Comment source: DOWNSTREAM

Matched terms: ['failed']

COMMENT:
Start testing on daos container redundant factor.

Test failed due to DAOS-6510: Unable to query container after rebuild.

Disruption Event: T_7bd6cffd2d2d7528b58ca9a2
Project: DAOS-I01
Upstream: DAOS-8231
Downstream: DAOS-8242
Disruption time: 2021-08-24 13:03:48+00:00
Comment time: 2021-08-24 13:02:49.461000+00:00
Comment ID: 85058
C

In [ ]:
high_priority["days_from_disruption"] = (
    high_priority["comment_time"]
    - high_priority["disruption_time"]
).dt.total_seconds() / 86400

high_priority["evidence_timing"] = (
    high_priority["days_from_disruption"].apply(
        lambda days:
            "BEFORE_OR_AT_OUTCOME"
            if days <= 0
            else "RETROSPECTIVE"
    )
)

display(
    high_priority[
        [
            "event_uid",
            "upstream_issue",
            "downstream_issue",
            "comment_id",
            "days_from_disruption",
            "evidence_timing",
            "body"
        ]
    ]
)

,event_uid,upstream_issue,downstream_issue,comment_id,days_from_disruption,evidence_timing,body
218,T_5c8facaa5312b00dea3049fe,DAOS-6510,DAOS-4836,34146,-5.767083,BEFORE_OR_AT_OUTCOME,tart testing on daos container redundant facto...
212,T_659d8724f9ff8b7644d9066a,DAOS-6510,DAOS-4837,34145,-5.768102,BEFORE_OR_AT_OUTCOME,Start testing on daos container redundant fact...
73,T_7bd6cffd2d2d7528b58ca9a2,DAOS-8231,DAOS-8242,85058,-0.000678,BEFORE_OR_AT_OUTCOME,*Soak harasser – offline exclude/reintegrate w...
93,T_b442f54be47cce2d5c9557a1,DAOS-8242,DAOS-7449,43017,-11.469803,BEFORE_OR_AT_OUTCOME,[~accountid:60357e914623c60069b22199] [~accoun...
153,T_d6a83a7ecd4f74fb2a63d561,DAOS-6808,DAOS-6743,36654,-0.005370,BEFORE_OR_AT_OUTCOME,Soak is currently blocked by DAOS-6808. The ...
123,T_e2740c38cce24c0127ca385a,DAOS-6808,DAOS-6743,36654,-0.005081,BEFORE_OR_AT_OUTCOME,Soak is currently blocked by DAOS-6808. The ...


In [ ]:
review_decisions = {
    "T_5c8facaa5312b00dea3049fe": "STRONG_CANDIDATE",
    "T_659d8724f9ff8b7644d9066a": "STRONG_CANDIDATE",
    "T_7bd6cffd2d2d7528b58ca9a2": "STRONG_CANDIDATE",
    "T_b442f54be47cce2d5c9557a1": "PROXY",
    "T_d6a83a7ecd4f74fb2a63d561": "CONFIRMED_BLOCKING_IMPACT",
    "T_e2740c38cce24c0127ca385a": "CONFIRMED_BLOCKING_IMPACT"
}

review_notes = {
    "T_5c8facaa5312b00dea3049fe":
        "Test failure explicitly attributed to DAOS-6510.",
    "T_659d8724f9ff8b7644d9066a":
        "Test failure explicitly attributed to DAOS-6510.",
    "T_7bd6cffd2d2d7528b58ca9a2":
        "Test failure explicitly attributed to DAOS-8231.",
    "T_b442f54be47cce2d5c9557a1":
        "Blocking relationship confirmed; specific setback causation unclear.",
    "T_d6a83a7ecd4f74fb2a63d561":
        "Testing explicitly blocked by DAOS-6808.",
    "T_e2740c38cce24c0127ca385a":
        "Same blocking evidence as another event; review episode duplication."
}

dependency_review_df = supported_outcomes.copy()

dependency_review_df["evidence_class"] = (
    dependency_review_df["event_uid"]
    .map(review_decisions)
    .fillna("UNREVIEWED")
)

dependency_review_df["review_notes"] = (
    dependency_review_df["event_uid"]
    .map(review_notes)
    .fillna("")
)

print("Evidence classification distribution:")
display(
    dependency_review_df["evidence_class"].value_counts()
)

Evidence classification distribution:


,count
evidence_class,
UNREVIEWED,23
STRONG_CANDIDATE,3
CONFIRMED_BLOCKING_IMPACT,2
PROXY,1


In [ ]:
# Select the six high-priority comments
reviewed_comments = high_priority[
    [
        "event_uid",
        "comment_id",
        "comment_time",
        "issue_key",
        "body"
    ]
].copy()

reviewed_comments = reviewed_comments.rename(
    columns={
        "comment_time": "evidence_time",
        "issue_key": "evidence_issue",
        "body": "evidence_text"
    }
)

dependency_review_df = dependency_review_df.merge(
    reviewed_comments,
    on="event_uid",
    how="left"
)

print("Outcomes with reviewed comment evidence:")
print(
    dependency_review_df["comment_id"]
    .notna()
    .sum()
)

display(
    dependency_review_df[
        [
            "event_uid",
            "instance_id",
            "outcome_time",
            "evidence_class",
            "comment_id",
            "review_notes"
        ]
    ].head(15)
)

Outcomes with reviewed comment evidence:
6


,event_uid,instance_id,outcome_time,evidence_class,comment_id,review_notes
0,T_7ad42d7799a9b2bdcd0d137f,JBEAP-I01,2016-01-12 07:26:52+00:00,UNREVIEWED,NaN,
1,T_3fa7a9ef832fd67b02be4d60,JBEAP-I01,2016-01-19 02:20:43+00:00,UNREVIEWED,NaN,
2,T_26bce03fe181b51967caede5,JBEAP-I01,2016-01-19 02:23:10+00:00,UNREVIEWED,NaN,
3,T_a2985d18e7d2ac3922214930,JBEAP-I01,2016-01-19 02:23:30+00:00,UNREVIEWED,NaN,
4,T_747b915ce87c8c5f17c850e4,JBEAP-I01,2016-01-19 02:33:50+00:00,UNREVIEWED,NaN,
5,T_8ba13d520d628c4bf8d2d996,JBEAP-I01,2016-01-19 02:34:25+00:00,UNREVIEWED,NaN,
6,T_020d268afc763c4d946f55ff,JBEAP-I01,2016-01-12 08:28:48+00:00,UNREVIEWED,NaN,
7,T_179c66df36e43af5e34d2a8a,JBEAP-I01,2016-01-12 09:41:48+00:00,UNREVIEWED,NaN,
8,T_c78c514d885da65279246103,JBEAP-I01,2016-01-19 02:18:48+00:00,UNREVIEWED,NaN,
9,T_269866c37032727f1a2aa256,JBEAP-I01,2016-01-19 02:19:29+00:00,UNREVIEWED,NaN,


In [ ]:
episode_df = dependency_review_df.copy()

episode_df["outcome_time"] = pd.to_datetime(
    episode_df["outcome_time"],
    utc=True
)

episode_df = episode_df.sort_values(
    [
        "instance_id",
        "downstream_issue",
        "outcome_time"
    ]
)

episode_df["previous_event_time"] = (
    episode_df.groupby(
        ["instance_id", "downstream_issue"]
    )["outcome_time"].shift()
)

episode_df["days_since_previous"] = (
    episode_df["outcome_time"]
    - episode_df["previous_event_time"]
).dt.total_seconds() / 86400

# Proposed review grouping: events within 24 hours
episode_df["new_episode"] = (
    episode_df["previous_event_time"].isna()
    |
    (episode_df["days_since_previous"] > 1)
)

episode_df["episode_number"] = (
    episode_df.groupby(
        ["instance_id", "downstream_issue"]
    )["new_episode"].cumsum()
)

episode_df["episode_id"] = (
    episode_df["instance_id"].astype(str)
    + "::"
    + episode_df["downstream_issue"].astype(str)
    + "::E"
    + episode_df["episode_number"].astype(str)
)

print("Unique candidate episodes:")
print(episode_df["episode_id"].nunique())

display(
    episode_df[
        [
            "event_uid",
            "downstream_issue",
            "outcome_time",
            "episode_id",
            "evidence_class"
        ]
    ].head(20)
)

Unique candidate episodes:
22


,event_uid,downstream_issue,outcome_time,episode_id,evidence_class
28,T_c5a3c203511123019f4cf53b,DAOS-4823,2021-01-20 19:26:13+00:00,DAOS-I01::DAOS-4823::E1,UNREVIEWED
27,T_5c8facaa5312b00dea3049fe,DAOS-4836,2021-01-20 19:26:00+00:00,DAOS-I01::DAOS-4836::E1,STRONG_CANDIDATE
26,T_659d8724f9ff8b7644d9066a,DAOS-4837,2021-01-20 19:26:22+00:00,DAOS-I01::DAOS-4837::E1,STRONG_CANDIDATE
24,T_73b1868c7e9658dca1264c21,DAOS-6600,2021-02-25 15:19:31+00:00,DAOS-I01::DAOS-6600::E1,UNREVIEWED
25,T_09694fd8a0ab7038e8123486,DAOS-6600,2021-02-25 15:20:06+00:00,DAOS-I01::DAOS-6600::E1,UNREVIEWED
23,T_95040ea9e674bb9aeeccfd37,DAOS-6674,2021-05-10 16:12:48+00:00,DAOS-I01::DAOS-6674::E1,UNREVIEWED
22,T_771fdfd1cc56155a6e54019a,DAOS-6679,2021-04-20 00:23:14+00:00,DAOS-I01::DAOS-6679::E1,UNREVIEWED
21,T_0f5a340b1201cfcb86aba7fa,DAOS-6680,2021-04-20 00:23:37+00:00,DAOS-I01::DAOS-6680::E1,UNREVIEWED
19,T_e2740c38cce24c0127ca385a,DAOS-6743,2021-02-25 15:19:52+00:00,DAOS-I01::DAOS-6743::E1,CONFIRMED_BLOCKING_IMPACT
20,T_d6a83a7ecd4f74fb2a63d561,DAOS-6743,2021-02-25 15:20:17+00:00,DAOS-I01::DAOS-6743::E1,CONFIRMED_BLOCKING_IMPACT


In [ ]:
output_path = (
    GC1_ROOT / "Temporal_snapshots"/"dependency_reviewed_outcomes.jsonl"
)

with open(output_path, "w", encoding="utf-8") as f:

    for record in episode_df.to_dict(orient="records"):

        for field in [
            "outcome_time",
            "evidence_time",
            "previous_event_time"
        ]:
            value = record.get(field)

            if pd.isna(value):
                record[field] = None
            else:
                record[field] = value.isoformat()

        f.write(
            json.dumps(record, default=str)
            + "\n"
        )

print("Saved:", output_path)

Saved: /content/drive/MyDrive/GC1/Temporal_snapshots/dependency_reviewed_outcomes.jsonl


In [ ]:
remaining_events = episode_df[
    episode_df["evidence_class"] == "UNREVIEWED"
].copy()

print("Remaining events:", len(remaining_events))

display(
    remaining_events[
        [
            "event_uid",
            "instance_id",
            "downstream_issue",
            "outcome_time",
            "disruption_type",
            "episode_id"
        ]
    ]
)

Remaining events: 23


,event_uid,instance_id,downstream_issue,outcome_time,disruption_type,episode_id
28,T_c5a3c203511123019f4cf53b,DAOS-I01,DAOS-4823,2021-01-20 19:26:13+00:00,BLOCKED_STATUS,DAOS-I01::DAOS-4823::E1
24,T_73b1868c7e9658dca1264c21,DAOS-I01,DAOS-6600,2021-02-25 15:19:31+00:00,BLOCKED_STATUS,DAOS-I01::DAOS-6600::E1
25,T_09694fd8a0ab7038e8123486,DAOS-I01,DAOS-6600,2021-02-25 15:20:06+00:00,BLOCKED_STATUS,DAOS-I01::DAOS-6600::E1
23,T_95040ea9e674bb9aeeccfd37,DAOS-I01,DAOS-6674,2021-05-10 16:12:48+00:00,BLOCKED_STATUS,DAOS-I01::DAOS-6674::E1
22,T_771fdfd1cc56155a6e54019a,DAOS-I01,DAOS-6679,2021-04-20 00:23:14+00:00,BLOCKED_STATUS,DAOS-I01::DAOS-6679::E1
21,T_0f5a340b1201cfcb86aba7fa,DAOS-I01,DAOS-6680,2021-04-20 00:23:37+00:00,BLOCKED_STATUS,DAOS-I01::DAOS-6680::E1
17,T_4fdf9c847f2e723c0544be06,DAOS-I01,DAOS-7778,2021-07-23 19:30:36+00:00,BLOCKED_STATUS,DAOS-I01::DAOS-7778::E1
12,T_c273da55ab379b9cca90c5aa,JBDS-I01,JBDS-4104,2016-11-03 09:00:23+00:00,REOPENED_WORK,JBDS-I01::JBDS-4104::E1
8,T_c78c514d885da65279246103,JBEAP-I01,JBEAP-1515,2016-01-19 02:18:48+00:00,REOPENED_WORK,JBEAP-I01::JBEAP-1515::E1
9,T_269866c37032727f1a2aa256,JBEAP-I01,JBEAP-1515,2016-01-19 02:19:29+00:00,REOPENED_WORK,JBEAP-I01::JBEAP-1515::E1


In [ ]:
remaining_ids = set(remaining_events["event_uid"])

remaining_comments = evidence_window_df[
    evidence_window_df["event_uid"].isin(remaining_ids)
].copy()

remaining_comments = remaining_comments.sort_values(
    ["event_uid", "comment_time"]
)

print("Remaining events with comments:",
      remaining_comments["event_uid"].nunique())

print("Total associated comments:",
      len(remaining_comments))

display(
    remaining_comments[
        [
            "event_uid",
            "upstream_issue",
            "downstream_issue",
            "comment_time",
            "comment_role",
            "body"
        ]
    ].head(30)
)

Remaining events with comments: 18
Total associated comments: 123


,event_uid,upstream_issue,downstream_issue,comment_time,comment_role,body
23,T_020d268afc763c4d946f55ff,JBEAP-2499,JBEAP-1707,2016-01-07 14:46:01+00:00,DOWNSTREAM,Actually this is because JGroupsBroadcastTest ...
24,T_020d268afc763c4d946f55ff,JBEAP-2499,JBEAP-1707,2016-01-07 20:58:50+00:00,DOWNSTREAM,PR sent:
25,T_020d268afc763c4d946f55ff,JBEAP-2499,JBEAP-1707,2016-01-18 06:25:15+00:00,DOWNSTREAM,"Since the test hasn't failed anymore, I have v..."
184,T_09694fd8a0ab7038e8123486,DAOS-6808,DAOS-6600,2021-02-16 13:38:04+00:00,UPSTREAM,[~accountid:60357ea525b84e00693d0418] Please h...
185,T_09694fd8a0ab7038e8123486,DAOS-6808,DAOS-6600,2021-02-16 14:56:29+00:00,UPSTREAM,[~accountid:60357ea525b84e00693d0418] I am onl...
186,T_09694fd8a0ab7038e8123486,DAOS-6808,DAOS-6600,2021-02-17 16:52:08+00:00,UPSTREAM,memleax was installed on the daos server nodes...
187,T_09694fd8a0ab7038e8123486,DAOS-6808,DAOS-6600,2021-02-17 22:04:30+00:00,UPSTREAM,The logs were obtained. I'm analyzing logs now...
188,T_09694fd8a0ab7038e8123486,DAOS-6808,DAOS-6600,2021-02-19 07:40:08+00:00,UPSTREAM,"[~accountid:6033e1eef6ccc0007024622b], could y..."
189,T_09694fd8a0ab7038e8123486,DAOS-6808,DAOS-6600,2021-02-19 15:56:05+00:00,UPSTREAM,[~accountid:60357ea525b84e00693d0418] I have ...
204,T_09694fd8a0ab7038e8123486,DAOS-6808,DAOS-6600,2021-02-19 16:53:51+00:00,DOWNSTREAM,It has been recommended to run soak exclude/re...


In [ ]:
def summarize_comments(group):

    records = []

    for _, row in group.iterrows():

        text = str(row["body"]).replace("\n", " ")

        records.append({
            "comment_id": row["comment_id"],
            "comment_time": str(row["comment_time"]),
            "role": row["comment_role"],
            "text": text[:500]
        })

    return records


comment_context = {}

for event_uid, group in remaining_comments.groupby("event_uid"):
    comment_context[event_uid] = summarize_comments(group)


review_records = []

for _, row in remaining_events.iterrows():

    event_uid = row["event_uid"]

    review_records.append({
        "event_uid": event_uid,
        "instance_id": row["instance_id"],
        "downstream_issue": row["downstream_issue"],
        "outcome_time": row["outcome_time"],
        "disruption_type": row["disruption_type"],
        "episode_id": row["episode_id"],
        "nearby_comments": comment_context.get(event_uid, []),
        "review_decision": "PENDING",
        "review_notes": ""
    })


remaining_review_df = pd.DataFrame(review_records)

print("Review records prepared:", len(remaining_review_df))

display(remaining_review_df.head())

Review records prepared: 23


,event_uid,instance_id,downstream_issue,outcome_time,disruption_type,episode_id,nearby_comments,review_decision,review_notes
0,T_c5a3c203511123019f4cf53b,DAOS-I01,DAOS-4823,2021-01-20 19:26:13+00:00,BLOCKED_STATUS,DAOS-I01::DAOS-4823::E1,"[{'comment_id': '34536', 'comment_time': '2021...",PENDING,
1,T_73b1868c7e9658dca1264c21,DAOS-I01,DAOS-6600,2021-02-25 15:19:31+00:00,BLOCKED_STATUS,DAOS-I01::DAOS-6600::E1,"[{'comment_id': '36177', 'comment_time': '2021...",PENDING,
2,T_09694fd8a0ab7038e8123486,DAOS-I01,DAOS-6600,2021-02-25 15:20:06+00:00,BLOCKED_STATUS,DAOS-I01::DAOS-6600::E1,"[{'comment_id': '36177', 'comment_time': '2021...",PENDING,
3,T_95040ea9e674bb9aeeccfd37,DAOS-I01,DAOS-6674,2021-05-10 16:12:48+00:00,BLOCKED_STATUS,DAOS-I01::DAOS-6674::E1,"[{'comment_id': '39272', 'comment_time': '2021...",PENDING,
4,T_771fdfd1cc56155a6e54019a,DAOS-I01,DAOS-6679,2021-04-20 00:23:14+00:00,BLOCKED_STATUS,DAOS-I01::DAOS-6679::E1,[],PENDING,


In [ ]:
import pandas as pd

# Start from the 23 unreviewed events
review_df = remaining_review_df.copy()

# Retrieve associated upstream issues
upstream_mapping = (
    verification_df.groupby("event_uid")["upstream_issue"]
    .apply(lambda x: sorted(set(x)))
    .to_dict()
)

review_df["upstream_issues"] = (
    review_df["event_uid"]
    .map(upstream_mapping)
)

# Ensure timestamps are valid
review_df["outcome_time"] = pd.to_datetime(
    review_df["outcome_time"],
    utc=True,
    errors="coerce"
)

print("Events ready for review:", len(review_df))

display(
    review_df[
        [
            "instance_id",
            "upstream_issues",
            "downstream_issue",
            "outcome_time",
            "disruption_type",
            "episode_id"
        ]
    ]
)

Events ready for review: 23


,instance_id,upstream_issues,downstream_issue,outcome_time,disruption_type,episode_id
0,DAOS-I01,"[DAOS-6462, DAOS-6510]",DAOS-4823,2021-01-20 19:26:13+00:00,BLOCKED_STATUS,DAOS-I01::DAOS-4823::E1
1,DAOS-I01,[DAOS-6808],DAOS-6600,2021-02-25 15:19:31+00:00,BLOCKED_STATUS,DAOS-I01::DAOS-6600::E1
2,DAOS-I01,[DAOS-6808],DAOS-6600,2021-02-25 15:20:06+00:00,BLOCKED_STATUS,DAOS-I01::DAOS-6600::E1
3,DAOS-I01,"[DAOS-6807, DAOS-7081]",DAOS-6674,2021-05-10 16:12:48+00:00,BLOCKED_STATUS,DAOS-I01::DAOS-6674::E1
4,DAOS-I01,"[DAOS-6177, DAOS-6367]",DAOS-6679,2021-04-20 00:23:14+00:00,BLOCKED_STATUS,DAOS-I01::DAOS-6679::E1
5,DAOS-I01,"[DAOS-6177, DAOS-6367]",DAOS-6680,2021-04-20 00:23:37+00:00,BLOCKED_STATUS,DAOS-I01::DAOS-6680::E1
6,DAOS-I01,[DAOS-7671],DAOS-7778,2021-07-23 19:30:36+00:00,BLOCKED_STATUS,DAOS-I01::DAOS-7778::E1
7,JBDS-I01,"[CDK-24, JBDS-4098]",JBDS-4104,2016-11-03 09:00:23+00:00,REOPENED_WORK,JBDS-I01::JBDS-4104::E1
8,JBEAP-I01,[JBEAP-2443],JBEAP-1515,2016-01-19 02:18:48+00:00,REOPENED_WORK,JBEAP-I01::JBEAP-1515::E1
9,JBEAP-I01,[JBEAP-2443],JBEAP-1515,2016-01-19 02:19:29+00:00,REOPENED_WORK,JBEAP-I01::JBEAP-1515::E1


In [ ]:
import re

REVIEW_TERMS = [
    "blocked",
    "blocking",
    "dependency",
    "depends on",
    "waiting for",
    "cannot proceed",
    "unable to",
    "failed due to",
    "delayed because",
    "reopened because",
    "regression"
]

def rank_comment(comment, upstream_issues):

    text = str(comment.get("text", ""))
    text_lower = text.lower()

    direct_reference = any(
        re.search(
            rf"(?<![A-Z0-9-]){re.escape(issue)}(?![A-Z0-9-])",
            text,
            re.IGNORECASE
        )
        for issue in upstream_issues
    )

    matched_terms = [
        term for term in REVIEW_TERMS
        if term in text_lower
    ]

    score = (
        2 * int(direct_reference)
        + int(bool(matched_terms))
    )

    return score, direct_reference, matched_terms


ranked_records = []

for _, row in review_df.iterrows():

    upstream_issues = row["upstream_issues"] or []

    for comment in row["nearby_comments"]:

        score, direct_ref, terms = rank_comment(
            comment,
            upstream_issues
        )

        ranked_records.append({
            "event_uid": row["event_uid"],
            "instance_id": row["instance_id"],
            "upstream_issues": upstream_issues,
            "downstream_issue": row["downstream_issue"],
            "outcome_time": row["outcome_time"],
            "episode_id": row["episode_id"],
            "comment_id": comment["comment_id"],
            "comment_time": comment["comment_time"],
            "comment_text": comment["text"],
            "direct_reference": direct_ref,
            "matched_terms": terms,
            "review_score": score
        })

ranked_review_df = pd.DataFrame(ranked_records)

print("Comments ranked:", len(ranked_review_df))

if not ranked_review_df.empty:
    display(
        ranked_review_df.sort_values(
            "review_score",
            ascending=False
        ).head(20)
    )

Comments ranked: 123


,event_uid,instance_id,upstream_issues,downstream_issue,outcome_time,episode_id,comment_id,comment_time,comment_text,direct_reference,matched_terms,review_score
14,T_73b1868c7e9658dca1264c21,DAOS-I01,[DAOS-6808],DAOS-6600,2021-02-25 15:19:31+00:00,DAOS-I01::DAOS-6600::E1,36469,2021-02-22 14:05:49+00:00,exclude/reintegrate was not run this weekend d...,True,[],2
37,T_09694fd8a0ab7038e8123486,DAOS-I01,[DAOS-6808],DAOS-6600,2021-02-25 15:20:06+00:00,DAOS-I01::DAOS-6600::E1,36469,2021-02-22 14:05:49+00:00,exclude/reintegrate was not run this weekend d...,True,[],2
1,T_c5a3c203511123019f4cf53b,DAOS-I01,"[DAOS-6462, DAOS-6510]",DAOS-4823,2021-01-20 19:26:13+00:00,DAOS-I01::DAOS-4823::E1,34598,2021-01-21 17:30:47+00:00,This ticket is marked as blocking a number of ...,False,[blocking],1
18,T_73b1868c7e9658dca1264c21,DAOS-I01,[DAOS-6808],DAOS-6600,2021-02-25 15:19:31+00:00,DAOS-I01::DAOS-6600::E1,36660,2021-02-25 15:50:54+00:00,[~accountid:6033e1eef6ccc0007024622b] Do we st...,False,[waiting for],1
21,T_73b1868c7e9658dca1264c21,DAOS-I01,[DAOS-6808],DAOS-6600,2021-02-25 15:19:31+00:00,DAOS-I01::DAOS-6600::E1,36739,2021-02-26 21:41:00+00:00,"Just to be clear, this ticket is very high pri...",False,[blocking],1
41,T_09694fd8a0ab7038e8123486,DAOS-I01,[DAOS-6808],DAOS-6600,2021-02-25 15:20:06+00:00,DAOS-I01::DAOS-6600::E1,36660,2021-02-25 15:50:54+00:00,[~accountid:6033e1eef6ccc0007024622b] Do we st...,False,[waiting for],1
0,T_c5a3c203511123019f4cf53b,DAOS-I01,"[DAOS-6462, DAOS-6510]",DAOS-4823,2021-01-20 19:26:13+00:00,DAOS-I01::DAOS-4823::E1,34536,2021-01-20 19:07:04+00:00,This issue is blocking req1.2 test.,False,[blocking],1
44,T_09694fd8a0ab7038e8123486,DAOS-I01,[DAOS-6808],DAOS-6600,2021-02-25 15:20:06+00:00,DAOS-I01::DAOS-6600::E1,36739,2021-02-26 21:41:00+00:00,"Just to be clear, this ticket is very high pri...",False,[blocking],1
83,T_269866c37032727f1a2aa256,JBEAP-I01,[JBEAP-2443],JBEAP-1515,2016-01-19 02:19:29+00:00,JBEAP-I01::JBEAP-1515::E1,13150300,2016-01-19 02:16:55+00:00,Thanks for info! I've changed jira type to bug...,False,[regression],1
79,T_c78c514d885da65279246103,JBEAP-I01,[JBEAP-2443],JBEAP-1515,2016-01-19 02:18:48+00:00,JBEAP-I01::JBEAP-1515::E1,13150300,2016-01-19 02:16:55+00:00,Thanks for info! I've changed jira type to bug...,False,[regression],1


In [ ]:
for _, event in review_df.iterrows():

    event_uid = event["event_uid"]

    print("\n" + "=" * 75)
    print("EVENT:", event_uid)
    print("Project:", event["instance_id"])
    print("Upstream:", event["upstream_issues"])
    print("Downstream:", event["downstream_issue"])
    print("Disruption:", event["outcome_time"])
    print("Type:", event["disruption_type"])

    if ranked_review_df.empty:
        print("No nearby comments.")
        continue

    comments = ranked_review_df[
        ranked_review_df["event_uid"] == event_uid
    ].sort_values(
        "review_score",
        ascending=False
    ).head(3)

    if comments.empty:
        print("No nearby comments.")
        continue

    for _, comment in comments.iterrows():

        print("\nComment ID:", comment["comment_id"])
        print("Comment time:", comment["comment_time"])
        print("Review score:", comment["review_score"])
        print("Direct reference:", comment["direct_reference"])
        print("Matched terms:", comment["matched_terms"])

        print("\nTEXT:")
        print(comment["comment_text"])


EVENT: T_c5a3c203511123019f4cf53b
Project: DAOS-I01
Upstream: ['DAOS-6462', 'DAOS-6510']
Downstream: DAOS-4823
Disruption: 2021-01-20 19:26:13+00:00
Type: BLOCKED_STATUS

Comment ID: 34536
Comment time: 2021-01-20 19:07:04+00:00
Review score: 1
Direct reference: False
Matched terms: ['blocking']

TEXT:
This issue is blocking req1.2 test.

Comment ID: 34598
Comment time: 2021-01-21 17:30:47+00:00
Review score: 1
Direct reference: False
Matched terms: ['blocking']

TEXT:
This ticket is marked as blocking a number of 1.1.3 tickets - should this ticket not also be in 1.1.3? Is it being prioritized appropriately?

Comment ID: 34746
Comment time: 2021-01-22 16:46:47+00:00
Review score: 0
Direct reference: False
Matched terms: []

TEXT:
[~accountid:6033e1f94623c600699ff14f]: After some discussion, it seems as if this is the expected behavior. My understanding is that [~accountid:60357e555ddf020069b1cc99] is working on an update to make the error more clear in DAOS-4884. I also understand tha

In [ ]:
# Preliminary classifications for the remaining events

remaining_classifications = {
    "DAOS-4823": "PROXY",
    "DAOS-6600": "STRONG_CANDIDATE",
    "DAOS-6674": "UNCERTAIN",
    "DAOS-6679": "UNCERTAIN",
    "DAOS-6680": "UNCERTAIN",
    "DAOS-7778": "UNCERTAIN",

    "JBDS-4104": "PROXY",

    "JBEAP-1515": "PROXY",
    "JBEAP-1707": "UNCERTAIN",
    "JBEAP-2100": "PROXY",
    "JBEAP-2544": "UNCERTAIN",
    "JBEAP-850": "UNCERTAIN",

    "JBIDE-3708": "UNCERTAIN",
    "SPARK-24432": "UNCERTAIN"
}

# Work from the existing episode-level event table
classified_events = episode_df.copy()

# Preserve classifications from the earlier six reviewed events.
mask = classified_events["evidence_class"] == "UNREVIEWED"

classified_events.loc[mask, "evidence_class"] = (
    classified_events.loc[mask, "downstream_issue"]
    .map(remaining_classifications)
    .fillna("UNCERTAIN")
)

print("Updated event classifications:")

display(
    classified_events["evidence_class"].value_counts()
)

Updated event classifications:


,count
evidence_class,
UNCERTAIN,15
PROXY,7
STRONG_CANDIDATE,5
CONFIRMED_BLOCKING_IMPACT,2


In [ ]:
# Map upstream blockers to downstream events
upstream_map = (
    verification_df.groupby("event_uid")["upstream_issue"]
    .apply(lambda x: sorted(set(x)))
    .to_dict()
)

classified_events["upstream_issue_ids"] = (
    classified_events["event_uid"]
    .map(upstream_map)
)

# Map timestamped comments
comment_map = (
    evidence_window_df.groupby("event_uid")["comment_id"]
    .apply(
        lambda x: sorted(set(
            str(v) for v in x if pd.notna(v)
        ))
    )
    .to_dict()
)

classified_events["comment_ids"] = (
    classified_events["event_uid"]
    .map(lambda x: comment_map.get(x, []))
)

display(
    classified_events[
        [
            "event_uid",
            "instance_id",
            "downstream_issue",
            "upstream_issue_ids",
            "evidence_class",
            "comment_ids"
        ]
    ].head(10)
)

,event_uid,instance_id,downstream_issue,upstream_issue_ids,evidence_class,comment_ids
28,T_c5a3c203511123019f4cf53b,DAOS-I01,DAOS-4823,"[DAOS-6462, DAOS-6510]",PROXY,"[34536, 34598, 34746, 34759, 34760]"
27,T_5c8facaa5312b00dea3049fe,DAOS-I01,DAOS-4836,[DAOS-6510],STRONG_CANDIDATE,"[34146, 34536, 34598, 34746, 34759, 34760]"
26,T_659d8724f9ff8b7644d9066a,DAOS-I01,DAOS-4837,[DAOS-6510],STRONG_CANDIDATE,"[34145, 34536, 34598, 34746, 34759, 34760]"
24,T_73b1868c7e9658dca1264c21,DAOS-I01,DAOS-6600,[DAOS-6808],STRONG_CANDIDATE,"[36177, 36189, 36274, 36318, 36401, 36418, 364..."
25,T_09694fd8a0ab7038e8123486,DAOS-I01,DAOS-6600,[DAOS-6808],STRONG_CANDIDATE,"[36177, 36189, 36274, 36318, 36401, 36418, 364..."
23,T_95040ea9e674bb9aeeccfd37,DAOS-I01,DAOS-6674,"[DAOS-6807, DAOS-7081]",UNCERTAIN,"[39272, 39314, 39548, 39679, 39684]"
22,T_771fdfd1cc56155a6e54019a,DAOS-I01,DAOS-6679,"[DAOS-6177, DAOS-6367]",UNCERTAIN,[]
21,T_0f5a340b1201cfcb86aba7fa,DAOS-I01,DAOS-6680,"[DAOS-6177, DAOS-6367]",UNCERTAIN,"[39008, 39060]"
19,T_e2740c38cce24c0127ca385a,DAOS-I01,DAOS-6743,"[DAOS-6519, DAOS-6714, DAOS-6808]",CONFIRMED_BLOCKING_IMPACT,"[36177, 36189, 36257, 36271, 36274, 36318, 364..."
20,T_d6a83a7ecd4f74fb2a63d561,DAOS-I01,DAOS-6743,"[DAOS-6519, DAOS-6714, DAOS-6808]",CONFIRMED_BLOCKING_IMPACT,"[36177, 36189, 36257, 36271, 36274, 36318, 364..."


In [ ]:
import pandas as pd

def unique_flatten(series):
    values = set()

    for item in series:
        if isinstance(item, list):
            values.update(str(x) for x in item if pd.notna(x))
        elif pd.notna(item):
            values.add(str(item))

    return sorted(values)


def episode_classification(values):
    classes = set(values)

    # Evidence classification priority
    if "CONFIRMED_BLOCKING_IMPACT" in classes:
        return "CONFIRMED_BLOCKING_IMPACT"

    if "STRONG_CANDIDATE" in classes:
        return "STRONG_CANDIDATE"

    if "PROXY" in classes:
        return "PROXY"

    return "UNCERTAIN"


episode_outcomes = (
    classified_events.groupby("episode_id")
    .agg(
        instance_id=("instance_id", "first"),
        downstream_issue=("downstream_issue", "first"),
        outcome_time=("outcome_time", "min"),
        event_uids=("event_uid", lambda x: sorted(set(x))),
        upstream_issue_ids=(
            "upstream_issue_ids", unique_flatten
        ),
        comment_ids=("comment_ids", unique_flatten),
        evidence_class=(
            "evidence_class", episode_classification
        ),
        event_count=("event_uid", "nunique")
    )
    .reset_index()
)

episode_outcomes["risk_type"] = "DEPENDENCY"

episode_outcomes["review_status"] = "PRELIMINARY"

print("Total candidate episodes:", len(episode_outcomes))

display(
    episode_outcomes["evidence_class"].value_counts()
)

display(episode_outcomes.head(10))

Total candidate episodes: 22


,count
evidence_class,
UNCERTAIN,12
PROXY,5
STRONG_CANDIDATE,4
CONFIRMED_BLOCKING_IMPACT,1


,episode_id,instance_id,downstream_issue,outcome_time,event_uids,upstream_issue_ids,comment_ids,evidence_class,event_count,risk_type,review_status
0,DAOS-I01::DAOS-4823::E1,DAOS-I01,DAOS-4823,2021-01-20 19:26:13+00:00,[T_c5a3c203511123019f4cf53b],"[DAOS-6462, DAOS-6510]","[34536, 34598, 34746, 34759, 34760]",PROXY,1,DEPENDENCY,PRELIMINARY
1,DAOS-I01::DAOS-4836::E1,DAOS-I01,DAOS-4836,2021-01-20 19:26:00+00:00,[T_5c8facaa5312b00dea3049fe],[DAOS-6510],"[34146, 34536, 34598, 34746, 34759, 34760]",STRONG_CANDIDATE,1,DEPENDENCY,PRELIMINARY
2,DAOS-I01::DAOS-4837::E1,DAOS-I01,DAOS-4837,2021-01-20 19:26:22+00:00,[T_659d8724f9ff8b7644d9066a],[DAOS-6510],"[34145, 34536, 34598, 34746, 34759, 34760]",STRONG_CANDIDATE,1,DEPENDENCY,PRELIMINARY
3,DAOS-I01::DAOS-6600::E1,DAOS-I01,DAOS-6600,2021-02-25 15:19:31+00:00,"[T_09694fd8a0ab7038e8123486, T_73b1868c7e9658d...",[DAOS-6808],"[36177, 36189, 36274, 36318, 36401, 36418, 364...",STRONG_CANDIDATE,2,DEPENDENCY,PRELIMINARY
4,DAOS-I01::DAOS-6674::E1,DAOS-I01,DAOS-6674,2021-05-10 16:12:48+00:00,[T_95040ea9e674bb9aeeccfd37],"[DAOS-6807, DAOS-7081]","[39272, 39314, 39548, 39679, 39684]",UNCERTAIN,1,DEPENDENCY,PRELIMINARY
5,DAOS-I01::DAOS-6679::E1,DAOS-I01,DAOS-6679,2021-04-20 00:23:14+00:00,[T_771fdfd1cc56155a6e54019a],"[DAOS-6177, DAOS-6367]",[],UNCERTAIN,1,DEPENDENCY,PRELIMINARY
6,DAOS-I01::DAOS-6680::E1,DAOS-I01,DAOS-6680,2021-04-20 00:23:37+00:00,[T_0f5a340b1201cfcb86aba7fa],"[DAOS-6177, DAOS-6367]","[39008, 39060]",UNCERTAIN,1,DEPENDENCY,PRELIMINARY
7,DAOS-I01::DAOS-6743::E1,DAOS-I01,DAOS-6743,2021-02-25 15:19:52+00:00,"[T_d6a83a7ecd4f74fb2a63d561, T_e2740c38cce24c0...","[DAOS-6519, DAOS-6714, DAOS-6808]","[36177, 36189, 36257, 36271, 36274, 36318, 364...",CONFIRMED_BLOCKING_IMPACT,2,DEPENDENCY,PRELIMINARY
8,DAOS-I01::DAOS-7449::E1,DAOS-I01,DAOS-7449,2021-08-17 22:34:10+00:00,[T_b442f54be47cce2d5c9557a1],"[DAOS-6378, DAOS-6380, DAOS-6381, DAOS-6811, D...","[42997, 43017, 85057, 85058]",PROXY,1,DEPENDENCY,PRELIMINARY
9,DAOS-I01::DAOS-7778::E1,DAOS-I01,DAOS-7778,2021-07-23 19:30:36+00:00,[T_4fdf9c847f2e723c0544be06],[DAOS-7671],"[41879, 41922, 41925, 41956, 41957, 42004, 420...",UNCERTAIN,1,DEPENDENCY,PRELIMINARY


In [ ]:
import json

output_path = (
    GC1_ROOT / "Temporal_snapshots"/"dependency_episode_outcomes.jsonl"
)

with open(output_path, "w", encoding="utf-8") as f:

    for record in episode_outcomes.to_dict(
        orient="records"
    ):

        record["outcome_time"] = (
            record["outcome_time"].isoformat()
            if pd.notna(record["outcome_time"])
            else None
        )

        f.write(
            json.dumps(record, default=str)
            + "\n"
        )

print("Saved:", output_path)

Saved: /content/drive/MyDrive/GC1/Temporal_snapshots/dependency_episode_outcomes.jsonl


### 2.3 Blocking-link escalation and bulk-update screening

In [ ]:
import pandas as pd

# Use the cleaned historical link timeline
new_blocking_events = clean_links[
    clean_links["action"] == "ADD"
].copy()

new_blocking_events["event_date"] = pd.to_datetime(
    new_blocking_events["observed_dt"],
    utc=True,
    errors="coerce"
)

new_blocking_events = new_blocking_events.dropna(
    subset=["event_date"]
)

project_escalation_coverage = (
    new_blocking_events.groupby("instance_id")
    .agg(
        new_blocking_events=("event_uid", "count"),
        unique_downstream_issues=("downstream_issue", "nunique"),
        unique_upstream_issues=("upstream_issue", "nunique"),
        first_event=("event_date", "min"),
        last_event=("event_date", "max")
    )
    .reset_index()
)

print(
    "Projects with newly recorded blocking dependencies:",
    len(project_escalation_coverage)
)

display(
    project_escalation_coverage.sort_values(
        "new_blocking_events",
        ascending=False
    )
)

Projects with newly recorded blocking dependencies: 18


,instance_id,new_blocking_events,unique_downstream_issues,unique_upstream_issues,first_event,last_event
11,JBIDE-I01,587,415,337,2020-09-13 05:02:31+00:00,2020-09-14 05:32:38+00:00
10,JBEAP-I01,407,151,295,2015-11-26 10:09:15+00:00,2016-07-29 08:19:03+00:00
5,ENTESB-I01,294,208,191,2020-04-08 07:32:26+00:00,2020-11-30 13:32:57+00:00
4,DAOS-I01,186,64,142,2021-01-06 16:16:24+00:00,2021-09-10 20:19:31+00:00
0,AEROGEAR-I01,40,20,35,2018-02-15 05:23:04+00:00,2018-09-04 06:15:06+00:00
7,HBASE-I01,28,7,28,2016-07-15 07:43:20+00:00,2017-02-14 00:20:25+00:00
2,BXMSDOC-I01,24,20,20,2020-04-14 11:50:44+00:00,2020-09-13 11:36:59+00:00
17,SPARK-I01,22,17,15,2018-11-23 02:22:56+00:00,2019-07-19 18:41:27+00:00
9,JBDS-I01,21,14,17,2016-03-22 04:51:32+00:00,2016-10-28 15:56:54+00:00
8,INFRA-I01,15,13,9,2017-09-21 16:03:09+00:00,2018-02-03 15:59:10+00:00


In [ ]:
import pandas as pd

adds = clean_links[
    clean_links["action"] == "ADD"
].copy()

adds["event_dt"] = pd.to_datetime(
    adds["observed_dt"],
    utc=True,
    errors="coerce"
)

adds = adds.dropna(subset=["event_dt"])

adds["event_day"] = adds["event_dt"].dt.floor("D")

daily_adds = (
    adds.groupby(["instance_id", "event_day"])
    .agg(
        add_count=("event_uid", "count"),
        unique_downstream=("downstream_issue", "nunique"),
        unique_upstream=("upstream_issue", "nunique")
    )
    .reset_index()
)

print("Top 20 project-days by blocking-link additions:")

display(
    daily_adds.sort_values(
        "add_count",
        ascending=False
    ).head(20)
)

Top 20 project-days by blocking-link additions:


,instance_id,event_day,add_count,unique_downstream,unique_upstream
276,JBIDE-I01,2020-09-13 00:00:00+00:00,572,404,326
116,ENTESB-I01,2020-09-13 00:00:00+00:00,234,173,148
23,BXMSDOC-I01,2020-09-13 00:00:00+00:00,22,18,18
228,JBEAP-I01,2016-04-04 00:00:00+00:00,22,2,22
184,JBEAP-I01,2016-01-06 00:00:00+00:00,21,2,21
186,JBEAP-I01,2016-01-12 00:00:00+00:00,19,11,12
277,JBIDE-I01,2020-09-14 00:00:00+00:00,15,15,15
215,JBEAP-I01,2016-03-09 00:00:00+00:00,15,14,5
202,JBEAP-I01,2016-02-12 00:00:00+00:00,15,3,15
198,JBEAP-I01,2016-02-08 00:00:00+00:00,14,1,14


In [ ]:
project_totals = (
    daily_adds.groupby("instance_id")["add_count"]
    .sum()
    .rename("total_adds")
)

project_peaks = (
    daily_adds.groupby("instance_id")["add_count"]
    .max()
    .rename("peak_day_adds")
)

concentration_df = pd.concat(
    [project_totals, project_peaks],
    axis=1
).reset_index()

concentration_df["peak_day_share"] = (
    concentration_df["peak_day_adds"]
    / concentration_df["total_adds"]
).round(3)

display(
    concentration_df.sort_values(
        "peak_day_share",
        ascending=False
    )
)

,instance_id,total_adds,peak_day_adds,peak_day_share
16,MNG-I01,1,1,1.000
3,CONFSERVER-I01,2,2,1.000
11,JBIDE-I01,587,572,0.974
2,BXMSDOC-I01,24,22,0.917
5,ENTESB-I01,294,234,0.796
14,JSDCLOUD-I01,3,2,0.667
12,JRACLOUD-I01,2,1,0.500
1,AMBARI-I01,8,4,0.500
7,HBASE-I01,28,11,0.393
13,JRASERVER-I01,3,1,0.333


In [ ]:
import pandas as pd
import json

suspect_projects = [
    "JBIDE-I01",
    "ENTESB-I01",
    "BXMSDOC-I01"
]

suspect_day = pd.Timestamp(
    "2020-09-13",
    tz="UTC"
)

suspect_adds = adds[
    adds["instance_id"].isin(suspect_projects)
    & (adds["event_day"] == suspect_day)
].copy()

# Parse original Event payloads
def parse_event_payload(value):
    if isinstance(value, dict):
        return value

    if isinstance(value, str):
        try:
            return json.loads(value)
        except json.JSONDecodeError:
            return {}

    return {}

event_lookup = (
    nodes_df[nodes_df["label"] == "Event"]
    .set_index("uid")["payload_json"]
    .to_dict()
)

records = []

for _, row in suspect_adds.iterrows():

    payload = parse_event_payload(
        event_lookup.get(row["event_uid"])
    )

    records.append({
        "instance_id": row["instance_id"],
        "event_uid": row["event_uid"],
        "event_time": row["event_dt"],
        "actor": payload.get("actor"),
        "from_value": payload.get("from_value"),
        "to_value": payload.get("to_value"),
        "upstream_issue": row["upstream_issue"],
        "downstream_issue": row["downstream_issue"]
    })

suspect_details = pd.DataFrame(records)

print("Suspected ADD events:", len(suspect_details))

print("\nActor distribution:")
display(
    suspect_details.groupby(
        ["instance_id", "actor"],
        dropna=False
    )
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(20)
)

print("\nSample original events:")
display(
    suspect_details[
        [
            "instance_id",
            "event_time",
            "actor",
            "from_value",
            "to_value"
        ]
    ].head(20)
)

Suspected ADD events: 828

Actor distribution:


,instance_id,actor,count
2,JBIDE-I01,<<|author_displayName|0106bf55-f45a-47e2-9c3f-...,572
1,ENTESB-I01,<<|author_displayName|0106bf55-f45a-47e2-9c3f-...,234
0,BXMSDOC-I01,<<|author_displayName|0106bf55-f45a-47e2-9c3f-...,22



Sample original events:


,instance_id,event_time,actor,from_value,to_value
0,BXMSDOC-I01,2020-09-13 11:36:59+00:00,<<|author_displayName|0106bf55-f45a-47e2-9c3f-...,None,This issue is blocked by-legacy AF-1664
1,BXMSDOC-I01,2020-09-13 11:24:47+00:00,<<|author_displayName|0106bf55-f45a-47e2-9c3f-...,None,This issue is blocked by-legacy BAPL-1003
2,BXMSDOC-I01,2020-09-13 11:26:34+00:00,<<|author_displayName|0106bf55-f45a-47e2-9c3f-...,None,This issue is blocked by-legacy BAPL-662
3,BXMSDOC-I01,2020-09-13 11:19:10+00:00,<<|author_displayName|0106bf55-f45a-47e2-9c3f-...,None,This issue is blocked by-legacy BXMSDOC-1317
4,BXMSDOC-I01,2020-09-13 11:19:10+00:00,<<|author_displayName|0106bf55-f45a-47e2-9c3f-...,None,This issue is blocked by-legacy BXMSDOC-1317
5,BXMSDOC-I01,2020-09-13 11:19:11+00:00,<<|author_displayName|0106bf55-f45a-47e2-9c3f-...,None,This issue is blocked by-legacy BXMSDOC-1317
6,BXMSDOC-I01,2020-09-13 11:19:11+00:00,<<|author_displayName|0106bf55-f45a-47e2-9c3f-...,None,This issue is blocked by-legacy BXMSDOC-1317
7,BXMSDOC-I01,2020-09-13 11:19:07+00:00,<<|author_displayName|0106bf55-f45a-47e2-9c3f-...,None,This issue blocks-legacy GUVNOR-2868
8,BXMSDOC-I01,2020-09-13 11:19:06+00:00,<<|author_displayName|0106bf55-f45a-47e2-9c3f-...,None,This issue is blocked by-legacy BXMSDOC-1431
9,BXMSDOC-I01,2020-09-13 11:19:07+00:00,<<|author_displayName|0106bf55-f45a-47e2-9c3f-...,None,This issue blocks-legacy BXMSDOC-1432


In [ ]:
# Work from original, pre-consolidation records
original_actions = links_with_text.copy()

original_actions["observed_dt"] = pd.to_datetime(
    original_actions["observed_dt"],
    utc=True
)

suspect_keys = suspect_adds[
    [
        "instance_id",
        "upstream_issue",
        "downstream_issue",
        "observed_dt"
    ]
].drop_duplicates()

matched_actions = original_actions.merge(
    suspect_keys,
    on=[
        "instance_id",
        "upstream_issue",
        "downstream_issue",
        "observed_dt"
    ],
    how="inner"
)

print("Original actions matching suspected additions:")

display(
    matched_actions["action"].value_counts()
)

print("\nLegacy removal records:")

legacy_removals = matched_actions[
    (matched_actions["action"] == "REMOVE")
    &
    matched_actions["from_value"]
    .fillna("")
    .str.contains("-legacy", case=False, regex=False)
]

print("Legacy removals:", len(legacy_removals))

display(
    legacy_removals[
        [
            "instance_id",
            "upstream_issue",
            "downstream_issue",
            "observed_dt",
            "from_value"
        ]
    ].head(15)
)

Original actions matching suspected additions:


,count
action,
ADD,828



Legacy removal records:
Legacy removals: 0


,instance_id,upstream_issue,downstream_issue,observed_dt,from_value


In [ ]:
import pandas as pd

# Preserve the complete cleaned history
dependency_history = clean_links.copy()

dependency_history["event_dt"] = pd.to_datetime(
    dependency_history["observed_dt"],
    utc=True,
    errors="coerce"
)

dependency_history["event_day"] = (
    dependency_history["event_dt"].dt.floor("D")
)

bulk_projects = {
    "JBIDE-I01",
    "ENTESB-I01",
    "BXMSDOC-I01"
}

bulk_day = pd.Timestamp("2020-09-13", tz="UTC")

dependency_history["suspected_bulk_update"] = (
    dependency_history["instance_id"].isin(bulk_projects)
    &
    (dependency_history["event_day"] == bulk_day)
    &
    (dependency_history["action"] == "ADD")
)

print(
    "Suspected bulk ADD events:",
    dependency_history["suspected_bulk_update"].sum()
)

display(
    dependency_history.groupby(
        ["action", "suspected_bulk_update"]
    ).size().reset_index(name="count")
)

Suspected bulk ADD events: 828


,action,suspected_bulk_update,count
0,ADD,False,831
1,ADD,True,828
2,MIGRATION,False,808
3,REMOVE,False,912


In [ ]:
eligible_dependency_adds = dependency_history[
    (dependency_history["action"] == "ADD")
    &
    (~dependency_history["suspected_bulk_update"])
    &
    (dependency_history["event_dt"].notna())
].copy()

print(
    "Escalation-eligible ADD events:",
    len(eligible_dependency_adds)
)

print(
    "Projects with eligible ADD events:",
    eligible_dependency_adds["instance_id"].nunique()
)

display(
    eligible_dependency_adds.groupby("instance_id")
    .agg(
        eligible_adds=("event_uid", "count"),
        affected_downstream=("downstream_issue", "nunique"),
        affected_upstream=("upstream_issue", "nunique")
    )
    .reset_index()
    .sort_values("eligible_adds", ascending=False)
)

Escalation-eligible ADD events: 831
Projects with eligible ADD events: 18


,instance_id,eligible_adds,affected_downstream,affected_upstream
10,JBEAP-I01,407,151,295
4,DAOS-I01,186,64,142
5,ENTESB-I01,60,42,46
0,AEROGEAR-I01,40,20,35
7,HBASE-I01,28,7,28
17,SPARK-I01,22,17,15
9,JBDS-I01,21,14,17
8,INFRA-I01,15,13,9
11,JBIDE-I01,15,15,15
15,MC-I01,13,13,10


In [ ]:
eligible_dependency_adds["event_week"] = (
    eligible_dependency_adds["event_dt"]
    .dt.to_period("W-SUN")
    .dt.start_time
)

weekly_escalations = (
    eligible_dependency_adds.groupby(
        ["instance_id", "event_week"]
    )
    .agg(
        new_blocking_links=("event_uid", "count"),
        affected_downstream=("downstream_issue", "nunique")
    )
    .reset_index()
)

print(
    "Project-weeks containing eligible additions:",
    len(weekly_escalations)
)

print(
    "Projects represented:",
    weekly_escalations["instance_id"].nunique()
)

display(
    weekly_escalations.sort_values(
        "new_blocking_links",
        ascending=False
    ).head(20)
)

Project-weeks containing eligible additions: 169
Projects represented: 18


/tmp/ipykernel_1144/600178321.py:3: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  .dt.to_period("W-SUN")


,instance_id,event_week,new_blocking_links,affected_downstream
114,JBEAP-I01,2016-02-08,44,11
122,JBEAP-I01,2016-04-04,39,10
47,DAOS-I01,2021-08-30,29,8
109,JBEAP-I01,2016-01-04,29,10
115,JBEAP-I01,2016-02-15,28,13
110,JBEAP-I01,2016-01-11,23,15
106,JBEAP-I01,2015-12-07,23,10
107,JBEAP-I01,2015-12-14,21,10
118,JBEAP-I01,2016-03-07,20,18
126,JBEAP-I01,2016-05-02,19,5


In [ ]:
import pandas as pd

weekly_dependency_metrics = (
    eligible_dependency_adds.groupby(
        ["instance_id", "event_week"]
    )
    .agg(
        new_blocking_links=("event_uid", "count"),
        newly_linked_downstream=("downstream_issue", "nunique"),
        unique_upstream=("upstream_issue", "nunique")
    )
    .reset_index()
)

weekly_dependency_metrics["links_per_downstream"] = (
    weekly_dependency_metrics["new_blocking_links"]
    / weekly_dependency_metrics["newly_linked_downstream"]
)

print("Weekly dependency escalation statistics:")

display(
    weekly_dependency_metrics[
        [
            "new_blocking_links",
            "newly_linked_downstream",
            "unique_upstream",
            "links_per_downstream"
        ]
    ].describe()
)

print("\nTop project-weeks:")

display(
    weekly_dependency_metrics.sort_values(
        "newly_linked_downstream",
        ascending=False
    ).head(20)
)

Weekly dependency escalation statistics:


,new_blocking_links,newly_linked_downstream,unique_upstream,links_per_downstream
count,169.000000,169.000000,169.000000,169.000000
mean,4.917160,2.875740,4.106509,1.679395
std,6.928564,3.227846,5.828952,1.666227
min,1.000000,1.000000,1.000000,1.000000
25%,1.000000,1.000000,1.000000,1.000000
50%,2.000000,1.000000,2.000000,1.000000
75%,6.000000,3.000000,5.000000,1.666667
max,44.000000,18.000000,41.000000,16.000000



Top project-weeks:


,instance_id,event_week,new_blocking_links,newly_linked_downstream,unique_upstream,links_per_downstream
118,JBEAP-I01,2016-03-07,20,18,8,1.111111
116,JBEAP-I01,2016-02-22,18,17,5,1.058824
139,JBIDE-I01,2020-09-14,15,15,15,1.000000
110,JBEAP-I01,2016-01-11,23,15,12,1.533333
115,JBEAP-I01,2016-02-15,28,13,19,2.153846
114,JBEAP-I01,2016-02-08,44,11,41,4.000000
112,JBEAP-I01,2016-01-25,11,11,3,1.000000
106,JBEAP-I01,2015-12-07,23,10,18,2.300000
122,JBEAP-I01,2016-04-04,39,10,34,3.900000
107,JBEAP-I01,2015-12-14,21,10,15,2.100000


In [ ]:
first_blocking_adds = (
    eligible_dependency_adds.sort_values("event_dt")
    .drop_duplicates(
        subset=["instance_id", "downstream_issue"],
        keep="first"
    )
    .copy()
)

first_blocking_adds["event_week"] = (
    first_blocking_adds["event_dt"]
    .dt.to_period("W-SUN")
    .dt.start_time
)

first_blocking_weekly = (
    first_blocking_adds.groupby(
        ["instance_id", "event_week"]
    )
    .agg(
        first_observed_blocked_issues=(
            "downstream_issue", "nunique"
        )
    )
    .reset_index()
)

weekly_dependency_metrics = (
    weekly_dependency_metrics.merge(
        first_blocking_weekly,
        on=["instance_id", "event_week"],
        how="left"
    )
)

weekly_dependency_metrics[
    "first_observed_blocked_issues"
] = (
    weekly_dependency_metrics[
        "first_observed_blocked_issues"
    ].fillna(0).astype(int)
)

print("First-observed blocked-issue statistics:")

display(
    weekly_dependency_metrics[
        "first_observed_blocked_issues"
    ].describe()
)

First-observed blocked-issue statistics:


/tmp/ipykernel_1144/2118235777.py:12: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  .dt.to_period("W-SUN")


,first_observed_blocked_issues
count,169.000000
mean,2.236686
std,2.745703
min,0.000000
25%,1.000000
50%,1.000000
75%,2.000000
max,15.000000


In [ ]:
project_coverage = (
    weekly_dependency_metrics.groupby("instance_id")
    .agg(
        active_weeks=("event_week", "nunique"),
        total_new_links=("new_blocking_links", "sum"),
        first_observed_blocked_issues=(
            "first_observed_blocked_issues", "sum"
        ),
        peak_week_links=("new_blocking_links", "max")
    )
    .reset_index()
)

display(
    project_coverage.sort_values(
        "first_observed_blocked_issues",
        ascending=False
    )
)

,instance_id,active_weeks,total_new_links,first_observed_blocked_issues,peak_week_links
10,JBEAP-I01,35,407,151,44
4,DAOS-I01,32,186,64,29
5,ENTESB-I01,23,60,42,10
0,AEROGEAR-I01,11,40,20,14
17,SPARK-I01,10,22,17,7
11,JBIDE-I01,1,15,15,15
9,JBDS-I01,12,21,14,5
15,MC-I01,11,13,13,3
8,INFRA-I01,7,15,13,5
7,HBASE-I01,10,28,7,11


In [ ]:
import pandas as pd

checkpoint_records = []

for _, project in project_timeline.iterrows():

    instance_id = project["instance_id"]

    start = pd.to_datetime(
        project["first_observation"],
        utc=True
    )

    end = pd.to_datetime(
        project["last_observation"],
        utc=True
    )

    checkpoint = start + pd.Timedelta(days=7)
    week_number = 1

    while checkpoint <= end:

        checkpoint_records.append({
            "instance_id": instance_id,
            "week_number": week_number,
            "checkpoint_time": checkpoint,
            "observation_end": end
        })

        checkpoint += pd.Timedelta(days=7)
        week_number += 1

checkpoints_df = pd.DataFrame(checkpoint_records)

print("Total weekly checkpoints:", len(checkpoints_df))
print("Projects:", checkpoints_df["instance_id"].nunique())

display(checkpoints_df.head(10))

Total weekly checkpoints: 1047
Projects: 30


,instance_id,week_number,checkpoint_time,observation_end
0,AEROGEAR-I01,1,2018-01-24 05:30:01+00:00,2018-09-25 09:29:15+00:00
1,AEROGEAR-I01,2,2018-01-31 05:30:01+00:00,2018-09-25 09:29:15+00:00
2,AEROGEAR-I01,3,2018-02-07 05:30:01+00:00,2018-09-25 09:29:15+00:00
3,AEROGEAR-I01,4,2018-02-14 05:30:01+00:00,2018-09-25 09:29:15+00:00
4,AEROGEAR-I01,5,2018-02-21 05:30:01+00:00,2018-09-25 09:29:15+00:00
5,AEROGEAR-I01,6,2018-02-28 05:30:01+00:00,2018-09-25 09:29:15+00:00
6,AEROGEAR-I01,7,2018-03-07 05:30:01+00:00,2018-09-25 09:29:15+00:00
7,AEROGEAR-I01,8,2018-03-14 05:30:01+00:00,2018-09-25 09:29:15+00:00
8,AEROGEAR-I01,9,2018-03-21 05:30:01+00:00,2018-09-25 09:29:15+00:00
9,AEROGEAR-I01,10,2018-03-28 05:30:01+00:00,2018-09-25 09:29:15+00:00


In [ ]:
HORIZON_DAYS = 14

label_records = []

for _, checkpoint in checkpoints_df.iterrows():

    instance_id = checkpoint["instance_id"]
    t = checkpoint["checkpoint_time"]

    horizon_end = t + pd.Timedelta(
        days=HORIZON_DAYS
    )

    # Only use checkpoints with a complete future window
    if horizon_end > checkpoint["observation_end"]:
        continue

    future_events = eligible_dependency_adds[
        (eligible_dependency_adds["instance_id"] == instance_id)
        &
        (eligible_dependency_adds["event_dt"] > t)
        &
        (eligible_dependency_adds["event_dt"] <= horizon_end)
    ]

    label_records.append({
        "instance_id": instance_id,
        "week_number": checkpoint["week_number"],
        "checkpoint_time": t,
        "horizon_end": horizon_end,
        "future_new_links": len(future_events),
        "future_affected_downstream": (
            future_events["downstream_issue"].nunique()
        )
    })

dependency_future_df = pd.DataFrame(label_records)

print("Eligible project-week checkpoints:",
      len(dependency_future_df))

display(dependency_future_df.head(10))

Eligible project-week checkpoints: 987


,instance_id,week_number,checkpoint_time,horizon_end,future_new_links,future_affected_downstream
0,AEROGEAR-I01,1,2018-01-24 05:30:01+00:00,2018-02-07 05:30:01+00:00,0,0
1,AEROGEAR-I01,2,2018-01-31 05:30:01+00:00,2018-02-14 05:30:01+00:00,0,0
2,AEROGEAR-I01,3,2018-02-07 05:30:01+00:00,2018-02-21 05:30:01+00:00,2,1
3,AEROGEAR-I01,4,2018-02-14 05:30:01+00:00,2018-02-28 05:30:01+00:00,2,1
4,AEROGEAR-I01,5,2018-02-21 05:30:01+00:00,2018-03-07 05:30:01+00:00,15,7
5,AEROGEAR-I01,6,2018-02-28 05:30:01+00:00,2018-03-14 05:30:01+00:00,18,9
6,AEROGEAR-I01,7,2018-03-07 05:30:01+00:00,2018-03-21 05:30:01+00:00,3,3
7,AEROGEAR-I01,8,2018-03-14 05:30:01+00:00,2018-03-28 05:30:01+00:00,0,0
8,AEROGEAR-I01,9,2018-03-21 05:30:01+00:00,2018-04-04 05:30:01+00:00,1,1
9,AEROGEAR-I01,10,2018-03-28 05:30:01+00:00,2018-04-11 05:30:01+00:00,6,2


In [ ]:
print("Future dependency activity distribution:")

display(
    dependency_future_df[
        [
            "future_new_links",
            "future_affected_downstream"
        ]
    ].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]
    )
)

print("\nCheckpoints with no future additions:")

print(
    (dependency_future_df["future_new_links"] == 0).sum()
)

print("\nProjects with future dependency activity:")

display(
    dependency_future_df.groupby("instance_id")
    .agg(
        checkpoints=("week_number", "count"),
        active_future_windows=(
            "future_new_links",
            lambda x: (x > 0).sum()
        ),
        peak_future_downstream=(
            "future_affected_downstream", "max"
        )
    )
    .reset_index()
)

Future dependency activity distribution:


,future_new_links,future_affected_downstream
count,987.000000,987.000000
mean,1.584600,0.879433
std,5.877527,2.953777
min,0.000000,0.000000
25%,0.000000,0.000000
50%,0.000000,0.000000
75%,0.000000,0.000000
90%,3.000000,2.000000
95%,10.000000,5.000000
max,71.000000,35.000000



Checkpoints with no future additions:
767

Projects with future dependency activity:


,instance_id,checkpoints,active_future_windows,peak_future_downstream
0,AEROGEAR-I01,33,18,9
1,AMBARI-I01,33,6,3
2,BXMSDOC-I01,33,4,1
3,CONFSERVER-I01,33,0,0
4,DAOS-I01,33,32,15
5,DOCS-I01,33,0,0
6,ENTESB-I01,33,27,9
7,EVG-I01,33,0,0
8,FLEX-I01,33,5,1
9,HBASE-I01,33,16,3


In [ ]:
import pandas as pd

threshold_results = []

for threshold in [1, 2, 3, 4, 5, 6]:

    positive = (
        dependency_future_df[
            "future_affected_downstream"
        ] >= threshold
    )

    positive_projects = (
        dependency_future_df.loc[
            positive, "instance_id"
        ].nunique()
    )

    threshold_results.append({
        "threshold": threshold,
        "positive_checkpoints": int(positive.sum()),
        "negative_checkpoints": int((~positive).sum()),
        "positive_pct": round(
            positive.mean() * 100, 2
        ),
        "positive_projects": positive_projects
    })

threshold_df = pd.DataFrame(threshold_results)

display(threshold_df)

,threshold,positive_checkpoints,negative_checkpoints,positive_pct,positive_projects
0,1,220,767,22.29,17
1,2,126,861,12.77,12
2,3,90,897,9.12,11
3,4,67,920,6.79,7
4,5,55,932,5.57,6
5,6,41,946,4.15,5


In [ ]:
PROVISIONAL_THRESHOLD = 3

dependency_future_df["provisional_label"] = (
    dependency_future_df[
        "future_affected_downstream"
    ] >= PROVISIONAL_THRESHOLD
).astype(int)

project_label_distribution = (
    dependency_future_df.groupby("instance_id")
    .agg(
        total_checkpoints=("week_number", "count"),
        positive_checkpoints=(
            "provisional_label", "sum"
        )
    )
    .reset_index()
)

project_label_distribution["positive_pct"] = (
    project_label_distribution["positive_checkpoints"]
    / project_label_distribution["total_checkpoints"]
    * 100
).round(2)

display(
    project_label_distribution.sort_values(
        "positive_checkpoints",
        ascending=False
    )
)

,instance_id,total_checkpoints,positive_checkpoints,positive_pct
12,JBEAP-I01,33,28,84.85
4,DAOS-I01,33,24,72.73
6,ENTESB-I01,33,12,36.36
0,AEROGEAR-I01,33,5,15.15
27,SPARK-I01,33,5,15.15
11,JBDS-I01,33,4,12.12
10,INFRA-I01,33,4,12.12
9,HBASE-I01,33,2,6.06
1,AMBARI-I01,33,2,6.06
13,JBIDE-I01,33,2,6.06


In [ ]:
label_sequence = dependency_future_df.sort_values(
    ["instance_id", "week_number"]
).copy()

label_sequence["previous_label"] = (
    label_sequence.groupby("instance_id")[
        "provisional_label"
    ].shift(fill_value=0)
)

label_sequence["new_positive_run"] = (
    (label_sequence["provisional_label"] == 1)
    &
    (label_sequence["previous_label"] == 0)
)

print(
    "Total positive checkpoints:",
    label_sequence["provisional_label"].sum()
)

print(
    "Distinct positive runs:",
    label_sequence["new_positive_run"].sum()
)

display(
    label_sequence.groupby("instance_id")
    .agg(
        positive_checkpoints=(
            "provisional_label", "sum"
        ),
        positive_runs=(
            "new_positive_run", "sum"
        )
    )
    .reset_index()
)

Total positive checkpoints: 90
Distinct positive runs: 29


,instance_id,positive_checkpoints,positive_runs
0,AEROGEAR-I01,5,2
1,AMBARI-I01,2,1
2,BXMSDOC-I01,0,0
3,CONFSERVER-I01,0,0
4,DAOS-I01,24,6
5,DOCS-I01,0,0
6,ENTESB-I01,12,4
7,EVG-I01,0,0
8,FLEX-I01,0,0
9,HBASE-I01,2,1


### 2.4 Final dependency risk labels

In [ ]:
positive_windows = dependency_future_df[
    dependency_future_df["provisional_label"] == 1
].copy()

positive_evidence = []

for _, row in positive_windows.iterrows():

    matching_events = eligible_dependency_adds[
        (eligible_dependency_adds["instance_id"] == row["instance_id"])
        &
        (eligible_dependency_adds["event_dt"] > row["checkpoint_time"])
        &
        (eligible_dependency_adds["event_dt"] <= row["horizon_end"])
    ]

    positive_evidence.append({
        "instance_id": row["instance_id"],
        "week_number": row["week_number"],
        "checkpoint_time": row["checkpoint_time"],
        "future_affected_downstream":
            row["future_affected_downstream"],
        "event_count": len(matching_events),
        "event_uids": matching_events["event_uid"].tolist(),
        "event_dates": matching_events["event_dt"].tolist(),
        "downstream_issues": sorted(
            matching_events["downstream_issue"].unique()
        )
    })

positive_evidence_df = pd.DataFrame(positive_evidence)

print("Positive windows audited:", len(positive_evidence_df))

display(positive_evidence_df.head())

Positive windows audited: 90


,instance_id,week_number,checkpoint_time,future_affected_downstream,event_count,event_uids,event_dates,downstream_issues
0,AEROGEAR-I01,5,2018-02-21 05:30:01+00:00,7,15,"[T_62d9e53fc092cf712ba348bf, T_008617171e2bba5...","[2018-03-01 09:46:57+00:00, 2018-03-05 07:30:4...","[AEROGEAR-2218, AEROGEAR-2242, AEROGEAR-2243, ..."
1,AEROGEAR-I01,6,2018-02-28 05:30:01+00:00,9,18,"[T_62d9e53fc092cf712ba348bf, T_008617171e2bba5...","[2018-03-01 09:46:57+00:00, 2018-03-05 07:30:4...","[AEROGEAR-2218, AEROGEAR-2238, AEROGEAR-2242, ..."
2,AEROGEAR-I01,7,2018-03-07 05:30:01+00:00,3,3,"[T_7fd58f15f1148a1b28f15360, T_2933479b2ebc22e...","[2018-03-08 05:09:57+00:00, 2018-03-09 10:56:0...","[AEROGEAR-2238, AEROGEAR-2250, AEROGEAR-5543]"
3,AEROGEAR-I01,19,2018-05-30 05:30:01+00:00,3,3,"[T_03aefd230e6392a811b00bab, T_517bf5cb25875ca...","[2018-06-12 09:34:26+00:00, 2018-06-12 09:34:3...","[AEROGEAR-3140, AEROGEAR-3202, AEROGEAR-3215]"
4,AEROGEAR-I01,20,2018-06-06 05:30:01+00:00,4,5,"[T_03aefd230e6392a811b00bab, T_517bf5cb25875ca...","[2018-06-12 09:34:26+00:00, 2018-06-12 09:34:3...","[AEROGEAR-2948, AEROGEAR-3140, AEROGEAR-3202, ..."


In [ ]:
# Count how many positive windows are affected
# by each individual event day

burst_records = []

for _, row in positive_evidence_df.iterrows():

    event_days = pd.Series(
        row["event_dates"],
        dtype="datetime64[ns, UTC]"
    ).dt.floor("D")

    if event_days.empty:
        continue

    counts = event_days.value_counts()

    burst_records.append({
        "instance_id": row["instance_id"],
        "week_number": row["week_number"],
        "total_events": len(event_days),
        "peak_day_events": int(counts.max()),
        "peak_day_share": round(
            counts.max() / len(event_days), 3
        )
    })

burst_df = pd.DataFrame(burst_records)

print("Positive windows with >=80% events on one day:")

display(
    burst_df[
        burst_df["peak_day_share"] >= 0.8
    ].sort_values(
        "total_events",
        ascending=False
    ).head(30)
)

Positive windows with >=80% events on one day:


,instance_id,week_number,total_events,peak_day_events,peak_day_share
82,JBIDE-I01,11,15,15,1.000
81,JBIDE-I01,10,15,15,1.000
86,SPARK-I01,18,7,7,1.000
87,SPARK-I01,19,7,7,1.000
34,ENTESB-I01,6,6,5,0.833
42,ENTESB-I01,28,6,5,0.833
52,JBDS-I01,30,6,5,0.833
37,ENTESB-I01,21,5,4,0.800
46,INFRA-I01,15,5,5,1.000
45,INFRA-I01,14,5,5,1.000


In [ ]:
jbide_remaining = eligible_dependency_adds[
    eligible_dependency_adds["instance_id"] == "JBIDE-I01"
].copy()

print("JBIDE remaining ADD events:", len(jbide_remaining))

display(
    jbide_remaining[
        [
            "event_uid",
            "upstream_issue",
            "downstream_issue",
            "event_dt"
        ]
    ].sort_values("event_dt")
)

JBIDE remaining ADD events: 15


,event_uid,upstream_issue,downstream_issue,event_dt
2757,T_61b8bea9b91c726c0f76a3ce,JBIDE-24966,JBIDE-24948,2020-09-14 05:28:20+00:00
2698,T_90c1bf13ca1b780b000173e7,JBIDE-24880,JBIDE-24484,2020-09-14 05:28:21+00:00
2628,T_74e6ff43157e5faa6e66c887,JBIDE-23317,JBIDE-24121,2020-09-14 05:28:22+00:00
2660,T_81a0d144b33856e153657e63,JBIDE-24007,JBIDE-23317,2020-09-14 05:28:23+00:00
2647,T_ff119893c4119e087f7d137d,JBIDE-23506,JBIDE-23026,2020-09-14 05:28:24+00:00
2490,T_662d6653c0f363b3b0b8e3c4,JBIDE-21029,JBIDE-19846,2020-09-14 05:28:33+00:00
2451,T_28ce65b1802ddf3374146176,JBIDE-19778,JBIDE-19590,2020-09-14 05:28:34+00:00
2320,T_937fba25ea58dbf8ccf026a4,JBIDE-18388,JBIDE-18413,2020-09-14 05:28:38+00:00
2070,T_973053aa881d91bad742787b,JBIDE-14002,JBIDE-16340,2020-09-14 05:28:45+00:00
2212,T_bbad20b2db4dbc943f4edf36,JBIDE-16290,JBIDE-15639,2020-09-14 05:28:48+00:00


In [ ]:
import json
import pandas as pd

# Retrieve original payloads for remaining JBIDE events
jbide_event_details = []

for _, row in jbide_remaining.iterrows():

    payload = parse_event_payload(
        event_lookup.get(row["event_uid"])
    )

    jbide_event_details.append({
        "event_uid": row["event_uid"],
        "event_time": row["event_dt"],
        "actor": payload.get("actor"),
        "from_value": payload.get("from_value"),
        "to_value": payload.get("to_value")
    })

jbide_details_df = pd.DataFrame(jbide_event_details)

print("Actor distribution:")
display(jbide_details_df["actor"].value_counts(dropna=False))

# Compare with the previously identified bulk-update actor
bulk_actors = set(
    suspect_details["actor"].dropna()
)

jbide_details_df["matches_bulk_actor"] = (
    jbide_details_df["actor"].isin(bulk_actors)
)

print(
    "\nEvents matching bulk-update actor:",
    jbide_details_df["matches_bulk_actor"].sum()
)

display(jbide_details_df)

Actor distribution:


,count
actor,
<<|author_displayName|082e60dc-fe91-45a3-adae-7927ef9e1288|>>,15



Events matching bulk-update actor: 0


,event_uid,event_time,actor,from_value,to_value,matches_bulk_actor
0,T_847dda1b9148567190304ab1,2020-09-14 05:29:02+00:00,<<|author_displayName|082e60dc-fe91-45a3-adae-...,None,This issue is blocked by JBIDE-10915,False
1,T_973053aa881d91bad742787b,2020-09-14 05:28:45+00:00,<<|author_displayName|082e60dc-fe91-45a3-adae-...,None,This issue is blocked by JBIDE-14002,False
2,T_bbad20b2db4dbc943f4edf36,2020-09-14 05:28:48+00:00,<<|author_displayName|082e60dc-fe91-45a3-adae-...,None,This issue is blocked by JBIDE-16290,False
3,T_937fba25ea58dbf8ccf026a4,2020-09-14 05:28:38+00:00,<<|author_displayName|082e60dc-fe91-45a3-adae-...,None,This issue blocks JBIDE-18413,False
4,T_28ce65b1802ddf3374146176,2020-09-14 05:28:34+00:00,<<|author_displayName|082e60dc-fe91-45a3-adae-...,None,This issue is blocked by JBIDE-19778,False
5,T_662d6653c0f363b3b0b8e3c4,2020-09-14 05:28:33+00:00,<<|author_displayName|082e60dc-fe91-45a3-adae-...,None,This issue blocks JBIDE-19846,False
6,T_74e6ff43157e5faa6e66c887,2020-09-14 05:28:22+00:00,<<|author_displayName|082e60dc-fe91-45a3-adae-...,None,This issue blocks JBIDE-24121,False
7,T_ff119893c4119e087f7d137d,2020-09-14 05:28:24+00:00,<<|author_displayName|082e60dc-fe91-45a3-adae-...,None,This issue is blocked by JBIDE-23506,False
8,T_81a0d144b33856e153657e63,2020-09-14 05:28:23+00:00,<<|author_displayName|082e60dc-fe91-45a3-adae-...,None,This issue is blocked by JBIDE-24007,False
9,T_90c1bf13ca1b780b000173e7,2020-09-14 05:28:21+00:00,<<|author_displayName|082e60dc-fe91-45a3-adae-...,None,This issue blocks JBIDE-24484,False


In [ ]:
# Known legacy migration relationships
migration_pairs = migration_df[
    [
        "instance_id",
        "upstream_issue",
        "downstream_issue"
    ]
].drop_duplicates()

# Remaining JBIDE additions
jbide_pairs = jbide_remaining[
    [
        "event_uid",
        "instance_id",
        "upstream_issue",
        "downstream_issue",
        "event_dt"
    ]
].copy()

comparison = jbide_pairs.merge(
    migration_pairs.assign(
        found_in_migration=True
    ),
    on=[
        "instance_id",
        "upstream_issue",
        "downstream_issue"
    ],
    how="left"
)

comparison["found_in_migration"] = (
    comparison["found_in_migration"]
    .fillna(False)
    .astype(bool)
)

print("Total remaining JBIDE additions:", len(comparison))

print(
    "Pairs also found in legacy migrations:",
    comparison["found_in_migration"].sum()
)

display(comparison)

Total remaining JBIDE additions: 15
Pairs also found in legacy migrations: 0


/tmp/ipykernel_1144/1506156476.py:35: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


,event_uid,instance_id,upstream_issue,downstream_issue,event_dt,found_in_migration
0,T_847dda1b9148567190304ab1,JBIDE-I01,JBIDE-10915,JBIDE-11157,2020-09-14 05:29:02+00:00,False
1,T_973053aa881d91bad742787b,JBIDE-I01,JBIDE-14002,JBIDE-16340,2020-09-14 05:28:45+00:00,False
2,T_bbad20b2db4dbc943f4edf36,JBIDE-I01,JBIDE-16290,JBIDE-15639,2020-09-14 05:28:48+00:00,False
3,T_937fba25ea58dbf8ccf026a4,JBIDE-I01,JBIDE-18388,JBIDE-18413,2020-09-14 05:28:38+00:00,False
4,T_28ce65b1802ddf3374146176,JBIDE-I01,JBIDE-19778,JBIDE-19590,2020-09-14 05:28:34+00:00,False
5,T_662d6653c0f363b3b0b8e3c4,JBIDE-I01,JBIDE-21029,JBIDE-19846,2020-09-14 05:28:33+00:00,False
6,T_74e6ff43157e5faa6e66c887,JBIDE-I01,JBIDE-23317,JBIDE-24121,2020-09-14 05:28:22+00:00,False
7,T_ff119893c4119e087f7d137d,JBIDE-I01,JBIDE-23506,JBIDE-23026,2020-09-14 05:28:24+00:00,False
8,T_81a0d144b33856e153657e63,JBIDE-I01,JBIDE-24007,JBIDE-23317,2020-09-14 05:28:23+00:00,False
9,T_90c1bf13ca1b780b000173e7,JBIDE-I01,JBIDE-24880,JBIDE-24484,2020-09-14 05:28:21+00:00,False


In [ ]:
issue_created_lookup = (
    jira_df.set_index("issue_key")["created_at"]
    .to_dict()
)

comparison["downstream_created_at"] = pd.to_datetime(
    comparison["downstream_issue"].map(
        issue_created_lookup
    ),
    utc=True,
    errors="coerce"
)

comparison["issue_age_days"] = (
    comparison["event_dt"]
    - comparison["downstream_created_at"]
).dt.total_seconds() / 86400

print("Downstream issue age at link addition:")

display(
    comparison[
        [
            "upstream_issue",
            "downstream_issue",
            "event_dt",
            "downstream_created_at",
            "issue_age_days",
            "found_in_migration"
        ]
    ]
)

display(
    comparison["issue_age_days"].describe()
)

Downstream issue age at link addition:


,upstream_issue,downstream_issue,event_dt,downstream_created_at,issue_age_days,found_in_migration
0,JBIDE-10915,JBIDE-11157,2020-09-14 05:29:02+00:00,2012-03-04 11:11:16+00:00,3115.762338,False
1,JBIDE-14002,JBIDE-16340,2020-09-14 05:28:45+00:00,2014-01-09 11:34:48+00:00,2439.745799,False
2,JBIDE-16290,JBIDE-15639,2020-09-14 05:28:48+00:00,2013-10-09 03:14:02+00:00,2532.093588,False
3,JBIDE-18388,JBIDE-18413,2020-09-14 05:28:38+00:00,NaT,NaN,False
4,JBIDE-19778,JBIDE-19590,2020-09-14 05:28:34+00:00,2015-04-14 12:30:09+00:00,1979.707234,False
5,JBIDE-21029,JBIDE-19846,2020-09-14 05:28:33+00:00,NaT,NaN,False
6,JBIDE-23317,JBIDE-24121,2020-09-14 05:28:22+00:00,NaT,NaN,False
7,JBIDE-23506,JBIDE-23026,2020-09-14 05:28:24+00:00,2016-08-22 10:55:08+00:00,1483.773102,False
8,JBIDE-24007,JBIDE-23317,2020-09-14 05:28:23+00:00,2016-10-11 13:26:03+00:00,1433.668287,False
9,JBIDE-24880,JBIDE-24484,2020-09-14 05:28:21+00:00,NaT,NaN,False


,issue_age_days
count,10.000000
mean,2405.720105
std,862.379872
min,1111.067639
25%,1607.756635
50%,2485.919693
75%,3235.027538
max,3398.697940


In [ ]:
# Preserve the existing dependency history
final_dependency_history = dependency_history.copy()

# Flag the 15 suspicious JBIDE events
additional_suspect_ids = set(
    jbide_remaining["event_uid"]
)

final_dependency_history["suspected_jbide_migration"] = (
    final_dependency_history["event_uid"]
    .isin(additional_suspect_ids)
)

# Combined exclusion flag
final_dependency_history["exclude_from_escalation"] = (
    final_dependency_history["suspected_bulk_update"]
    |
    final_dependency_history["suspected_jbide_migration"]
)

print("Original bulk exclusions:",
      final_dependency_history["suspected_bulk_update"].sum())

print("Additional JBIDE exclusions:",
      final_dependency_history["suspected_jbide_migration"].sum())

print("Total unique exclusions:",
      final_dependency_history["exclude_from_escalation"].sum())

Original bulk exclusions: 828
Additional JBIDE exclusions: 15
Total unique exclusions: 843


In [ ]:
final_eligible_adds = final_dependency_history[
    (final_dependency_history["action"] == "ADD")
    &
    (~final_dependency_history["exclude_from_escalation"])
    &
    (final_dependency_history["event_dt"].notna())
].copy()

print("Final eligible ADD events:", len(final_eligible_adds))

print(
    "Projects with eligible additions:",
    final_eligible_adds["instance_id"].nunique()
)

display(
    final_eligible_adds.groupby("instance_id")
    .size()
    .reset_index(name="eligible_adds")
    .sort_values("eligible_adds", ascending=False)
)

Final eligible ADD events: 816
Projects with eligible additions: 17


,instance_id,eligible_adds
10,JBEAP-I01,407
4,DAOS-I01,186
5,ENTESB-I01,60
0,AEROGEAR-I01,40
7,HBASE-I01,28
16,SPARK-I01,22
9,JBDS-I01,21
8,INFRA-I01,15
14,MC-I01,13
1,AMBARI-I01,8


In [ ]:
from pathlib import Path
import pandas as pd

BASE = Path(
    "/content/drive/MyDrive/GC1/Temporal_snapshots"
)

OUT = BASE / "dependency_label_handover"
OUT.mkdir(parents=True, exist_ok=True)

assert "final_eligible_adds" in globals(), (
    "Run the dependency eligibility cells first."
)

eligible = final_eligible_adds.copy()

required = {
    "instance_id",
    "event_dt",
    "downstream_issue",
    "action"
}

missing = required - set(eligible.columns)
assert not missing, f"Missing columns: {missing}"

assert eligible["action"].eq("ADD").all()
assert eligible["event_dt"].notna().all()

if "exclude_from_escalation" in eligible.columns:
    assert not eligible[
        "exclude_from_escalation"
    ].fillna(False).any()

parquet_path = OUT / "final_eligible_dependency_adds.parquet"
csv_path = OUT / "final_eligible_dependency_adds.csv"

# Prevent accidental overwriting.
assert not parquet_path.exists(), parquet_path
assert not csv_path.exists(), csv_path

eligible.to_parquet(parquet_path, index=False)
eligible.to_csv(csv_path, index=False)

print("Exported eligible ADD events:", len(eligible))
print("Projects:", eligible["instance_id"].nunique())
print("Columns:", eligible.columns.tolist())
print("Parquet:", parquet_path)
print("CSV:", csv_path)

Exported eligible ADD events: 816
Projects: 17
Columns: ['instance_id', 'event_uid', 'observed_at', 'action', 'upstream_issue', 'downstream_issue', 'legacy', 'observed_dt', 'uid', 'from_value', 'to_value', 'source_id', 'source_event_uids', 'event_dt', 'event_day', 'suspected_bulk_update', 'suspected_jbide_migration', 'exclude_from_escalation']
Parquet: /content/drive/MyDrive/GC1/Temporal_snapshots/dependency_label_handover/final_eligible_dependency_adds.parquet
CSV: /content/drive/MyDrive/GC1/Temporal_snapshots/dependency_label_handover/final_eligible_dependency_adds.csv


In [ ]:
# ============================================================
# EXPORT COMPLETE DEPENDENCY HISTORY
# ============================================================

from pathlib import Path

OUT = Path(
    "/content/drive/MyDrive/GC1/Temporal_snapshots/"
    "dependency_label_handover"
)

OUT.mkdir(parents=True, exist_ok=True)

assert "final_dependency_history" in globals()

history = final_dependency_history.copy()

print("Total dependency events:", len(history))
print("ADD/REMOVE distribution:")
print(history["action"].value_counts())

print("\nExclusion distribution:")
print(history["exclude_from_escalation"].value_counts())

display(history.head())

# Export without overwriting existing files
csv_path = OUT / "final_dependency_history.csv"
parquet_path = OUT / "final_dependency_history.parquet"

assert not csv_path.exists(), f"Already exists: {csv_path}"
assert not parquet_path.exists(), f"Already exists: {parquet_path}"

history.to_csv(csv_path, index=False)
history.to_parquet(parquet_path, index=False)

print("\nCOMPLETE DEPENDENCY HISTORY EXPORTED")
print("CSV:", csv_path)
print("Parquet:", parquet_path)

Total dependency events: 3379
ADD/REMOVE distribution:
action
ADD          1659
REMOVE        912
MIGRATION     808
Name: count, dtype: int64

Exclusion distribution:
exclude_from_escalation
False    2536
True      843
Name: count, dtype: int64


,instance_id,event_uid,observed_at,action,upstream_issue,downstream_issue,legacy,observed_dt,uid,from_value,to_value,source_id,source_event_uids,event_dt,event_day,suspected_bulk_update,suspected_jbide_migration,exclude_from_escalation
0,AEROGEAR-I01,T_62d9e53fc092cf712ba348bf,2018-03-01T09:46:57+00:00,ADD,AEROGEAR-2210,AEROGEAR-2218,False,2018-03-01 09:46:57+00:00,T_62d9e53fc092cf712ba348bf,None,This issue is blocked by AEROGEAR-2210,AEROGEAR-2218,NaN,2018-03-01 09:46:57+00:00,2018-03-01 00:00:00+00:00,False,False,False
1,AEROGEAR-I01,T_008617171e2bba5c69d039a8,2018-03-05T07:30:44+00:00,ADD,AEROGEAR-2243,AEROGEAR-2242,False,2018-03-05 07:30:44+00:00,T_008617171e2bba5c69d039a8,None,This issue is blocked by AEROGEAR-2243,AEROGEAR-2242,NaN,2018-03-05 07:30:44+00:00,2018-03-05 00:00:00+00:00,False,False,False
2,AEROGEAR-I01,T_36fe574bd0cf69a23db63895,2018-03-05T07:30:43+00:00,ADD,AEROGEAR-2244,AEROGEAR-2242,False,2018-03-05 07:30:43+00:00,T_36fe574bd0cf69a23db63895,None,This issue is blocked by AEROGEAR-2244,AEROGEAR-2242,NaN,2018-03-05 07:30:43+00:00,2018-03-05 00:00:00+00:00,False,False,False
3,AEROGEAR-I01,T_43e8d8780eb1370115794c18,2018-03-05T07:30:21+00:00,ADD,AEROGEAR-2244,AEROGEAR-2243,False,2018-03-05 07:30:21+00:00,T_43e8d8780eb1370115794c18,None,This issue blocks AEROGEAR-2243,AEROGEAR-2244,NaN,2018-03-05 07:30:21+00:00,2018-03-05 00:00:00+00:00,False,False,False
4,AEROGEAR-I01,T_397ec8597f2000ddb67c0c9c,2018-03-06T04:23:32+00:00,ADD,AEROGEAR-2248,AEROGEAR-2263,False,2018-03-06 04:23:32+00:00,T_397ec8597f2000ddb67c0c9c,None,This issue blocks AEROGEAR-2263,AEROGEAR-2248,NaN,2018-03-06 04:23:32+00:00,2018-03-06 00:00:00+00:00,False,False,False



COMPLETE DEPENDENCY HISTORY EXPORTED
CSV: /content/drive/MyDrive/GC1/Temporal_snapshots/dependency_label_handover/final_dependency_history.csv
Parquet: /content/drive/MyDrive/GC1/Temporal_snapshots/dependency_label_handover/final_dependency_history.parquet


In [ ]:
# ============================================================
# INSPECT JIRA ISSUE-TO-INSTANCE MAPPING
# ============================================================

import pandas as pd

for name, value in list(globals().items()):

    if not isinstance(value, pd.DataFrame):
        continue

    columns = set(value.columns)

    if "instance_id" in columns and (
        "issue_key" in columns
        or "upstream_issue" in columns
        or "downstream_issue" in columns
    ):
        print("\nDataFrame:", name)
        print("Records:", len(value))
        print("Columns:", value.columns.tolist())

        display(value.head(3))


DataFrame: historical_links_df
Records: 4563
Columns: ['instance_id', 'event_uid', 'observed_at', 'action', 'upstream_issue', 'downstream_issue', 'observed_dt']


,instance_id,event_uid,observed_at,action,upstream_issue,downstream_issue,observed_dt
0,JBEAP-I01,T_60b9a9e53ee877d3b82c4598,2016-07-29T07:55:38+00:00,ADD,JBEAP-5465,JBEAP-5464,2016-07-29 07:55:38+00:00
1,JBEAP-I01,T_7d01fe67dc9d47a055a621f8,2016-07-29T08:16:19+00:00,ADD,JBEAP-5466,JBEAP-5464,2016-07-29 08:16:19+00:00
2,JBEAP-I01,T_e27ceb1862f05fc0731a3e53,2016-07-29T08:19:03+00:00,ADD,JBEAP-5467,JBEAP-5464,2016-07-29 08:19:03+00:00



DataFrame: history
Records: 3379
Columns: ['instance_id', 'event_uid', 'observed_at', 'action', 'upstream_issue', 'downstream_issue', 'legacy', 'observed_dt', 'uid', 'from_value', 'to_value', 'source_id', 'source_event_uids', 'event_dt', 'event_day', 'suspected_bulk_update', 'suspected_jbide_migration', 'exclude_from_escalation']


,instance_id,event_uid,observed_at,action,upstream_issue,downstream_issue,legacy,observed_dt,uid,from_value,to_value,source_id,source_event_uids,event_dt,event_day,suspected_bulk_update,suspected_jbide_migration,exclude_from_escalation
0,AEROGEAR-I01,T_62d9e53fc092cf712ba348bf,2018-03-01T09:46:57+00:00,ADD,AEROGEAR-2210,AEROGEAR-2218,False,2018-03-01 09:46:57+00:00,T_62d9e53fc092cf712ba348bf,None,This issue is blocked by AEROGEAR-2210,AEROGEAR-2218,NaN,2018-03-01 09:46:57+00:00,2018-03-01 00:00:00+00:00,False,False,False
1,AEROGEAR-I01,T_008617171e2bba5c69d039a8,2018-03-05T07:30:44+00:00,ADD,AEROGEAR-2243,AEROGEAR-2242,False,2018-03-05 07:30:44+00:00,T_008617171e2bba5c69d039a8,None,This issue is blocked by AEROGEAR-2243,AEROGEAR-2242,NaN,2018-03-05 07:30:44+00:00,2018-03-05 00:00:00+00:00,False,False,False
2,AEROGEAR-I01,T_36fe574bd0cf69a23db63895,2018-03-05T07:30:43+00:00,ADD,AEROGEAR-2244,AEROGEAR-2242,False,2018-03-05 07:30:43+00:00,T_36fe574bd0cf69a23db63895,None,This issue is blocked by AEROGEAR-2244,AEROGEAR-2242,NaN,2018-03-05 07:30:43+00:00,2018-03-05 00:00:00+00:00,False,False,False



DataFrame: intervals
Records: 2465
Columns: ['instance_id', 'upstream_issue', 'downstream_issue', 'active_from', 'first_confirmed_at', 'active_to', 'start_known', 'start_event_uid', 'end_event_uid']


...

,instance_id,event_uid,observed_at,action,upstream_issue,downstream_issue,legacy,observed_dt,uid,from_value,to_value,source_id,source_event_uids,event_dt,event_day,suspected_bulk_update,suspected_jbide_migration,exclude_from_escalation
0,AEROGEAR-I01,T_62d9e53fc092cf712ba348bf,2018-03-01T09:46:57+00:00,ADD,AEROGEAR-2210,AEROGEAR-2218,False,2018-03-01 09:46:57+00:00,T_62d9e53fc092cf712ba348bf,None,This issue is blocked by AEROGEAR-2210,AEROGEAR-2218,NaN,2018-03-01 09:46:57+00:00,2018-03-01 00:00:00+00:00,False,False,False
1,AEROGEAR-I01,T_008617171e2bba5c69d039a8,2018-03-05T07:30:44+00:00,ADD,AEROGEAR-2243,AEROGEAR-2242,False,2018-03-05 07:30:44+00:00,T_008617171e2bba5c69d039a8,None,This issue is blocked by AEROGEAR-2243,AEROGEAR-2242,NaN,2018-03-05 07:30:44+00:00,2018-03-05 00:00:00+00:00,False,False,False
2,AEROGEAR-I01,T_36fe574bd0cf69a23db63895,2018-03-05T07:30:43+00:00,ADD,AEROGEAR-2244,AEROGEAR-2242,False,2018-03-05 07:30:43+00:00,T_36fe574bd0cf69a23db63895,None,This issue is blocked by AEROGEAR-2244,AEROGEAR-2242,NaN,2018-03-05 07:30:43+00:00,2018-03-05 00:00:00+00:00,False,False,False



DataFrame: final_eligible_adds
Records: 816
Columns: ['instance_id', 'event_uid', 'observed_at', 'action', 'upstream_issue', 'downstream_issue', 'legacy', 'observed_dt', 'uid', 'from_value', 'to_value', 'source_id', 'source_event_uids', 'event_dt', 'event_day', 'suspected_bulk_update', 'suspected_jbide_migration', 'exclude_from_escalation']


,instance_id,event_uid,observed_at,action,upstream_issue,downstream_issue,legacy,observed_dt,uid,from_value,to_value,source_id,source_event_uids,event_dt,event_day,suspected_bulk_update,suspected_jbide_migration,exclude_from_escalation
0,AEROGEAR-I01,T_62d9e53fc092cf712ba348bf,2018-03-01T09:46:57+00:00,ADD,AEROGEAR-2210,AEROGEAR-2218,False,2018-03-01 09:46:57+00:00,T_62d9e53fc092cf712ba348bf,None,This issue is blocked by AEROGEAR-2210,AEROGEAR-2218,NaN,2018-03-01 09:46:57+00:00,2018-03-01 00:00:00+00:00,False,False,False
1,AEROGEAR-I01,T_008617171e2bba5c69d039a8,2018-03-05T07:30:44+00:00,ADD,AEROGEAR-2243,AEROGEAR-2242,False,2018-03-05 07:30:44+00:00,T_008617171e2bba5c69d039a8,None,This issue is blocked by AEROGEAR-2243,AEROGEAR-2242,NaN,2018-03-05 07:30:44+00:00,2018-03-05 00:00:00+00:00,False,False,False
2,AEROGEAR-I01,T_36fe574bd0cf69a23db63895,2018-03-05T07:30:43+00:00,ADD,AEROGEAR-2244,AEROGEAR-2242,False,2018-03-05 07:30:43+00:00,T_36fe574bd0cf69a23db63895,None,This issue is blocked by AEROGEAR-2244,AEROGEAR-2242,NaN,2018-03-05 07:30:43+00:00,2018-03-05 00:00:00+00:00,False,False,False



DataFrame: eligible
Records: 816
Columns: ['instance_id', 'event_uid', 'observed_at', 'action', 'upstream_issue', 'downstream_issue', 'legacy', 'observed_dt', 'uid', 'from_value', 'to_value', 'source_id', 'source_event_uids', 'event_dt', 'event_day', 'suspected_bulk_update', 'suspected_jbide_migration', 'exclude_from_escalation']


,instance_id,event_uid,observed_at,action,upstream_issue,downstream_issue,legacy,observed_dt,uid,from_value,to_value,source_id,source_event_uids,event_dt,event_day,suspected_bulk_update,suspected_jbide_migration,exclude_from_escalation
0,AEROGEAR-I01,T_62d9e53fc092cf712ba348bf,2018-03-01T09:46:57+00:00,ADD,AEROGEAR-2210,AEROGEAR-2218,False,2018-03-01 09:46:57+00:00,T_62d9e53fc092cf712ba348bf,None,This issue is blocked by AEROGEAR-2210,AEROGEAR-2218,NaN,2018-03-01 09:46:57+00:00,2018-03-01 00:00:00+00:00,False,False,False
1,AEROGEAR-I01,T_008617171e2bba5c69d039a8,2018-03-05T07:30:44+00:00,ADD,AEROGEAR-2243,AEROGEAR-2242,False,2018-03-05 07:30:44+00:00,T_008617171e2bba5c69d039a8,None,This issue is blocked by AEROGEAR-2243,AEROGEAR-2242,NaN,2018-03-05 07:30:44+00:00,2018-03-05 00:00:00+00:00,False,False,False
2,AEROGEAR-I01,T_36fe574bd0cf69a23db63895,2018-03-05T07:30:43+00:00,ADD,AEROGEAR-2244,AEROGEAR-2242,False,2018-03-05 07:30:43+00:00,T_36fe574bd0cf69a23db63895,None,This issue is blocked by AEROGEAR-2244,AEROGEAR-2242,NaN,2018-03-05 07:30:43+00:00,2018-03-05 00:00:00+00:00,False,False,False


In [ ]:
###################################DO NOT RERUN AFTER THIS ###########################################

In [ ]:
final_label_records = []

for _, checkpoint in checkpoints_df.iterrows():

    instance_id = checkpoint["instance_id"]
    t = checkpoint["checkpoint_time"]

    horizon_end = t + pd.Timedelta(days=14)

    if horizon_end > checkpoint["observation_end"]:
        continue

    future_events = final_eligible_adds[
        (final_eligible_adds["instance_id"] == instance_id)
        &
        (final_eligible_adds["event_dt"] > t)
        &
        (final_eligible_adds["event_dt"] <= horizon_end)
    ]

    final_label_records.append({
        "instance_id": instance_id,
        "week_number": checkpoint["week_number"],
        "checkpoint_time": t,
        "horizon_end": horizon_end,
        "future_new_links": len(future_events),
        "future_affected_downstream":
            future_events["downstream_issue"].nunique()
    })

final_dependency_labels = pd.DataFrame(final_label_records)

# Operational threshold selected in the previous step
DEPENDENCY_THRESHOLD = 3

final_dependency_labels["dependency_risk_label"] = (
    final_dependency_labels["future_affected_downstream"]
    >= DEPENDENCY_THRESHOLD
).astype(int)

print("Total eligible checkpoints:", len(final_dependency_labels))

print("\nLabel distribution:")
display(
    final_dependency_labels["dependency_risk_label"]
    .value_counts()
)

print("\nProjects with positive labels:")
print(
    final_dependency_labels.loc[
        final_dependency_labels["dependency_risk_label"] == 1,
        "instance_id"
    ].nunique()
)

Total eligible checkpoints: 987

Label distribution:


,count
dependency_risk_label,
0,899
1,88



Projects with positive labels:
10


In [ ]:
# Check for duplicate project-week records
duplicates = final_dependency_labels.duplicated(
    subset=["instance_id", "week_number"]
).sum()

# Check for missing labels
missing_labels = (
    final_dependency_labels["dependency_risk_label"]
    .isna()
    .sum()
)

# Check for incomplete prediction windows
invalid_windows = (
    final_dependency_labels["horizon_end"]
    >
    final_dependency_labels["instance_id"].map(
        project_timeline.set_index(
            "instance_id"
        )["last_observation"]
    )
).sum()

print("Duplicate project-weeks:", duplicates)
print("Missing labels:", missing_labels)
print("Incomplete future windows:", invalid_windows)

assert duplicates == 0
assert missing_labels == 0
assert invalid_windows == 0

print("Structural label validation passed.")

Duplicate project-weeks: 0
Missing labels: 0
Incomplete future windows: 0
Structural label validation passed.


In [ ]:
output_path = (
    GC1_ROOT /"Temporal_snapshots"/ "dependency_escalation_labels.csv"
)

final_dependency_labels.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)

Saved: /content/drive/MyDrive/GC1/Temporal_snapshots/dependency_escalation_labels.csv


## Schedule Risk — Ground-Truth Labelling

Identify future workflow disruption events and create 14-day Schedule Risk labels.

In [ ]:
import pandas as pd

schedule_events = events[
    events["event_type"] == "STATUS_CHANGE"
].copy()

schedule_events["event_dt"] = pd.to_datetime(
    schedule_events["observed_at"],
    utc=True,
    errors="coerce"
)

schedule_events["from_status"] = (
    schedule_events["from_value"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

schedule_events["to_status"] = (
    schedule_events["to_value"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

def classify_schedule_disruption(row):

    old = row["from_status"]
    new = row["to_status"]

    # Ignore no-op transitions
    if not old or not new or old == new:
        return None

    # Previously completed work reopened
    if old in {
        "resolved", "closed", "done", "verified"
    } and new in {
        "reopened", "open", "in progress"
    }:
        return "REOPENED_WORK"

    # Work moved backward
    if old in {
        "in progress",
        "coding in progress",
        "in code review"
    } and new in {
        "open", "new", "reopened"
    }:
        return "WORKFLOW_REGRESSION"

    # Work entered an explicitly blocked state
    if new in {
        "blocked",
        "on hold",
        "waiting for dependency",
        "waiting for infra"
    } and old not in {
        "blocked",
        "on hold",
        "waiting for dependency",
        "waiting for infra"
    }:
        return "BLOCKED_PROGRESS"

    return None


schedule_events["schedule_event_type"] = (
    schedule_events.apply(
        classify_schedule_disruption,
        axis=1
    )
)

schedule_candidates = schedule_events[
    schedule_events["schedule_event_type"].notna()
    &
    schedule_events["event_dt"].notna()
].copy()

print("Candidate schedule-disruption events:",
      len(schedule_candidates))

display(
    schedule_candidates["schedule_event_type"]
    .value_counts()
)

display(
    schedule_candidates[
        [
            "instance_id",
            "source_id",
            "event_dt",
            "from_value",
            "to_value",
            "schedule_event_type"
        ]
    ].head(15)
)

Candidate schedule-disruption events: 1270


,count
schedule_event_type,
REOPENED_WORK,813
WORKFLOW_REGRESSION,230
BLOCKED_PROGRESS,227


,instance_id,source_id,event_dt,from_value,to_value,schedule_event_type
40875,JBEAP-I01,JBEAP-4518,2016-07-20 10:26:26+00:00,Closed,Reopened,REOPENED_WORK
40994,JBEAP-I01,JBEAP-4347,2016-05-01 22:00:30+00:00,Resolved,Reopened,REOPENED_WORK
40999,JBEAP-I01,JBEAP-4347,2016-05-03 02:25:51+00:00,Resolved,Reopened,REOPENED_WORK
41078,JBEAP-I01,JBEAP-4162,2016-05-03 04:26:45+00:00,Resolved,Reopened,REOPENED_WORK
41255,JBEAP-I01,JBEAP-3969,2016-05-31 12:02:09+00:00,Resolved,Reopened,REOPENED_WORK
41266,JBEAP-I01,JBEAP-3969,2016-07-20 01:59:33+00:00,Coding In Progress,New,WORKFLOW_REGRESSION
41302,JBEAP-I01,JBEAP-3862,2016-03-28 10:46:51+00:00,Coding In Progress,Open,WORKFLOW_REGRESSION
41420,JBEAP-I01,JBEAP-3825,2016-08-01 14:26:33+00:00,Resolved,Reopened,REOPENED_WORK
41567,JBEAP-I01,JBEAP-3593,2016-03-09 20:25:48+00:00,Coding In Progress,Open,WORKFLOW_REGRESSION
41702,JBEAP-I01,JBEAP-3459,2016-03-07 08:09:50+00:00,Closed,Reopened,REOPENED_WORK


In [ ]:
schedule_candidates = (
    schedule_candidates
    .sort_values("event_dt")
    .drop_duplicates(
        subset=[
            "instance_id",
            "source_id",
            "event_dt",
            "from_status",
            "to_status"
        ]
    )
    .reset_index(drop=True)
)

# Preserve source event IDs for provenance
schedule_candidates["event_uid"] = (
    schedule_candidates["uid"]
)

print("Unique schedule-disruption events:",
      len(schedule_candidates))

print(
    "Projects with schedule disruptions:",
    schedule_candidates["instance_id"].nunique()
)

display(
    schedule_candidates.groupby("instance_id")
    .agg(
        disruption_events=("event_uid", "count"),
        affected_issues=("source_id", "nunique")
    )
    .reset_index()
    .sort_values(
        "disruption_events",
        ascending=False
    )
)

Unique schedule-disruption events: 1270
Projects with schedule disruptions: 29


,instance_id,disruption_events,affected_issues
12,JBEAP-I01,153,69
16,MC-I01,150,57
10,INFRA-I01,115,33
4,DAOS-I01,92,61
28,WT-I01,85,56
25,SERVER-I01,66,29
1,AMBARI-I01,61,40
11,JBDS-I01,58,32
0,AEROGEAR-I01,52,33
8,FLEX-I01,52,49


In [ ]:
HORIZON_DAYS = 14

schedule_label_records = []

for _, checkpoint in checkpoints_df.iterrows():

    instance_id = checkpoint["instance_id"]
    t = checkpoint["checkpoint_time"]

    horizon_end = t + pd.Timedelta(
        days=HORIZON_DAYS
    )

    if horizon_end > checkpoint["observation_end"]:
        continue

    future_events = schedule_candidates[
        (schedule_candidates["instance_id"] == instance_id)
        &
        (schedule_candidates["event_dt"] > t)
        &
        (schedule_candidates["event_dt"] <= horizon_end)
    ]

    schedule_label_records.append({
        "instance_id": instance_id,
        "week_number": checkpoint["week_number"],
        "checkpoint_time": t,
        "horizon_end": horizon_end,

        "future_schedule_events": len(future_events),

        "future_affected_issues": (
            future_events["source_id"].nunique()
        ),

        "future_reopened_issues": (
            future_events.loc[
                future_events["schedule_event_type"]
                == "REOPENED_WORK",
                "source_id"
            ].nunique()
        ),

        "future_regressed_issues": (
            future_events.loc[
                future_events["schedule_event_type"]
                == "WORKFLOW_REGRESSION",
                "source_id"
            ].nunique()
        ),

        "future_blocked_issues": (
            future_events.loc[
                future_events["schedule_event_type"]
                == "BLOCKED_PROGRESS",
                "source_id"
            ].nunique()
        )
    })

schedule_future_df = pd.DataFrame(
    schedule_label_records
)

print("Eligible schedule checkpoints:",
      len(schedule_future_df))

display(schedule_future_df.head(10))

Eligible schedule checkpoints: 987


,instance_id,week_number,checkpoint_time,horizon_end,future_schedule_events,future_affected_issues,future_reopened_issues,future_regressed_issues,future_blocked_issues
0,AEROGEAR-I01,1,2018-01-24 05:30:01+00:00,2018-02-07 05:30:01+00:00,1,1,0,1,0
1,AEROGEAR-I01,2,2018-01-31 05:30:01+00:00,2018-02-14 05:30:01+00:00,3,2,1,1,0
2,AEROGEAR-I01,3,2018-02-07 05:30:01+00:00,2018-02-21 05:30:01+00:00,4,3,1,2,0
3,AEROGEAR-I01,4,2018-02-14 05:30:01+00:00,2018-02-28 05:30:01+00:00,3,3,0,3,0
4,AEROGEAR-I01,5,2018-02-21 05:30:01+00:00,2018-03-07 05:30:01+00:00,5,3,0,3,0
5,AEROGEAR-I01,6,2018-02-28 05:30:01+00:00,2018-03-14 05:30:01+00:00,10,7,1,6,0
6,AEROGEAR-I01,7,2018-03-07 05:30:01+00:00,2018-03-21 05:30:01+00:00,8,6,2,4,0
7,AEROGEAR-I01,8,2018-03-14 05:30:01+00:00,2018-03-28 05:30:01+00:00,3,2,1,1,0
8,AEROGEAR-I01,9,2018-03-21 05:30:01+00:00,2018-04-04 05:30:01+00:00,4,2,1,1,0
9,AEROGEAR-I01,10,2018-03-28 05:30:01+00:00,2018-04-11 05:30:01+00:00,4,3,2,1,0


In [ ]:
print("Future schedule-disruption distribution:")

display(
    schedule_future_df[
        [
            "future_schedule_events",
            "future_affected_issues",
            "future_reopened_issues",
            "future_regressed_issues",
            "future_blocked_issues"
        ]
    ].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]
    )
)

threshold_results = []

for threshold in [1, 2, 3, 4, 5, 6, 8, 10]:

    positive = (
        schedule_future_df["future_affected_issues"]
        >= threshold
    )

    threshold_results.append({
        "threshold": threshold,
        "positive_checkpoints": int(positive.sum()),
        "negative_checkpoints": int((~positive).sum()),
        "positive_pct": round(
            positive.mean() * 100, 2
        ),
        "positive_projects": (
            schedule_future_df.loc[
                positive, "instance_id"
            ].nunique()
        )
    })

schedule_threshold_df = pd.DataFrame(
    threshold_results
)

print("\nSchedule threshold sensitivity:")
display(schedule_threshold_df)

Future schedule-disruption distribution:


,future_schedule_events,future_affected_issues,future_reopened_issues,future_regressed_issues,future_blocked_issues
count,987.000000,987.000000,987.000000,987.000000,987.000000
mean,2.345491,1.747720,1.115502,0.374873,0.300912
std,3.898976,2.277454,1.861000,0.818954,1.063706
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,1.000000,1.000000,0.000000,0.000000
75%,3.000000,3.000000,2.000000,0.000000,0.000000
90%,6.000000,4.000000,3.000000,1.000000,1.000000
95%,8.000000,6.000000,4.000000,2.000000,3.000000
max,45.000000,24.000000,23.000000,6.000000,9.000000



Schedule threshold sensitivity:


,threshold,positive_checkpoints,negative_checkpoints,positive_pct,positive_projects
0,1,623,364,63.12,29
1,2,406,581,41.13,27
2,3,265,722,26.85,24
3,4,156,831,15.81,19
4,5,89,898,9.02,17
5,6,63,924,6.38,13
6,8,24,963,2.43,7
7,10,11,976,1.11,5


In [ ]:
# ============================================================
# SCHEDULE RISK — CONSOLIDATED VALIDATION
# ============================================================

import pandas as pd

validation_results = []

def add_check(name, passed, details):
    validation_results.append({
        "check": name,
        "status": "PASS" if passed else "FAIL",
        "details": details
    })

# 1. Project coverage
project_count = schedule_future_df["instance_id"].nunique()

add_check(
    "All 30 projects represented",
    project_count == 30,
    f"{project_count} projects"
)

# 2. Expected checkpoint alignment
dependency_keys = set(
    zip(
        final_dependency_labels["instance_id"],
        final_dependency_labels["week_number"]
    )
)

schedule_keys = set(
    zip(
        schedule_future_df["instance_id"],
        schedule_future_df["week_number"]
    )
)

add_check(
    "Matches Dependency Risk checkpoints",
    schedule_keys == dependency_keys,
    f"{len(schedule_keys)} schedule checkpoints"
)

# 3. Duplicate records
duplicate_count = schedule_future_df.duplicated(
    subset=["instance_id", "week_number"]
).sum()

add_check(
    "No duplicate project-week records",
    duplicate_count == 0,
    f"{duplicate_count} duplicates"
)

# 4. Missing values
required_cols = [
    "instance_id",
    "week_number",
    "checkpoint_time",
    "horizon_end",
    "future_schedule_events",
    "future_affected_issues"
]

missing_count = (
    schedule_future_df[required_cols]
    .isna()
    .sum()
    .sum()
)

add_check(
    "No missing required values",
    missing_count == 0,
    f"{missing_count} missing values"
)

# 5. Prediction horizon
horizon_days = (
    schedule_future_df["horizon_end"]
    - schedule_future_df["checkpoint_time"]
).dt.total_seconds() / 86400

invalid_horizons = (
    (horizon_days != 14)
).sum()

add_check(
    "All horizons are 14 days",
    invalid_horizons == 0,
    f"{invalid_horizons} invalid horizons"
)

# 6. Future observation boundaries
project_end = (
    checkpoints_df.groupby("instance_id")[
        "observation_end"
    ].max()
)

mapped_end = schedule_future_df[
    "instance_id"
].map(project_end)

invalid_windows = (
    schedule_future_df["horizon_end"]
    > mapped_end
).sum()

add_check(
    "No incomplete future windows",
    invalid_windows == 0,
    f"{invalid_windows} invalid windows"
)

# 7. Event timestamps
missing_event_times = (
    schedule_candidates["event_dt"]
    .isna()
    .sum()
)

add_check(
    "All candidate events timestamped",
    missing_event_times == 0,
    f"{missing_event_times} missing timestamps"
)

# 8. Source event identifiers
missing_event_ids = (
    schedule_candidates["event_uid"]
    .isna()
    .sum()
)

add_check(
    "All events have provenance IDs",
    missing_event_ids == 0,
    f"{missing_event_ids} missing IDs"
)

# 9. Event ID uniqueness
duplicate_event_ids = (
    schedule_candidates["event_uid"]
    .duplicated()
    .sum()
)

add_check(
    "Unique event identifiers",
    duplicate_event_ids == 0,
    f"{duplicate_event_ids} duplicate IDs"
)

# 10. Valid outcome counts
count_columns = [
    "future_schedule_events",
    "future_affected_issues",
    "future_reopened_issues",
    "future_regressed_issues",
    "future_blocked_issues"
]

negative_counts = (
    schedule_future_df[count_columns] < 0
).sum().sum()

add_check(
    "No negative outcome counts",
    negative_counts == 0,
    f"{negative_counts} negative values"
)

# 11. Unique affected issues cannot exceed event count
invalid_affected = (
    schedule_future_df["future_affected_issues"]
    > schedule_future_df["future_schedule_events"]
).sum()

add_check(
    "Affected issues <= event count",
    invalid_affected == 0,
    f"{invalid_affected} invalid records"
)

# 12. No-op status transitions
noop_events = (
    schedule_candidates["from_status"]
    == schedule_candidates["to_status"]
).sum()

add_check(
    "No no-op transitions",
    noop_events == 0,
    f"{noop_events} no-op events"
)

# 13. Event source integrity
known_event_ids = set(events["uid"].dropna())

invalid_source_ids = (
    ~schedule_candidates["event_uid"]
    .isin(known_event_ids)
).sum()

add_check(
    "All events traceable to Phase 1 KG",
    invalid_source_ids == 0,
    f"{invalid_source_ids} invalid source IDs"
)

# 14. Project distribution
projects_with_events = (
    schedule_candidates["instance_id"].nunique()
)

add_check(
    "Schedule candidates span multiple projects",
    projects_with_events >= 2,
    f"{projects_with_events} projects"
)

# ------------------------------------------------------------
# VALIDATION REPORT
# ------------------------------------------------------------

validation_df = pd.DataFrame(validation_results)

display(validation_df)

failed_checks = validation_df[
    validation_df["status"] == "FAIL"
]

print("\n" + "=" * 60)

if failed_checks.empty:
    print("ALL STRUCTURAL VALIDATIONS PASSED")
else:
    print(
        f"{len(failed_checks)} VALIDATION CHECKS FAILED"
    )

print("=" * 60)

print("\nSchedule candidate events:",
      len(schedule_candidates))

print("Eligible checkpoints:",
      len(schedule_future_df))

print("Projects with schedule events:",
      projects_with_events)

print("\nCandidate event distribution:")
display(
    schedule_candidates["schedule_event_type"]
    .value_counts()
)

,check,status,details
0,All 30 projects represented,PASS,30 projects
1,Matches Dependency Risk checkpoints,PASS,987 schedule checkpoints
2,No duplicate project-week records,PASS,0 duplicates
3,No missing required values,PASS,0 missing values
4,All horizons are 14 days,PASS,0 invalid horizons
5,No incomplete future windows,PASS,0 invalid windows
6,All candidate events timestamped,PASS,0 missing timestamps
7,All events have provenance IDs,PASS,0 missing IDs
8,Unique event identifiers,PASS,0 duplicate IDs
9,No negative outcome counts,PASS,0 negative values



ALL STRUCTURAL VALIDATIONS PASSED

Schedule candidate events: 1270
Eligible checkpoints: 987
Projects with schedule events: 29

Candidate event distribution:


,count
schedule_event_type,
REOPENED_WORK,813
WORKFLOW_REGRESSION,230
BLOCKED_PROGRESS,227


In [ ]:
# ============================================================
# STEP 2.5.6 — GENERATE SCHEDULE RISK LABELS
# ============================================================

import pandas as pd

SCHEDULE_THRESHOLD = 3

final_schedule_labels = schedule_future_df.copy()

final_schedule_labels["schedule_risk_label"] = (
    final_schedule_labels["future_affected_issues"]
    >= SCHEDULE_THRESHOLD
).astype(int)

print("Total labelled checkpoints:", len(final_schedule_labels))

print("\nSchedule Risk label distribution:")
display(
    final_schedule_labels["schedule_risk_label"]
    .value_counts()
)

print("\nProjects with positive labels:")
print(
    final_schedule_labels.loc[
        final_schedule_labels["schedule_risk_label"] == 1,
        "instance_id"
    ].nunique()
)

print("\nPositive percentage:")
print(
    round(
        final_schedule_labels["schedule_risk_label"].mean() * 100,
        2
    ),
    "%"
)

Total labelled checkpoints: 987

Schedule Risk label distribution:


,count
schedule_risk_label,
0,722
1,265



Projects with positive labels:
24

Positive percentage:
26.85 %


In [ ]:
# ============================================================
# COMPARE SCHEDULE AND DEPENDENCY LABELS
# ============================================================

label_keys = [
    "instance_id",
    "week_number",
    "checkpoint_time"
]

combined_risk_labels = (
    final_schedule_labels[
        label_keys + ["schedule_risk_label"]
    ]
    .merge(
        final_dependency_labels[
            label_keys + ["dependency_risk_label"]
        ],
        on=label_keys,
        how="inner",
        validate="one_to_one"
    )
)

print("Combined project-week records:",
      len(combined_risk_labels))

print("\nRisk label combinations:")

display(
    combined_risk_labels.groupby(
        ["schedule_risk_label", "dependency_risk_label"]
    )
    .size()
    .reset_index(name="count")
)

print("\nBoth risks positive:")

print(
    (
        (combined_risk_labels["schedule_risk_label"] == 1)
        &
        (combined_risk_labels["dependency_risk_label"] == 1)
    ).sum()
)

Combined project-week records: 987

Risk label combinations:


,schedule_risk_label,dependency_risk_label,count
0,0,0,691
1,0,1,31
2,1,0,208
3,1,1,57



Both risks positive:
57


In [ ]:
# ============================================================
# SCHEDULE RISK — FINAL CONSOLIDATED VALIDATION
# ============================================================

import pandas as pd

checks = []

def check(name, condition, detail):
    checks.append({
        "validation": name,
        "status": "PASS" if condition else "FAIL",
        "details": detail
    })

keys = ["instance_id", "week_number", "checkpoint_time"]

# 1. Total records
check(
    "987 labelled checkpoints",
    len(final_schedule_labels) == 987,
    len(final_schedule_labels)
)

# 2. Project coverage
check(
    "All 30 projects represented",
    final_schedule_labels["instance_id"].nunique() == 30,
    final_schedule_labels["instance_id"].nunique()
)

# 3. Unique checkpoints
duplicates = final_schedule_labels.duplicated(keys).sum()

check(
    "No duplicate checkpoints",
    duplicates == 0,
    duplicates
)

# 4. Missing labels
missing = final_schedule_labels["schedule_risk_label"].isna().sum()

check(
    "No missing labels",
    missing == 0,
    missing
)

# 5. Binary labels
valid_binary = final_schedule_labels[
    "schedule_risk_label"
].isin([0, 1]).all()

check(
    "All labels binary",
    valid_binary,
    "0 or 1"
)

# 6. Label-rule consistency
expected = (
    final_schedule_labels["future_affected_issues"]
    >= SCHEDULE_THRESHOLD
).astype(int)

mismatches = (
    expected != final_schedule_labels["schedule_risk_label"]
).sum()

check(
    "Labels follow threshold rule",
    mismatches == 0,
    mismatches
)

# 7. Horizon consistency
horizon = (
    final_schedule_labels["horizon_end"]
    - final_schedule_labels["checkpoint_time"]
)

invalid_horizons = (
    horizon != pd.Timedelta(days=14)
).sum()

check(
    "14-day prediction horizon",
    invalid_horizons == 0,
    invalid_horizons
)

# 8. Complete future windows
end_lookup = checkpoints_df.groupby("instance_id")[
    "observation_end"
].max()

invalid_windows = (
    final_schedule_labels["horizon_end"]
    >
    final_schedule_labels["instance_id"].map(end_lookup)
).sum()

check(
    "Complete future windows",
    invalid_windows == 0,
    invalid_windows
)

# 9. Exact dependency checkpoint alignment
schedule_index = pd.MultiIndex.from_frame(
    final_schedule_labels[keys]
)

dependency_index = pd.MultiIndex.from_frame(
    final_dependency_labels[keys]
)

check(
    "Matches Dependency Risk checkpoints",
    schedule_index.equals(dependency_index)
    or (
        len(schedule_index) == len(dependency_index)
        and set(schedule_index) == set(dependency_index)
    ),
    f"{len(schedule_index)} schedule checkpoints"
)

# 10. Combined table completeness
check(
    "Combined table has 987 records",
    len(combined_risk_labels) == 987,
    len(combined_risk_labels)
)

# 11. No missing combined labels
combined_missing = combined_risk_labels[
    ["schedule_risk_label", "dependency_risk_label"]
].isna().sum().sum()

check(
    "No missing combined labels",
    combined_missing == 0,
    combined_missing
)

# 12. Event provenance
missing_uids = schedule_candidates["event_uid"].isna().sum()

check(
    "Schedule events have provenance IDs",
    missing_uids == 0,
    missing_uids
)

# ------------------------------------------------------------
# REPORT
# ------------------------------------------------------------

validation_report = pd.DataFrame(checks)

display(validation_report)

failed = validation_report[
    validation_report["status"] == "FAIL"
]

print("\n" + "=" * 60)

if failed.empty:
    print("ALL FINAL SCHEDULE LABEL VALIDATIONS PASSED")
else:
    print(f"{len(failed)} VALIDATIONS FAILED")

print("=" * 60)

print("\nFinal Schedule Risk distribution:")

display(
    final_schedule_labels["schedule_risk_label"]
    .value_counts()
)

,validation,status,details
0,987 labelled checkpoints,PASS,987
1,All 30 projects represented,PASS,30
2,No duplicate checkpoints,PASS,0
3,No missing labels,PASS,0
4,All labels binary,PASS,0 or 1
5,Labels follow threshold rule,PASS,0
6,14-day prediction horizon,PASS,0
7,Complete future windows,PASS,0
8,Matches Dependency Risk checkpoints,PASS,987 schedule checkpoints
9,Combined table has 987 records,PASS,987



ALL FINAL SCHEDULE LABEL VALIDATIONS PASSED

Final Schedule Risk distribution:


,count
schedule_risk_label,
0,722
1,265


In [ ]:
# Save Schedule Risk labels
schedule_output = GC1_ROOT / "Temporal_snapshots"/"schedule_risk_labels.csv"

final_schedule_labels.to_csv(
    schedule_output,
    index=False
)

# Save combined Schedule + Dependency labels
combined_output = GC1_ROOT /"Temporal_snapshots"/ "phase2_partial_risk_labels.csv"

combined_risk_labels.to_csv(
    combined_output,
    index=False
)

print("Saved:", schedule_output)
print("Saved:", combined_output)

Saved: /content/drive/MyDrive/GC1/Temporal_snapshots/schedule_risk_labels.csv
Saved: /content/drive/MyDrive/GC1/Temporal_snapshots/phase2_partial_risk_labels.csv


## Scope Risk — Ground-Truth Labelling

Classify structural scope events, audit administrative changes, and export the final historical scope-event dataset.

### 4.1 Candidate scope events

In [ ]:
import pandas as pd

SCOPE_EVENT_TYPES = [
    "DESCRIPTION_CHANGE",
    "SUMMARY_CHANGE",
    "ISSUE_TYPE_CHANGE",
    "STORY_POINT_CHANGE",
    "EPIC_CHILD_CHANGE",
    "EPIC_LINK_CHANGE",
    "EPIC_NAME_CHANGE",
    "EPIC_STATUS_CHANGE",
    "PRIORITY_CHANGE"
]

scope_events = events[
    events["event_type"].isin(SCOPE_EVENT_TYPES)
].copy()

scope_events["event_dt"] = pd.to_datetime(
    scope_events["observed_at"],
    utc=True,
    errors="coerce"
)

scope_events = scope_events.dropna(
    subset=["event_dt"]
)

print("Total scope-related candidate events:",
      len(scope_events))

print("\nEvents by type:")

display(
    scope_events["event_type"]
    .value_counts()
    .reset_index(name="count")
)

print("\nProjects represented:",
      scope_events["instance_id"].nunique())

display(
    scope_events[
        [
            "instance_id",
            "source_id",
            "event_type",
            "from_value",
            "to_value",
            "event_dt"
        ]
    ].head(15)
)

Total scope-related candidate events: 3863

Events by type:


,event_type,count
0,DESCRIPTION_CHANGE,1425
1,SUMMARY_CHANGE,694
2,EPIC_CHILD_CHANGE,410
3,EPIC_LINK_CHANGE,367
4,PRIORITY_CHANGE,364
5,STORY_POINT_CHANGE,355
6,ISSUE_TYPE_CHANGE,190
7,EPIC_NAME_CHANGE,38
8,EPIC_STATUS_CHANGE,20



Projects represented: 30


,instance_id,source_id,event_type,from_value,to_value,event_dt
40528,JBEAP-I01,JBEAP-5248,SUMMARY_CHANGE,Excluded contexts not specific to a host shoul...,(7.0.z) Excluded contexts not specific to a ho...,2016-07-29 00:22:33+00:00
40555,JBEAP-I01,JBEAP-5161,DESCRIPTION_CHANGE,This is a tracker JIRA fr bean validation rela...,This is a tracker JIRA for bean validation rel...,2016-07-07 02:19:54+00:00
40619,JBEAP-I01,JBEAP-4962,DESCRIPTION_CHANGE,EAP7 Management CLI Guide issues tracker\n\nht...,EAP7 Management CLI Guide issues tracker\n\nht...,2016-06-14 09:27:56+00:00
40723,JBEAP-I01,JBEAP-4742,SUMMARY_CHANGE,Duplicate messages in replicated HA topology w...,(7.0.z) Duplicate messages in replicated HA to...,2016-07-28 13:08:19+00:00
40744,JBEAP-I01,JBEAP-4617,SUMMARY_CHANGE,Docs for configuraing EAP on Azure,Docs for configuring EAP on Azure,2016-05-18 19:42:04+00:00
40746,JBEAP-I01,JBEAP-4617,PRIORITY_CHANGE,Major,Blocker,2016-06-21 08:41:07+00:00
40791,JBEAP-I01,JBEAP-4530,DESCRIPTION_CHANGE,When defining new broadcast-group in Configura...,When defining new broadcast-group in Configura...,2016-05-05 06:02:16+00:00
40792,JBEAP-I01,JBEAP-4530,SUMMARY_CHANGE,Broadcast group is possible to create with jus...,Broadcast/Discovery group is possible to creat...,2016-05-05 06:03:50+00:00
40819,JBEAP-I01,JBEAP-4519,DESCRIPTION_CHANGE,Tracker bug for jboss-ec2-eap for EAP 7.0.0,Tracker bug for jboss-ec2-eap for EAP 7.0.1,2016-05-04 11:54:01+00:00
40917,JBEAP-I01,JBEAP-4453,SUMMARY_CHANGE,Document for RN,Document JBEAP-3998 for RN,2016-06-21 07:50:14+00:00


In [ ]:
from difflib import SequenceMatcher

text_scope_events = scope_events[
    scope_events["event_type"].isin([
        "DESCRIPTION_CHANGE",
        "SUMMARY_CHANGE"
    ])
].copy()

def calculate_text_change(old, new):

    old = "" if pd.isna(old) else str(old)
    new = "" if pd.isna(new) else str(new)

    old = " ".join(old.split())
    new = " ".join(new.split())

    if old == new:
        return pd.Series({
            "similarity": 1.0,
            "length_change": 0,
            "relative_length_change": 0.0,
            "change_magnitude": 0.0
        })

    similarity = SequenceMatcher(
        None, old, new,
        autojunk=False
    ).ratio()

    length_change = len(new) - len(old)

    relative_length_change = (
        length_change / max(len(old), 1)
    )

    return pd.Series({
        "similarity": similarity,
        "length_change": length_change,
        "relative_length_change": relative_length_change,
        "change_magnitude": 1 - similarity
    })


text_change_metrics = text_scope_events.apply(
    lambda row: calculate_text_change(
        row["from_value"],
        row["to_value"]
    ),
    axis=1
)

text_scope_events = pd.concat(
    [
        text_scope_events.reset_index(drop=True),
        text_change_metrics.reset_index(drop=True)
    ],
    axis=1
)

print("Text-change events:", len(text_scope_events))

display(
    text_scope_events[
        [
            "similarity",
            "length_change",
            "relative_length_change",
            "change_magnitude"
        ]
    ].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]
    )
)

print("\nLargest textual changes:")

display(
    text_scope_events.sort_values(
        "change_magnitude",
        ascending=False
    )[
        [
            "instance_id",
            "source_id",
            "event_type",
            "similarity",
            "length_change",
            "change_magnitude"
        ]
    ].head(20)
)

Text-change events: 2119


,similarity,length_change,relative_length_change,change_magnitude
count,2119.000000,2119.000000,2119.000000,2119.000000
mean,0.760529,155.716376,40.826238,0.239471
std,0.293111,3052.023894,461.444172,0.293111
min,0.000000,-33921.000000,-1.000000,0.000000
25%,0.639815,-1.000000,-0.000700,0.019953
50%,0.892857,12.000000,0.040487,0.107143
75%,0.980047,85.000000,0.353231,0.360185
90%,0.996935,327.400000,1.455433,0.730203
95%,0.999264,583.800000,28.328360,1.000000
max,1.000000,126254.000000,14851.000000,1.000000



Largest textual changes:


,instance_id,source_id,event_type,similarity,length_change,change_magnitude
443,AMBARI-I01,AMBARI-15655,DESCRIPTION_CHANGE,0.0,329.0,1.0
445,AMBARI-I01,AMBARI-15538,DESCRIPTION_CHANGE,0.0,740.0,1.0
402,AMBARI-I01,AMBARI-18692,DESCRIPTION_CHANGE,0.0,243.0,1.0
407,AMBARI-I01,AMBARI-18423,DESCRIPTION_CHANGE,0.0,76.0,1.0
416,AMBARI-I01,AMBARI-17951,DESCRIPTION_CHANGE,0.0,1313.0,1.0
1941,RUBY-I01,RUBY-2645,DESCRIPTION_CHANGE,0.0,3423.0,1.0
1955,EVG-I01,EVG-13905,DESCRIPTION_CHANGE,0.0,65.0,1.0
401,AMBARI-I01,AMBARI-18836,DESCRIPTION_CHANGE,0.0,412.0,1.0
694,OSSRH-I01,OSSRH-74040,DESCRIPTION_CHANGE,0.0,56.0,1.0
714,INFRA-I01,INFRA-15138,DESCRIPTION_CHANGE,0.0,77.0,1.0


In [ ]:
STRUCTURAL_SCOPE_TYPES = [
    "EPIC_CHILD_CHANGE",
    "EPIC_LINK_CHANGE",
    "ISSUE_TYPE_CHANGE",
    "STORY_POINT_CHANGE"
]

structural_scope_events = scope_events[
    scope_events["event_type"].isin(
        STRUCTURAL_SCOPE_TYPES
    )
].copy()

print("Structural scope candidate events:",
      len(structural_scope_events))

display(
    structural_scope_events["event_type"]
    .value_counts()
)

print("\nProject distribution:")

display(
    structural_scope_events.groupby("instance_id")
    .agg(
        structural_events=("uid", "count"),
        affected_issues=("source_id", "nunique")
    )
    .reset_index()
    .sort_values(
        "structural_events",
        ascending=False
    )
)

print("\nSample structural changes:")

display(
    structural_scope_events[
        [
            "instance_id",
            "source_id",
            "event_type",
            "from_value",
            "to_value",
            "event_dt"
        ]
    ].head(25)
)

Structural scope candidate events: 1322


,count
event_type,
EPIC_CHILD_CHANGE,410
EPIC_LINK_CHANGE,367
STORY_POINT_CHANGE,355
ISSUE_TYPE_CHANGE,190



Project distribution:


,instance_id,structural_events,affected_issues
21,RUBY-I01,160,38
4,DAOS-I01,118,95
5,DOCS-I01,111,45
25,WT-I01,106,64
0,AEROGEAR-I01,105,40
11,JBDS-I01,96,47
2,BXMSDOC-I01,86,47
7,EVG-I01,81,57
16,MONGOID-I01,61,33
17,NEXUS-I01,56,17



Sample structural changes:


,instance_id,source_id,event_type,from_value,to_value,event_dt
41782,JBEAP-I01,JBEAP-3444,ISSUE_TYPE_CHANGE,Enhancement,Bug,2016-02-16 11:43:10+00:00
41886,JBEAP-I01,JBEAP-3426,ISSUE_TYPE_CHANGE,Feature Request,Sub-task,2016-03-22 02:12:44+00:00
42677,JBEAP-I01,JBEAP-3028,ISSUE_TYPE_CHANGE,Feature Request,Component Upgrade,2016-01-26 04:47:43+00:00
43322,JBEAP-I01,JBEAP-2786,ISSUE_TYPE_CHANGE,Bug,Sub-task,2016-01-14 09:42:31+00:00
44162,JBEAP-I01,JBEAP-2544,ISSUE_TYPE_CHANGE,Feature Request,Bug,2016-01-19 02:11:19+00:00
44207,JBEAP-I01,JBEAP-2532,ISSUE_TYPE_CHANGE,Feature Request,Bug,2016-01-04 07:13:20+00:00
45680,JBEAP-I01,JBEAP-1905,ISSUE_TYPE_CHANGE,Feature Request,Enhancement,2015-12-08 11:39:29+00:00
45769,JBEAP-I01,JBEAP-1848,ISSUE_TYPE_CHANGE,Documentation,Bug,2015-12-01 12:44:05+00:00
46010,JBEAP-I01,JBEAP-1686,ISSUE_TYPE_CHANGE,Documentation,Bug,2015-12-01 15:32:34+00:00
46090,JBEAP-I01,JBEAP-1681,ISSUE_TYPE_CHANGE,Documentation,Bug,2015-12-01 12:27:34+00:00


In [ ]:
def normalize_change_value(value):

    if value is None:
        return ""

    return " ".join(
        str(value).strip().lower().split()
    )


scope_events["old_normalized"] = (
    scope_events["from_value"]
    .apply(normalize_change_value)
)

scope_events["new_normalized"] = (
    scope_events["to_value"]
    .apply(normalize_change_value)
)

scope_events["has_actual_change"] = (
    scope_events["old_normalized"]
    != scope_events["new_normalized"]
)

scope_events["has_both_values"] = (
    scope_events["old_normalized"].ne("")
    &
    scope_events["new_normalized"].ne("")
)

print("Events with changed values:",
      scope_events["has_actual_change"].sum())

print("Events with both before/after values:",
      scope_events["has_both_values"].sum())

print("\nScreening summary by event type:")

display(
    scope_events.groupby("event_type")
    .agg(
        total=("uid", "count"),
        changed=("has_actual_change", "sum"),
        both_values=("has_both_values", "sum"),
        projects=("instance_id", "nunique")
    )
    .reset_index()
)

Events with changed values: 3799
Events with both before/after values: 2735

Screening summary by event type:


,event_type,total,changed,both_values,projects
0,DESCRIPTION_CHANGE,1425,1380,1315,29
1,EPIC_CHILD_CHANGE,410,410,0,14
2,EPIC_LINK_CHANGE,367,367,71,17
3,EPIC_NAME_CHANGE,38,37,25,12
4,EPIC_STATUS_CHANGE,20,20,8,9
5,ISSUE_TYPE_CHANGE,190,190,190,25
6,PRIORITY_CHANGE,364,364,353,27
7,STORY_POINT_CHANGE,355,355,79,9
8,SUMMARY_CHANGE,694,676,694,29


In [ ]:
# ============================================================
# SCOPE RISK — CONSOLIDATED VALIDATION
# ============================================================

import pandas as pd

checks = []

def add_check(name, condition, details):
    checks.append({
        "validation": name,
        "status": "PASS" if condition else "FAIL",
        "details": details
    })

# 1. Required schema
required_columns = [
    "uid", "instance_id", "source_id",
    "event_type", "event_dt",
    "from_value", "to_value"
]

missing_columns = [
    c for c in required_columns
    if c not in scope_events.columns
]

add_check(
    "Required event fields available",
    len(missing_columns) == 0,
    str(missing_columns)
)

# 2. Event timestamps
missing_times = scope_events["event_dt"].isna().sum()

add_check(
    "No missing event timestamps",
    missing_times == 0,
    int(missing_times)
)

# 3. Event IDs
missing_ids = scope_events["uid"].isna().sum()

add_check(
    "No missing event IDs",
    missing_ids == 0,
    int(missing_ids)
)

# 4. Duplicate event IDs
duplicate_ids = scope_events["uid"].duplicated().sum()

add_check(
    "Unique event IDs",
    duplicate_ids == 0,
    int(duplicate_ids)
)

# 5. Valid project instances
valid_projects = set(
    checkpoints_df["instance_id"]
)

invalid_projects = (
    ~scope_events["instance_id"].isin(valid_projects)
).sum()

add_check(
    "All events belong to known projects",
    invalid_projects == 0,
    int(invalid_projects)
)

# 6. Event types
invalid_types = (
    ~scope_events["event_type"].isin(
        SCOPE_EVENT_TYPES
    )
).sum()

add_check(
    "Only candidate scope event types",
    invalid_types == 0,
    int(invalid_types)
)

# 7. Phase 1 provenance
known_event_ids = set(events["uid"].dropna())

invalid_provenance = (
    ~scope_events["uid"].isin(known_event_ids)
).sum()

add_check(
    "All events traceable to Phase 1",
    invalid_provenance == 0,
    int(invalid_provenance)
)

# 8. Text-change metric range
valid_similarity = (
    text_scope_events["similarity"]
    .between(0, 1)
    .all()
)

add_check(
    "Text similarity between 0 and 1",
    valid_similarity,
    "Range [0, 1]"
)

# 9. Change magnitude range
valid_magnitude = (
    text_scope_events["change_magnitude"]
    .between(0, 1)
    .all()
)

add_check(
    "Text-change magnitude between 0 and 1",
    valid_magnitude,
    "Range [0, 1]"
)

# 10. Structural events
valid_structural = (
    structural_scope_events["event_type"]
    .isin(STRUCTURAL_SCOPE_TYPES)
    .all()
)

add_check(
    "Structural event types valid",
    valid_structural,
    len(structural_scope_events)
)

# 11. Actual change indicator
valid_change_flags = (
    scope_events["has_actual_change"]
    .isin([True, False])
    .all()
)

add_check(
    "Actual-change flags valid",
    valid_change_flags,
    "Boolean values"
)

# 12. Project coverage
project_count = scope_events["instance_id"].nunique()

add_check(
    "Scope events span multiple projects",
    project_count >= 2,
    f"{project_count} projects"
)

# 13. Checkpoint baseline alignment
checkpoint_count = len(
    final_schedule_labels
)

dependency_count = len(
    final_dependency_labels
)

add_check(
    "Schedule and Dependency checkpoints aligned",
    checkpoint_count == dependency_count == 987,
    f"Schedule={checkpoint_count}, Dependency={dependency_count}"
)

# 14. No future data introduced into features
# This step produces candidate outcome events only.
# Actual leakage validation must occur after feature engineering.
add_check(
    "No predictive features generated in this step",
    True,
    "Outcome-candidate inspection only"
)

# ------------------------------------------------------------
# REPORT
# ------------------------------------------------------------

scope_validation_df = pd.DataFrame(checks)

display(scope_validation_df)

failed = scope_validation_df[
    scope_validation_df["status"] == "FAIL"
]

print("\n" + "=" * 60)

if failed.empty:
    print("ALL SCOPE INSPECTION VALIDATIONS PASSED")
else:
    print(f"{len(failed)} VALIDATIONS FAILED")

print("=" * 60)

print("\nTotal candidate events:", len(scope_events))
print("Projects represented:", project_count)

print("\nEvent-type distribution:")
display(scope_events["event_type"].value_counts())

,validation,status,details
0,Required event fields available,PASS,[]
1,No missing event timestamps,PASS,0
2,No missing event IDs,PASS,0
3,Unique event IDs,PASS,0
4,All events belong to known projects,PASS,0
5,Only candidate scope event types,PASS,0
6,All events traceable to Phase 1,PASS,0
7,Text similarity between 0 and 1,PASS,"Range [0, 1]"
8,Text-change magnitude between 0 and 1,PASS,"Range [0, 1]"
9,Structural event types valid,PASS,1322



ALL SCOPE INSPECTION VALIDATIONS PASSED

Total candidate events: 3863
Projects represented: 30

Event-type distribution:


,count
event_type,
DESCRIPTION_CHANGE,1425
SUMMARY_CHANGE,694
EPIC_CHILD_CHANGE,410
EPIC_LINK_CHANGE,367
PRIORITY_CHANGE,364
STORY_POINT_CHANGE,355
ISSUE_TYPE_CHANGE,190
EPIC_NAME_CHANGE,38
EPIC_STATUS_CHANGE,20


### 4.2 Structural scope classification

In [ ]:
import pandas as pd

def classify_structural_scope(row):

    event_type = row["event_type"]

    old = row["old_normalized"]
    new = row["new_normalized"]

    if old == new:
        return "NO_CHANGE"

    if event_type == "EPIC_CHILD_CHANGE":

        if not old and new:
            return "EPIC_CHILD_ADDED"

        if old and not new:
            return "EPIC_CHILD_REMOVED"

        return "EPIC_CHILD_MODIFIED"

    if event_type == "EPIC_LINK_CHANGE":

        if not old and new:
            return "EPIC_ASSIGNED"

        if old and not new:
            return "EPIC_UNASSIGNED"

        return "EPIC_REASSIGNED"

    if event_type == "ISSUE_TYPE_CHANGE":
        return "ISSUE_RECLASSIFIED"

    if event_type == "STORY_POINT_CHANGE":

        if not old and new:
            return "INITIAL_ESTIMATION"

        if old and new:
            return "ESTIMATE_REVISED"

        return "ESTIMATE_REMOVED"

    return "OTHER"


structural_candidates = scope_events[
    scope_events["event_type"].isin(
        STRUCTURAL_SCOPE_TYPES
    )
].copy()

structural_candidates["structural_class"] = (
    structural_candidates.apply(
        classify_structural_scope,
        axis=1
    )
)

print("Structural change classifications:")

display(
    structural_candidates["structural_class"]
    .value_counts()
)

print("\nProjects by structural class:")

display(
    structural_candidates.groupby("structural_class")
    .agg(
        events=("uid", "count"),
        projects=("instance_id", "nunique"),
        affected_issues=("source_id", "nunique")
    )
    .reset_index()
)

Structural change classifications:


,count
structural_class,
EPIC_CHILD_ADDED,374
INITIAL_ESTIMATION,263
EPIC_ASSIGNED,247
ISSUE_RECLASSIFIED,190
ESTIMATE_REVISED,79
EPIC_REASSIGNED,71
EPIC_UNASSIGNED,49
EPIC_CHILD_REMOVED,36
ESTIMATE_REMOVED,13



Projects by structural class:


,structural_class,events,projects,affected_issues
0,EPIC_ASSIGNED,247,17,244
1,EPIC_CHILD_ADDED,374,14,43
2,EPIC_CHILD_REMOVED,36,9,16
3,EPIC_REASSIGNED,71,12,56
4,EPIC_UNASSIGNED,49,13,48
5,ESTIMATE_REMOVED,13,3,12
6,ESTIMATE_REVISED,79,7,66
7,INITIAL_ESTIMATION,263,9,258
8,ISSUE_RECLASSIFIED,190,25,168


In [ ]:
story_changes = structural_candidates[
    structural_candidates["event_type"]
    == "STORY_POINT_CHANGE"
].copy()

def parse_story_points(value):

    try:
        return float(value)
    except (ValueError, TypeError):
        return None


story_changes["old_points"] = (
    story_changes["from_value"]
    .apply(parse_story_points)
)

story_changes["new_points"] = (
    story_changes["to_value"]
    .apply(parse_story_points)
)

story_changes["point_difference"] = (
    story_changes["new_points"]
    - story_changes["old_points"]
)

revised_estimates = story_changes[
    story_changes["structural_class"]
    == "ESTIMATE_REVISED"
].copy()

print("Total story-point changes:", len(story_changes))

print("Actual estimate revisions:",
      len(revised_estimates))

print("\nPoint difference distribution:")

display(
    revised_estimates["point_difference"]
    .describe()
)

print("\nEstimate increases:")

display(
    revised_estimates[
        revised_estimates["point_difference"] > 0
    ][
        [
            "instance_id",
            "source_id",
            "old_points",
            "new_points",
            "point_difference"
        ]
    ].head(20)
)

Total story-point changes: 355
Actual estimate revisions: 79

Point difference distribution:


,point_difference
count,79.000000
mean,-0.787975
std,4.900472
min,-19.000000
25%,-3.000000
50%,-1.000000
75%,2.000000
max,8.000000



Estimate increases:


,instance_id,source_id,old_points,new_points,point_difference
58003,JBDS-I01,JBDS-3822,5.00,13.00,8.00
58967,JBDS-I01,JBDS-3653,5.00,13.00,8.00
114241,BXMSDOC-I01,BXMSDOC-6576,1.00,3.00,2.00
166011,WT-I01,WT-8169,0.00,5.00,5.00
166044,WT-I01,WT-8165,0.00,8.00,8.00
166202,WT-I01,WT-8093,3.00,5.00,2.00
166496,WT-I01,WT-8020,8.00,13.00,5.00
167094,WT-I01,WT-7848,1.00,8.00,7.00
167314,WT-I01,WT-7757,5.00,13.00,8.00
167715,WT-I01,WT-7616,3.00,5.00,2.00


In [ ]:
text_candidates = text_scope_events.copy()

text_candidates["old_text"] = (
    text_candidates["from_value"]
    .fillna("")
    .astype(str)
    .str.strip()
)

text_candidates["new_text"] = (
    text_candidates["to_value"]
    .fillna("")
    .astype(str)
    .str.strip()
)

text_candidates["substantial_change"] = (
    text_candidates["old_text"].ne("")
    &
    text_candidates["new_text"].ne("")
    &
    (
        text_candidates["change_magnitude"]
        >= 0.30
    )
)

substantial_text_changes = text_candidates[
    text_candidates["substantial_change"]
].copy()

print("Total text changes:", len(text_candidates))

print("Substantial text-change candidates:",
      len(substantial_text_changes))

print("\nProject coverage:")

print(
    substantial_text_changes["instance_id"]
    .nunique()
)

display(
    substantial_text_changes.groupby("event_type")
    .agg(
        events=("uid", "count"),
        projects=("instance_id", "nunique")
    )
    .reset_index()
)

print("\nSample substantial changes:")

for _, row in substantial_text_changes.head(10).iterrows():

    print("\n" + "=" * 60)

    print("Project:", row["instance_id"])
    print("Issue:", row["source_id"])
    print("Type:", row["event_type"])

    print("\nBEFORE:")
    print(str(row["from_value"])[:350])

    print("\nAFTER:")
    print(str(row["to_value"])[:350])

Total text changes: 2119
Substantial text-change candidates: 503

Project coverage:
29


,event_type,events,projects
0,DESCRIPTION_CHANGE,241,28
1,SUMMARY_CHANGE,262,27



Sample substantial changes:

Project: JBEAP-I01
Issue: JBEAP-4962
Type: DESCRIPTION_CHANGE

BEFORE:
EAP7 Management CLI Guide issues tracker

https://access.stage.redhat.com/documentation/en/red-hat-jboss-enterprise-application-platform/version-7.0/management-cli-guide/

AFTER:
EAP7 Management CLI Guide issues tracker

https://access.stage.redhat.com/documentation/en/red-hat-jboss-enterprise-application-platform/version-7.0/management-cli-guide/

Apart from new features, we should go through the following resources and transform as much as possible into the product documentation.
 \- http://www.mastertheboss.com/jboss-ser

Project: JBEAP-I01
Issue: JBEAP-3969
Type: SUMMARY_CHANGE

BEFORE:
Session draining always takes maximum configured timeout since listener is always notified before session is removed

AFTER:
Session draining always takes maximum configured timeout

Project: JBEAP-I01
Issue: JBEAP-3459
Type: SUMMARY_CHANGE

BEFORE:
Server returns 204, when javax.validation.Constrain

In [ ]:
# ============================================================
# SCOPE CANDIDATE SCREENING — ALL VALIDATIONS
# ============================================================

checks = []

def check(name, passed, details):
    checks.append({
        "validation": name,
        "status": "PASS" if passed else "FAIL",
        "details": details
    })


check(
    "Structural event count preserved",
    len(structural_candidates) == len(
        structural_scope_events
    ),
    len(structural_candidates)
)

check(
    "All structural events classified",
    structural_candidates["structural_class"]
    .notna().all(),
    structural_candidates["structural_class"]
    .isna().sum()
)

check(
    "No duplicate structural event IDs",
    structural_candidates["uid"]
    .duplicated().sum() == 0,
    structural_candidates["uid"]
    .duplicated().sum()
)

check(
    "Story-point classifications complete",
    len(story_changes) == 355,
    len(story_changes)
)

check(
    "No negative screening magnitudes",
    (text_candidates["change_magnitude"] >= 0).all(),
    "Checked"
)

check(
    "Substantial changes meet threshold",
    (
        substantial_text_changes["change_magnitude"]
        >= 0.30
    ).all(),
    len(substantial_text_changes)
)

check(
    "Substantial changes have both values",
    (
        substantial_text_changes["old_text"].ne("")
        &
        substantial_text_changes["new_text"].ne("")
    ).all(),
    "Checked"
)

check(
    "Text candidate count preserved",
    len(text_candidates) == len(text_scope_events),
    len(text_candidates)
)

check(
    "All candidates traceable to Phase 1",
    set(
        structural_candidates["uid"]
    ).union(
        set(substantial_text_changes["uid"])
    ).issubset(set(events["uid"])),
    "Checked"
)

validation_df = pd.DataFrame(checks)

display(validation_df)

failed = validation_df[
    validation_df["status"] == "FAIL"
]

if failed.empty:
    print("ALL SCOPE SCREENING VALIDATIONS PASSED")
else:
    print(f"{len(failed)} VALIDATIONS FAILED")

,validation,status,details
0,Structural event count preserved,PASS,1322
1,All structural events classified,PASS,0
2,No duplicate structural event IDs,PASS,0
3,Story-point classifications complete,PASS,355
4,No negative screening magnitudes,PASS,Checked
5,Substantial changes meet threshold,PASS,503
6,Substantial changes have both values,PASS,Checked
7,Text candidate count preserved,PASS,2119
8,All candidates traceable to Phase 1,PASS,Checked


ALL SCOPE SCREENING VALIDATIONS PASSED


### 4.3 Eligible structural scope events

In [ ]:
import pandas as pd

ELIGIBLE_SCOPE_CLASSES = {
    "EPIC_CHILD_ADDED",
    "EPIC_CHILD_REMOVED",
    "EPIC_ASSIGNED",
    "EPIC_REASSIGNED",
    "EPIC_UNASSIGNED",
    "ISSUE_RECLASSIFIED"
}

scope_structural_eligible = structural_candidates[
    structural_candidates["structural_class"].isin(
        ELIGIBLE_SCOPE_CLASSES
    )
].copy()

scope_structural_eligible["event_dt"] = pd.to_datetime(
    scope_structural_eligible["event_dt"],
    utc=True,
    errors="coerce"
)

print("Eligible structural scope events:",
      len(scope_structural_eligible))

print("Projects represented:",
      scope_structural_eligible["instance_id"].nunique())

display(
    scope_structural_eligible["structural_class"]
    .value_counts()
)

Eligible structural scope events: 967
Projects represented: 26


,count
structural_class,
EPIC_CHILD_ADDED,374
EPIC_ASSIGNED,247
ISSUE_RECLASSIFIED,190
EPIC_REASSIGNED,71
EPIC_UNASSIGNED,49
EPIC_CHILD_REMOVED,36


In [ ]:
import re

ISSUE_PATTERN = re.compile(
    r"\b[A-Z][A-Z0-9]+-\d+\b",
    re.IGNORECASE
)

def get_affected_issue(row):

    event_type = row["event_type"]

    if event_type == "EPIC_CHILD_CHANGE":

        value = (
            row["to_value"]
            if row["structural_class"] == "EPIC_CHILD_ADDED"
            else row["from_value"]
        )

        matches = ISSUE_PATTERN.findall(str(value))

        if len(matches) == 1:
            return matches[0].upper()

        return None

    return row["source_id"]


scope_structural_eligible["affected_issue"] = (
    scope_structural_eligible.apply(
        get_affected_issue,
        axis=1
    )
)

print("Events with identifiable affected issues:",
      scope_structural_eligible["affected_issue"].notna().sum())

print("Events without identifiable affected issues:",
      scope_structural_eligible["affected_issue"].isna().sum())

display(
    scope_structural_eligible[
        [
            "instance_id",
            "source_id",
            "structural_class",
            "from_value",
            "to_value",
            "affected_issue"
        ]
    ].head(20)
)

Events with identifiable affected issues: 967
Events without identifiable affected issues: 0


,instance_id,source_id,structural_class,from_value,to_value,affected_issue
41782,JBEAP-I01,JBEAP-3444,ISSUE_RECLASSIFIED,Enhancement,Bug,JBEAP-3444
41886,JBEAP-I01,JBEAP-3426,ISSUE_RECLASSIFIED,Feature Request,Sub-task,JBEAP-3426
42677,JBEAP-I01,JBEAP-3028,ISSUE_RECLASSIFIED,Feature Request,Component Upgrade,JBEAP-3028
43322,JBEAP-I01,JBEAP-2786,ISSUE_RECLASSIFIED,Bug,Sub-task,JBEAP-2786
44162,JBEAP-I01,JBEAP-2544,ISSUE_RECLASSIFIED,Feature Request,Bug,JBEAP-2544
44207,JBEAP-I01,JBEAP-2532,ISSUE_RECLASSIFIED,Feature Request,Bug,JBEAP-2532
45680,JBEAP-I01,JBEAP-1905,ISSUE_RECLASSIFIED,Feature Request,Enhancement,JBEAP-1905
45769,JBEAP-I01,JBEAP-1848,ISSUE_RECLASSIFIED,Documentation,Bug,JBEAP-1848
46010,JBEAP-I01,JBEAP-1686,ISSUE_RECLASSIFIED,Documentation,Bug,JBEAP-1686
46090,JBEAP-I01,JBEAP-1681,ISSUE_RECLASSIFIED,Documentation,Bug,JBEAP-1681


In [ ]:
scope_structural_clean = (
    scope_structural_eligible
    .dropna(subset=["affected_issue", "event_dt"])
    .sort_values("event_dt")
    .drop_duplicates(
        subset=[
            "instance_id",
            "uid"
        ]
    )
    .copy()
)

scope_structural_clean["event_week"] = (
    scope_structural_clean["event_dt"]
    .dt.tz_localize(None)
    .dt.to_period("W-SUN")
    .dt.start_time
)

weekly_scope_activity = (
    scope_structural_clean.groupby(
        ["instance_id", "event_week"]
    )
    .agg(
        structural_events=("uid", "count"),
        affected_issues=("affected_issue", "nunique"),
        change_types=("structural_class", "nunique")
    )
    .reset_index()
)

print("Project-weeks with structural changes:",
      len(weekly_scope_activity))

print("Projects with structural changes:",
      weekly_scope_activity["instance_id"].nunique())

display(
    weekly_scope_activity[
        [
            "structural_events",
            "affected_issues",
            "change_types"
        ]
    ].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]
    )
)

display(
    weekly_scope_activity.sort_values(
        "affected_issues",
        ascending=False
    ).head(20)
)

Project-weeks with structural changes: 319
Projects with structural changes: 26


,structural_events,affected_issues,change_types
count,319.000000,319.000000,319.000000
mean,3.031348,2.721003,1.460815
std,4.192568,3.955002,0.791583
min,1.000000,1.000000,1.000000
25%,1.000000,1.000000,1.000000
50%,2.000000,1.000000,1.000000
75%,3.000000,3.000000,2.000000
90%,6.200000,5.200000,2.200000
95%,10.100000,9.100000,3.000000
max,37.000000,37.000000,5.000000


,instance_id,event_week,structural_events,affected_issues,change_types
164,JBIDE-I01,2020-07-13,37,37,1
225,OSSRH-I01,2021-05-03,29,29,1
19,AEROGEAR-I01,2018-09-17,23,23,1
260,RUBY-I01,2021-07-26,25,20,3
4,AEROGEAR-I01,2018-03-05,18,18,2
243,QTCREATORBUG-I01,2018-09-17,18,18,2
258,RUBY-I01,2021-07-12,17,17,2
251,RUBY-I01,2021-05-24,15,14,2
77,DOCS-I01,2016-08-29,20,14,5
34,BXMSDOC-I01,2020-04-06,14,13,2


In [ ]:
# ============================================================
# STRUCTURAL SCOPE OUTCOME — CONSOLIDATED VALIDATION
# ============================================================

checks = []

def check(name, passed, detail):
    checks.append({
        "validation": name,
        "status": "PASS" if passed else "FAIL",
        "details": detail
    })

check(
    "Eligible event count",
    len(scope_structural_eligible) == 967,
    len(scope_structural_eligible)
)

check(
    "No missing event timestamps",
    scope_structural_eligible["event_dt"].notna().all(),
    scope_structural_eligible["event_dt"].isna().sum()
)

check(
    "Unique cleaned event IDs",
    not scope_structural_clean["uid"].duplicated().any(),
    scope_structural_clean["uid"].duplicated().sum()
)

check(
    "Valid affected issue identifiers",
    scope_structural_clean["affected_issue"]
    .astype(str)
    .str.fullmatch(r"[A-Z][A-Z0-9]+-\d+")
    .all(),
    len(scope_structural_clean)
)

check(
    "All eligible classes valid",
    scope_structural_eligible["structural_class"]
    .isin(ELIGIBLE_SCOPE_CLASSES).all(),
    "Checked"
)

check(
    "All cleaned events traceable to Phase 1",
    set(scope_structural_clean["uid"])
    .issubset(set(events["uid"])),
    "Checked"
)

check(
    "Weekly affected issues <= event count",
    (
        weekly_scope_activity["affected_issues"]
        <= weekly_scope_activity["structural_events"]
    ).all(),
    "Checked"
)

check(
    "No negative weekly counts",
    (
        weekly_scope_activity[
            ["structural_events", "affected_issues"]
        ] >= 0
    ).all().all(),
    "Checked"
)

check(
    "Baseline checkpoints remain aligned",
    len(final_schedule_labels)
    == len(final_dependency_labels)
    == 987,
    "987 checkpoints"
)

validation_df = pd.DataFrame(checks)

display(validation_df)

failed = validation_df[
    validation_df["status"] == "FAIL"
]

if failed.empty:
    print("ALL STRUCTURAL SCOPE VALIDATIONS PASSED")
else:
    print(f"{len(failed)} VALIDATIONS FAILED")

,validation,status,details
0,Eligible event count,PASS,967
1,No missing event timestamps,PASS,0
2,Unique cleaned event IDs,PASS,0
3,Valid affected issue identifiers,PASS,967
4,All eligible classes valid,PASS,Checked
5,All cleaned events traceable to Phase 1,PASS,Checked
6,Weekly affected issues <= event count,PASS,Checked
7,No negative weekly counts,PASS,Checked
8,Baseline checkpoints remain aligned,PASS,987 checkpoints


ALL STRUCTURAL SCOPE VALIDATIONS PASSED


### 4.4 Future scope outcomes

In [ ]:
import pandas as pd

HORIZON_DAYS = 14

scope_label_records = []

# Reuse the exact eligible checkpoints
base_checkpoints = final_schedule_labels[
    [
        "instance_id",
        "week_number",
        "checkpoint_time",
        "horizon_end"
    ]
].copy()

for _, checkpoint in base_checkpoints.iterrows():

    instance_id = checkpoint["instance_id"]
    t = checkpoint["checkpoint_time"]
    horizon_end = checkpoint["horizon_end"]

    future_events = scope_structural_clean[
        (scope_structural_clean["instance_id"] == instance_id)
        &
        (scope_structural_clean["event_dt"] > t)
        &
        (scope_structural_clean["event_dt"] <= horizon_end)
    ]

    scope_label_records.append({
        "instance_id": instance_id,
        "week_number": checkpoint["week_number"],
        "checkpoint_time": t,
        "horizon_end": horizon_end,

        "future_scope_events": len(future_events),

        "future_affected_issues": (
            future_events["affected_issue"].nunique()
        ),

        "future_epic_child_changes": (
            future_events[
                future_events["structural_class"].isin([
                    "EPIC_CHILD_ADDED",
                    "EPIC_CHILD_REMOVED"
                ])
            ]["affected_issue"].nunique()
        ),

        "future_epic_link_changes": (
            future_events[
                future_events["structural_class"].isin([
                    "EPIC_ASSIGNED",
                    "EPIC_REASSIGNED",
                    "EPIC_UNASSIGNED"
                ])
            ]["affected_issue"].nunique()
        ),

        "future_reclassified_issues": (
            future_events[
                future_events["structural_class"]
                == "ISSUE_RECLASSIFIED"
            ]["affected_issue"].nunique()
        )
    })

scope_future_df = pd.DataFrame(scope_label_records)

print("Eligible Scope Risk checkpoints:", len(scope_future_df))
print("Projects:", scope_future_df["instance_id"].nunique())

display(scope_future_df.head(10))

Eligible Scope Risk checkpoints: 987
Projects: 30


,instance_id,week_number,checkpoint_time,horizon_end,future_scope_events,future_affected_issues,future_epic_child_changes,future_epic_link_changes,future_reclassified_issues
0,AEROGEAR-I01,1,2018-01-24 05:30:01+00:00,2018-02-07 05:30:01+00:00,0,0,0,0,0
1,AEROGEAR-I01,2,2018-01-31 05:30:01+00:00,2018-02-14 05:30:01+00:00,2,2,0,2,0
2,AEROGEAR-I01,3,2018-02-07 05:30:01+00:00,2018-02-21 05:30:01+00:00,2,2,0,2,0
3,AEROGEAR-I01,4,2018-02-14 05:30:01+00:00,2018-02-28 05:30:01+00:00,1,1,0,1,0
4,AEROGEAR-I01,5,2018-02-21 05:30:01+00:00,2018-03-07 05:30:01+00:00,17,17,8,9,0
5,AEROGEAR-I01,6,2018-02-28 05:30:01+00:00,2018-03-14 05:30:01+00:00,20,19,8,11,0
6,AEROGEAR-I01,7,2018-03-07 05:30:01+00:00,2018-03-21 05:30:01+00:00,4,3,0,3,0
7,AEROGEAR-I01,8,2018-03-14 05:30:01+00:00,2018-03-28 05:30:01+00:00,0,0,0,0,0
8,AEROGEAR-I01,9,2018-03-21 05:30:01+00:00,2018-04-04 05:30:01+00:00,0,0,0,0,0
9,AEROGEAR-I01,10,2018-03-28 05:30:01+00:00,2018-04-11 05:30:01+00:00,2,2,1,1,0


In [ ]:
print("FUTURE SCOPE CHANGE DISTRIBUTION")

display(
    scope_future_df[
        [
            "future_scope_events",
            "future_affected_issues",
            "future_epic_child_changes",
            "future_epic_link_changes",
            "future_reclassified_issues"
        ]
    ].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]
    )
)

threshold_results = []

for threshold in [1, 2, 3, 4, 5, 6, 8, 10]:

    positive = (
        scope_future_df["future_affected_issues"]
        >= threshold
    )

    threshold_results.append({
        "threshold": threshold,
        "positive_checkpoints": int(positive.sum()),
        "negative_checkpoints": int((~positive).sum()),
        "positive_pct": round(
            positive.mean() * 100, 2
        ),
        "positive_projects": (
            scope_future_df.loc[
                positive, "instance_id"
            ].nunique()
        )
    })

scope_threshold_df = pd.DataFrame(threshold_results)

print("\nSCOPE THRESHOLD SENSITIVITY")
display(scope_threshold_df)

FUTURE SCOPE CHANGE DISTRIBUTION


,future_scope_events,future_affected_issues,future_epic_child_changes,future_epic_link_changes,future_reclassified_issues
count,987.000000,987.000000,987.000000,987.000000,987.000000
mean,1.790274,1.583587,0.709220,0.626140,0.328267
std,4.063091,3.714979,2.908954,1.520331,1.441081
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000
75%,2.000000,2.000000,0.000000,1.000000,0.000000
90%,5.000000,4.000000,2.000000,2.000000,1.000000
95%,8.000000,7.000000,3.700000,3.000000,1.000000
max,37.000000,37.000000,37.000000,16.000000,29.000000



SCOPE THRESHOLD SENSITIVITY


,threshold,positive_checkpoints,negative_checkpoints,positive_pct,positive_projects
0,1,451,536,45.69,26
1,2,261,726,26.44,24
2,3,169,818,17.12,21
3,4,112,875,11.35,20
4,5,84,903,8.51,17
5,6,61,926,6.18,17
6,8,44,943,4.46,12
7,10,30,957,3.04,9


In [ ]:
scope_event_audit = scope_structural_clean.copy()

scope_event_audit["event_day"] = (
    scope_event_audit["event_dt"].dt.floor("D")
)

daily_scope_activity = (
    scope_event_audit.groupby(
        ["instance_id", "event_day"]
    )
    .agg(
        event_count=("uid", "count"),
        affected_issues=("affected_issue", "nunique"),
        epic_child_adds=(
            "structural_class",
            lambda x: (x == "EPIC_CHILD_ADDED").sum()
        ),
        epic_assignments=(
            "structural_class",
            lambda x: (x == "EPIC_ASSIGNED").sum()
        )
    )
    .reset_index()
)

print("TOP STRUCTURAL CHANGE DAYS")

display(
    daily_scope_activity.sort_values(
        "event_count",
        ascending=False
    ).head(20)
)

print("\nPROJECT-LEVEL CONCENTRATION")

project_scope_totals = (
    daily_scope_activity.groupby("instance_id")
    .agg(
        total_events=("event_count", "sum"),
        peak_day_events=("event_count", "max"),
        active_days=("event_day", "nunique")
    )
    .reset_index()
)

project_scope_totals["peak_day_share"] = (
    project_scope_totals["peak_day_events"]
    / project_scope_totals["total_events"]
).round(3)

display(
    project_scope_totals.sort_values(
        "peak_day_share",
        ascending=False
    )
)

TOP STRUCTURAL CHANGE DAYS


,instance_id,event_day,event_count,affected_issues,epic_child_adds,epic_assignments
319,OSSRH-I01,2021-05-05 00:00:00+00:00,29,29,0,0
231,JBIDE-I01,2020-07-15 00:00:00+00:00,26,26,26,0
31,AEROGEAR-I01,2018-09-18 00:00:00+00:00,21,21,21,0
6,AEROGEAR-I01,2018-03-05 00:00:00+00:00,14,14,8,6
389,RUBY-I01,2021-07-27 00:00:00+00:00,14,14,0,14
117,DOCS-I01,2016-09-01 00:00:00+00:00,12,6,1,5
232,JBIDE-I01,2020-07-16 00:00:00+00:00,11,11,11,0
363,RUBY-I01,2021-05-28 00:00:00+00:00,11,10,10,1
453,SPARK-I01,2019-04-04 00:00:00+00:00,10,8,7,2
196,JBDS-I01,2016-03-08 00:00:00+00:00,10,10,3,7



PROJECT-LEVEL CONCENTRATION


,instance_id,total_events,peak_day_events,active_days,peak_day_share
18,OSSRH-I01,31,29,3,0.935
15,JSDCLOUD-I01,5,4,2,0.800
13,JBIDE-I01,49,26,8,0.531
24,SPR-I01,5,2,4,0.400
12,JBEAP-I01,13,5,8,0.385
1,AMBARI-I01,24,8,13,0.333
3,CONFSERVER-I01,3,1,3,0.333
20,QTCREATORBUG-I01,31,10,13,0.323
9,HBASE-I01,7,2,6,0.286
23,SPARK-I01,40,10,21,0.250


In [ ]:
# ============================================================
# SCOPE RISK — CONSOLIDATED OUTCOME VALIDATION
# ============================================================

import pandas as pd

checks = []

def add_check(name, condition, details):
    checks.append({
        "validation": name,
        "status": "PASS" if condition else "FAIL",
        "details": details
    })

keys = [
    "instance_id",
    "week_number",
    "checkpoint_time"
]

# 1. Expected checkpoint count
add_check(
    "987 eligible checkpoints",
    len(scope_future_df) == 987,
    len(scope_future_df)
)

# 2. Project coverage
add_check(
    "All 30 projects represented",
    scope_future_df["instance_id"].nunique() == 30,
    scope_future_df["instance_id"].nunique()
)

# 3. Duplicate checkpoints
duplicates = scope_future_df.duplicated(keys).sum()

add_check(
    "No duplicate checkpoints",
    duplicates == 0,
    duplicates
)

# 4. Missing values
required = keys + [
    "horizon_end",
    "future_scope_events",
    "future_affected_issues"
]

missing = scope_future_df[required].isna().sum().sum()

add_check(
    "No missing required values",
    missing == 0,
    missing
)

# 5. Prediction horizon
horizon = (
    scope_future_df["horizon_end"]
    - scope_future_df["checkpoint_time"]
)

invalid_horizons = (
    horizon != pd.Timedelta(days=14)
).sum()

add_check(
    "All prediction horizons are 14 days",
    invalid_horizons == 0,
    invalid_horizons
)

# 6. Alignment with Schedule Risk
schedule_keys = pd.MultiIndex.from_frame(
    final_schedule_labels[keys]
)

scope_keys = pd.MultiIndex.from_frame(
    scope_future_df[keys]
)

add_check(
    "Exact Schedule Risk checkpoint alignment",
    len(scope_keys) == len(schedule_keys)
    and set(scope_keys) == set(schedule_keys),
    len(scope_keys)
)

# 7. Alignment with Dependency Risk
dependency_keys = pd.MultiIndex.from_frame(
    final_dependency_labels[keys]
)

add_check(
    "Exact Dependency Risk checkpoint alignment",
    len(scope_keys) == len(dependency_keys)
    and set(scope_keys) == set(dependency_keys),
    len(scope_keys)
)

# 8. Count integrity
count_cols = [
    "future_scope_events",
    "future_affected_issues",
    "future_epic_child_changes",
    "future_epic_link_changes",
    "future_reclassified_issues"
]

negative_counts = (
    scope_future_df[count_cols] < 0
).sum().sum()

add_check(
    "No negative outcome counts",
    negative_counts == 0,
    negative_counts
)

# 9. Distinct affected issues <= events
invalid_affected = (
    scope_future_df["future_affected_issues"]
    > scope_future_df["future_scope_events"]
).sum()

add_check(
    "Affected issues do not exceed event count",
    invalid_affected == 0,
    invalid_affected
)

# 10. Event provenance
known_ids = set(events["uid"].dropna())

invalid_event_ids = (
    ~scope_structural_clean["uid"].isin(known_ids)
).sum()

add_check(
    "All structural events traceable to Phase 1",
    invalid_event_ids == 0,
    invalid_event_ids
)

# 11. Source event uniqueness
duplicate_events = (
    scope_structural_clean["uid"].duplicated().sum()
)

add_check(
    "No duplicate structural event IDs",
    duplicate_events == 0,
    duplicate_events
)

# 12. No future windows beyond observation end
end_lookup = (
    checkpoints_df.groupby("instance_id")[
        "observation_end"
    ].max()
)

invalid_windows = (
    scope_future_df["horizon_end"]
    >
    scope_future_df["instance_id"].map(end_lookup)
).sum()

add_check(
    "Complete future observation windows",
    invalid_windows == 0,
    invalid_windows
)

# 13. Affected issue identifiers
missing_issues = (
    scope_structural_clean["affected_issue"]
    .isna().sum()
)

add_check(
    "All retained events have affected issues",
    missing_issues == 0,
    missing_issues
)

# ------------------------------------------------------------
# REPORT
# ------------------------------------------------------------

validation_df = pd.DataFrame(checks)

display(validation_df)

failed = validation_df[
    validation_df["status"] == "FAIL"
]

print("\n" + "=" * 60)

if failed.empty:
    print("ALL SCOPE OUTCOME VALIDATIONS PASSED")
else:
    print(f"{len(failed)} VALIDATIONS FAILED")

print("=" * 60)

print("\nEligible checkpoints:", len(scope_future_df))
print("Projects:", scope_future_df["instance_id"].nunique())
print("Structural events:", len(scope_structural_clean))

,validation,status,details
0,987 eligible checkpoints,PASS,987
1,All 30 projects represented,PASS,30
2,No duplicate checkpoints,PASS,0
3,No missing required values,PASS,0
4,All prediction horizons are 14 days,PASS,0
5,Exact Schedule Risk checkpoint alignment,PASS,987
6,Exact Dependency Risk checkpoint alignment,PASS,987
7,No negative outcome counts,PASS,0
8,Affected issues do not exceed event count,PASS,0
9,All structural events traceable to Phase 1,PASS,0



ALL SCOPE OUTCOME VALIDATIONS PASSED

Eligible checkpoints: 987
Projects: 30
Structural events: 967


In [ ]:
import pandas as pd

ossrh_burst = scope_structural_clean[
    (scope_structural_clean["instance_id"] == "OSSRH-I01")
    &
    (
        scope_structural_clean["event_dt"].dt.floor("D")
        == pd.Timestamp("2021-05-05", tz="UTC")
    )
].copy()

print("OSSRH burst events:", len(ossrh_burst))

print("\nEvent classification:")
display(ossrh_burst["structural_class"].value_counts())

print("\nOriginal changes:")
display(
    ossrh_burst[
        [
            "source_id",
            "affected_issue",
            "event_type",
            "structural_class",
            "from_value",
            "to_value",
            "event_dt"
        ]
    ].sort_values("event_dt")
)

OSSRH burst events: 29

Event classification:


,count
structural_class,
ISSUE_RECLASSIFIED,29



Original changes:


,source_id,affected_issue,event_type,structural_class,from_value,to_value,event_dt
109503,OSSRH-5652,OSSRH-5652,ISSUE_TYPE_CHANGE,ISSUE_RECLASSIFIED,Task,Publishing Support,2021-05-05 14:37:27+00:00
109520,OSSRH-5613,OSSRH-5613,ISSUE_TYPE_CHANGE,ISSUE_RECLASSIFIED,Task,Publishing Support,2021-05-05 14:43:42+00:00
109490,OSSRH-5675,OSSRH-5675,ISSUE_TYPE_CHANGE,ISSUE_RECLASSIFIED,Task,Publishing Support,2021-05-05 14:47:23+00:00
109454,OSSRH-9768,OSSRH-9768,ISSUE_TYPE_CHANGE,ISSUE_RECLASSIFIED,Task,Publishing Support,2021-05-05 15:00:23+00:00
109412,OSSRH-15600,OSSRH-15600,ISSUE_TYPE_CHANGE,ISSUE_RECLASSIFIED,Task,Publishing Support,2021-05-05 15:00:52+00:00
109474,OSSRH-7231,OSSRH-7231,ISSUE_TYPE_CHANGE,ISSUE_RECLASSIFIED,Task,Publishing Support,2021-05-05 15:02:18+00:00
109401,OSSRH-15743,OSSRH-15743,ISSUE_TYPE_CHANGE,ISSUE_RECLASSIFIED,Task,Publishing Support,2021-05-05 15:03:05+00:00
109308,OSSRH-24765,OSSRH-24765,ISSUE_TYPE_CHANGE,ISSUE_RECLASSIFIED,Task,Publishing Support,2021-05-05 15:17:05+00:00
109338,OSSRH-22981,OSSRH-22981,ISSUE_TYPE_CHANGE,ISSUE_RECLASSIFIED,Task,Publishing Support,2021-05-05 15:20:01+00:00
109353,OSSRH-21705,OSSRH-21705,ISSUE_TYPE_CHANGE,ISSUE_RECLASSIFIED,Task,Publishing Support,2021-05-05 15:20:10+00:00


In [ ]:
large_scope_days = daily_scope_activity[
    daily_scope_activity["event_count"] >= 10
].copy()

print("Project-days with at least 10 structural events:",
      len(large_scope_days))

display(
    large_scope_days.sort_values(
        "event_count",
        ascending=False
    )
)

# Examine event-class composition for each large day
burst_details = []

for _, row in large_scope_days.iterrows():

    subset = scope_structural_clean[
        (scope_structural_clean["instance_id"] == row["instance_id"])
        &
        (
            scope_structural_clean["event_dt"].dt.floor("D")
            == row["event_day"]
        )
    ]

    for event_class, count in (
        subset["structural_class"].value_counts().items()
    ):
        burst_details.append({
            "instance_id": row["instance_id"],
            "event_day": row["event_day"],
            "structural_class": event_class,
            "count": count
        })

burst_details_df = pd.DataFrame(burst_details)

print("\nBurst composition:")
display(
    burst_details_df.sort_values(
        ["instance_id", "event_day", "count"],
        ascending=[True, True, False]
    )
)

Project-days with at least 10 structural events: 13


,instance_id,event_day,event_count,affected_issues,epic_child_adds,epic_assignments
319,OSSRH-I01,2021-05-05 00:00:00+00:00,29,29,0,0
231,JBIDE-I01,2020-07-15 00:00:00+00:00,26,26,26,0
31,AEROGEAR-I01,2018-09-18 00:00:00+00:00,21,21,21,0
6,AEROGEAR-I01,2018-03-05 00:00:00+00:00,14,14,8,6
389,RUBY-I01,2021-07-27 00:00:00+00:00,14,14,0,14
117,DOCS-I01,2016-09-01 00:00:00+00:00,12,6,1,5
232,JBIDE-I01,2020-07-16 00:00:00+00:00,11,11,11,0
363,RUBY-I01,2021-05-28 00:00:00+00:00,11,10,10,1
12,AEROGEAR-I01,2018-04-16 00:00:00+00:00,10,8,8,2
196,JBDS-I01,2016-03-08 00:00:00+00:00,10,10,3,7



Burst composition:


,instance_id,event_day,structural_class,count
0,AEROGEAR-I01,2018-03-05 00:00:00+00:00,EPIC_CHILD_ADDED,8
1,AEROGEAR-I01,2018-03-05 00:00:00+00:00,EPIC_ASSIGNED,6
2,AEROGEAR-I01,2018-04-16 00:00:00+00:00,EPIC_CHILD_ADDED,8
3,AEROGEAR-I01,2018-04-16 00:00:00+00:00,EPIC_ASSIGNED,2
4,AEROGEAR-I01,2018-09-18 00:00:00+00:00,EPIC_CHILD_ADDED,21
5,DOCS-I01,2016-09-01 00:00:00+00:00,EPIC_ASSIGNED,5
6,DOCS-I01,2016-09-01 00:00:00+00:00,EPIC_UNASSIGNED,3
7,DOCS-I01,2016-09-01 00:00:00+00:00,ISSUE_RECLASSIFIED,2
8,DOCS-I01,2016-09-01 00:00:00+00:00,EPIC_CHILD_ADDED,1
9,DOCS-I01,2016-09-01 00:00:00+00:00,EPIC_CHILD_REMOVED,1


In [ ]:
import json

event_payload_lookup = (
    nodes_df[nodes_df["label"] == "Event"]
    .set_index("uid")["payload_json"]
    .to_dict()
)

def parse_payload(value):
    if isinstance(value, dict):
        return value

    if isinstance(value, str):
        try:
            return json.loads(value)
        except (json.JSONDecodeError, TypeError):
            return {}

    return {}


actor_records = []

for _, row in ossrh_burst.iterrows():

    payload = parse_payload(
        event_payload_lookup.get(row["uid"])
    )

    actor_records.append({
        "event_uid": row["uid"],
        "source_id": row["source_id"],
        "event_dt": row["event_dt"],
        "structural_class": row["structural_class"],
        "actor": payload.get("actor"),
        "from_value": row["from_value"],
        "to_value": row["to_value"]
    })

ossrh_actor_df = pd.DataFrame(actor_records)

print("Actor distribution:")
display(
    ossrh_actor_df["actor"].value_counts(dropna=False)
)

print("\nTime range:")
print("First:", ossrh_actor_df["event_dt"].min())
print("Last:", ossrh_actor_df["event_dt"].max())

print("\nEvent details:")
display(ossrh_actor_df.sort_values("event_dt"))

Actor distribution:


,count
actor,
<<|author_displayName|6e00fffe-e4ef-4304-b693-cf01a164d890|>>,29



Time range:
First: 2021-05-05 14:37:27+00:00
Last: 2021-05-05 16:59:05+00:00

Event details:


,event_uid,source_id,event_dt,structural_class,actor,from_value,to_value
0,T_f5fef2567a370c6107483b89,OSSRH-5652,2021-05-05 14:37:27+00:00,ISSUE_RECLASSIFIED,<<|author_displayName|6e00fffe-e4ef-4304-b693-...,Task,Publishing Support
1,T_b3efee02387966892950d932,OSSRH-5613,2021-05-05 14:43:42+00:00,ISSUE_RECLASSIFIED,<<|author_displayName|6e00fffe-e4ef-4304-b693-...,Task,Publishing Support
2,T_14ea00f1e69262d024548ee0,OSSRH-5675,2021-05-05 14:47:23+00:00,ISSUE_RECLASSIFIED,<<|author_displayName|6e00fffe-e4ef-4304-b693-...,Task,Publishing Support
3,T_7234be88df1fc7f97064b681,OSSRH-9768,2021-05-05 15:00:23+00:00,ISSUE_RECLASSIFIED,<<|author_displayName|6e00fffe-e4ef-4304-b693-...,Task,Publishing Support
4,T_e8a0df008a06aff9e92228ed,OSSRH-15600,2021-05-05 15:00:52+00:00,ISSUE_RECLASSIFIED,<<|author_displayName|6e00fffe-e4ef-4304-b693-...,Task,Publishing Support
5,T_51d239db55fd1ee0c61d219a,OSSRH-7231,2021-05-05 15:02:18+00:00,ISSUE_RECLASSIFIED,<<|author_displayName|6e00fffe-e4ef-4304-b693-...,Task,Publishing Support
6,T_5095cc15ee6b4458369ceb18,OSSRH-15743,2021-05-05 15:03:05+00:00,ISSUE_RECLASSIFIED,<<|author_displayName|6e00fffe-e4ef-4304-b693-...,Task,Publishing Support
7,T_789c7d146c474594515265d2,OSSRH-24765,2021-05-05 15:17:05+00:00,ISSUE_RECLASSIFIED,<<|author_displayName|6e00fffe-e4ef-4304-b693-...,Task,Publishing Support
8,T_19745a72bf3e33e51913d2aa,OSSRH-22981,2021-05-05 15:20:01+00:00,ISSUE_RECLASSIFIED,<<|author_displayName|6e00fffe-e4ef-4304-b693-...,Task,Publishing Support
9,T_100af9b1f06566c7430e765d,OSSRH-21705,2021-05-05 15:20:10+00:00,ISSUE_RECLASSIFIED,<<|author_displayName|6e00fffe-e4ef-4304-b693-...,Task,Publishing Support


In [ ]:
# ============================================================
# SCOPE BURST AUDIT — CONSOLIDATED VALIDATION
# ============================================================

checks = []

def check(name, passed, detail):
    checks.append({
        "validation": name,
        "status": "PASS" if passed else "FAIL",
        "details": detail
    })

check(
    "OSSRH burst count matches daily summary",
    len(ossrh_burst) == 29,
    len(ossrh_burst)
)

check(
    "OSSRH event IDs unique",
    ossrh_burst["uid"].is_unique,
    ossrh_burst["uid"].duplicated().sum()
)

check(
    "OSSRH events have timestamps",
    ossrh_burst["event_dt"].notna().all(),
    ossrh_burst["event_dt"].isna().sum()
)

check(
    "All burst events traceable to Phase 1",
    set(ossrh_burst["uid"]).issubset(
        set(events["uid"])
    ),
    "Checked"
)

check(
    "Actor inspection covers all OSSRH events",
    len(ossrh_actor_df) == len(ossrh_burst),
    len(ossrh_actor_df)
)

check(
    "All audited days have at least 10 events",
    (large_scope_days["event_count"] >= 10).all(),
    len(large_scope_days)
)

check(
    "No duplicate project-day records",
    not daily_scope_activity.duplicated(
        ["instance_id", "event_day"]
    ).any(),
    "Checked"
)

check(
    "987 Scope Risk checkpoints preserved",
    len(scope_future_df) == 987,
    len(scope_future_df)
)

validation_df = pd.DataFrame(checks)

display(validation_df)

failed = validation_df[
    validation_df["status"] == "FAIL"
]

if failed.empty:
    print("ALL SCOPE BURST AUDIT VALIDATIONS PASSED")
else:
    print(f"{len(failed)} VALIDATIONS FAILED")

,validation,status,details
0,OSSRH burst count matches daily summary,PASS,29
1,OSSRH event IDs unique,PASS,0
2,OSSRH events have timestamps,PASS,0
3,All burst events traceable to Phase 1,PASS,Checked
4,Actor inspection covers all OSSRH events,PASS,29
5,All audited days have at least 10 events,PASS,13
6,No duplicate project-day records,PASS,Checked
7,987 Scope Risk checkpoints preserved,PASS,987


ALL SCOPE BURST AUDIT VALIDATIONS PASSED


In [ ]:
# ============================================================
# STEP 2.6.10 — OSSRH RECLASSIFICATION INSPECTION
# ============================================================

ossrh_transitions = ossrh_burst.copy()

ossrh_transitions["transition"] = (
    ossrh_transitions["from_value"].astype(str)
    + " -> "
    + ossrh_transitions["to_value"].astype(str)
)

print("OSSRH issue-type transitions:")

display(
    ossrh_transitions["transition"]
    .value_counts()
    .reset_index(name="count")
)

print("\nIndividual issue changes:")

display(
    ossrh_transitions[
        [
            "source_id",
            "from_value",
            "to_value",
            "event_dt"
        ]
    ].sort_values("event_dt")
)

OSSRH issue-type transitions:


,transition,count
0,Task -> Publishing Support,29



Individual issue changes:


,source_id,from_value,to_value,event_dt
109503,OSSRH-5652,Task,Publishing Support,2021-05-05 14:37:27+00:00
109520,OSSRH-5613,Task,Publishing Support,2021-05-05 14:43:42+00:00
109490,OSSRH-5675,Task,Publishing Support,2021-05-05 14:47:23+00:00
109454,OSSRH-9768,Task,Publishing Support,2021-05-05 15:00:23+00:00
109412,OSSRH-15600,Task,Publishing Support,2021-05-05 15:00:52+00:00
109474,OSSRH-7231,Task,Publishing Support,2021-05-05 15:02:18+00:00
109401,OSSRH-15743,Task,Publishing Support,2021-05-05 15:03:05+00:00
109308,OSSRH-24765,Task,Publishing Support,2021-05-05 15:17:05+00:00
109338,OSSRH-22981,Task,Publishing Support,2021-05-05 15:20:01+00:00
109353,OSSRH-21705,Task,Publishing Support,2021-05-05 15:20:10+00:00


In [ ]:
# ============================================================
# ISSUE AGE AT RECLASSIFICATION
# ============================================================

issue_created_lookup = (
    jira_df.set_index("issue_key")["created_at"]
    .to_dict()
)

ossrh_transitions["issue_created_at"] = (
    pd.to_datetime(
        ossrh_transitions["source_id"].map(
            issue_created_lookup
        ),
        utc=True,
        errors="coerce"
    )
)

ossrh_transitions["issue_age_days"] = (
    ossrh_transitions["event_dt"]
    - ossrh_transitions["issue_created_at"]
).dt.total_seconds() / 86400

print("Issue-age distribution:")

display(
    ossrh_transitions["issue_age_days"]
    .describe()
)

print("\nIssue-level details:")

display(
    ossrh_transitions[
        [
            "source_id",
            "from_value",
            "to_value",
            "issue_age_days"
        ]
    ]
)

Issue-age distribution:


,issue_age_days
count,29.000000
mean,1302.717303
std,909.471271
min,22.853437
25%,632.788171
50%,985.867465
75%,1852.925139
max,2979.880475



Issue-level details:


,source_id,from_value,to_value,issue_age_days
109503,OSSRH-5652,Task,Publishing Support,2974.780532
109520,OSSRH-5613,Task,Publishing Support,2979.880475
109490,OSSRH-5675,Task,Publishing Support,2972.568623
109454,OSSRH-9768,Task,Publishing Support,2554.216412
109412,OSSRH-15600,Task,Publishing Support,2178.937257
109474,OSSRH-7231,Task,Publishing Support,2791.664028
109401,OSSRH-15743,Task,Publishing Support,2177.909167
109308,OSSRH-24765,Task,Publishing Support,1705.575637
109338,OSSRH-22981,Task,Publishing Support,1790.291840
109353,OSSRH-21705,Task,Publishing Support,1852.925139


In [ ]:
# ============================================================
# EPIC CHILD BURST INSPECTION
# ============================================================

epic_burst_projects = [
    "JBIDE-I01",
    "AEROGEAR-I01",
    "RUBY-I01"
]

epic_burst_events = scope_structural_clean[
    (
        scope_structural_clean["instance_id"]
        .isin(epic_burst_projects)
    )
    &
    (
        scope_structural_clean["structural_class"]
        == "EPIC_CHILD_ADDED"
    )
].copy()

epic_burst_events["event_day"] = (
    epic_burst_events["event_dt"].dt.floor("D")
)

epic_grouping = (
    epic_burst_events.groupby(
        [
            "instance_id",
            "event_day",
            "source_id"
        ]
    )
    .agg(
        children_added=("affected_issue", "nunique"),
        first_event=("event_dt", "min"),
        last_event=("event_dt", "max")
    )
    .reset_index()
)

print("Largest epic-child addition groups:")

display(
    epic_grouping.sort_values(
        "children_added",
        ascending=False
    ).head(20)
)

Largest epic-child addition groups:


,instance_id,event_day,source_id,children_added,first_event,last_event
15,JBIDE-I01,2020-07-15 00:00:00+00:00,JBIDE-27384,26,2020-07-15 06:16:42+00:00,2020-07-15 10:09:52+00:00
11,AEROGEAR-I01,2018-09-18 00:00:00+00:00,AEROGEAR-7942,12,2018-09-18 11:19:20+00:00,2018-09-18 11:36:25+00:00
16,JBIDE-I01,2020-07-16 00:00:00+00:00,JBIDE-27384,11,2020-07-16 03:18:03+00:00,2020-07-16 03:31:53+00:00
40,RUBY-I01,2021-05-28 00:00:00+00:00,RUBY-2515,9,2021-05-28 16:34:29+00:00,2021-05-28 21:26:58+00:00
2,AEROGEAR-I01,2018-03-05 00:00:00+00:00,AEROGEAR-2446,8,2018-03-05 07:48:39+00:00,2018-03-05 09:20:40+00:00
4,AEROGEAR-I01,2018-04-16 00:00:00+00:00,AEROGEAR-2505,8,2018-04-16 11:39:12+00:00,2018-04-16 16:12:10+00:00
9,AEROGEAR-I01,2018-05-17 00:00:00+00:00,AEROGEAR-2505,7,2018-05-17 11:02:30+00:00,2018-05-17 13:12:53+00:00
22,JBIDE-I01,2021-03-01 00:00:00+00:00,JBIDE-27384,6,2021-03-01 04:36:44+00:00,2021-03-01 04:44:23+00:00
12,AEROGEAR-I01,2018-09-18 00:00:00+00:00,AEROGEAR-7943,6,2018-09-18 11:21:11+00:00,2018-09-18 11:36:42+00:00
0,AEROGEAR-I01,2018-01-19 00:00:00+00:00,AEROGEAR-1904,4,2018-01-19 05:43:40+00:00,2018-01-19 07:09:39+00:00


In [ ]:
# ============================================================
# SCOPE BURST REVIEW — CONSOLIDATED VALIDATION
# ============================================================

checks = []

def check(name, passed, detail):
    checks.append({
        "validation": name,
        "status": "PASS" if passed else "FAIL",
        "details": detail
    })

check(
    "OSSRH burst contains 29 events",
    len(ossrh_transitions) == 29,
    len(ossrh_transitions)
)

check(
    "OSSRH events are issue reclassifications",
    (
        ossrh_transitions["structural_class"]
        == "ISSUE_RECLASSIFIED"
    ).all(),
    "Checked"
)

check(
    "OSSRH event IDs unique",
    ossrh_transitions["uid"].is_unique,
    ossrh_transitions["uid"].duplicated().sum()
)

check(
    "OSSRH transition values available",
    ossrh_transitions[
        ["from_value", "to_value"]
    ].notna().all().all(),
    "Checked"
)

check(
    "Issue ages nonnegative where available",
    (
        ossrh_transitions["issue_age_days"]
        .dropna() >= 0
    ).all(),
    "Checked"
)

check(
    "Epic burst events are child additions",
    (
        epic_burst_events["structural_class"]
        == "EPIC_CHILD_ADDED"
    ).all(),
    len(epic_burst_events)
)

check(
    "Epic-child identifiers available",
    epic_burst_events["affected_issue"]
    .notna().all(),
    "Checked"
)

check(
    "Scope checkpoint count preserved",
    len(scope_future_df) == 987,
    len(scope_future_df)
)

validation_df = pd.DataFrame(checks)

display(validation_df)

failed = validation_df[
    validation_df["status"] == "FAIL"
]

if failed.empty:
    print("ALL SCOPE BURST REVIEW VALIDATIONS PASSED")
else:
    print(f"{len(failed)} VALIDATIONS FAILED")

,validation,status,details
0,OSSRH burst contains 29 events,PASS,29
1,OSSRH events are issue reclassifications,PASS,Checked
2,OSSRH event IDs unique,PASS,0
3,OSSRH transition values available,PASS,Checked
4,Issue ages nonnegative where available,PASS,Checked
5,Epic burst events are child additions,PASS,205
6,Epic-child identifiers available,PASS,Checked
7,Scope checkpoint count preserved,PASS,987


ALL SCOPE BURST REVIEW VALIDATIONS PASSED


### 4.5 Exclude administrative reclassifications

In [ ]:
import pandas as pd

# Preserve original structural events
final_scope_history = scope_structural_clean.copy()

# Flag the OSSRH administrative reclassification burst
ossrh_day = pd.Timestamp("2021-05-05", tz="UTC")

final_scope_history["suspected_admin_reclassification"] = (
    (final_scope_history["instance_id"] == "OSSRH-I01")
    &
    (final_scope_history["event_dt"].dt.floor("D") == ossrh_day)
    &
    (final_scope_history["structural_class"] == "ISSUE_RECLASSIFIED")
    &
    (
        final_scope_history["from_value"]
        .astype(str).str.strip().str.lower() == "task"
    )
    &
    (
        final_scope_history["to_value"]
        .astype(str).str.strip().str.lower()
        == "publishing support"
    )
)

print(
    "Flagged administrative reclassifications:",
    final_scope_history["suspected_admin_reclassification"].sum()
)

final_scope_eligible = final_scope_history[
    ~final_scope_history["suspected_admin_reclassification"]
].copy()

print("Final eligible structural events:", len(final_scope_eligible))
print("Projects represented:", final_scope_eligible["instance_id"].nunique())

Flagged administrative reclassifications: 29
Final eligible structural events: 938
Projects represented: 26


#### Export final historical structural scope events

Required by Notebook 2. Exports the **938 final eligible events**, after removing 29 suspected administrative reclassifications.

In [ ]:
# ============================================================
# EXPORT HISTORICAL STRUCTURAL SCOPE EVENTS FOR NOTEBOOK 2
# ============================================================
from pathlib import Path
import pandas as pd

scope_export_dir = Path("/content/drive/MyDrive/GC1/Temporal_snapshots")
scope_export_dir.mkdir(parents=True, exist_ok=True)

scope_events_export = final_scope_eligible.copy()
required_scope_columns = {"instance_id", "uid", "event_dt", "structural_class", "affected_issue"}
assert required_scope_columns.issubset(scope_events_export.columns), (
    f"Missing scope columns: {required_scope_columns - set(scope_events_export.columns)}"
)
assert len(scope_events_export) == 938, f"Expected 938 final scope events, got {len(scope_events_export)}"
assert scope_events_export["uid"].is_unique
assert scope_events_export["event_dt"].notna().all()
assert not scope_events_export["suspected_admin_reclassification"].fillna(False).astype(bool).any()

scope_export_path = scope_export_dir / "historical_structural_scope_events.parquet"
scope_events_export.to_parquet(scope_export_path, index=False)
print("Final historical scope events exported:", len(scope_events_export))
print("Projects:", scope_events_export["instance_id"].nunique())
print("File:", scope_export_path)


### 4.6 Final scope labels

In [ ]:
scope_final_records = []

for _, checkpoint in base_checkpoints.iterrows():

    project = checkpoint["instance_id"]
    t = checkpoint["checkpoint_time"]
    horizon_end = checkpoint["horizon_end"]

    future = final_scope_eligible[
        (final_scope_eligible["instance_id"] == project)
        &
        (final_scope_eligible["event_dt"] > t)
        &
        (final_scope_eligible["event_dt"] <= horizon_end)
    ]

    scope_final_records.append({
        "instance_id": project,
        "week_number": checkpoint["week_number"],
        "checkpoint_time": t,
        "horizon_end": horizon_end,
        "future_scope_events": len(future),
        "future_affected_issues": future["affected_issue"].nunique(),
        "future_epic_child_changes": future.loc[
            future["structural_class"].isin([
                "EPIC_CHILD_ADDED", "EPIC_CHILD_REMOVED"
            ]),
            "affected_issue"
        ].nunique(),
        "future_epic_link_changes": future.loc[
            future["structural_class"].isin([
                "EPIC_ASSIGNED",
                "EPIC_REASSIGNED",
                "EPIC_UNASSIGNED"
            ]),
            "affected_issue"
        ].nunique(),
        "future_reclassified_issues": future.loc[
            future["structural_class"] == "ISSUE_RECLASSIFIED",
            "affected_issue"
        ].nunique()
    })

final_scope_labels = pd.DataFrame(scope_final_records)

SCOPE_THRESHOLD = 3

final_scope_labels["scope_risk_label"] = (
    final_scope_labels["future_affected_issues"]
    >= SCOPE_THRESHOLD
).astype(int)

print("Total labelled checkpoints:", len(final_scope_labels))

print("\nScope Risk label distribution:")
display(final_scope_labels["scope_risk_label"].value_counts())

print("\nProjects with positive labels:")
print(
    final_scope_labels.loc[
        final_scope_labels["scope_risk_label"] == 1,
        "instance_id"
    ].nunique()
)

Total labelled checkpoints: 987

Scope Risk label distribution:


,count
scope_risk_label,
0,820
1,167



Projects with positive labels:
20


In [ ]:
keys = ["instance_id", "week_number", "checkpoint_time"]

combined_three_risks = (
    final_schedule_labels[
        keys + ["schedule_risk_label"]
    ]
    .merge(
        final_dependency_labels[
            keys + ["dependency_risk_label"]
        ],
        on=keys,
        how="inner",
        validate="one_to_one"
    )
    .merge(
        final_scope_labels[
            keys + ["scope_risk_label"]
        ],
        on=keys,
        how="inner",
        validate="one_to_one"
    )
)

print("Combined project-week records:", len(combined_three_risks))

print("\nRisk label combinations:")

display(
    combined_three_risks.groupby(
        [
            "schedule_risk_label",
            "dependency_risk_label",
            "scope_risk_label"
        ]
    )
    .size()
    .reset_index(name="count")
)

print("\nPositive counts:")
display(
    combined_three_risks[
        [
            "schedule_risk_label",
            "dependency_risk_label",
            "scope_risk_label"
        ]
    ].sum()
)

Combined project-week records: 987

Risk label combinations:


,schedule_risk_label,dependency_risk_label,scope_risk_label,count
0,0,0,0,567
1,0,0,1,124
2,0,1,0,23
3,0,1,1,8
4,1,0,0,185
5,1,0,1,23
6,1,1,0,45
7,1,1,1,12



Positive counts:


,0
schedule_risk_label,265
dependency_risk_label,88
scope_risk_label,167


In [ ]:
# ============================================================
# SCOPE RISK — FINAL CONSOLIDATED VALIDATION
# ============================================================

checks = []

def check(name, passed, details):
    checks.append({
        "validation": name,
        "status": "PASS" if passed else "FAIL",
        "details": details
    })

keys = ["instance_id", "week_number", "checkpoint_time"]

check(
    "29 OSSRH events flagged",
    final_scope_history["suspected_admin_reclassification"].sum() == 29,
    int(final_scope_history["suspected_admin_reclassification"].sum())
)

check(
    "938 eligible structural events",
    len(final_scope_eligible) == 938,
    len(final_scope_eligible)
)

check(
    "987 eligible checkpoints",
    len(final_scope_labels) == 987,
    len(final_scope_labels)
)

check(
    "All 30 projects represented",
    final_scope_labels["instance_id"].nunique() == 30,
    final_scope_labels["instance_id"].nunique()
)

check(
    "No duplicate checkpoints",
    not final_scope_labels.duplicated(keys).any(),
    int(final_scope_labels.duplicated(keys).sum())
)

check(
    "No missing labels",
    final_scope_labels["scope_risk_label"].notna().all(),
    int(final_scope_labels["scope_risk_label"].isna().sum())
)

check(
    "Binary labels",
    final_scope_labels["scope_risk_label"].isin([0, 1]).all(),
    "0 or 1"
)

check(
    "Labels follow threshold rule",
    (
        final_scope_labels["scope_risk_label"]
        ==
        (final_scope_labels["future_affected_issues"] >= SCOPE_THRESHOLD).astype(int)
    ).all(),
    f"Threshold = {SCOPE_THRESHOLD}"
)

check(
    "14-day prediction horizon",
    (
        final_scope_labels["horizon_end"]
        - final_scope_labels["checkpoint_time"]
        == pd.Timedelta(days=14)
    ).all(),
    "14 days"
)

check(
    "Complete future windows",
    (
        final_scope_labels["horizon_end"]
        <= final_scope_labels["instance_id"].map(
            checkpoints_df.groupby("instance_id")["observation_end"].max()
        )
    ).all(),
    "Checked"
)

reference_keys = set(
    map(tuple, final_schedule_labels[keys].itertuples(index=False, name=None))
)

scope_keys = set(
    map(tuple, final_scope_labels[keys].itertuples(index=False, name=None))
)

check(
    "Exact Schedule Risk alignment",
    scope_keys == reference_keys,
    len(scope_keys)
)

dependency_keys = set(
    map(tuple, final_dependency_labels[keys].itertuples(index=False, name=None))
)

check(
    "Exact Dependency Risk alignment",
    scope_keys == dependency_keys,
    len(scope_keys)
)

check(
    "Combined three-risk table complete",
    len(combined_three_risks) == 987,
    len(combined_three_risks)
)

check(
    "No missing combined labels",
    combined_three_risks[
        [
            "schedule_risk_label",
            "dependency_risk_label",
            "scope_risk_label"
        ]
    ].notna().all().all(),
    "Checked"
)

check(
    "No negative future outcome counts",
    (
        final_scope_labels[
            ["future_scope_events", "future_affected_issues"]
        ] >= 0
    ).all().all(),
    "Checked"
)

check(
    "Affected issues do not exceed event count",
    (
        final_scope_labels["future_affected_issues"]
        <= final_scope_labels["future_scope_events"]
    ).all(),
    "Checked"
)

validation_report = pd.DataFrame(checks)

display(validation_report)

failed = validation_report[
    validation_report["status"] == "FAIL"
]

print("\n" + "=" * 60)

if failed.empty:
    print("ALL FINAL SCOPE LABEL VALIDATIONS PASSED")
else:
    print(f"{len(failed)} VALIDATIONS FAILED")

print("=" * 60)

,validation,status,details
0,29 OSSRH events flagged,PASS,29
1,938 eligible structural events,PASS,938
2,987 eligible checkpoints,PASS,987
3,All 30 projects represented,PASS,30
4,No duplicate checkpoints,PASS,0
5,No missing labels,PASS,0
6,Binary labels,PASS,0 or 1
7,Labels follow threshold rule,PASS,Threshold = 3
8,14-day prediction horizon,PASS,14 days
9,Complete future windows,PASS,Checked



ALL FINAL SCOPE LABEL VALIDATIONS PASSED


In [ ]:
assert (validation_report["status"] == "PASS").all(), \
    "Validation failed. Do not export labels."

scope_output = GC1_ROOT / "Temporal_snapshots"/"scope_risk_labels.csv"
combined_output = GC1_ROOT / "Temporal_snapshots"/"phase2_three_risk_labels.csv"

final_scope_labels.to_csv(scope_output, index=False)
combined_three_risks.to_csv(combined_output, index=False)

print("Saved:", scope_output)
print("Saved:", combined_output)

Saved: /content/drive/MyDrive/GC1/Temporal_snapshots/scope_risk_labels.csv
Saved: /content/drive/MyDrive/GC1/Temporal_snapshots/phase2_three_risk_labels.csv


## Point-in-Time Temporal Snapshot Reconstruction

Reconstruct historical issue status and dependencies, create event–entity graph registries and build 987 checkpoint views.

### 5.1 Load temporal KG and checkpoints

In [ ]:
# ============================================================
# STEP 3.1 — LOAD PHASE 1 KG AND WEEKLY CHECKPOINTS
# ============================================================

import pandas as pd
import json
from pathlib import Path

GC1_ROOT = Path("/content/drive/MyDrive/GC1")

KG_DIR = GC1_ROOT / "entity_resolution_output"

print("KG directory:", KG_DIR)

# ------------------------------------------------------------
# 1. Locate exported KG files
# ------------------------------------------------------------

candidate_dirs = [
    KG_DIR / "temporal_kg_v2_output",
    KG_DIR / "temporal_kg_output",
    KG_DIR / "neo4j_import"
]

print("\nAvailable graph files:")

for folder in candidate_dirs:
    if folder.exists():
        print(f"\n{folder.name}")
        for file in folder.rglob("*"):
            if file.is_file():
                print(" ", file.relative_to(folder))

# ------------------------------------------------------------
# 2. Reuse loaded nodes if available
# ------------------------------------------------------------

if "nodes_df" in globals():
    snapshot_nodes = nodes_df.copy()
    print("\nNodes loaded from existing nodes_df")
else:
    snapshot_nodes = None
    print("\nNode DataFrame needs to be loaded.")

# ------------------------------------------------------------
# 3. Reuse relationships if available
# ------------------------------------------------------------

if "relationships_df" in globals():
    snapshot_edges = relationships_df.copy()

elif "edges_df" in globals():
    snapshot_edges = edges_df.copy()

elif "rels_df" in globals():
    snapshot_edges = rels_df.copy()

else:
    snapshot_edges = None
    print("Relationship DataFrame needs to be loaded.")

# ------------------------------------------------------------
# 4. Load weekly checkpoints
# ------------------------------------------------------------

snapshot_checkpoints = final_schedule_labels[
    [
        "instance_id",
        "week_number",
        "checkpoint_time",
        "horizon_end"
    ]
].copy()

snapshot_checkpoints["checkpoint_time"] = pd.to_datetime(
    snapshot_checkpoints["checkpoint_time"],
    utc=True
)

snapshot_checkpoints["horizon_end"] = pd.to_datetime(
    snapshot_checkpoints["horizon_end"],
    utc=True
)

print("\nCheckpoints:", len(snapshot_checkpoints))
print("Projects:", snapshot_checkpoints["instance_id"].nunique())

if snapshot_nodes is not None:
    print("Nodes:", len(snapshot_nodes))

if snapshot_edges is not None:
    print("Relationships:", len(snapshot_edges))

KG directory: /content/drive/MyDrive/GC1/entity_resolution_output

Available graph files:

temporal_kg_v2_output
  nodes.jsonl
  links.jsonl
  manifest.json
  README.md
  nodes/ActionItem.csv
  nodes/Blocker.csv
  nodes/CanonicalEntity.csv
  nodes/Decision.csv
  nodes/Dependency.csv
  nodes/Event.csv
  nodes/Evidence.csv
  nodes/Fact.csv
  nodes/SourceRecord.csv
  edges/ASSERTS.csv
  edges/FROM_ENTITY.csv
  edges/HAS_ACTIONITEM.csv
  edges/HAS_BLOCKER.csv
  edges/HAS_DECISION.csv
  edges/HAS_DEPENDENCY.csv
  edges/HAS_EVENT.csv
  edges/HAS_EVIDENCE.csv
  edges/MENTIONS.csv
  edges/REFERS_TO.csv
  edges/SUPPORTED_BY.csv
  edges/TO_ENTITY.csv
  neo4j/01_constraints.cypher
  neo4j/02_import.cypher
  neo4j/03_verify.cypher
  neo4j/04_visualize.cypher
  neo4j/05_point_in_time.cypher
  neo4j/06_event_timeline.cypher
  validation/review_items.jsonl
  validation/missing_observation_times.jsonl
  validation/jira_mapping_diagnostics.jsonl
  validation/document_diagnostics.jsonl
  validation/temp

In [ ]:
# ============================================================
# STEP 3.1 — LOAD PHASE 1 KG RELATIONSHIPS
# ============================================================

import pandas as pd
from pathlib import Path

KG_OUTPUT = (
    Path("/content/drive/MyDrive/GC1")
    / "entity_resolution_output"
    / "temporal_kg_v2_output"
)

# Load relationships from finalized Phase 1 export
snapshot_edges = pd.read_json(
    KG_OUTPUT / "links.jsonl",
    lines=True
)

# Preserve existing nodes
snapshot_nodes = nodes_df.copy()

print("Nodes loaded:", len(snapshot_nodes))
print("Relationships loaded:", len(snapshot_edges))
print("Weekly checkpoints:", len(snapshot_checkpoints))

print("\nNode fields:")
print(snapshot_nodes.columns.tolist())

print("\nRelationship fields:")
print(snapshot_edges.columns.tolist())

print("\nRelationship types:")
display(snapshot_edges["type"].value_counts())

# Basic integrity checks
assert len(snapshot_nodes) == 206634, \
    "Unexpected node count"

assert len(snapshot_edges) == 600623, \
    "Unexpected relationship count"

assert len(snapshot_checkpoints) == 987, \
    "Unexpected checkpoint count"

assert snapshot_edges["uid"].is_unique, \
    "Duplicate relationship IDs detected"

assert snapshot_edges["from_uid"].isin(
    snapshot_nodes["uid"]
).all(), "Invalid relationship source detected"

assert snapshot_edges["to_uid"].isin(
    snapshot_nodes["uid"]
).all(), "Invalid relationship target detected"

print("\nPHASE 1 KG LOADED SUCCESSFULLY")
print("Basic integrity checks passed.")

Nodes loaded: 206634
Relationships loaded: 600623
Weekly checkpoints: 987

Node fields:
['canonical_id', 'entity_type', 'instance_id', 'label', 'name', 'payload_json', 'project_key', 'uid', 'created_at', 'observed_at', 'temporal_basis', 'temporal_status', 'chunk_id', 'source_id', 'source_timestamp', 'source_type', 'evidence_text', 'event_time', 'valid_from', 'valid_to', 'relation_type']

Relationship fields:
['from_uid', 'instance_id', 'to_uid', 'type', 'uid', 'role']

Relationship types:


,count
type,
SUPPORTED_BY,154655
REFERS_TO,97561
HAS_EVENT,82577
TO_ENTITY,63305
ASSERTS,63305
FROM_ENTITY,63305
MENTIONS,61315
HAS_DEPENDENCY,7709
HAS_EVIDENCE,5827



PHASE 1 KG LOADED SUCCESSFULLY
Basic integrity checks passed.


In [ ]:
# ============================================================
# STEP 3.2.1 — DEFINE TEMPORAL NODE POLICIES
# ============================================================

import pandas as pd
import numpy as np

TEMPORAL_NODE_POLICIES = {
    "Event": "OBSERVATION_GATED",
    "Decision": "OBSERVATION_GATED",
    "ActionItem": "OBSERVATION_GATED",
    "Blocker": "OBSERVATION_GATED",
    "Dependency": "OBSERVATION_GATED",
    "Fact": "OBSERVATION_GATED",
    "Evidence": "SOURCE_GATED",
    "SourceRecord": "SOURCE_GATED",
    "CanonicalEntity": "FIRST_SUPPORTED_OBSERVATION"
}

snapshot_nodes["temporal_policy"] = (
    snapshot_nodes["label"].map(
        TEMPORAL_NODE_POLICIES
    )
)

print("TEMPORAL NODE POLICIES")

display(
    snapshot_nodes.groupby(
        ["label", "temporal_policy"],
        dropna=False
    )
    .size()
    .reset_index(name="node_count")
)

TEMPORAL NODE POLICIES


,label,temporal_policy,node_count
0,ActionItem,OBSERVATION_GATED,207
1,Blocker,OBSERVATION_GATED,443
2,CanonicalEntity,FIRST_SUPPORTED_OBSERVATION,40325
3,Decision,OBSERVATION_GATED,414
4,Dependency,OBSERVATION_GATED,7709
5,Event,OBSERVATION_GATED,82577
6,Evidence,SOURCE_GATED,5827
7,Fact,OBSERVATION_GATED,63305
8,SourceRecord,SOURCE_GATED,5827


In [ ]:
# ============================================================
# STEP 3.2.2 — INITIAL NODE AVAILABILITY TIMESTAMPS
# ============================================================

snapshot_nodes["snapshot_available_at"] = pd.to_datetime(
    snapshot_nodes["observed_at"],
    utc=True,
    errors="coerce",
    format="mixed"
)

snapshot_nodes["availability_basis"] = np.where(
    snapshot_nodes["snapshot_available_at"].notna(),
    "EXPLICIT_OBSERVED_AT",
    "UNKNOWN"
)

# Inspect temporal availability
availability_summary = (
    snapshot_nodes.groupby("label")
    .agg(
        total_nodes=("uid", "count"),
        available_nodes=("snapshot_available_at", "count")
    )
    .reset_index()
)

availability_summary["unknown_nodes"] = (
    availability_summary["total_nodes"]
    - availability_summary["available_nodes"]
)

availability_summary["coverage_pct"] = (
    availability_summary["available_nodes"]
    / availability_summary["total_nodes"] * 100
).round(2)

print("INITIAL AVAILABILITY COVERAGE")

display(availability_summary)

INITIAL AVAILABILITY COVERAGE


,label,total_nodes,available_nodes,unknown_nodes,coverage_pct
0,ActionItem,207,207,0,100.00
1,Blocker,443,167,276,37.70
2,CanonicalEntity,40325,5926,34399,14.70
3,Decision,414,414,0,100.00
4,Dependency,7709,237,7472,3.07
5,Event,82577,82396,181,99.78
6,Evidence,5827,67,5760,1.15
7,Fact,63305,3809,59496,6.02
8,SourceRecord,5827,67,5760,1.15


In [ ]:
# ============================================================
# STEP 3.2.3 — INSPECT SOURCE RECORD AVAILABILITY
# ============================================================

source_nodes = snapshot_nodes[
    snapshot_nodes["label"] == "SourceRecord"
].copy()

SOURCE_TIME_FIELDS = [
    "observed_at",
    "source_timestamp",
    "created_at"
]

for field in SOURCE_TIME_FIELDS:

    source_nodes[field + "_parsed"] = pd.to_datetime(
        source_nodes[field],
        utc=True,
        errors="coerce",
        format="mixed"
    )

print("SOURCE RECORD TEMPORAL COVERAGE")

display(
    pd.DataFrame({
        "field": SOURCE_TIME_FIELDS,
        "populated": [
            source_nodes[f + "_parsed"].notna().sum()
            for f in SOURCE_TIME_FIELDS
        ]
    })
)

print("\nSOURCE TYPES")

display(
    source_nodes["source_type"]
    .value_counts(dropna=False)
)

print("\nSAMPLE SOURCE RECORDS")

display(
    source_nodes[
        [
            "uid",
            "instance_id",
            "source_id",
            "source_type",
            "observed_at",
            "source_timestamp",
            "temporal_basis",
            "temporal_status"
        ]
    ].head(20)
)

SOURCE RECORD TEMPORAL COVERAGE


,field,populated
0,observed_at,67
1,source_timestamp,67
2,created_at,0



SOURCE TYPES


,count
source_type,
JIRA,5737
MEETING,60
PRD,30



SAMPLE SOURCE RECORDS


,uid,instance_id,source_id,source_type,observed_at,source_timestamp,temporal_basis,temporal_status
40325,R_f20150ac2dd405a99ad2c161,JBEAP-I01,JBEAP-7698,JIRA,None,None,unavailable,UNKNOWN
40340,R_de766dac549e33345755ad8f,JBEAP-I01,JBEAP-7694,JIRA,None,None,unavailable,UNKNOWN
40357,R_b5c38debbc7a5a9be2391f0f,JBEAP-I01,JBEAP-5471,JIRA,None,None,unavailable,UNKNOWN
40418,R_63a7cf156590b38f0744548d,JBEAP-I01,JBEAP-5464,JIRA,None,None,unavailable,UNKNOWN
40470,R_72b8d8d8fc658bf747c66b21,JBEAP-I01,JBEAP-5322,JIRA,None,None,unavailable,UNKNOWN
40481,R_e815119fa287c4e0b1ef9548,JBEAP-I01,JBEAP-5275,JIRA,None,None,unavailable,UNKNOWN
40503,R_9af21691a815aa38db2a09fb,JBEAP-I01,JBEAP-5248,JIRA,None,None,unavailable,UNKNOWN
40542,R_a727e128c6640dd1570558f0,JBEAP-I01,JBEAP-5161,JIRA,None,None,unavailable,UNKNOWN
40573,R_5cacbce653dce00802cc3b46,JBEAP-I01,JBEAP-5136,JIRA,None,None,unavailable,UNKNOWN
40602,R_996117a4409d4426acd4a7f6,JBEAP-I01,JBEAP-4962,JIRA,None,None,unavailable,UNKNOWN


In [ ]:
# ============================================================
# STEP 3.2.4 — CLASSIFY RELATIONSHIP SEMANTICS
# ============================================================

node_labels = (
    snapshot_nodes.set_index("uid")["label"]
)

snapshot_edges["source_label"] = (
    snapshot_edges["from_uid"].map(node_labels)
)

snapshot_edges["target_label"] = (
    snapshot_edges["to_uid"].map(node_labels)
)

PROVENANCE_EDGE_TYPES = {
    "SUPPORTED_BY",
    "HAS_EVIDENCE",
    "ASSERTS",
    "MENTIONS"
}

EVENT_EDGE_TYPES = {
    "HAS_EVENT",
    "HAS_DECISION",
    "HAS_ACTIONITEM",
    "HAS_BLOCKER",
    "HAS_DEPENDENCY"
}

ENTITY_EDGE_TYPES = {
    "REFERS_TO",
    "FROM_ENTITY",
    "TO_ENTITY"
}

def classify_edge(edge_type):

    if edge_type in PROVENANCE_EDGE_TYPES:
        return "PROVENANCE"

    if edge_type in EVENT_EDGE_TYPES:
        return "EVENT_ASSOCIATION"

    if edge_type in ENTITY_EDGE_TYPES:
        return "ENTITY_ASSOCIATION"

    return "OTHER"


snapshot_edges["edge_category"] = (
    snapshot_edges["type"].apply(classify_edge)
)

print("EDGE CATEGORY DISTRIBUTION")

display(
    snapshot_edges["edge_category"]
    .value_counts()
)

print("\nPROVENANCE RELATIONSHIP DIRECTIONS")

display(
    snapshot_edges[
        snapshot_edges["edge_category"] == "PROVENANCE"
    ]
    .groupby(
        ["type", "source_label", "target_label"]
    )
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

EDGE CATEGORY DISTRIBUTION


,count
edge_category,
PROVENANCE,285102
ENTITY_ASSOCIATION,224171
EVENT_ASSOCIATION,91350



PROVENANCE RELATIONSHIP DIRECTIONS


,type,source_label,target_label,count
7,SUPPORTED_BY,Event,Evidence,82577
0,ASSERTS,SourceRecord,Fact,63305
8,SUPPORTED_BY,Fact,Evidence,63305
2,MENTIONS,SourceRecord,CanonicalEntity,61315
6,SUPPORTED_BY,Dependency,Evidence,7709
1,HAS_EVIDENCE,SourceRecord,Evidence,5827
4,SUPPORTED_BY,Blocker,Evidence,443
5,SUPPORTED_BY,Decision,Evidence,414
3,SUPPORTED_BY,ActionItem,Evidence,207


In [ ]:
# ============================================================
# STEP 3.2.5 — TEMPORAL ELIGIBILITY STATUS
# ============================================================

snapshot_nodes["temporal_eligibility"] = np.where(
    snapshot_nodes["snapshot_available_at"].notna(),
    "TIMESTAMPED",
    "UNRESOLVED"
)

print("NODE TEMPORAL ELIGIBILITY")

display(
    pd.crosstab(
        snapshot_nodes["label"],
        snapshot_nodes["temporal_eligibility"]
    )
)

print("\nTOTAL TIMESTAMPED NODES:",
      (snapshot_nodes["temporal_eligibility"] == "TIMESTAMPED").sum())

print("TOTAL UNRESOLVED NODES:",
      (snapshot_nodes["temporal_eligibility"] == "UNRESOLVED").sum())

NODE TEMPORAL ELIGIBILITY


temporal_eligibility,TIMESTAMPED,UNRESOLVED
label,,
ActionItem,207,0
Blocker,167,276
CanonicalEntity,5926,34399
Decision,414,0
Dependency,237,7472
Event,82396,181
Evidence,67,5760
Fact,3809,59496
SourceRecord,67,5760



TOTAL TIMESTAMPED NODES: 93290
TOTAL UNRESOLVED NODES: 113344


In [ ]:
# ============================================================
# STEP 3.2 — CONSOLIDATED TEMPORAL POLICY VALIDATION
# ============================================================

checks = []

def check(name, passed, detail):
    checks.append({
        "validation": name,
        "status": "PASS" if passed else "FAIL",
        "details": detail
    })

# 1. Node count
check(
    "Node count preserved",
    len(snapshot_nodes) == 206634,
    len(snapshot_nodes)
)

# 2. Edge count
check(
    "Relationship count preserved",
    len(snapshot_edges) == 600623,
    len(snapshot_edges)
)

# 3. All node policies assigned
check(
    "All node types have temporal policies",
    snapshot_nodes["temporal_policy"].notna().all(),
    snapshot_nodes["temporal_policy"].isna().sum()
)

# 4. All nodes have eligibility status
check(
    "All nodes classified by temporal eligibility",
    snapshot_nodes["temporal_eligibility"].notna().all(),
    snapshot_nodes["temporal_eligibility"].isna().sum()
)

# 5. Timestamped nodes have availability timestamps
timestamped = snapshot_nodes[
    snapshot_nodes["temporal_eligibility"] == "TIMESTAMPED"
]

check(
    "Timestamped nodes have valid timestamps",
    timestamped["snapshot_available_at"].notna().all(),
    len(timestamped)
)

# 6. Unresolved nodes have no assigned timestamp
unresolved = snapshot_nodes[
    snapshot_nodes["temporal_eligibility"] == "UNRESOLVED"
]

check(
    "Unresolved nodes remain untimestamped",
    unresolved["snapshot_available_at"].isna().all(),
    len(unresolved)
)

# 7. No future timestamps artificially assigned
check(
    "No timestamps assigned without observation evidence",
    (
        snapshot_nodes.loc[
            snapshot_nodes["availability_basis"]
            == "EXPLICIT_OBSERVED_AT",
            "snapshot_available_at"
        ].notna().all()
    ),
    "Explicit observations only"
)

# 8. All edges classified
check(
    "All relationship categories assigned",
    snapshot_edges["edge_category"].notna().all(),
    snapshot_edges["edge_category"].isna().sum()
)

# 9. Relationship endpoints valid
node_ids = set(snapshot_nodes["uid"])

check(
    "All edge sources valid",
    snapshot_edges["from_uid"].isin(node_ids).all(),
    "Checked"
)

check(
    "All edge targets valid",
    snapshot_edges["to_uid"].isin(node_ids).all(),
    "Checked"
)

# 10. Checkpoint count
check(
    "987 eligible checkpoints preserved",
    len(snapshot_checkpoints) == 987,
    len(snapshot_checkpoints)
)

# 11. Project count
check(
    "30 projects represented",
    snapshot_checkpoints["instance_id"].nunique() == 30,
    snapshot_checkpoints["instance_id"].nunique()
)

# 12. Unknown timestamps are not silently imputed
check(
    "Unknown availability remains unresolved",
    (
        snapshot_nodes.loc[
            snapshot_nodes["observed_at"].isna(),
            "snapshot_available_at"
        ].isna().all()
    ),
    "No automatic fallback"
)

validation_report = pd.DataFrame(checks)

display(validation_report)

failed = validation_report[
    validation_report["status"] == "FAIL"
]

print("\n" + "=" * 60)

if failed.empty:
    print("ALL TEMPORAL POLICY VALIDATIONS PASSED")
else:
    print(f"{len(failed)} VALIDATIONS FAILED")

print("=" * 60)

print("\nTimestamped nodes:", len(timestamped))
print("Unresolved nodes:", len(unresolved))

,validation,status,details
0,Node count preserved,PASS,206634
1,Relationship count preserved,PASS,600623
2,All node types have temporal policies,PASS,0
3,All nodes classified by temporal eligibility,PASS,0
4,Timestamped nodes have valid timestamps,PASS,93290
5,Unresolved nodes remain untimestamped,PASS,113344
6,No timestamps assigned without observation evi...,PASS,Explicit observations only
7,All relationship categories assigned,PASS,0
8,All edge sources valid,PASS,Checked
9,All edge targets valid,PASS,Checked



ALL TEMPORAL POLICY VALIDATIONS PASSED

Timestamped nodes: 93290
Unresolved nodes: 113344


### 5.2 Historical Jira status reconstruction

In [ ]:
# ============================================================
# STEP 3.3.1 — PREPARE JIRA STATUS HISTORY
# ============================================================

import pandas as pd
import numpy as np
import json
from pathlib import Path

GC1_ROOT = Path("/content/drive/MyDrive/GC1")

# Reuse existing Jira DataFrame
jira_history_df = jira_df.copy()

# Parse issue creation timestamps
jira_history_df["created_dt"] = pd.to_datetime(
    jira_history_df["created_at"],
    utc=True,
    errors="coerce",
    format="mixed"
)

jira_history_df["updated_dt"] = pd.to_datetime(
    jira_history_df["updated_at"],
    utc=True,
    errors="coerce",
    format="mixed"
)

jira_history_df["resolved_dt"] = pd.to_datetime(
    jira_history_df["resolved_at"],
    utc=True,
    errors="coerce",
    format="mixed"
)

# Map Jira project keys to Phase 1 instance IDs
project_instance_lookup = {
    instance.split("-I")[0]: instance
    for instance in snapshot_checkpoints["instance_id"].unique()
}

jira_history_df["instance_id"] = (
    jira_history_df["project_key"]
    .map(project_instance_lookup)
)

print("Total Jira tickets:", len(jira_history_df))

print(
    "Projects represented:",
    jira_history_df["instance_id"].nunique()
)

print(
    "Tickets with creation timestamps:",
    jira_history_df["created_dt"].notna().sum()
)

print(
    "Tickets with nonempty status histories:",
    jira_history_df["status_history"]
    .apply(lambda x: isinstance(x, list) and len(x) > 0)
    .sum()
)

display(
    jira_history_df[
        [
            "instance_id",
            "issue_key",
            "status",
            "created_dt",
            "updated_dt",
            "resolved_dt"
        ]
    ].head(10)
)

Total Jira tickets: 9000
Projects represented: 30
Tickets with creation timestamps: 9000
Tickets with nonempty status histories: 8774


,instance_id,issue_key,status,created_dt,updated_dt,resolved_dt
0,JBEAP-I01,JBEAP-22995,Pull Request Sent,2021-12-21 14:39:41+00:00,2021-12-24 12:33:29+00:00,NaT
1,JBEAP-I01,JBEAP-22981,Closed,2021-12-17 05:27:13+00:00,2021-12-27 12:20:46+00:00,2021-12-23 11:41:14+00:00
2,JBEAP-I01,JBEAP-22948,Verified,2021-12-14 11:47:45+00:00,2021-12-21 06:13:45+00:00,2021-12-14 15:05:09+00:00
3,JBEAP-I01,JBEAP-22945,Pull Request Sent,2021-12-13 15:16:58+00:00,2022-01-04 14:36:51+00:00,NaT
4,JBEAP-I01,JBEAP-22917,Verified,2021-12-10 06:17:20+00:00,2021-12-20 08:13:36+00:00,2021-12-17 11:51:07+00:00
5,JBEAP-I01,JBEAP-22916,Reopened,2021-12-09 20:45:32+00:00,2021-12-21 08:21:13+00:00,NaT
6,JBEAP-I01,JBEAP-22915,Verified,2021-12-09 08:13:35+00:00,2021-12-20 08:06:03+00:00,2021-12-14 11:47:50+00:00
7,JBEAP-I01,JBEAP-22914,Verified,2021-12-09 06:19:35+00:00,2021-12-20 08:24:21+00:00,2021-12-17 06:09:59+00:00
8,JBEAP-I01,JBEAP-22888,Closed,2021-12-06 19:29:16+00:00,2021-12-10 15:37:43+00:00,2021-12-10 15:36:31+00:00
9,JBEAP-I01,JBEAP-22885,Resolved,2021-12-06 14:40:11+00:00,2021-12-09 15:35:00+00:00,2021-12-09 15:25:36+00:00


In [ ]:
# ============================================================
# STEP 3.3.2 — EXTRACT STATUS TRANSITIONS
# ============================================================

transition_records = []

for _, issue in jira_history_df.iterrows():

    history = issue["status_history"]

    if not isinstance(history, list):
        continue

    for index, change in enumerate(history):

        if not isinstance(change, dict):
            continue

        transition_records.append({
            "instance_id": issue["instance_id"],
            "issue_key": issue["issue_key"],
            "transition_index": index,
            "changed_at": change.get("changed_at"),
            "from_status": change.get("from_status"),
            "to_status": change.get("to_status")
        })

status_transitions = pd.DataFrame(
    transition_records,
    columns=[
        "instance_id",
        "issue_key",
        "transition_index",
        "changed_at",
        "from_status",
        "to_status"
    ]
)

status_transitions["changed_dt"] = pd.to_datetime(
    status_transitions["changed_at"],
    utc=True,
    errors="coerce",
    format="mixed"
)

print("Total status transitions:", len(status_transitions))

print(
    "Transitions with missing timestamps:",
    status_transitions["changed_dt"].isna().sum()
)

print(
    "Transitions with missing FROM status:",
    status_transitions["from_status"].isna().sum()
)

print(
    "Transitions with missing TO status:",
    status_transitions["to_status"].isna().sum()
)

print("\nMost common transitions:")

display(
    status_transitions.groupby(
        ["from_status", "to_status"],
        dropna=False
    )
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(25)
)

Total status transitions: 63337
Transitions with missing timestamps: 0
Transitions with missing FROM status: 0
Transitions with missing TO status: 0

Most common transitions:


,from_status,to_status,count
430,Open,In Progress,3671
648,Resolved,Closed,3186
664,Resolved,Reopened,3091
460,Open,Resolved,2640
630,Reopened,Resolved,2488
628,Reopened,Reopened,2242
445,Open,Open,1834
86,Closed,Reopened,1527
446,Open,Patch Available,1522
417,Open,Closed,1419


In [ ]:
# ============================================================
# STEP 3.3.3 — RECONSTRUCT STATUS INTERVALS
# ============================================================

status_interval_records = []
status_reconstruction_issues = []

transitions_by_issue = {
    key: group.sort_values(
        ["changed_dt", "transition_index"],
        kind="stable"
    )
    for key, group in status_transitions.groupby(
        ["instance_id", "issue_key"]
    )
}

for _, issue in jira_history_df.iterrows():

    instance_id = issue["instance_id"]
    issue_key = issue["issue_key"]
    created_dt = issue["created_dt"]

    if pd.isna(created_dt):
        status_reconstruction_issues.append({
            "instance_id": instance_id,
            "issue_key": issue_key,
            "reason": "MISSING_CREATION_TIME"
        })
        continue

    history = transitions_by_issue.get(
        (instance_id, issue_key)
    )

    if history is None or history.empty:

        status_interval_records.append({
            "instance_id": instance_id,
            "issue_key": issue_key,
            "valid_from": created_dt,
            "valid_to": pd.NaT,
            "historical_status": "UNKNOWN",
            "status_basis": "NO_STATUS_HISTORY"
        })

        continue

    valid_history = history[
        history["changed_dt"].notna()
    ].copy()

    valid_history = valid_history[
        valid_history["changed_dt"] >= created_dt
    ]

    if valid_history.empty:

        status_interval_records.append({
            "instance_id": instance_id,
            "issue_key": issue_key,
            "valid_from": created_dt,
            "valid_to": pd.NaT,
            "historical_status": "UNKNOWN",
            "status_basis": "NO_VALID_TRANSITIONS"
        })

        continue

    first_change = valid_history.iloc[0]

    initial_status = first_change["from_status"]

    if pd.isna(initial_status) or str(initial_status).strip() == "":
        initial_status = "UNKNOWN"

    current_status = str(initial_status)
    current_start = created_dt

    for _, change in valid_history.iterrows():

        change_time = change["changed_dt"]
        new_status = change["to_status"]

        if pd.isna(new_status) or str(new_status).strip() == "":
            status_reconstruction_issues.append({
                "instance_id": instance_id,
                "issue_key": issue_key,
                "reason": "MISSING_TARGET_STATUS"
            })
            continue

        # Close previous status interval
        if change_time > current_start:

            status_interval_records.append({
                "instance_id": instance_id,
                "issue_key": issue_key,
                "valid_from": current_start,
                "valid_to": change_time,
                "historical_status": current_status,
                "status_basis": "RECONSTRUCTED_HISTORY"
            })

        # Begin new status interval
        current_status = str(new_status)
        current_start = change_time

    # Final interval remains open
    status_interval_records.append({
        "instance_id": instance_id,
        "issue_key": issue_key,
        "valid_from": current_start,
        "valid_to": pd.NaT,
        "historical_status": current_status,
        "status_basis": "RECONSTRUCTED_HISTORY"
    })


status_intervals_df = pd.DataFrame(status_interval_records)

status_intervals_df["valid_from"] = pd.to_datetime(
    status_intervals_df["valid_from"],
    utc=True
)

status_intervals_df["valid_to"] = pd.to_datetime(
    status_intervals_df["valid_to"],
    utc=True
)

status_reconstruction_issues_df = pd.DataFrame(
    status_reconstruction_issues,
    columns=["instance_id", "issue_key", "reason"]
)

print("Reconstructed status intervals:", len(status_intervals_df))

print(
    "Issues with reconstruction warnings:",
    status_reconstruction_issues_df["issue_key"].nunique()
)

print("\nStatus basis distribution:")

display(status_intervals_df["status_basis"].value_counts())

print("\nSample intervals:")

display(
    status_intervals_df.sort_values(
        ["instance_id", "issue_key", "valid_from"]
    ).head(20)
)

Reconstructed status intervals: 72256
Issues with reconstruction warnings: 0

Status basis distribution:


,count
status_basis,
RECONSTRUCTED_HISTORY,72030
NO_STATUS_HISTORY,226



Sample intervals:


,instance_id,issue_key,valid_from,valid_to,historical_status,status_basis
53432,AEROGEAR-I01,AEROGEAR-10005,2019-10-04 06:26:26+00:00,2019-10-29 05:05:46+00:00,Open,RECONSTRUCTED_HISTORY
53433,AEROGEAR-I01,AEROGEAR-10005,2019-10-29 05:05:46+00:00,2019-11-05 05:37:13+00:00,Coding In Progress,RECONSTRUCTED_HISTORY
53434,AEROGEAR-I01,AEROGEAR-10005,2019-11-05 05:37:13+00:00,2019-11-05 10:04:00+00:00,Resolved,RECONSTRUCTED_HISTORY
53435,AEROGEAR-I01,AEROGEAR-10005,2019-11-05 10:04:00+00:00,2019-11-05 10:04:43+00:00,Reopened,RECONSTRUCTED_HISTORY
53436,AEROGEAR-I01,AEROGEAR-10005,2019-11-05 10:04:43+00:00,2019-11-05 10:08:56+00:00,Pull Request Sent,RECONSTRUCTED_HISTORY
53437,AEROGEAR-I01,AEROGEAR-10005,2019-11-05 10:08:56+00:00,2019-11-05 10:09:23+00:00,Coding In Progress,RECONSTRUCTED_HISTORY
53438,AEROGEAR-I01,AEROGEAR-10005,2019-11-05 10:09:23+00:00,2019-11-05 10:19:33+00:00,Pull Request Sent,RECONSTRUCTED_HISTORY
53439,AEROGEAR-I01,AEROGEAR-10005,2019-11-05 10:19:33+00:00,2019-11-05 10:19:43+00:00,Coding In Progress,RECONSTRUCTED_HISTORY
53440,AEROGEAR-I01,AEROGEAR-10005,2019-11-05 10:19:43+00:00,2019-11-11 04:47:43+00:00,Pull Request Sent,RECONSTRUCTED_HISTORY
53441,AEROGEAR-I01,AEROGEAR-10005,2019-11-11 04:47:43+00:00,NaT,Resolved,RECONSTRUCTED_HISTORY


In [ ]:
# ============================================================
# STEP 3.3.4 — AUDIT STATUS HISTORY CONSISTENCY
# ============================================================

transition_anomalies = []

for (instance_id, issue_key), group in status_transitions.groupby(
    ["instance_id", "issue_key"]
):

    group = group.sort_values(
        ["changed_dt", "transition_index"],
        kind="stable"
    )

    previous_to = None

    for _, row in group.iterrows():

        old_status = row["from_status"]
        new_status = row["to_status"]

        if (
            pd.notna(old_status)
            and pd.notna(new_status)
            and str(old_status).strip().lower()
            == str(new_status).strip().lower()
        ):
            transition_anomalies.append({
                "instance_id": instance_id,
                "issue_key": issue_key,
                "reason": "NO_OP_TRANSITION",
                "changed_dt": row["changed_dt"]
            })

        if (
            previous_to is not None
            and pd.notna(old_status)
            and str(previous_to).strip().lower()
            != str(old_status).strip().lower()
        ):
            transition_anomalies.append({
                "instance_id": instance_id,
                "issue_key": issue_key,
                "reason": "STATUS_CHAIN_MISMATCH",
                "changed_dt": row["changed_dt"]
            })

        if pd.notna(new_status):
            previous_to = new_status


transition_anomalies_df = pd.DataFrame(
    transition_anomalies,
    columns=[
        "instance_id",
        "issue_key",
        "reason",
        "changed_dt"
    ]
)

print("Transition anomalies:", len(transition_anomalies_df))

display(
    transition_anomalies_df["reason"]
    .value_counts()
)

print("\nAffected projects:")

display(
    transition_anomalies_df.groupby(
        ["instance_id", "reason"]
    )
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(20)
)

Transition anomalies: 5369


,count
reason,
NO_OP_TRANSITION,5181
STATUS_CHAIN_MISMATCH,188



Affected projects:


,instance_id,reason,count
18,MC-I01,NO_OP_TRANSITION,3899
23,QTBUG-I01,NO_OP_TRANSITION,207
30,SERVER-I01,NO_OP_TRANSITION,206
11,JBDS-I01,NO_OP_TRANSITION,170
12,JBEAP-I01,NO_OP_TRANSITION,167
6,DOCS-I01,NO_OP_TRANSITION,87
8,EVG-I01,NO_OP_TRANSITION,79
25,QTCREATORBUG-I01,NO_OP_TRANSITION,66
29,SAK-I01,STATUS_CHAIN_MISMATCH,66
13,JBIDE-I01,NO_OP_TRANSITION,61


In [ ]:
# ============================================================
# STEP 3.3.5 — GENERATE WEEKLY JIRA ISSUE STATES
# ============================================================

checkpoint_issue_records = []

# Group intervals by project for efficient filtering
intervals_by_project = {
    project: group
    for project, group in status_intervals_df.groupby("instance_id")
}

for _, checkpoint in snapshot_checkpoints.iterrows():

    instance_id = checkpoint["instance_id"]
    checkpoint_time = checkpoint["checkpoint_time"]
    week_number = checkpoint["week_number"]

    project_intervals = intervals_by_project.get(instance_id)

    if project_intervals is None:
        continue

    # Include intervals active at the checkpoint
    active_intervals = project_intervals[
        (project_intervals["valid_from"] <= checkpoint_time)
        &
        (
            project_intervals["valid_to"].isna()
            |
            (project_intervals["valid_to"] > checkpoint_time)
        )
    ]

    for _, issue in active_intervals.iterrows():

        checkpoint_issue_records.append({
            "instance_id": instance_id,
            "week_number": week_number,
            "checkpoint_time": checkpoint_time,
            "issue_key": issue["issue_key"],
            "historical_status": issue["historical_status"],
            "status_basis": issue["status_basis"]
        })


checkpoint_issue_states_df = pd.DataFrame(
    checkpoint_issue_records,
    columns=[
        "instance_id",
        "week_number",
        "checkpoint_time",
        "issue_key",
        "historical_status",
        "status_basis"
    ]
)

print(
    "Total checkpoint-issue state records:",
    len(checkpoint_issue_states_df)
)

print(
    "Unique Jira issues represented:",
    checkpoint_issue_states_df["issue_key"].nunique()
)

print(
    "Project-week checkpoints represented:",
    checkpoint_issue_states_df[
        ["instance_id", "week_number"]
    ].drop_duplicates().shape[0]
)

print("\nHistorical status distribution:")

display(
    checkpoint_issue_states_df["historical_status"]
    .value_counts()
    .head(20)
)

print("\nSample checkpoint states:")

display(checkpoint_issue_states_df.head(20))

Total checkpoint-issue state records: 228255
Unique Jira issues represented: 7363
Project-week checkpoints represented: 987

Historical status distribution:


,count
historical_status,
Closed,118465
Resolved,50819
Open,23598
Reopened,8133
UNKNOWN,4288
Gathering Interest,3732
In Progress,2850
Done,2485
Verified,1831



Sample checkpoint states:


,instance_id,week_number,checkpoint_time,issue_key,historical_status,status_basis
0,AEROGEAR-I01,1,2018-01-24 05:30:01+00:00,AEROGEAR-7517,Resolved,RECONSTRUCTED_HISTORY
1,AEROGEAR-I01,1,2018-01-24 05:30:01+00:00,AEROGEAR-7491,Closed,RECONSTRUCTED_HISTORY
2,AEROGEAR-I01,1,2018-01-24 05:30:01+00:00,AEROGEAR-7474,Closed,RECONSTRUCTED_HISTORY
3,AEROGEAR-I01,1,2018-01-24 05:30:01+00:00,AEROGEAR-7390,Closed,RECONSTRUCTED_HISTORY
4,AEROGEAR-I01,1,2018-01-24 05:30:01+00:00,AEROGEAR-7383,Closed,RECONSTRUCTED_HISTORY
5,AEROGEAR-I01,1,2018-01-24 05:30:01+00:00,AEROGEAR-7344,Closed,RECONSTRUCTED_HISTORY
6,AEROGEAR-I01,1,2018-01-24 05:30:01+00:00,AEROGEAR-7256,Resolved,RECONSTRUCTED_HISTORY
7,AEROGEAR-I01,1,2018-01-24 05:30:01+00:00,AEROGEAR-7200,Closed,RECONSTRUCTED_HISTORY
8,AEROGEAR-I01,1,2018-01-24 05:30:01+00:00,AEROGEAR-7184,Closed,RECONSTRUCTED_HISTORY
9,AEROGEAR-I01,1,2018-01-24 05:30:01+00:00,AEROGEAR-7159,Closed,RECONSTRUCTED_HISTORY


In [ ]:
# ============================================================
# STEP 3.3 — CONSOLIDATED HISTORICAL STATE VALIDATION
# ============================================================

checks = []

def check(name, passed, detail):
    checks.append({
        "validation": name,
        "status": "PASS" if passed else "FAIL",
        "details": detail
    })

# 1. Jira ticket count
check(
    "9,000 Jira tickets preserved",
    len(jira_history_df) == 9000,
    len(jira_history_df)
)

# 2. Project coverage
check(
    "30 projects represented",
    jira_history_df["instance_id"].nunique() == 30,
    jira_history_df["instance_id"].nunique()
)

# 3. Creation timestamps
check(
    "All Jira creation timestamps available",
    jira_history_df["created_dt"].notna().all(),
    jira_history_df["created_dt"].isna().sum()
)

# 4. Unique Jira issue keys
check(
    "Unique Jira issue keys",
    jira_history_df["issue_key"].is_unique,
    jira_history_df["issue_key"].duplicated().sum()
)

# 5. Status interval start timestamps
check(
    "All intervals have start timestamps",
    status_intervals_df["valid_from"].notna().all(),
    status_intervals_df["valid_from"].isna().sum()
)

# 6. No negative intervals
invalid_intervals = (
    status_intervals_df["valid_to"].notna()
    &
    (
        status_intervals_df["valid_to"]
        < status_intervals_df["valid_from"]
    )
).sum()

check(
    "No negative status intervals",
    invalid_intervals == 0,
    invalid_intervals
)

# 7. All intervals have status
check(
    "All intervals have a status value",
    status_intervals_df["historical_status"].notna().all(),
    status_intervals_df["historical_status"].isna().sum()
)

# 8. All interval issues are known
known_issues = set(jira_history_df["issue_key"])

check(
    "All interval issues belong to Jira dataset",
    status_intervals_df["issue_key"].isin(known_issues).all(),
    "Checked"
)

# 9. No duplicate checkpoint-issue states
duplicate_states = checkpoint_issue_states_df.duplicated(
    [
        "instance_id",
        "week_number",
        "checkpoint_time",
        "issue_key"
    ]
).sum()

check(
    "No duplicate checkpoint-issue states",
    duplicate_states == 0,
    duplicate_states
)

# 10. No future-created issues
creation_lookup = (
    jira_history_df.set_index("issue_key")["created_dt"]
)

future_issues = (
    checkpoint_issue_states_df["issue_key"]
    .map(creation_lookup)
    >
    checkpoint_issue_states_df["checkpoint_time"]
).sum()

check(
    "No issues appear before creation",
    future_issues == 0,
    future_issues
)

# 11. All checkpoint states belong to known checkpoints
checkpoint_keys = set(
    map(
        tuple,
        snapshot_checkpoints[
            ["instance_id", "week_number", "checkpoint_time"]
        ].itertuples(index=False, name=None)
    )
)

state_keys = set(
    map(
        tuple,
        checkpoint_issue_states_df[
            ["instance_id", "week_number", "checkpoint_time"]
        ].itertuples(index=False, name=None)
    )
)

check(
    "All reconstructed states belong to valid checkpoints",
    state_keys.issubset(checkpoint_keys),
    len(state_keys)
)

# 12. No missing checkpoint statuses
check(
    "No missing historical status values",
    checkpoint_issue_states_df["historical_status"].notna().all(),
    checkpoint_issue_states_df["historical_status"].isna().sum()
)

# 13. 987 checkpoints retained
check(
    "987 original checkpoints preserved",
    len(snapshot_checkpoints) == 987,
    len(snapshot_checkpoints)
)

# 14. No final-state fallback for missing histories
no_history_intervals = status_intervals_df[
    status_intervals_df["status_basis"] == "NO_STATUS_HISTORY"
]

check(
    "Missing histories marked UNKNOWN",
    (
        no_history_intervals["historical_status"]
        == "UNKNOWN"
    ).all(),
    len(no_history_intervals)
)

# ------------------------------------------------------------
# REPORT
# ------------------------------------------------------------

validation_df = pd.DataFrame(checks)

display(validation_df)

failed = validation_df[
    validation_df["status"] == "FAIL"
]

print("\n" + "=" * 60)

if failed.empty:
    print("ALL HISTORICAL JIRA STATE VALIDATIONS PASSED")
else:
    print(f"{len(failed)} VALIDATIONS FAILED")

print("=" * 60)

print("\nStatus transitions:", len(status_transitions))
print("Status intervals:", len(status_intervals_df))
print("Checkpoint issue states:", len(checkpoint_issue_states_df))
print("Transition anomalies:", len(transition_anomalies_df))

,validation,status,details
0,"9,000 Jira tickets preserved",PASS,9000
1,30 projects represented,PASS,30
2,All Jira creation timestamps available,PASS,0
3,Unique Jira issue keys,PASS,0
4,All intervals have start timestamps,PASS,0
5,No negative status intervals,PASS,0
6,All intervals have a status value,PASS,0
7,All interval issues belong to Jira dataset,PASS,Checked
8,No duplicate checkpoint-issue states,PASS,0
9,No issues appear before creation,PASS,0



ALL HISTORICAL JIRA STATE VALIDATIONS PASSED

Status transitions: 63337
Status intervals: 72256
Checkpoint issue states: 228255
Transition anomalies: 5369


In [ ]:
# ============================================================
# EXPORT HISTORICAL JIRA STATES
# ============================================================

assert (validation_df["status"] == "PASS").all(), \
    "Validation failed. Review before exporting."

output_dir = GC1_ROOT / "entity_resolution_output"

status_intervals_df.to_csv(
    output_dir / "jira_status_intervals.csv",
    index=False
)

checkpoint_issue_states_df.to_csv(
    output_dir / "jira_checkpoint_issue_states.csv",
    index=False
)

transition_anomalies_df.to_csv(
    output_dir / "jira_status_transition_anomalies.csv",
    index=False
)

print("Historical Jira states exported successfully.")

Historical Jira states exported successfully.


In [ ]:
# ============================================================
# STEP 3.3.8 — INSPECT STATUS-CHAIN MISMATCHES
# ============================================================

import pandas as pd

mismatch_details = []

for (instance_id, issue_key), group in status_transitions.groupby(
    ["instance_id", "issue_key"]
):

    group = group.sort_values(
        ["changed_dt", "transition_index"],
        kind="stable"
    ).reset_index(drop=True)

    for i in range(1, len(group)):

        previous = group.iloc[i - 1]
        current = group.iloc[i]

        previous_to = str(previous["to_status"]).strip().lower()
        current_from = str(current["from_status"]).strip().lower()

        if previous_to != current_from:

            mismatch_details.append({
                "instance_id": instance_id,
                "issue_key": issue_key,
                "previous_time": previous["changed_dt"],
                "current_time": current["changed_dt"],
                "previous_from": previous["from_status"],
                "previous_to": previous["to_status"],
                "current_from": current["from_status"],
                "current_to": current["to_status"],
                "gap_hours": (
                    current["changed_dt"]
                    - previous["changed_dt"]
                ).total_seconds() / 3600
            })

mismatch_details_df = pd.DataFrame(mismatch_details)

print("Total status-chain mismatches:", len(mismatch_details_df))

print(
    "Affected issues:",
    mismatch_details_df["issue_key"].nunique()
)

print(
    "Affected projects:",
    mismatch_details_df["instance_id"].nunique()
)

print("\nSample mismatches:")

display(mismatch_details_df.head(25))

Total status-chain mismatches: 188
Affected issues: 132
Affected projects: 11

Sample mismatches:


,instance_id,issue_key,previous_time,current_time,previous_from,previous_to,current_from,current_to,gap_hours
0,CONFSERVER-I01,CONFSERVER-192,2003-09-19 18:15:34.970000+00:00,2004-01-26 18:51:56.210000+00:00,Unassigned,Assigned,Open,Closed,3096.605900
1,DAOS-I01,DAOS-397,2017-11-21 02:28:35+00:00,2018-05-11 12:15:17+00:00,Open,Resolved,Reopened,Closed,4113.778333
2,DAOS-I01,DAOS-397,2018-05-11 12:15:17+00:00,2018-05-11 12:15:17+00:00,Reopened,Closed,Resolved,Resolved,0.000000
3,DAOS-I01,DAOS-397,2018-05-11 12:15:17+00:00,2019-10-09 21:17:47+00:00,Resolved,Reopened,Closed,Resolved,12393.041667
4,EVG-I01,EVG-13030,2021-11-16 20:26:58+00:00,2021-11-18 16:56:57+00:00,Open,In Progress,In Code Review,In Progress,44.499722
5,EVG-I01,EVG-13030,2021-11-18 16:56:57+00:00,2021-11-18 19:33:48+00:00,In Code Review,In Progress,In Code Review,In Progress,2.614167
6,EVG-I01,EVG-13060,2021-06-08 19:46:33+00:00,2021-07-01 15:05:41+00:00,Open,In Progress,In Code Review,Closed,547.318889
7,EVG-I01,EVG-13178,2021-11-18 22:16:04+00:00,2021-12-02 15:24:08+00:00,Open,In Progress,In Code Review,In Progress,329.134444
8,EVG-I01,EVG-13205,2021-08-11 15:25:12+00:00,2021-08-31 18:39:23+00:00,Open,In Progress,In Code Review,In Progress,483.236389
9,EVG-I01,EVG-13205,2021-08-31 18:39:23+00:00,2021-09-07 14:41:48+00:00,In Code Review,In Progress,In Code Review,In Progress,164.040278


In [ ]:
# ============================================================
# STEP 3.3.9 — ANALYSE MISMATCH PATTERNS
# ============================================================

print("TIME GAP DISTRIBUTION")

display(
    mismatch_details_df["gap_hours"].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]
    )
)

print("\nMISMATCHES WITH IDENTICAL TIMESTAMPS")

same_time = mismatch_details_df[
    mismatch_details_df["gap_hours"] == 0
]

print("Count:", len(same_time))

print("\nMOST COMMON MISMATCH PATTERNS")

display(
    mismatch_details_df.groupby(
        ["previous_to", "current_from"]
    )
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(20)
)

print("\nMISMATCHES BY PROJECT")

display(
    mismatch_details_df.groupby("instance_id")
    .agg(
        mismatches=("issue_key", "count"),
        affected_issues=("issue_key", "nunique")
    )
    .reset_index()
    .sort_values("mismatches", ascending=False)
)

TIME GAP DISTRIBUTION


,gap_hours
count,188.000000
mean,3545.564672
std,14709.269821
min,0.000000
25%,0.610243
50%,166.766111
75%,1219.029653
90%,6814.754833
95%,11808.217915
max,146252.171347



MISMATCHES WITH IDENTICAL TIMESTAMPS
Count: 2

MOST COMMON MISMATCH PATTERNS


,previous_to,current_from,count
20,Reopened,Resolved,31
12,Needs Scheduling,Open,30
15,Open,Needs Verification,29
8,In Progress,In Code Review,27
10,In Progress,Resolved,16
16,Open,Resolved,14
28,Waiting (Blocked),In Code Review,8
13,Open,In Code Review,6
19,Reopened,Closed,3
25,ToDo,Open,3



MISMATCHES BY PROJECT


,instance_id,mismatches,affected_issues
8,SAK-I01,66,55
9,SERVER-I01,47,23
2,EVG-I01,44,31
10,WT-I01,13,7
3,JRASERVER-I01,10,10
1,DAOS-I01,3,1
0,CONFSERVER-I01,1,1
6,QTBUG-I01,1,1
5,NEXUS-I01,1,1
4,MONGOID-I01,1,1


In [ ]:
# ============================================================
# STEP 3.3.10 — CHECKPOINT IMPACT ANALYSIS
# ============================================================

affected_checkpoint_records = []

for _, mismatch in mismatch_details_df.iterrows():

    project_checkpoints = snapshot_checkpoints[
        snapshot_checkpoints["instance_id"]
        == mismatch["instance_id"]
    ]

    affected = project_checkpoints[
        (
            project_checkpoints["checkpoint_time"]
            >= mismatch["previous_time"]
        )
        &
        (
            project_checkpoints["checkpoint_time"]
            < mismatch["current_time"]
        )
    ]

    for _, checkpoint in affected.iterrows():

        affected_checkpoint_records.append({
            "instance_id": mismatch["instance_id"],
            "issue_key": mismatch["issue_key"],
            "week_number": checkpoint["week_number"],
            "checkpoint_time": checkpoint["checkpoint_time"],
            "previous_status": mismatch["previous_to"],
            "next_recorded_from_status": mismatch["current_from"]
        })

affected_mismatch_checkpoints = pd.DataFrame(
    affected_checkpoint_records,
    columns=[
        "instance_id",
        "issue_key",
        "week_number",
        "checkpoint_time",
        "previous_status",
        "next_recorded_from_status"
    ]
)

print(
    "Potentially affected checkpoint-issue records:",
    len(affected_mismatch_checkpoints)
)

print(
    "Unique affected issues:",
    affected_mismatch_checkpoints["issue_key"].nunique()
)

print(
    "Unique affected project-weeks:",
    affected_mismatch_checkpoints[
        ["instance_id", "week_number"]
    ].drop_duplicates().shape[0]
)

display(affected_mismatch_checkpoints.head(20))

Potentially affected checkpoint-issue records: 182
Unique affected issues: 13
Unique affected project-weeks: 104


,instance_id,issue_key,week_number,checkpoint_time,previous_status,next_recorded_from_status
0,JRASERVER-I01,JRASERVER-1,1,2018-01-26 00:22:27+00:00,Assigned,Resolved
1,JRASERVER-I01,JRASERVER-1,2,2018-02-02 00:22:27+00:00,Assigned,Resolved
2,JRASERVER-I01,JRASERVER-1,3,2018-02-09 00:22:27+00:00,Assigned,Resolved
3,JRASERVER-I01,JRASERVER-1,4,2018-02-16 00:22:27+00:00,Assigned,Resolved
4,JRASERVER-I01,JRASERVER-1,5,2018-02-23 00:22:27+00:00,Assigned,Resolved
5,JRASERVER-I01,JRASERVER-1,6,2018-03-02 00:22:27+00:00,Assigned,Resolved
6,JRASERVER-I01,JRASERVER-1,7,2018-03-09 00:22:27+00:00,Assigned,Resolved
7,JRASERVER-I01,JRASERVER-1,8,2018-03-16 00:22:27+00:00,Assigned,Resolved
8,JRASERVER-I01,JRASERVER-1,9,2018-03-23 00:22:27+00:00,Assigned,Resolved
9,JRASERVER-I01,JRASERVER-1,10,2018-03-30 00:22:27+00:00,Assigned,Resolved


In [ ]:
# ============================================================
# STATUS MISMATCH AUDIT — CONSOLIDATED VALIDATION
# ============================================================

checks = []

def check(name, passed, detail):
    checks.append({
        "validation": name,
        "status": "PASS" if passed else "FAIL",
        "details": detail
    })

check(
    "Mismatch count agrees with previous audit",
    len(mismatch_details_df) == 188,
    len(mismatch_details_df)
)

check(
    "All mismatch timestamps available",
    mismatch_details_df[
        ["previous_time", "current_time"]
    ].notna().all().all(),
    "Checked"
)

check(
    "No negative mismatch time gaps",
    (mismatch_details_df["gap_hours"] >= 0).all(),
    "Checked"
)

check(
    "All mismatched issues exist in Jira dataset",
    mismatch_details_df["issue_key"].isin(
        jira_history_df["issue_key"]
    ).all(),
    "Checked"
)

check(
    "All affected checkpoints are valid",
    set(
        map(
            tuple,
            affected_mismatch_checkpoints[
                ["instance_id", "week_number", "checkpoint_time"]
            ].itertuples(index=False, name=None)
        )
    ).issubset(
        set(
            map(
                tuple,
                snapshot_checkpoints[
                    ["instance_id", "week_number", "checkpoint_time"]
                ].itertuples(index=False, name=None)
            )
        )
    ),
    "Checked"
)

check(
    "Original status intervals preserved",
    len(status_intervals_df) == 72256,
    len(status_intervals_df)
)

check(
    "Original checkpoint states preserved",
    len(checkpoint_issue_states_df) == 228255,
    len(checkpoint_issue_states_df)
)

validation_df = pd.DataFrame(checks)

display(validation_df)

failed = validation_df[
    validation_df["status"] == "FAIL"
]

if failed.empty:
    print("ALL STATUS MISMATCH AUDIT VALIDATIONS PASSED")
else:
    print(f"{len(failed)} VALIDATIONS FAILED")

,validation,status,details
0,Mismatch count agrees with previous audit,PASS,188
1,All mismatch timestamps available,PASS,Checked
2,No negative mismatch time gaps,PASS,Checked
3,All mismatched issues exist in Jira dataset,PASS,Checked
4,All affected checkpoints are valid,PASS,Checked
5,Original status intervals preserved,PASS,72256
6,Original checkpoint states preserved,PASS,228255


ALL STATUS MISMATCH AUDIT VALIDATIONS PASSED


In [ ]:
# ============================================================
# STEP 3.3.11 — FLAG UNCERTAIN CHECKPOINT STATES
# ============================================================

import pandas as pd

checkpoint_issue_states_final = (
    checkpoint_issue_states_df.copy()
)

# Start with all reconstructed statuses as unflagged
checkpoint_issue_states_final["status_uncertain"] = False

# Identify checkpoint-issue combinations affected by mismatches
uncertain_keys = (
    affected_mismatch_checkpoints[
        [
            "instance_id",
            "week_number",
            "checkpoint_time",
            "issue_key"
        ]
    ]
    .drop_duplicates()
    .assign(status_uncertain=True)
)

keys = [
    "instance_id",
    "week_number",
    "checkpoint_time",
    "issue_key"
]

checkpoint_issue_states_final = (
    checkpoint_issue_states_final.drop(
        columns=["status_uncertain"]
    )
    .merge(
        uncertain_keys,
        on=keys,
        how="left",
        validate="one_to_one"
    )
)

checkpoint_issue_states_final["status_uncertain"] = (
    checkpoint_issue_states_final["status_uncertain"]
    .fillna(False)
    .astype(bool)
)

# Preserve reconstructed status separately
checkpoint_issue_states_final["recorded_historical_status"] = (
    checkpoint_issue_states_final["historical_status"]
)

# Conservative feature-safe status
checkpoint_issue_states_final["feature_safe_status"] = (
    checkpoint_issue_states_final["historical_status"]
    .mask(
        checkpoint_issue_states_final["status_uncertain"],
        "UNKNOWN"
    )
)

print(
    "Total checkpoint issue records:",
    len(checkpoint_issue_states_final)
)

print(
    "Uncertain checkpoint issue records:",
    checkpoint_issue_states_final["status_uncertain"].sum()
)

print(
    "Affected project-weeks:",
    checkpoint_issue_states_final.loc[
        checkpoint_issue_states_final["status_uncertain"],
        ["instance_id", "week_number"]
    ].drop_duplicates().shape[0]
)

display(
    checkpoint_issue_states_final[
        checkpoint_issue_states_final["status_uncertain"]
    ].head(20)
)

Total checkpoint issue records: 228255
Uncertain checkpoint issue records: 182
Affected project-weeks: 104


/tmp/ipykernel_4111/3928930886.py:49: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


,instance_id,week_number,checkpoint_time,issue_key,historical_status,status_basis,status_uncertain,recorded_historical_status,feature_safe_status
104849,JRASERVER-I01,1,2018-01-26 00:22:27+00:00,JRASERVER-1,Assigned,RECONSTRUCTED_HISTORY,True,Assigned,UNKNOWN
105127,JRASERVER-I01,2,2018-02-02 00:22:27+00:00,JRASERVER-1,Assigned,RECONSTRUCTED_HISTORY,True,Assigned,UNKNOWN
105406,JRASERVER-I01,3,2018-02-09 00:22:27+00:00,JRASERVER-1,Assigned,RECONSTRUCTED_HISTORY,True,Assigned,UNKNOWN
105685,JRASERVER-I01,4,2018-02-16 00:22:27+00:00,JRASERVER-1,Assigned,RECONSTRUCTED_HISTORY,True,Assigned,UNKNOWN
105964,JRASERVER-I01,5,2018-02-23 00:22:27+00:00,JRASERVER-1,Assigned,RECONSTRUCTED_HISTORY,True,Assigned,UNKNOWN
106243,JRASERVER-I01,6,2018-03-02 00:22:27+00:00,JRASERVER-1,Assigned,RECONSTRUCTED_HISTORY,True,Assigned,UNKNOWN
106522,JRASERVER-I01,7,2018-03-09 00:22:27+00:00,JRASERVER-1,Assigned,RECONSTRUCTED_HISTORY,True,Assigned,UNKNOWN
106801,JRASERVER-I01,8,2018-03-16 00:22:27+00:00,JRASERVER-1,Assigned,RECONSTRUCTED_HISTORY,True,Assigned,UNKNOWN
107080,JRASERVER-I01,9,2018-03-23 00:22:27+00:00,JRASERVER-1,Assigned,RECONSTRUCTED_HISTORY,True,Assigned,UNKNOWN
107359,JRASERVER-I01,10,2018-03-30 00:22:27+00:00,JRASERVER-1,Assigned,RECONSTRUCTED_HISTORY,True,Assigned,UNKNOWN


In [ ]:
# ============================================================
# STEP 3.3.12 — INSPECT UNCERTAIN CHECKPOINT STATES
# ============================================================

uncertain_status_summary = (
    checkpoint_issue_states_final[
        checkpoint_issue_states_final["status_uncertain"]
    ]
    .groupby("instance_id")
    .agg(
        uncertain_records=("issue_key", "count"),
        affected_issues=("issue_key", "nunique"),
        affected_weeks=("week_number", "nunique")
    )
    .reset_index()
    .sort_values(
        "uncertain_records",
        ascending=False
    )
)

print("UNCERTAINTY DISTRIBUTION BY PROJECT")
display(uncertain_status_summary)

print("\nFEATURE-SAFE STATUS DISTRIBUTION")

display(
    checkpoint_issue_states_final[
        "feature_safe_status"
    ].value_counts().head(20)
)

UNCERTAINTY DISTRIBUTION BY PROJECT


,instance_id,uncertain_records,affected_issues,affected_weeks
1,SAK-I01,106,8,33
2,SERVER-I01,37,3,32
0,JRASERVER-I01,33,1,33
3,WT-I01,6,1,6



FEATURE-SAFE STATUS DISTRIBUTION


,count
feature_safe_status,
Closed,118465
Resolved,50819
Open,23597
Reopened,8106
UNKNOWN,4470
Gathering Interest,3732
In Progress,2805
Done,2485
Verified,1831


In [ ]:
# ============================================================
# STEP 3.3 — FINAL STATUS UNCERTAINTY VALIDATION
# ============================================================

checks = []

def check(name, passed, detail):
    checks.append({
        "validation": name,
        "status": "PASS" if passed else "FAIL",
        "details": detail
    })

keys = [
    "instance_id",
    "week_number",
    "checkpoint_time",
    "issue_key"
]

check(
    "Original checkpoint record count preserved",
    len(checkpoint_issue_states_final) == 228255,
    len(checkpoint_issue_states_final)
)

check(
    "No duplicate checkpoint-issue records",
    not checkpoint_issue_states_final.duplicated(keys).any(),
    int(checkpoint_issue_states_final.duplicated(keys).sum())
)

check(
    "Uncertainty flags are valid",
    checkpoint_issue_states_final[
        "status_uncertain"
    ].isin([True, False]).all(),
    "Checked"
)

check(
    "Original historical statuses preserved",
    checkpoint_issue_states_final[
        "recorded_historical_status"
    ].equals(
        checkpoint_issue_states_final["historical_status"]
    ),
    "Checked"
)

check(
    "Uncertain statuses masked",
    (
        checkpoint_issue_states_final.loc[
            checkpoint_issue_states_final["status_uncertain"],
            "feature_safe_status"
        ] == "UNKNOWN"
    ).all(),
    "Checked"
)

check(
    "Certain statuses unchanged",
    (
        checkpoint_issue_states_final.loc[
            ~checkpoint_issue_states_final["status_uncertain"],
            "feature_safe_status"
        ]
        ==
        checkpoint_issue_states_final.loc[
            ~checkpoint_issue_states_final["status_uncertain"],
            "historical_status"
        ]
    ).all(),
    "Checked"
)

check(
    "987 original checkpoints preserved",
    len(snapshot_checkpoints) == 987,
    len(snapshot_checkpoints)
)

check(
    "All issue records belong to valid checkpoints",
    set(
        map(
            tuple,
            checkpoint_issue_states_final[
                ["instance_id", "week_number", "checkpoint_time"]
            ].itertuples(index=False, name=None)
        )
    ).issubset(
        set(
            map(
                tuple,
                snapshot_checkpoints[
                    ["instance_id", "week_number", "checkpoint_time"]
                ].itertuples(index=False, name=None)
            )
        )
    ),
    "Checked"
)

validation_df = pd.DataFrame(checks)
display(validation_df)

failed = validation_df[
    validation_df["status"] == "FAIL"
]

print("\n" + "=" * 60)

if failed.empty:
    print("ALL FINAL JIRA STATUS VALIDATIONS PASSED")
else:
    print(f"{len(failed)} VALIDATIONS FAILED")

print("=" * 60)

,validation,status,details
0,Original checkpoint record count preserved,PASS,228255
1,No duplicate checkpoint-issue records,PASS,0
2,Uncertainty flags are valid,PASS,Checked
3,Original historical statuses preserved,PASS,Checked
4,Uncertain statuses masked,PASS,Checked
5,Certain statuses unchanged,PASS,Checked
6,987 original checkpoints preserved,PASS,987
7,All issue records belong to valid checkpoints,PASS,Checked



ALL FINAL JIRA STATUS VALIDATIONS PASSED


### 5.3 Dependency state reconstruction

In [ ]:
# ============================================================
# STEP 3.4.1 — LOCATE HISTORICAL DEPENDENCY INTERVALS
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

GC1_ROOT = Path("/content/drive/MyDrive/GC1")

candidate_names = [
    "corrected_intervals",
    "corrected_intervals_df",
    "dependency_intervals",
    "dependency_intervals_df",
    "link_intervals",
    "link_intervals_df",
    "historical_link_intervals",
    "historical_link_intervals_df"
]

available = []

for name in candidate_names:
    if name in globals():
        obj = globals()[name]

        if isinstance(obj, pd.DataFrame):
            available.append({
                "variable": name,
                "rows": len(obj),
                "columns": obj.columns.tolist()
            })

print("Available dependency interval DataFrames:")

for item in available:
    print("\nVariable:", item["variable"])
    print("Rows:", item["rows"])
    print("Columns:", item["columns"])

if not available:
    print(
        "\nNo matching interval DataFrame found. "
        "Restore the saved interval table or rerun "
        "the dependency interval reconstruction cells."
    )

Available dependency interval DataFrames:

Variable: corrected_intervals_df
Rows: 2465
Columns: ['instance_id', 'upstream_issue', 'downstream_issue', 'active_from', 'first_confirmed_at', 'active_to', 'start_known', 'start_event_uid', 'end_event_uid']


In [ ]:
# ============================================================
# STEP 3.4.2 — STANDARDIZE DEPENDENCY INTERVALS
# ============================================================

import pandas as pd
import numpy as np

dependency_history = corrected_intervals_df.copy()

required_columns = [
    "instance_id",
    "upstream_issue",
    "downstream_issue",
    "active_from",
    "first_confirmed_at",
    "active_to",
    "start_known",
    "start_event_uid",
    "end_event_uid"
]

missing = [
    col for col in required_columns
    if col not in dependency_history.columns
]

assert not missing, f"Missing columns: {missing}"

# Parse timestamps
for col in ["active_from", "first_confirmed_at", "active_to"]:
    dependency_history[col] = pd.to_datetime(
        dependency_history[col],
        utc=True,
        errors="coerce",
        format="mixed"
    )

# Ensure start_known is genuinely boolean
assert dependency_history["start_known"].map(
    lambda x: isinstance(x, (bool, np.bool_))
).all(), "start_known contains non-boolean values"

dependency_history["temporal_class"] = np.where(
    dependency_history["start_known"],
    "KNOWN_START",
    "UNKNOWN_START"
)

print("Total dependency intervals:", len(dependency_history))

print("\nTemporal classification:")
display(dependency_history["temporal_class"].value_counts())

print("\nInterval statistics:")
print("Known starts:", dependency_history["start_known"].sum())
print("Unknown starts:", (~dependency_history["start_known"]).sum())
print("Open intervals:", dependency_history["active_to"].isna().sum())
print("Closed intervals:", dependency_history["active_to"].notna().sum())

display(dependency_history.head(10))

Total dependency intervals: 2465

Temporal classification:


,count
temporal_class,
KNOWN_START,1657
UNKNOWN_START,808



Interval statistics:
Known starts: 1657
Unknown starts: 808
Open intervals: 2371
Closed intervals: 94


,instance_id,upstream_issue,downstream_issue,active_from,first_confirmed_at,active_to,start_known,start_event_uid,end_event_uid,temporal_class
0,AEROGEAR-I01,AEROGEAR-2210,AEROGEAR-2218,2018-03-01 09:46:57+00:00,2018-03-01 09:46:57+00:00,NaT,True,T_62d9e53fc092cf712ba348bf,None,KNOWN_START
1,AEROGEAR-I01,AEROGEAR-2243,AEROGEAR-2242,2018-03-05 07:30:44+00:00,2018-03-05 07:30:44+00:00,NaT,True,T_008617171e2bba5c69d039a8,None,KNOWN_START
2,AEROGEAR-I01,AEROGEAR-2244,AEROGEAR-2242,2018-03-05 07:30:43+00:00,2018-03-05 07:30:43+00:00,NaT,True,T_36fe574bd0cf69a23db63895,None,KNOWN_START
3,AEROGEAR-I01,AEROGEAR-2244,AEROGEAR-2243,2018-03-05 07:30:21+00:00,2018-03-05 07:30:21+00:00,NaT,True,T_43e8d8780eb1370115794c18,None,KNOWN_START
4,AEROGEAR-I01,AEROGEAR-2248,AEROGEAR-2263,2018-03-06 04:23:32+00:00,2018-03-06 04:23:32+00:00,NaT,True,T_397ec8597f2000ddb67c0c9c,None,KNOWN_START
5,AEROGEAR-I01,AEROGEAR-2249,AEROGEAR-2263,2018-03-06 04:23:33+00:00,2018-03-06 04:23:33+00:00,NaT,True,T_23afb9522f5c47289b99f7a7,None,KNOWN_START
6,AEROGEAR-I01,AEROGEAR-2250,AEROGEAR-2248,2018-03-05 10:50:41+00:00,2018-03-05 10:50:41+00:00,NaT,True,T_0477e50f9ea3446798d732bb,None,KNOWN_START
7,AEROGEAR-I01,AEROGEAR-2250,AEROGEAR-2249,2018-03-05 10:50:56+00:00,2018-03-05 10:50:56+00:00,NaT,True,T_11e719c87e5093c8f005ad77,None,KNOWN_START
8,AEROGEAR-I01,AEROGEAR-2250,AEROGEAR-2263,2018-03-06 04:23:32+00:00,2018-03-06 04:23:32+00:00,NaT,True,T_98aca45816a244e8941f958d,None,KNOWN_START
9,AEROGEAR-I01,AEROGEAR-2266,AEROGEAR-2218,2018-03-07 03:23:43+00:00,2018-03-07 03:23:43+00:00,NaT,True,T_91c202b163b627817ca23d0d,None,KNOWN_START


In [ ]:
# ============================================================
# STEP 3.4.3 — CLASSIFY TEMPORAL RELIABILITY
# ============================================================

known_dependency_intervals = dependency_history[
    dependency_history["start_known"]
].copy()

unknown_dependency_intervals = dependency_history[
    ~dependency_history["start_known"]
].copy()

# Check interval consistency
invalid_intervals = known_dependency_intervals[
    known_dependency_intervals["active_from"].isna()
    |
    (
        known_dependency_intervals["active_to"].notna()
        &
        (
            known_dependency_intervals["active_to"]
            < known_dependency_intervals["active_from"]
        )
    )
]

print("Known-start intervals:", len(known_dependency_intervals))
print("Unknown-start intervals:", len(unknown_dependency_intervals))
print("Invalid known-start intervals:", len(invalid_intervals))

print("\nKnown-start project coverage:")
display(
    known_dependency_intervals["instance_id"].value_counts()
)

print("\nUnknown-start project coverage:")
display(
    unknown_dependency_intervals["instance_id"].value_counts()
)

print("\nUnknown-start first-confirmation coverage:")
print(
    unknown_dependency_intervals["first_confirmed_at"]
    .notna().sum()
)

Known-start intervals: 1657
Unknown-start intervals: 808
Invalid known-start intervals: 0

Known-start project coverage:


,count
instance_id,
JBIDE-I01,587
JBEAP-I01,407
ENTESB-I01,293
DAOS-I01,186
AEROGEAR-I01,40
HBASE-I01,28
BXMSDOC-I01,23
SPARK-I01,22
JBDS-I01,21



Unknown-start project coverage:


,count
instance_id,
JBIDE-I01,557
ENTESB-I01,230
BXMSDOC-I01,21



Unknown-start first-confirmation coverage:
808


In [ ]:
# ============================================================
# STEP 3.4.4 — RECONSTRUCT ACTIVE DEPENDENCIES
# ============================================================

checkpoint_dependency_records = []

intervals_by_project = {
    project: group
    for project, group in known_dependency_intervals.groupby(
        "instance_id"
    )
}

for _, checkpoint in snapshot_checkpoints.iterrows():

    instance_id = checkpoint["instance_id"]
    week_number = checkpoint["week_number"]
    checkpoint_time = checkpoint["checkpoint_time"]

    project_intervals = intervals_by_project.get(instance_id)

    if project_intervals is None:
        continue

    active = project_intervals[
        (project_intervals["active_from"] <= checkpoint_time)
        &
        (
            project_intervals["active_to"].isna()
            |
            (project_intervals["active_to"] > checkpoint_time)
        )
    ]

    for _, relation in active.iterrows():

        checkpoint_dependency_records.append({
            "instance_id": instance_id,
            "week_number": week_number,
            "checkpoint_time": checkpoint_time,
            "upstream_issue": relation["upstream_issue"],
            "downstream_issue": relation["downstream_issue"],
            "active_from": relation["active_from"],
            "first_confirmed_at": relation["first_confirmed_at"],
            "active_to": relation["active_to"],
            "start_event_uid": relation["start_event_uid"],
            "end_event_uid": relation["end_event_uid"],
            "temporal_basis": "KNOWN_START"
        })

checkpoint_dependencies_df = pd.DataFrame(
    checkpoint_dependency_records,
    columns=[
        "instance_id",
        "week_number",
        "checkpoint_time",
        "upstream_issue",
        "downstream_issue",
        "active_from",
        "first_confirmed_at",
        "active_to",
        "start_event_uid",
        "end_event_uid",
        "temporal_basis"
    ]
)

print("Checkpoint dependency records:", len(checkpoint_dependencies_df))

print(
    "Project-weeks containing active dependencies:",
    checkpoint_dependencies_df[
        ["instance_id", "week_number"]
    ].drop_duplicates().shape[0]
)

print(
    "Projects with active dependencies:",
    checkpoint_dependencies_df["instance_id"].nunique()
)

display(checkpoint_dependencies_df.head(20))

Checkpoint dependency records: 28340
Project-weeks containing active dependencies: 471
Projects with active dependencies: 17


,instance_id,week_number,checkpoint_time,upstream_issue,downstream_issue,active_from,first_confirmed_at,active_to,start_event_uid,end_event_uid,temporal_basis
0,AEROGEAR-I01,5,2018-02-21 05:30:01+00:00,AEROGEAR-4347,AEROGEAR-1968,2018-02-15 05:23:04+00:00,2018-02-15 05:23:04+00:00,NaT,T_7d51b32c251b512fb86d1559,None,KNOWN_START
1,AEROGEAR-I01,5,2018-02-21 05:30:01+00:00,AGDROID-683,AEROGEAR-1968,2018-02-15 05:23:04+00:00,2018-02-15 05:23:04+00:00,NaT,T_2f188ec3f5fc9f698213b945,None,KNOWN_START
2,AEROGEAR-I01,6,2018-02-28 05:30:01+00:00,AEROGEAR-4347,AEROGEAR-1968,2018-02-15 05:23:04+00:00,2018-02-15 05:23:04+00:00,NaT,T_7d51b32c251b512fb86d1559,None,KNOWN_START
3,AEROGEAR-I01,6,2018-02-28 05:30:01+00:00,AGDROID-683,AEROGEAR-1968,2018-02-15 05:23:04+00:00,2018-02-15 05:23:04+00:00,NaT,T_2f188ec3f5fc9f698213b945,None,KNOWN_START
4,AEROGEAR-I01,7,2018-03-07 05:30:01+00:00,AEROGEAR-2210,AEROGEAR-2218,2018-03-01 09:46:57+00:00,2018-03-01 09:46:57+00:00,NaT,T_62d9e53fc092cf712ba348bf,None,KNOWN_START
5,AEROGEAR-I01,7,2018-03-07 05:30:01+00:00,AEROGEAR-2243,AEROGEAR-2242,2018-03-05 07:30:44+00:00,2018-03-05 07:30:44+00:00,NaT,T_008617171e2bba5c69d039a8,None,KNOWN_START
6,AEROGEAR-I01,7,2018-03-07 05:30:01+00:00,AEROGEAR-2244,AEROGEAR-2242,2018-03-05 07:30:43+00:00,2018-03-05 07:30:43+00:00,NaT,T_36fe574bd0cf69a23db63895,None,KNOWN_START
7,AEROGEAR-I01,7,2018-03-07 05:30:01+00:00,AEROGEAR-2244,AEROGEAR-2243,2018-03-05 07:30:21+00:00,2018-03-05 07:30:21+00:00,NaT,T_43e8d8780eb1370115794c18,None,KNOWN_START
8,AEROGEAR-I01,7,2018-03-07 05:30:01+00:00,AEROGEAR-2248,AEROGEAR-2263,2018-03-06 04:23:32+00:00,2018-03-06 04:23:32+00:00,NaT,T_397ec8597f2000ddb67c0c9c,None,KNOWN_START
9,AEROGEAR-I01,7,2018-03-07 05:30:01+00:00,AEROGEAR-2249,AEROGEAR-2263,2018-03-06 04:23:33+00:00,2018-03-06 04:23:33+00:00,NaT,T_23afb9522f5c47289b99f7a7,None,KNOWN_START


In [ ]:
# ============================================================
# STEP 3.4.5 — FEATURE-SAFE DEPENDENCY RECORDS
# ============================================================

checkpoint_dependencies_safe = (
    checkpoint_dependencies_df[
        [
            "instance_id",
            "week_number",
            "checkpoint_time",
            "upstream_issue",
            "downstream_issue",
            "active_from",
            "start_event_uid",
            "temporal_basis"
        ]
    ].copy()
)

# Remove duplicate active pairs within a checkpoint
checkpoint_dependencies_safe = (
    checkpoint_dependencies_safe
    .sort_values(
        ["instance_id", "week_number", "active_from"]
    )
    .drop_duplicates(
        subset=[
            "instance_id",
            "week_number",
            "upstream_issue",
            "downstream_issue"
        ],
        keep="last"
    )
    .reset_index(drop=True)
)

print(
    "Feature-safe dependency records:",
    len(checkpoint_dependencies_safe)
)

print(
    "Projects represented:",
    checkpoint_dependencies_safe["instance_id"].nunique()
)

display(checkpoint_dependencies_safe.head(20))

Feature-safe dependency records: 28340
Projects represented: 17


,instance_id,week_number,checkpoint_time,upstream_issue,downstream_issue,active_from,start_event_uid,temporal_basis
0,AEROGEAR-I01,5,2018-02-21 05:30:01+00:00,AEROGEAR-4347,AEROGEAR-1968,2018-02-15 05:23:04+00:00,T_7d51b32c251b512fb86d1559,KNOWN_START
1,AEROGEAR-I01,5,2018-02-21 05:30:01+00:00,AGDROID-683,AEROGEAR-1968,2018-02-15 05:23:04+00:00,T_2f188ec3f5fc9f698213b945,KNOWN_START
2,AEROGEAR-I01,6,2018-02-28 05:30:01+00:00,AEROGEAR-4347,AEROGEAR-1968,2018-02-15 05:23:04+00:00,T_7d51b32c251b512fb86d1559,KNOWN_START
3,AEROGEAR-I01,6,2018-02-28 05:30:01+00:00,AGDROID-683,AEROGEAR-1968,2018-02-15 05:23:04+00:00,T_2f188ec3f5fc9f698213b945,KNOWN_START
4,AEROGEAR-I01,7,2018-03-07 05:30:01+00:00,AEROGEAR-4347,AEROGEAR-1968,2018-02-15 05:23:04+00:00,T_7d51b32c251b512fb86d1559,KNOWN_START
5,AEROGEAR-I01,7,2018-03-07 05:30:01+00:00,AGDROID-683,AEROGEAR-1968,2018-02-15 05:23:04+00:00,T_2f188ec3f5fc9f698213b945,KNOWN_START
6,AEROGEAR-I01,7,2018-03-07 05:30:01+00:00,AGIOS-664,AEROGEAR-5543,2018-02-28 06:40:19+00:00,T_7a73950976fd2d4df865f05b,KNOWN_START
7,AEROGEAR-I01,7,2018-03-07 05:30:01+00:00,AGIOS-668,AEROGEAR-5543,2018-02-28 06:40:19+00:00,T_481558a77e9a6f86df8d8355,KNOWN_START
8,AEROGEAR-I01,7,2018-03-07 05:30:01+00:00,AGIOS-665,AEROGEAR-5543,2018-02-28 06:40:20+00:00,T_68391c67ce39e9cb4110b310,KNOWN_START
9,AEROGEAR-I01,7,2018-03-07 05:30:01+00:00,AEROGEAR-2210,AEROGEAR-2218,2018-03-01 09:46:57+00:00,T_62d9e53fc092cf712ba348bf,KNOWN_START


In [ ]:
# ============================================================
# STEP 3.4.6 — DEPENDENCY CHECKPOINT MANIFEST
# ============================================================

dependency_checkpoint_manifest = (
    checkpoint_dependencies_safe.groupby(
        ["instance_id", "week_number", "checkpoint_time"]
    )
    .agg(
        active_dependency_links=("upstream_issue", "count"),
        blocked_downstream_issues=("downstream_issue", "nunique"),
        unique_upstream_issues=("upstream_issue", "nunique")
    )
    .reset_index()
)

# Include checkpoints with zero active dependencies
dependency_checkpoint_manifest = (
    snapshot_checkpoints[
        ["instance_id", "week_number", "checkpoint_time"]
    ]
    .merge(
        dependency_checkpoint_manifest,
        on=["instance_id", "week_number", "checkpoint_time"],
        how="left",
        validate="one_to_one"
    )
)

count_columns = [
    "active_dependency_links",
    "blocked_downstream_issues",
    "unique_upstream_issues"
]

dependency_checkpoint_manifest[count_columns] = (
    dependency_checkpoint_manifest[count_columns]
    .fillna(0)
    .astype(int)
)

print("Total checkpoints:", len(dependency_checkpoint_manifest))

print("\nActive dependency statistics:")
display(
    dependency_checkpoint_manifest[count_columns].describe()
)

print("\nProject-level summary:")
display(
    dependency_checkpoint_manifest.groupby("instance_id")
    .agg(
        max_active_links=("active_dependency_links", "max"),
        mean_active_links=("active_dependency_links", "mean"),
        peak_blocked_issues=("blocked_downstream_issues", "max")
    )
    .reset_index()
    .sort_values("max_active_links", ascending=False)
)

Total checkpoints: 987

Active dependency statistics:


,active_dependency_links,blocked_downstream_issues,unique_upstream_issues
count,987.000000,987.000000,987.000000
mean,28.713273,17.888551,18.886525
std,98.992736,65.684720,59.932072
min,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000
75%,7.000000,6.000000,6.000000
max,587.000000,415.000000,337.000000



Project-level summary:


,instance_id,max_active_links,mean_active_links,peak_blocked_issues
13,JBIDE-I01,587,391.333333,415
12,JBEAP-I01,353,210.757576,135
6,ENTESB-I01,258,96.606061,188
4,DAOS-I01,142,76.545455,55
0,AEROGEAR-I01,39,24.787879,20
9,HBASE-I01,28,8.969697,7
2,BXMSDOC-I01,21,6.303030,18
27,SPARK-I01,19,10.575758,16
11,JBDS-I01,17,6.878788,12
17,MC-I01,13,8.090909,13


In [ ]:
# ============================================================
# STEP 3.4 — CONSOLIDATED VALIDATION
# ============================================================

checks = []

def check(name, passed, detail):
    checks.append({
        "validation": name,
        "status": "PASS" if passed else "FAIL",
        "details": detail
    })

check(
    "Original dependency intervals preserved",
    len(dependency_history) == 2465,
    len(dependency_history)
)

check(
    "Known-start intervals",
    len(known_dependency_intervals) == 1657,
    len(known_dependency_intervals)
)

check(
    "Unknown-start intervals",
    len(unknown_dependency_intervals) == 808,
    len(unknown_dependency_intervals)
)

check(
    "Known-start flags valid",
    known_dependency_intervals["start_known"].all(),
    "Checked"
)

check(
    "Unknown-start flags valid",
    (~unknown_dependency_intervals["start_known"]).all(),
    "Checked"
)

check(
    "Known-start timestamps complete",
    known_dependency_intervals["active_from"].notna().all(),
    "Checked"
)

check(
    "No invalid known-start intervals",
    len(invalid_intervals) == 0,
    len(invalid_intervals)
)

check(
    "No dependencies before known start",
    (
        checkpoint_dependencies_safe["active_from"]
        <= checkpoint_dependencies_safe["checkpoint_time"]
    ).all(),
    "Checked"
)

check(
    "No duplicate active dependency pairs",
    not checkpoint_dependencies_safe.duplicated(
        [
            "instance_id",
            "week_number",
            "upstream_issue",
            "downstream_issue"
        ]
    ).any(),
    "Checked"
)

check(
    "All active dependencies have start provenance",
    checkpoint_dependencies_safe["start_event_uid"].notna().all(),
    "Checked"
)

check(
    "Start event IDs exist in Phase 1 KG",
    checkpoint_dependencies_safe["start_event_uid"]
    .isin(snapshot_nodes["uid"]).all(),
    "Checked"
)

check(
    "All dependency projects valid",
    set(checkpoint_dependencies_safe["instance_id"])
    .issubset(set(snapshot_checkpoints["instance_id"])),
    "Checked"
)

check(
    "987 checkpoint manifest records",
    len(dependency_checkpoint_manifest) == 987,
    len(dependency_checkpoint_manifest)
)

check(
    "No missing manifest values",
    dependency_checkpoint_manifest[count_columns]
    .notna().all().all(),
    "Checked"
)

check(
    "No negative dependency counts",
    (
        dependency_checkpoint_manifest[count_columns] >= 0
    ).all().all(),
    "Checked"
)

check(
    "Blocked issues do not exceed active links",
    (
        dependency_checkpoint_manifest["blocked_downstream_issues"]
        <= dependency_checkpoint_manifest["active_dependency_links"]
    ).all(),
    "Checked"
)

# Check that future removal information is absent
check(
    "Feature-safe table excludes future removal fields",
    (
        "active_to" not in checkpoint_dependencies_safe.columns
        and "end_event_uid" not in checkpoint_dependencies_safe.columns
    ),
    "Checked"
)

validation_df = pd.DataFrame(checks)

display(validation_df)

failed = validation_df[
    validation_df["status"] == "FAIL"
]

print("\n" + "=" * 60)

if failed.empty:
    print("ALL HISTORICAL DEPENDENCY VALIDATIONS PASSED")
else:
    print(f"{len(failed)} VALIDATIONS FAILED")

print("=" * 60)

,validation,status,details
0,Original dependency intervals preserved,PASS,2465
1,Known-start intervals,PASS,1657
2,Unknown-start intervals,PASS,808
3,Known-start flags valid,PASS,Checked
4,Unknown-start flags valid,PASS,Checked
5,Known-start timestamps complete,PASS,Checked
6,No invalid known-start intervals,PASS,0
7,No dependencies before known start,PASS,Checked
8,No duplicate active dependency pairs,PASS,Checked
9,All active dependencies have start provenance,PASS,Checked



ALL HISTORICAL DEPENDENCY VALIDATIONS PASSED


In [ ]:
# ============================================================
# STEP 3.4 — EXPORT HISTORICAL DEPENDENCY SNAPSHOT DATA
# ============================================================

from pathlib import Path

assert (validation_df["status"] == "PASS").all(), \
    "Validation failed. Review before exporting."

# Existing Phase 2 output directory
output_dir = (
    Path("/content/drive/MyDrive/GC1")
    / "Temporal_snapshots"
)

assert output_dir.is_dir(), \
    f"Temporal snapshots folder not found: {output_dir}"

# Save active dependency relationships
checkpoint_dependencies_safe.to_csv(
    output_dir / "checkpoint_active_dependencies.csv",
    index=False
)

# Save checkpoint-level dependency statistics
dependency_checkpoint_manifest.to_csv(
    output_dir / "dependency_checkpoint_manifest.csv",
    index=False
)

# Save unknown-start relationships for auditing
unknown_dependency_intervals.to_csv(
    output_dir / "dependency_unknown_start_audit.csv",
    index=False
)

print("Historical dependency data exported successfully.")
print("Output directory:", output_dir)

print("\nFiles saved:")
for filename in [
    "checkpoint_active_dependencies.csv",
    "dependency_checkpoint_manifest.csv",
    "dependency_unknown_start_audit.csv"
]:
    print("-", filename)

Historical dependency data exported successfully.
Output directory: /content/drive/MyDrive/GC1/Temporal_snapshots

Files saved:
- checkpoint_active_dependencies.csv
- dependency_checkpoint_manifest.csv
- dependency_unknown_start_audit.csv


### 5.4 Temporal node and edge registries

In [ ]:
# ============================================================
# STEP 3.5.1 — PREPARE SNAPSHOT INPUTS
# ============================================================

import pandas as pd
import numpy as np
import json
from pathlib import Path

GC1_ROOT = Path("/content/drive/MyDrive/GC1")

KG_DIR = (
    GC1_ROOT
    / "entity_resolution_output"
    / "temporal_kg_v2_output"
)

OUTPUT_DIR = GC1_ROOT / "Temporal_snapshots"

assert OUTPUT_DIR.is_dir(), (
    f"Output folder does not exist: {OUTPUT_DIR}"
)

# ------------------------------------------------------------
# Phase 1 KG
# ------------------------------------------------------------

if "snapshot_nodes" not in globals():
    snapshot_nodes = pd.read_json(
        KG_DIR / "nodes.jsonl",
        lines=True
    )

if "snapshot_edges" not in globals():
    snapshot_edges = pd.read_json(
        KG_DIR / "links.jsonl",
        lines=True
    )

# ------------------------------------------------------------
# Weekly checkpoints
# ------------------------------------------------------------

if "snapshot_checkpoints" not in globals():
    raise RuntimeError(
        "snapshot_checkpoints is missing. "
        "Run Step 3.1 first."
    )

snapshot_checkpoints = snapshot_checkpoints.copy()

for col in ["checkpoint_time", "horizon_end"]:
    snapshot_checkpoints[col] = pd.to_datetime(
        snapshot_checkpoints[col],
        utc=True,
        errors="coerce",
        format="mixed"
    )

# ------------------------------------------------------------
# Historical Jira issue states
# ------------------------------------------------------------

if "checkpoint_issue_states_final" not in globals():
    checkpoint_issue_states_final = pd.read_csv(
        OUTPUT_DIR / "jira_checkpoint_issue_states_final.csv"
    )

checkpoint_issue_states_final["checkpoint_time"] = pd.to_datetime(
    checkpoint_issue_states_final["checkpoint_time"],
    utc=True,
    format="mixed"
)

# ------------------------------------------------------------
# Active dependencies
# ------------------------------------------------------------

if "checkpoint_dependencies_safe" not in globals():
    checkpoint_dependencies_safe = pd.read_csv(
        OUTPUT_DIR / "checkpoint_active_dependencies.csv"
    )

for col in ["checkpoint_time", "active_from"]:
    checkpoint_dependencies_safe[col] = pd.to_datetime(
        checkpoint_dependencies_safe[col],
        utc=True,
        format="mixed"
    )

print("INPUT SUMMARY")
print("=" * 55)
print("KG nodes:", len(snapshot_nodes))
print("KG relationships:", len(snapshot_edges))
print("Weekly checkpoints:", len(snapshot_checkpoints))
print("Jira checkpoint states:", len(checkpoint_issue_states_final))
print("Checkpoint dependencies:", len(checkpoint_dependencies_safe))

INPUT SUMMARY
KG nodes: 206634
KG relationships: 600623
Weekly checkpoints: 987
Jira checkpoint states: 228255
Checkpoint dependencies: 28340


In [ ]:
# ============================================================
# STEP 3.5.2 — BUILD TEMPORAL NODE REGISTRY
# ============================================================

ALLOWED_NODE_TYPES = {
    "Event",
    "Decision",
    "ActionItem",
    "Blocker",
    "Dependency",
    "Fact"
}

kg_node_registry = snapshot_nodes[
    snapshot_nodes["label"].isin(ALLOWED_NODE_TYPES)
].copy()

kg_node_registry["available_at"] = pd.to_datetime(
    kg_node_registry["observed_at"],
    utc=True,
    errors="coerce",
    format="mixed"
)

kg_node_registry = kg_node_registry[
    kg_node_registry["available_at"].notna()
].copy()

# Keep only identity and temporal metadata.
# Do not expose unrestricted payload_json.
kg_node_registry = kg_node_registry[
    [
        "uid",
        "instance_id",
        "label",
        "available_at",
        "source_id",
        "source_type",
        "temporal_basis",
        "temporal_status"
    ]
].copy()

kg_node_registry = kg_node_registry.rename(
    columns={"uid": "node_uid"}
)

kg_node_registry = kg_node_registry.drop_duplicates(
    subset=["node_uid"]
)

print("Strictly timestamped KG nodes:",
      len(kg_node_registry))

print("\nNode types:")
display(
    kg_node_registry["label"].value_counts()
)

print("\nProject coverage:")
display(
    kg_node_registry.groupby("instance_id")
    .agg(
        nodes=("node_uid", "count"),
        first_observation=("available_at", "min"),
        last_observation=("available_at", "max")
    )
    .reset_index()
)

Strictly timestamped KG nodes: 87230

Node types:


,count
label,
Event,82396
Fact,3809
Decision,414
Dependency,237
ActionItem,207
Blocker,167



Project coverage:


,instance_id,nodes,first_observation,last_observation
0,AEROGEAR-I01,1766,2018-01-17 05:30:01+00:00,2018-09-25 09:29:15+00:00
1,AMBARI-I01,1977,2016-03-03 00:46:21+00:00,2016-11-09 22:26:15+00:00
2,BXMSDOC-I01,1265,2020-03-20 04:45:40+00:00,2020-11-26 03:06:05+00:00
3,CONFSERVER-I01,6796,2016-03-17 00:01:19+00:00,2016-11-23 14:23:28+00:00
4,DAOS-I01,3419,2021-01-06 09:40:54+00:00,2021-09-14 23:40:04+00:00
5,DOCS-I01,1462,2016-07-05 19:07:09+00:00,2017-03-13 01:42:03+00:00
6,ENTESB-I01,3972,2020-04-08 07:32:26+00:00,2020-12-14 10:17:29+00:00
7,EVG-I01,3401,2020-05-27 14:34:54+00:00,2021-02-02 23:07:55+00:00
8,FLEX-I01,1282,2013-09-02 04:32:52+00:00,2014-05-06 13:02:31+00:00
9,HBASE-I01,1624,2016-07-01 02:40:06+00:00,2017-03-09 15:00:42+00:00


In [ ]:
# ============================================================
# STEP 3.5.3 — BUILD TEMPORAL EDGE REGISTRY
# ============================================================

node_times = kg_node_registry[
    ["node_uid", "available_at", "instance_id"]
].copy()

source_times = node_times.rename(columns={
    "node_uid": "from_uid",
    "available_at": "source_available_at",
    "instance_id": "source_instance"
})

target_times = node_times.rename(columns={
    "node_uid": "to_uid",
    "available_at": "target_available_at",
    "instance_id": "target_instance"
})

# Only edges between strictly eligible nodes
kg_edge_registry = (
    snapshot_edges
    .merge(source_times, on="from_uid", how="inner")
    .merge(target_times, on="to_uid", how="inner")
)

# Prevent cross-project relationships
kg_edge_registry = kg_edge_registry[
    (kg_edge_registry["instance_id"]
     == kg_edge_registry["source_instance"])
    &
    (kg_edge_registry["instance_id"]
     == kg_edge_registry["target_instance"])
].copy()

# Candidate structural/provenance edges
SAFE_EDGE_TYPES = {
    "SUPPORTED_BY",
    "ASSERTS",
    "HAS_EVENT",
    "HAS_DECISION",
    "HAS_ACTIONITEM",
    "HAS_BLOCKER",
    "HAS_EVIDENCE"
}

kg_edge_registry = kg_edge_registry[
    kg_edge_registry["type"].isin(SAFE_EDGE_TYPES)
].copy()

kg_edge_registry["available_at"] = (
    kg_edge_registry[
        ["source_available_at", "target_available_at"]
    ].max(axis=1)
)

kg_edge_registry = kg_edge_registry[
    [
        "uid",
        "instance_id",
        "from_uid",
        "to_uid",
        "type",
        "available_at"
    ]
].rename(columns={
    "uid": "edge_uid"
})

kg_edge_registry = kg_edge_registry.drop_duplicates(
    subset=["edge_uid"]
)

print("Eligible KG relationships:", len(kg_edge_registry))

print("\nRelationship types:")
display(
    kg_edge_registry["type"].value_counts()
)

Eligible KG relationships: 0

Relationship types:


,count
type,


In [ ]:
# ============================================================
# STEP 3.5.3 — DIAGNOSE ZERO KG RELATIONSHIPS
# ============================================================

import pandas as pd

print("=" * 65)
print("TEMPORAL KG RELATIONSHIP DIAGNOSTIC")
print("=" * 65)

print("\n1. ORIGINAL GRAPH")
print("Total nodes:", len(snapshot_nodes))
print("Total relationships:", len(snapshot_edges))

print("\n2. ELIGIBLE NODE REGISTRY")
print("Eligible nodes:", len(kg_node_registry))

print("\nEligible node types:")
display(kg_node_registry["label"].value_counts())

# ------------------------------------------------------------
# Check original relationship endpoint types
# ------------------------------------------------------------

node_labels = snapshot_nodes.set_index("uid")["label"]

diagnostic_edges = snapshot_edges.copy()

diagnostic_edges["source_label"] = (
    diagnostic_edges["from_uid"].map(node_labels)
)

diagnostic_edges["target_label"] = (
    diagnostic_edges["to_uid"].map(node_labels)
)

print("\n3. ORIGINAL RELATIONSHIP ENDPOINT TYPES")

display(
    diagnostic_edges.groupby(
        ["type", "source_label", "target_label"]
    )
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

# ------------------------------------------------------------
# Check endpoint eligibility
# ------------------------------------------------------------

eligible_uids = set(kg_node_registry["node_uid"])

diagnostic_edges["source_eligible"] = (
    diagnostic_edges["from_uid"].isin(eligible_uids)
)

diagnostic_edges["target_eligible"] = (
    diagnostic_edges["to_uid"].isin(eligible_uids)
)

print("\n4. ENDPOINT ELIGIBILITY")

print(
    "Relationships with eligible source:",
    diagnostic_edges["source_eligible"].sum()
)

print(
    "Relationships with eligible target:",
    diagnostic_edges["target_eligible"].sum()
)

both_eligible = diagnostic_edges[
    diagnostic_edges["source_eligible"]
    &
    diagnostic_edges["target_eligible"]
]

print(
    "Relationships with BOTH endpoints eligible:",
    len(both_eligible)
)

print("\n5. RELATIONSHIP TYPES AFTER ENDPOINT FILTERING")

display(both_eligible["type"].value_counts())

# ------------------------------------------------------------
# Check allowed edge types
# ------------------------------------------------------------

print("\n6. FINAL FILTER")

print("Allowed relationship types:", SAFE_EDGE_TYPES)

final_candidates = both_eligible[
    both_eligible["type"].isin(SAFE_EDGE_TYPES)
]

print(
    "Relationships surviving all filters:",
    len(final_candidates)
)

# ------------------------------------------------------------
# Diagnose excluded endpoint types
# ------------------------------------------------------------

excluded_edges = diagnostic_edges[
    ~(
        diagnostic_edges["source_eligible"]
        &
        diagnostic_edges["target_eligible"]
    )
]

print("\n7. MOST COMMON EXCLUDED RELATIONSHIPS")

display(
    excluded_edges.groupby(
        ["type", "source_label", "target_label"]
    )
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(20)
)

print("\nDIAGNOSTIC COMPLETE")

TEMPORAL KG RELATIONSHIP DIAGNOSTIC

1. ORIGINAL GRAPH
Total nodes: 206634
Total relationships: 600623

2. ELIGIBLE NODE REGISTRY
Eligible nodes: 87230

Eligible node types:


,count
label,
Event,82396
Fact,3809
Decision,414
Dependency,237
ActionItem,207
Blocker,167



3. ORIGINAL RELATIONSHIP ENDPOINT TYPES


,type,source_label,target_label,count
16,SUPPORTED_BY,Event,Evidence,82577
11,REFERS_TO,Event,CanonicalEntity,82577
6,HAS_EVENT,SourceRecord,Event,82577
18,TO_ENTITY,Fact,CanonicalEntity,63305
1,FROM_ENTITY,Fact,CanonicalEntity,63305
0,ASSERTS,SourceRecord,Fact,63305
17,SUPPORTED_BY,Fact,Evidence,63305
8,MENTIONS,SourceRecord,CanonicalEntity,61315
10,REFERS_TO,Dependency,CanonicalEntity,14708
15,SUPPORTED_BY,Dependency,Evidence,7709



4. ENDPOINT ELIGIBILITY
Relationships with eligible source: 177244
Relationships with eligible target: 87230
Relationships with BOTH endpoints eligible: 0

5. RELATIONSHIP TYPES AFTER ENDPOINT FILTERING


,count
type,



6. FINAL FILTER
Allowed relationship types: {'HAS_ACTIONITEM', 'HAS_EVIDENCE', 'HAS_DECISION', 'HAS_EVENT', 'HAS_BLOCKER', 'ASSERTS', 'SUPPORTED_BY'}
Relationships surviving all filters: 0

7. MOST COMMON EXCLUDED RELATIONSHIPS


,type,source_label,target_label,count
16,SUPPORTED_BY,Event,Evidence,82577
11,REFERS_TO,Event,CanonicalEntity,82577
6,HAS_EVENT,SourceRecord,Event,82577
18,TO_ENTITY,Fact,CanonicalEntity,63305
1,FROM_ENTITY,Fact,CanonicalEntity,63305
0,ASSERTS,SourceRecord,Fact,63305
17,SUPPORTED_BY,Fact,Evidence,63305
8,MENTIONS,SourceRecord,CanonicalEntity,61315
10,REFERS_TO,Dependency,CanonicalEntity,14708
15,SUPPORTED_BY,Dependency,Evidence,7709



DIAGNOSTIC COMPLETE


In [ ]:
# ============================================================
# STEP 3.5.2A — INSPECT EXCLUDED NODE AVAILABILITY
# ============================================================

import pandas as pd

excluded_types = [
    "SourceRecord",
    "Evidence",
    "CanonicalEntity"
]

excluded_nodes = snapshot_nodes[
    snapshot_nodes["label"].isin(excluded_types)
].copy()

print("EXCLUDED NODE COUNTS")
display(excluded_nodes["label"].value_counts())

print("\nAVAILABLE NODE COLUMNS")
print(snapshot_nodes.columns.tolist())

# Inspect possible timestamp fields
candidate_fields = [
    "observed_at",
    "source_timestamp",
    "created_at",
    "event_time",
    "valid_from",
    "temporal_basis",
    "temporal_status",
    "source_type"
]

available_fields = [
    c for c in candidate_fields
    if c in excluded_nodes.columns
]

print("\nTIMESTAMP AND METADATA COVERAGE")

for field in available_fields:

    print("\nFIELD:", field)

    display(
        excluded_nodes.groupby("label")[field]
        .agg(
            total="size",
            non_null="count"
        )
        .reset_index()
    )

print("\nSAMPLE SOURCE RECORDS")

display(
    excluded_nodes[
        excluded_nodes["label"] == "SourceRecord"
    ].head(5).T
)

print("\nSAMPLE EVIDENCE NODES")

display(
    excluded_nodes[
        excluded_nodes["label"] == "Evidence"
    ].head(5).T
)

print("\nSAMPLE CANONICAL ENTITIES")

display(
    excluded_nodes[
        excluded_nodes["label"] == "CanonicalEntity"
    ].head(5).T
)

EXCLUDED NODE COUNTS


,count
label,
CanonicalEntity,40325
SourceRecord,5827
Evidence,5827



AVAILABLE NODE COLUMNS
['canonical_id', 'entity_type', 'instance_id', 'label', 'name', 'payload_json', 'project_key', 'uid', 'created_at', 'observed_at', 'temporal_basis', 'temporal_status', 'chunk_id', 'source_id', 'source_timestamp', 'source_type', 'evidence_text', 'event_time', 'valid_from', 'valid_to', 'relation_type', 'temporal_policy', 'snapshot_available_at', 'availability_basis', 'temporal_eligibility']

TIMESTAMP AND METADATA COVERAGE

FIELD: observed_at


,label,total,non_null
0,CanonicalEntity,40325,5926
1,Evidence,5827,67
2,SourceRecord,5827,67



FIELD: source_timestamp


...

,40325,40340,40357,40418,40470
canonical_id,NaN,NaN,NaN,NaN,NaN
entity_type,NaN,NaN,NaN,NaN,NaN
instance_id,JBEAP-I01,JBEAP-I01,JBEAP-I01,JBEAP-I01,JBEAP-I01
label,SourceRecord,SourceRecord,SourceRecord,SourceRecord,SourceRecord
name,NaN,NaN,NaN,NaN,NaN
payload_json,"{""provenance"": {""chunk_id"": null, ""evidence_te...","{""provenance"": {""chunk_id"": null, ""evidence_te...","{""provenance"": {""chunk_id"": null, ""evidence_te...","{""provenance"": {""chunk_id"": null, ""evidence_te...","{""provenance"": {""chunk_id"": null, ""evidence_te..."
project_key,JBEAP,JBEAP,JBEAP,JBEAP,JBEAP
uid,R_f20150ac2dd405a99ad2c161,R_de766dac549e33345755ad8f,R_b5c38debbc7a5a9be2391f0f,R_63a7cf156590b38f0744548d,R_72b8d8d8fc658bf747c66b21
created_at,NaN,NaN,NaN,NaN,NaN
observed_at,None,None,None,None,None



SAMPLE EVIDENCE NODES


,40326,40341,40358,40419,40471
canonical_id,NaN,NaN,NaN,NaN,NaN
entity_type,NaN,NaN,NaN,NaN,NaN
instance_id,JBEAP-I01,JBEAP-I01,JBEAP-I01,JBEAP-I01,JBEAP-I01
label,Evidence,Evidence,Evidence,Evidence,Evidence
name,NaN,NaN,NaN,NaN,NaN
payload_json,"{""chunk_id"": null, ""evidence_text"": ""[DOC RFE]...","{""chunk_id"": null, ""evidence_text"": ""[DOC RFE]...","{""chunk_id"": null, ""evidence_text"": ""(7.0.z) U...","{""chunk_id"": null, ""evidence_text"": ""Hibernate...","{""chunk_id"": null, ""evidence_text"": ""Tracker b..."
project_key,NaN,NaN,NaN,NaN,NaN
uid,E_f20150ac2dd405a99ad2c161,E_de766dac549e33345755ad8f,E_b5c38debbc7a5a9be2391f0f,E_63a7cf156590b38f0744548d,E_72b8d8d8fc658bf747c66b21
created_at,NaN,NaN,NaN,NaN,NaN
observed_at,None,None,None,None,None



SAMPLE CANONICAL ENTITIES


,0,1,2,3,4
canonical_id,COMPONENT::AEROGEAR-I01::admin-server::73ed893...,COMPONENT::AEROGEAR-I01::admin-ui-module::3a3e...,COMPONENT::AEROGEAR-I01::admin-ui::b280ae5b97f6,COMPONENT::AEROGEAR-I01::build-farm-components...,COMPONENT::AEROGEAR-I01::data-sync::6c66d7790a93
entity_type,COMPONENT,COMPONENT,COMPONENT,COMPONENT,COMPONENT
instance_id,AEROGEAR-I01,AEROGEAR-I01,AEROGEAR-I01,AEROGEAR-I01,AEROGEAR-I01
label,CanonicalEntity,CanonicalEntity,CanonicalEntity,CanonicalEntity,CanonicalEntity
name,admin server,admin-ui module,admin UI,Build farm components,data-sync
payload_json,"{""aliases"": [""admin server""], ""canonical_id"": ...","{""aliases"": [""admin-ui module""], ""canonical_id...","{""aliases"": [""admin UI""], ""canonical_id"": ""COM...","{""aliases"": [""Build farm components""], ""canoni...","{""aliases"": [""data-sync""], ""canonical_id"": ""CO..."
project_key,AEROGEAR,AEROGEAR,AEROGEAR,AEROGEAR,AEROGEAR
uid,C_COMPONENT::AEROGEAR-I01::admin-server::73ed8...,C_COMPONENT::AEROGEAR-I01::admin-ui-module::3a...,C_COMPONENT::AEROGEAR-I01::admin-ui::b280ae5b97f6,C_COMPONENT::AEROGEAR-I01::build-farm-componen...,C_COMPONENT::AEROGEAR-I01::data-sync::6c66d779...
created_at,NaN,NaN,NaN,NaN,NaN
observed_at,NaN,NaN,NaN,NaN,NaN


In [ ]:
# ============================================================
# STEP 3.5.2B — INSPECT TEMPORAL PROVENANCE PATHS
# ============================================================

node_metadata = snapshot_nodes[
    ["uid", "label", "observed_at"]
].copy()

node_metadata["observed_dt"] = pd.to_datetime(
    node_metadata["observed_at"],
    utc=True,
    errors="coerce",
    format="mixed"
)

source_meta = node_metadata.rename(columns={
    "uid": "from_uid",
    "label": "source_label",
    "observed_dt": "source_observed"
})

target_meta = node_metadata.rename(columns={
    "uid": "to_uid",
    "label": "target_label",
    "observed_dt": "target_observed"
})

edge_temporal_diagnostic = (
    snapshot_edges[
        ["uid", "type", "from_uid", "to_uid"]
    ]
    .merge(
        source_meta[
            ["from_uid", "source_label", "source_observed"]
        ],
        on="from_uid",
        how="left"
    )
    .merge(
        target_meta[
            ["to_uid", "target_label", "target_observed"]
        ],
        on="to_uid",
        how="left"
    )
)

provenance_coverage = (
    edge_temporal_diagnostic.groupby(
        ["type", "source_label", "target_label"]
    )
    .agg(
        relationships=("uid", "count"),
        timestamped_sources=("source_observed", "count"),
        timestamped_targets=("target_observed", "count")
    )
    .reset_index()
    .sort_values("relationships", ascending=False)
)

print("PROVENANCE TIMESTAMP COVERAGE")
display(provenance_coverage)

PROVENANCE TIMESTAMP COVERAGE


,type,source_label,target_label,relationships,timestamped_sources,timestamped_targets
16,SUPPORTED_BY,Event,Evidence,82577,82396,343
11,REFERS_TO,Event,CanonicalEntity,82577,82396,82053
6,HAS_EVENT,SourceRecord,Event,82577,343,82396
18,TO_ENTITY,Fact,CanonicalEntity,63305,3809,9127
1,FROM_ENTITY,Fact,CanonicalEntity,63305,3809,51808
0,ASSERTS,SourceRecord,Fact,63305,3809,3809
17,SUPPORTED_BY,Fact,Evidence,63305,3809,3809
8,MENTIONS,SourceRecord,CanonicalEntity,61315,3291,9008
10,REFERS_TO,Dependency,CanonicalEntity,14708,0,8730
15,SUPPORTED_BY,Dependency,Evidence,7709,237,237


In [ ]:
# ============================================================
# STEP 3.5 — AUDIT EVENT TEMPORAL RELIABILITY
# ============================================================

import pandas as pd

events = snapshot_nodes[
    snapshot_nodes["label"] == "Event"
].copy()

events["observed_dt"] = pd.to_datetime(
    events["observed_at"],
    utc=True,
    errors="coerce",
    format="mixed"
)

print("TOTAL EVENT NODES:", len(events))

print("\nTEMPORAL BASIS DISTRIBUTION")
display(
    events.groupby(
        ["temporal_basis", "temporal_status"],
        dropna=False
    )
    .agg(
        events=("uid", "count"),
        timestamped=("observed_dt", "count"),
        projects=("instance_id", "nunique")
    )
    .reset_index()
    .sort_values("events", ascending=False)
)

print("\nEVENT SOURCE TYPES")
display(
    events.groupby(
        ["source_type", "temporal_basis"],
        dropna=False
    )
    .agg(
        events=("uid", "count"),
        timestamped=("observed_dt", "count")
    )
    .reset_index()
    .sort_values("events", ascending=False)
)

print("\nSAMPLE TIMESTAMPED EVENTS")
display(
    events.loc[
        events["observed_dt"].notna(),
        [
            "uid",
            "instance_id",
            "source_id",
            "source_type",
            "event_time",
            "observed_at",
            "temporal_basis",
            "temporal_status"
        ]
    ].head(15)
)

TOTAL EVENT NODES: 82577

TEMPORAL BASIS DISTRIBUTION


,temporal_basis,temporal_status,events,timestamped,projects
0,jira.full_changelog.created_or_status_history....,EXACT,82053,82053,30
1,original_document_header,EXACT,343,343,30
2,unavailable,UNKNOWN,181,0,10



EVENT SOURCE TYPES


,source_type,temporal_basis,events,timestamped
0,JIRA,jira.full_changelog.created_or_status_history....,82053,82053
1,MEETING,original_document_header,319,319
3,PRD,unavailable,181,0
2,PRD,original_document_header,24,24



SAMPLE TIMESTAMPED EVENTS


,uid,instance_id,source_id,source_type,event_time,observed_at,temporal_basis,temporal_status
40359,T_78880e35c6d61ac7fd1cb6c0,JBEAP-I01,JBEAP-5471,JIRA,2016-07-29T13:23:29+00:00,2016-07-29T13:23:29+00:00,jira.full_changelog.created_or_status_history....,EXACT
40360,T_139d055c4525c8627deec04c,JBEAP-I01,JBEAP-5471,JIRA,2016-07-29T13:23:29+00:00,2016-07-29T13:23:29+00:00,jira.full_changelog.created_or_status_history....,EXACT
40361,T_6af67e67ab49c420d887a0bc,JBEAP-I01,JBEAP-5471,JIRA,2016-07-29T13:23:29+00:00,2016-07-29T13:23:29+00:00,jira.full_changelog.created_or_status_history....,EXACT
40362,T_c6fc7eef22cb6b4a12e9e61a,JBEAP-I01,JBEAP-5471,JIRA,2016-07-29T13:23:57+00:00,2016-07-29T13:23:57+00:00,jira.full_changelog.created_or_status_history....,EXACT
40363,T_5230344df3c92f1bdc70b8b5,JBEAP-I01,JBEAP-5471,JIRA,2016-07-29T13:24:04+00:00,2016-07-29T13:24:04+00:00,jira.full_changelog.created_or_status_history....,EXACT
40364,T_a4542f08492478e41210aefc,JBEAP-I01,JBEAP-5471,JIRA,2016-07-29T13:24:10+00:00,2016-07-29T13:24:10+00:00,jira.full_changelog.created_or_status_history....,EXACT
40365,T_6a818f3912f462828609eb7a,JBEAP-I01,JBEAP-5471,JIRA,2016-07-29T13:28:37+00:00,2016-07-29T13:28:37+00:00,jira.full_changelog.created_or_status_history....,EXACT
40366,T_e92d7d41d5fffc9971418783,JBEAP-I01,JBEAP-5471,JIRA,2016-07-29T13:28:38+00:00,2016-07-29T13:28:38+00:00,jira.full_changelog.created_or_status_history....,EXACT
40367,T_c24fe47f7e2db90fddbe0ead,JBEAP-I01,JBEAP-5471,JIRA,2016-07-29T13:28:38+00:00,2016-07-29T13:28:38+00:00,jira.full_changelog.created_or_status_history....,EXACT
40368,T_74ad08f9778aa4b4630cdc7f,JBEAP-I01,JBEAP-5471,JIRA,2016-07-29T13:28:38+00:00,2016-07-29T13:28:38+00:00,jira.full_changelog.created_or_status_history....,EXACT


In [ ]:
# ============================================================
# STEP 3.5.2 — HISTORICAL JIRA EVENT REGISTRY
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

GC1_ROOT = Path("/content/drive/MyDrive/GC1")

OUTPUT_DIR = GC1_ROOT / "temporal_snapshots"

# Use Phase 1 graph already loaded
assert len(snapshot_nodes) == 206634
assert len(snapshot_edges) == 600623

jira_event_registry = snapshot_nodes[
    (snapshot_nodes["label"] == "Event")
    &
    (snapshot_nodes["source_type"] == "JIRA")
    &
    (snapshot_nodes["temporal_status"] == "EXACT")
].copy()

jira_event_registry["available_at"] = pd.to_datetime(
    jira_event_registry["observed_at"],
    utc=True,
    errors="coerce",
    format="mixed"
)

jira_event_registry = jira_event_registry[
    jira_event_registry["available_at"].notna()
].copy()

jira_event_registry = jira_event_registry[
    [
        "uid",
        "instance_id",
        "label",
        "available_at",
        "source_id",
        "source_type",
        "temporal_basis",
        "temporal_status"
    ]
].rename(columns={"uid": "node_uid"})

jira_event_registry = jira_event_registry.drop_duplicates(
    "node_uid"
)

print("Historical Jira events:", len(jira_event_registry))
print("Projects:", jira_event_registry["instance_id"].nunique())

display(
    jira_event_registry.groupby("instance_id")
    .agg(
        events=("node_uid", "count"),
        first_event=("available_at", "min"),
        last_event=("available_at", "max")
    )
    .reset_index()
)

Historical Jira events: 82053
Projects: 30


,instance_id,events,first_event,last_event
0,AEROGEAR-I01,1592,2018-01-17 05:30:01+00:00,2018-09-25 09:29:15+00:00
1,AMBARI-I01,1817,2016-03-03 00:46:21+00:00,2016-11-09 22:26:15+00:00
2,BXMSDOC-I01,1118,2020-03-20 04:45:40+00:00,2020-11-26 03:06:05+00:00
3,CONFSERVER-I01,6639,2016-03-17 00:01:19+00:00,2016-11-23 14:23:28+00:00
4,DAOS-I01,3289,2021-01-06 09:40:54+00:00,2021-09-14 23:40:04+00:00
5,DOCS-I01,1314,2016-07-05 19:07:09+00:00,2017-03-13 01:42:03+00:00
6,ENTESB-I01,3828,2020-04-08 07:32:26+00:00,2020-12-14 10:17:29+00:00
7,EVG-I01,3245,2020-05-27 14:34:54+00:00,2021-02-02 23:07:55+00:00
8,FLEX-I01,1149,2013-09-02 04:32:52+00:00,2014-05-06 13:02:31+00:00
9,HBASE-I01,1379,2016-07-01 02:40:06+00:00,2017-03-09 15:00:42+00:00


In [ ]:
# ============================================================
# STEP 3.5.3 — RECOVER HISTORICAL ENTITY REFERENCES
# ============================================================

# Event-to-entity relationships
event_entity_edges = snapshot_edges[
    snapshot_edges["type"] == "REFERS_TO"
].copy()

event_times = jira_event_registry[
    ["node_uid", "instance_id", "available_at"]
].rename(columns={
    "node_uid": "from_uid",
    "instance_id": "event_instance",
    "available_at": "event_available_at"
})

# Restrict to verified Jira events
event_entity_edges = event_entity_edges.merge(
    event_times,
    on="from_uid",
    how="inner"
)

# Canonical entity metadata
canonical_nodes = snapshot_nodes[
    snapshot_nodes["label"] == "CanonicalEntity"
][
    ["uid", "instance_id", "label"]
].rename(columns={
    "uid": "to_uid",
    "instance_id": "entity_instance"
})

event_entity_edges = event_entity_edges.merge(
    canonical_nodes,
    on="to_uid",
    how="inner"
)

# Keep only project-consistent references
event_entity_edges = event_entity_edges[
    (
        event_entity_edges["instance_id"]
        == event_entity_edges["event_instance"]
    )
    &
    (
        event_entity_edges["instance_id"]
        == event_entity_edges["entity_instance"]
    )
].copy()

# An entity becomes visible at its first verified event reference
entity_first_seen = (
    event_entity_edges.groupby(
        ["to_uid", "instance_id"],
        as_index=False
    )["event_available_at"]
    .min()
)

entity_first_seen = entity_first_seen.rename(columns={
    "to_uid": "node_uid",
    "event_available_at": "available_at"
})

entity_first_seen["label"] = "CanonicalEntity"
entity_first_seen["source_type"] = "JIRA"
entity_first_seen["temporal_basis"] = (
    "FIRST_VERIFIED_JIRA_EVENT_REFERENCE"
)
entity_first_seen["temporal_status"] = "DERIVED"

# Registry compatible with event registry
entity_registry = entity_first_seen[
    [
        "node_uid",
        "instance_id",
        "label",
        "available_at",
        "source_type",
        "temporal_basis",
        "temporal_status"
    ]
].copy()

entity_registry["source_id"] = None

entity_registry = entity_registry[
    jira_event_registry.columns
]

# Relationship availability is the supporting event timestamp
historical_reference_edges = event_entity_edges[
    [
        "uid",
        "instance_id",
        "from_uid",
        "to_uid",
        "type",
        "event_available_at"
    ]
].rename(columns={
    "uid": "edge_uid",
    "event_available_at": "available_at"
})

historical_reference_edges = (
    historical_reference_edges.drop_duplicates("edge_uid")
)

print("Recovered canonical entities:", len(entity_registry))
print("Historical event-entity relationships:",
      len(historical_reference_edges))

display(entity_registry.head())
display(historical_reference_edges.head())

Recovered canonical entities: 5434
Historical event-entity relationships: 82053


,node_uid,instance_id,label,available_at,source_id,source_type,temporal_basis,temporal_status
0,C_ISSUE::AEROGEAR-I01::aerogear-1878::d58c0f25...,AEROGEAR-I01,CanonicalEntity,2018-01-17 05:30:01+00:00,None,JIRA,FIRST_VERIFIED_JIRA_EVENT_REFERENCE,DERIVED
1,C_ISSUE::AEROGEAR-I01::aerogear-1904::622547d0...,AEROGEAR-I01,CanonicalEntity,2018-01-19 05:09:18+00:00,None,JIRA,FIRST_VERIFIED_JIRA_EVENT_REFERENCE,DERIVED
2,C_ISSUE::AEROGEAR-I01::aerogear-1905::ed1475c7...,AEROGEAR-I01,CanonicalEntity,2018-01-19 05:13:14+00:00,None,JIRA,FIRST_VERIFIED_JIRA_EVENT_REFERENCE,DERIVED
3,C_ISSUE::AEROGEAR-I01::aerogear-1932::e61af0e0...,AEROGEAR-I01,CanonicalEntity,2018-01-19 07:09:39+00:00,None,JIRA,FIRST_VERIFIED_JIRA_EVENT_REFERENCE,DERIVED
4,C_ISSUE::AEROGEAR-I01::aerogear-1968::b3c11682...,AEROGEAR-I01,CanonicalEntity,2018-01-23 09:54:15+00:00,None,JIRA,FIRST_VERIFIED_JIRA_EVENT_REFERENCE,DERIVED


,edge_uid,instance_id,from_uid,to_uid,type,available_at
0,L_785f992910162ef696975bc1,JBEAP-I01,T_78880e35c6d61ac7fd1cb6c0,C_ISSUE::JBEAP-I01::jbeap-5471::a39b18f1d399,REFERS_TO,2016-07-29 13:23:29+00:00
1,L_8e4cedabbc0607aa6837e3de,JBEAP-I01,T_139d055c4525c8627deec04c,C_ISSUE::JBEAP-I01::jbeap-5471::a39b18f1d399,REFERS_TO,2016-07-29 13:23:29+00:00
2,L_942b21ba879021b88d317e06,JBEAP-I01,T_6af67e67ab49c420d887a0bc,C_ISSUE::JBEAP-I01::jbeap-5471::a39b18f1d399,REFERS_TO,2016-07-29 13:23:29+00:00
3,L_90c31acefb43e8396dc97c01,JBEAP-I01,T_c6fc7eef22cb6b4a12e9e61a,C_ISSUE::JBEAP-I01::jbeap-5471::a39b18f1d399,REFERS_TO,2016-07-29 13:23:57+00:00
4,L_ef914a21af634a62ec871d5a,JBEAP-I01,T_5230344df3c92f1bdc70b8b5,C_ISSUE::JBEAP-I01::jbeap-5471::a39b18f1d399,REFERS_TO,2016-07-29 13:24:04+00:00


In [ ]:
# ============================================================
# STEP 3.5.4 — HISTORICAL JIRA ISSUE IDENTITY REGISTRY
# ============================================================

jira_issue_registry = jira_history_df[
    [
        "instance_id",
        "issue_key",
        "created_dt"
    ]
].copy()

jira_issue_registry = jira_issue_registry.rename(columns={
    "created_dt": "available_at"
})

jira_issue_registry["node_uid"] = (
    "HIST_ISSUE::"
    + jira_issue_registry["instance_id"]
    + "::"
    + jira_issue_registry["issue_key"]
)

jira_issue_registry["label"] = "HistoricalIssue"
jira_issue_registry["source_id"] = jira_issue_registry["issue_key"]
jira_issue_registry["source_type"] = "JIRA"
jira_issue_registry["temporal_basis"] = "JIRA_CREATED_AT"
jira_issue_registry["temporal_status"] = "EXACT"

jira_issue_registry = jira_issue_registry[
    jira_event_registry.columns
].drop_duplicates("node_uid")

print("Historical Jira issue nodes:", len(jira_issue_registry))

display(jira_issue_registry.head())

Historical Jira issue nodes: 9000


,node_uid,instance_id,label,available_at,source_id,source_type,temporal_basis,temporal_status
0,HIST_ISSUE::JBEAP-I01::JBEAP-22995,JBEAP-I01,HistoricalIssue,2021-12-21 14:39:41+00:00,JBEAP-22995,JIRA,JIRA_CREATED_AT,EXACT
1,HIST_ISSUE::JBEAP-I01::JBEAP-22981,JBEAP-I01,HistoricalIssue,2021-12-17 05:27:13+00:00,JBEAP-22981,JIRA,JIRA_CREATED_AT,EXACT
2,HIST_ISSUE::JBEAP-I01::JBEAP-22948,JBEAP-I01,HistoricalIssue,2021-12-14 11:47:45+00:00,JBEAP-22948,JIRA,JIRA_CREATED_AT,EXACT
3,HIST_ISSUE::JBEAP-I01::JBEAP-22945,JBEAP-I01,HistoricalIssue,2021-12-13 15:16:58+00:00,JBEAP-22945,JIRA,JIRA_CREATED_AT,EXACT
4,HIST_ISSUE::JBEAP-I01::JBEAP-22917,JBEAP-I01,HistoricalIssue,2021-12-10 06:17:20+00:00,JBEAP-22917,JIRA,JIRA_CREATED_AT,EXACT


In [ ]:
# ============================================================
# STEP 3.5.5 — DOCUMENT EVENT REGISTRY
# ============================================================

document_event_registry = snapshot_nodes[
    (snapshot_nodes["label"] == "Event")
    &
    (snapshot_nodes["source_type"].isin(["MEETING", "PRD"]))
    &
    (snapshot_nodes["temporal_status"] == "EXACT")
].copy()

document_event_registry["available_at"] = pd.to_datetime(
    document_event_registry["observed_at"],
    utc=True,
    errors="coerce",
    format="mixed"
)

document_event_registry = document_event_registry[
    document_event_registry["available_at"].notna()
].copy()

document_event_registry = document_event_registry[
    [
        "uid",
        "instance_id",
        "label",
        "available_at",
        "source_id",
        "source_type",
        "temporal_basis",
        "temporal_status"
    ]
].rename(columns={"uid": "node_uid"})

document_event_registry = (
    document_event_registry.drop_duplicates("node_uid")
)

# Conservative handling: keep document events in a separate
# evidence registry until document availability is verified.
document_event_registry["availability_verified"] = False

print("Timestamped document events:",
      len(document_event_registry))

display(
    document_event_registry["source_type"].value_counts()
)

print(
    "Document events pending availability verification:",
    (~document_event_registry["availability_verified"]).sum()
)

Timestamped document events: 343


,count
source_type,
MEETING,319
PRD,24


Document events pending availability verification: 343


In [ ]:
# ============================================================
# STEP 3.5.6 — CONSOLIDATED TEMPORAL REGISTRIES
# ============================================================

# ------------------------------------------------------------
# Combine nodes
# ------------------------------------------------------------

kg_node_registry = pd.concat(
    [
        jira_event_registry,
        entity_registry,
        jira_issue_registry
    ],
    ignore_index=True
)

kg_node_registry = (
    kg_node_registry
    .sort_values("available_at")
    .drop_duplicates("node_uid", keep="first")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Build historical issue-to-event edges
# ------------------------------------------------------------

issue_event_links = jira_event_registry[
    [
        "node_uid",
        "instance_id",
        "source_id",
        "available_at"
    ]
].copy()

issue_event_links = issue_event_links.rename(columns={
    "node_uid": "to_uid",
    "source_id": "issue_key"
})

issue_lookup = jira_history_df[
    ["instance_id", "issue_key", "created_dt"]
].copy()

issue_lookup["from_uid"] = (
    "HIST_ISSUE::"
    + issue_lookup["instance_id"]
    + "::"
    + issue_lookup["issue_key"]
)

issue_event_links = issue_event_links.merge(
    issue_lookup,
    on=["instance_id", "issue_key"],
    how="inner"
)

# Events cannot precede their issue creation
issue_event_links = issue_event_links[
    issue_event_links["available_at"]
    >= issue_event_links["created_dt"]
].copy()

issue_event_links["type"] = "HAS_HISTORICAL_EVENT"

issue_event_links["edge_uid"] = (
    "HIST_EDGE::"
    + issue_event_links["from_uid"]
    + "::"
    + issue_event_links["to_uid"]
)

issue_event_links = issue_event_links[
    [
        "edge_uid",
        "instance_id",
        "from_uid",
        "to_uid",
        "type",
        "available_at"
    ]
]

# ------------------------------------------------------------
# Combine edges
# ------------------------------------------------------------

kg_edge_registry = pd.concat(
    [
        historical_reference_edges,
        issue_event_links
    ],
    ignore_index=True
)

kg_edge_registry = (
    kg_edge_registry
    .drop_duplicates("edge_uid")
    .reset_index(drop=True)
)

print("FINAL TEMPORAL REGISTRY")

print("Nodes:", len(kg_node_registry))
print("Edges:", len(kg_edge_registry))

print("\nNode types:")
display(kg_node_registry["label"].value_counts())

print("\nRelationship types:")
display(kg_edge_registry["type"].value_counts())

FINAL TEMPORAL REGISTRY
Nodes: 96487
Edges: 164101

Node types:


,count
label,
Event,82053
HistoricalIssue,9000
CanonicalEntity,5434



Relationship types:


,count
type,
REFERS_TO,82053
HAS_HISTORICAL_EVENT,82048


In [ ]:
# ============================================================
# STEP 3.5 — CORRECTED TEMPORAL KG VALIDATIONS
# ============================================================

checks = []

def check(name, passed, detail):
    checks.append({
        "validation": name,
        "status": "PASS" if passed else "FAIL",
        "details": detail
    })

node_ids = set(kg_node_registry["node_uid"])

node_time_lookup = (
    kg_node_registry.set_index("node_uid")["available_at"]
)

check(
    "Historical Jira events recovered",
    len(jira_event_registry) == 82053,
    len(jira_event_registry)
)

check(
    "Historical issue identities preserved",
    len(jira_issue_registry) == 9000,
    len(jira_issue_registry)
)

check(
    "No duplicate node IDs",
    kg_node_registry["node_uid"].is_unique,
    kg_node_registry["node_uid"].duplicated().sum()
)

check(
    "No duplicate edge IDs",
    kg_edge_registry["edge_uid"].is_unique,
    kg_edge_registry["edge_uid"].duplicated().sum()
)

check(
    "All edges have valid source nodes",
    kg_edge_registry["from_uid"].isin(node_ids).all(),
    "Checked"
)

check(
    "All edges have valid target nodes",
    kg_edge_registry["to_uid"].isin(node_ids).all(),
    "Checked"
)

check(
    "All nodes timestamped",
    kg_node_registry["available_at"].notna().all(),
    "Checked"
)

check(
    "All edges timestamped",
    kg_edge_registry["available_at"].notna().all(),
    "Checked"
)

check(
    "Edges do not precede source nodes",
    (
        kg_edge_registry["available_at"]
        >= kg_edge_registry["from_uid"].map(node_time_lookup)
    ).all(),
    "Checked"
)

check(
    "Edges do not precede target nodes",
    (
        kg_edge_registry["available_at"]
        >= kg_edge_registry["to_uid"].map(node_time_lookup)
    ).all(),
    "Checked"
)

check(
    "All projects represented",
    kg_node_registry["instance_id"].nunique() == 30,
    kg_node_registry["instance_id"].nunique()
)

check(
    "No document events in strict predictive registry",
    not kg_node_registry["node_uid"].isin(
        document_event_registry["node_uid"]
    ).any(),
    "Checked"
)

check(
    "987 checkpoints preserved",
    len(snapshot_checkpoints) == 987,
    len(snapshot_checkpoints)
)

# Validate issue-event time consistency
check(
    "Issue-event edges do not precede issue creation",
    (
        issue_event_links["available_at"]
        >= issue_event_links["created_dt"]
    ).all()
    if "created_dt" in issue_event_links.columns
    else True,
    "Checked"
)

validation_df = pd.DataFrame(checks)

display(validation_df)

failed = validation_df[
    validation_df["status"] == "FAIL"
]

print("\n" + "=" * 60)

if failed.empty:
    print("ALL CORRECTED TEMPORAL KG VALIDATIONS PASSED")
else:
    print(f"{len(failed)} VALIDATIONS FAILED")

print("=" * 60)

,validation,status,details
0,Historical Jira events recovered,PASS,82053
1,Historical issue identities preserved,PASS,9000
2,No duplicate node IDs,PASS,0
3,No duplicate edge IDs,PASS,0
4,All edges have valid source nodes,PASS,Checked
5,All edges have valid target nodes,PASS,Checked
6,All nodes timestamped,PASS,Checked
7,All edges timestamped,PASS,Checked
8,Edges do not precede source nodes,PASS,Checked
9,Edges do not precede target nodes,PASS,Checked



ALL CORRECTED TEMPORAL KG VALIDATIONS PASSED


In [ ]:
# ============================================================
# STEP 3.5 — EXPORT CORRECTED TEMPORAL REGISTRIES
# ============================================================

assert (validation_df["status"] == "PASS").all(), \
    "Temporal KG validation failed."

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/GC1/Temporal_snapshots"
)

assert OUTPUT_DIR.is_dir()

kg_node_registry.to_parquet(
    OUTPUT_DIR / "temporal_node_registry.parquet",
    index=False
)

kg_edge_registry.to_parquet(
    OUTPUT_DIR / "temporal_edge_registry.parquet",
    index=False
)

document_event_registry.to_parquet(
    OUTPUT_DIR / "document_event_availability_audit.parquet",
    index=False
)

print("Corrected temporal registries exported.")
print("Output:", OUTPUT_DIR)

print("\nSaved files:")
print("- temporal_node_registry.parquet")
print("- temporal_edge_registry.parquet")
print("- document_event_availability_audit.parquet")

Corrected temporal registries exported.
Output: /content/drive/MyDrive/GC1/Temporal_snapshots

Saved files:
- temporal_node_registry.parquet
- temporal_edge_registry.parquet
- document_event_availability_audit.parquet


In [ ]:
# ============================================================
# STEP 3.5 — FINAL EVENT-ISSUE LINK AUDIT
# ============================================================

linked_event_ids = set(issue_event_links["to_uid"])

unmatched_events = jira_event_registry[
    ~jira_event_registry["node_uid"].isin(linked_event_ids)
].copy()

print("Jira events:", len(jira_event_registry))
print("Issue-event relationships:", len(issue_event_links))
print("Unmatched events:", len(unmatched_events))

display(
    unmatched_events[
        [
            "node_uid",
            "instance_id",
            "source_id",
            "available_at"
        ]
    ]
)

# Check whether each unmatched event has a matching Jira issue
jira_lookup = jira_history_df[
    ["instance_id", "issue_key", "created_dt"]
].drop_duplicates()

unmatched_audit = unmatched_events.merge(
    jira_lookup,
    left_on=["instance_id", "source_id"],
    right_on=["instance_id", "issue_key"],
    how="left"
)

unmatched_audit["reason"] = np.select(
    [
        unmatched_audit["created_dt"].isna(),
        unmatched_audit["available_at"] <
        unmatched_audit["created_dt"]
    ],
    [
        "ISSUE_RECORD_UNAVAILABLE",
        "EVENT_BEFORE_ISSUE_CREATION"
    ],
    default="OTHER_LINKING_ISSUE"
)

print("\nUNMATCHED EVENT REASONS")
display(unmatched_audit["reason"].value_counts())

display(unmatched_audit)

Jira events: 82053
Issue-event relationships: 82048
Unmatched events: 5


,node_uid,instance_id,source_id,available_at
186168,T_2d08991deef8893a183c95fc,DAOS-I01,DAOS-8465,2021-09-01 20:57:11+00:00
186169,T_593dc1aaa0ef80d464727a72,DAOS-I01,DAOS-8465,2021-09-01 20:57:11+00:00
186333,T_61af3fd6ceee265cc1e90408,DAOS-I01,DAOS-8391,2021-08-24 04:12:42+00:00
186370,T_5dc3f6614f8f92fda6dad038,DAOS-I01,DAOS-8390,2021-08-24 04:06:08+00:00
186478,T_924b36dff44de7e141c45fcc,DAOS-I01,DAOS-8365,2021-08-24 02:41:18+00:00



UNMATCHED EVENT REASONS


,count
reason,
EVENT_BEFORE_ISSUE_CREATION,5


,node_uid,instance_id,label,available_at,source_id,source_type,temporal_basis,temporal_status,issue_key,created_dt,reason
0,T_2d08991deef8893a183c95fc,DAOS-I01,Event,2021-09-01 20:57:11+00:00,DAOS-8465,JIRA,jira.full_changelog.created_or_status_history....,EXACT,DAOS-8465,2021-09-01 20:57:11.320000+00:00,EVENT_BEFORE_ISSUE_CREATION
1,T_593dc1aaa0ef80d464727a72,DAOS-I01,Event,2021-09-01 20:57:11+00:00,DAOS-8465,JIRA,jira.full_changelog.created_or_status_history....,EXACT,DAOS-8465,2021-09-01 20:57:11.320000+00:00,EVENT_BEFORE_ISSUE_CREATION
2,T_61af3fd6ceee265cc1e90408,DAOS-I01,Event,2021-08-24 04:12:42+00:00,DAOS-8391,JIRA,jira.full_changelog.created_or_status_history....,EXACT,DAOS-8391,2021-08-24 04:12:42.091000+00:00,EVENT_BEFORE_ISSUE_CREATION
3,T_5dc3f6614f8f92fda6dad038,DAOS-I01,Event,2021-08-24 04:06:08+00:00,DAOS-8390,JIRA,jira.full_changelog.created_or_status_history....,EXACT,DAOS-8390,2021-08-24 04:06:08.337000+00:00,EVENT_BEFORE_ISSUE_CREATION
4,T_924b36dff44de7e141c45fcc,DAOS-I01,Event,2021-08-24 02:41:18+00:00,DAOS-8365,JIRA,jira.full_changelog.created_or_status_history....,EXACT,DAOS-8365,2021-08-24 02:41:18.233000+00:00,EVENT_BEFORE_ISSUE_CREATION


### 5.5 Weekly snapshot construction

In [ ]:
# ============================================================
# STEP 3.5.7 — LOAD SNAPSHOT COMPONENTS
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/GC1")
OUTPUT_DIR = ROOT / "Temporal_snapshots"

assert OUTPUT_DIR.is_dir(), "Temporal_snapshots folder not found"

# Load finalized temporal registries
kg_node_registry = pd.read_parquet(
    OUTPUT_DIR / "temporal_node_registry.parquet"
)

kg_edge_registry = pd.read_parquet(
    OUTPUT_DIR / "temporal_edge_registry.parquet"
)

# Reuse checkpoint data
checkpoints = snapshot_checkpoints.copy()

# Reuse historical Jira states
if "checkpoint_issue_states_final" in globals():
    jira_states = checkpoint_issue_states_final.copy()
else:
    jira_states = pd.read_csv(
        OUTPUT_DIR / "jira_checkpoint_issue_states_final.csv"
    )

# Reuse dependency states
if "checkpoint_dependencies_safe" in globals():
    dependency_states = checkpoint_dependencies_safe.copy()
else:
    dependency_states = pd.read_csv(
        OUTPUT_DIR / "checkpoint_active_dependencies.csv"
    )

# Normalize timestamps
for df, columns in [
    (kg_node_registry, ["available_at"]),
    (kg_edge_registry, ["available_at"]),
    (checkpoints, ["checkpoint_time", "horizon_end"]),
    (jira_states, ["checkpoint_time"]),
    (dependency_states, ["checkpoint_time", "active_from"])
]:
    for col in columns:
        df[col] = pd.to_datetime(
            df[col],
            utc=True,
            errors="coerce",
            format="mixed"
        )

KEYS = ["instance_id", "week_number", "checkpoint_time"]

assert len(checkpoints) == 987
assert checkpoints["instance_id"].nunique() == 30

print("Temporal nodes:", len(kg_node_registry))
print("Temporal edges:", len(kg_edge_registry))
print("Checkpoints:", len(checkpoints))
print("Jira states:", len(jira_states))
print("Dependency states:", len(dependency_states))

Temporal nodes: 96487
Temporal edges: 164101
Checkpoints: 987
Jira states: 228255
Dependency states: 28340


In [ ]:
# ============================================================
# STEP 3.5.8 — CREATE WEEKLY SNAPSHOT MANIFEST
# ============================================================

snapshot_manifest = checkpoints[
    ["instance_id", "week_number",
     "checkpoint_time", "horizon_end"]
].copy()

snapshot_manifest["snapshot_id"] = (
    snapshot_manifest["instance_id"]
    + "::W"
    + snapshot_manifest["week_number"]
    .astype(int).astype(str).str.zfill(2)
)

snapshot_manifest = snapshot_manifest[
    [
        "snapshot_id",
        "instance_id",
        "week_number",
        "checkpoint_time",
        "horizon_end"
    ]
].sort_values(
    ["instance_id", "checkpoint_time"]
).reset_index(drop=True)

print("Snapshots:", len(snapshot_manifest))
print("Projects:", snapshot_manifest["instance_id"].nunique())

display(snapshot_manifest.head(10))

Snapshots: 987
Projects: 30


,snapshot_id,instance_id,week_number,checkpoint_time,horizon_end
0,AEROGEAR-I01::W01,AEROGEAR-I01,1,2018-01-24 05:30:01+00:00,2018-02-07 05:30:01+00:00
1,AEROGEAR-I01::W02,AEROGEAR-I01,2,2018-01-31 05:30:01+00:00,2018-02-14 05:30:01+00:00
2,AEROGEAR-I01::W03,AEROGEAR-I01,3,2018-02-07 05:30:01+00:00,2018-02-21 05:30:01+00:00
3,AEROGEAR-I01::W04,AEROGEAR-I01,4,2018-02-14 05:30:01+00:00,2018-02-28 05:30:01+00:00
4,AEROGEAR-I01::W05,AEROGEAR-I01,5,2018-02-21 05:30:01+00:00,2018-03-07 05:30:01+00:00
5,AEROGEAR-I01::W06,AEROGEAR-I01,6,2018-02-28 05:30:01+00:00,2018-03-14 05:30:01+00:00
6,AEROGEAR-I01::W07,AEROGEAR-I01,7,2018-03-07 05:30:01+00:00,2018-03-21 05:30:01+00:00
7,AEROGEAR-I01::W08,AEROGEAR-I01,8,2018-03-14 05:30:01+00:00,2018-03-28 05:30:01+00:00
8,AEROGEAR-I01::W09,AEROGEAR-I01,9,2018-03-21 05:30:01+00:00,2018-04-04 05:30:01+00:00
9,AEROGEAR-I01::W10,AEROGEAR-I01,10,2018-03-28 05:30:01+00:00,2018-04-11 05:30:01+00:00


In [ ]:
# ============================================================
# STEP 3.5.9 — HISTORICAL GRAPH SNAPSHOT LOADER
# ============================================================

nodes_by_project = {
    p: g.copy()
    for p, g in kg_node_registry.groupby("instance_id")
}

edges_by_project = {
    p: g.copy()
    for p, g in kg_edge_registry.groupby("instance_id")
}

jira_by_checkpoint = {
    key: group.copy()
    for key, group in jira_states.groupby(
        ["instance_id", "week_number"]
    )
}

deps_by_checkpoint = {
    key: group.copy()
    for key, group in dependency_states.groupby(
        ["instance_id", "week_number"]
    )
}

def get_temporal_snapshot(instance_id, week_number):

    match = snapshot_manifest[
        (snapshot_manifest["instance_id"] == instance_id)
        &
        (snapshot_manifest["week_number"] == week_number)
    ]

    if len(match) != 1:
        raise ValueError("Snapshot not found or duplicated")

    checkpoint = match.iloc[0]
    t = checkpoint["checkpoint_time"]

    nodes = nodes_by_project[instance_id]
    nodes = nodes[
        nodes["available_at"] <= t
    ].copy()

    node_ids = set(nodes["node_uid"])

    edges = edges_by_project.get(
        instance_id,
        kg_edge_registry.iloc[0:0]
    )

    edges = edges[
        (edges["available_at"] <= t)
        &
        (edges["from_uid"].isin(node_ids))
        &
        (edges["to_uid"].isin(node_ids))
    ].copy()

    jira = jira_by_checkpoint.get(
        (instance_id, week_number),
        jira_states.iloc[0:0]
    ).copy()

    dependencies = deps_by_checkpoint.get(
        (instance_id, week_number),
        dependency_states.iloc[0:0]
    ).copy()

    # Remove retrospective status-quality fields.
    # The original reconstructed status is retained.
    jira = jira.drop(
        columns=[
            "status_uncertain",
            "feature_safe_status"
        ],
        errors="ignore"
    )

    return {
        "snapshot_id": checkpoint["snapshot_id"],
        "checkpoint_time": t,
        "nodes": nodes,
        "edges": edges,
        "jira_states": jira,
        "dependencies": dependencies
    }


# Test one snapshot
sample = snapshot_manifest.iloc[0]

test_snapshot = get_temporal_snapshot(
    sample["instance_id"],
    sample["week_number"]
)

print("Snapshot:", test_snapshot["snapshot_id"])
print("Nodes:", len(test_snapshot["nodes"]))
print("Edges:", len(test_snapshot["edges"]))
print("Jira states:", len(test_snapshot["jira_states"]))
print("Dependencies:", len(test_snapshot["dependencies"]))

Snapshot: AEROGEAR-I01::W01
Nodes: 239
Edges: 142
Jira states: 162
Dependencies: 0


In [ ]:
# ============================================================
# STEP 3.5.10 — GENERATE SNAPSHOT STATISTICS
# ============================================================

snapshot_records = []

for _, checkpoint in snapshot_manifest.iterrows():

    snap = get_temporal_snapshot(
        checkpoint["instance_id"],
        checkpoint["week_number"]
    )

    nodes = snap["nodes"]
    edges = snap["edges"]
    jira = snap["jira_states"]
    deps = snap["dependencies"]

    snapshot_records.append({
        "snapshot_id": snap["snapshot_id"],
        "instance_id": checkpoint["instance_id"],
        "week_number": checkpoint["week_number"],
        "checkpoint_time": checkpoint["checkpoint_time"],

        "kg_nodes": len(nodes),
        "kg_edges": len(edges),

        "historical_issues": jira["issue_key"].nunique(),

        "active_dependency_links": len(deps),
        "blocked_issues": (
            deps["downstream_issue"].nunique()
        ),
        "upstream_issues": (
            deps["upstream_issue"].nunique()
        )
    })

snapshot_statistics = pd.DataFrame(snapshot_records)

print("Snapshot statistics generated:",
      len(snapshot_statistics))

display(snapshot_statistics.head(10))

print("\nSummary:")
display(
    snapshot_statistics[
        [
            "kg_nodes",
            "kg_edges",
            "historical_issues",
            "active_dependency_links",
            "blocked_issues",
            "upstream_issues"
        ]
    ].describe()
)

Snapshot statistics generated: 987


,snapshot_id,instance_id,week_number,checkpoint_time,kg_nodes,kg_edges,historical_issues,active_dependency_links,blocked_issues,upstream_issues
0,AEROGEAR-I01::W01,AEROGEAR-I01,1,2018-01-24 05:30:01+00:00,239,142,162,0,0,0
1,AEROGEAR-I01::W02,AEROGEAR-I01,2,2018-01-31 05:30:01+00:00,271,202,162,0,0,0
2,AEROGEAR-I01::W03,AEROGEAR-I01,3,2018-02-07 05:30:01+00:00,284,228,162,0,0,0
3,AEROGEAR-I01::W04,AEROGEAR-I01,4,2018-02-14 05:30:01+00:00,328,308,164,0,0,0
4,AEROGEAR-I01::W05,AEROGEAR-I01,5,2018-02-21 05:30:01+00:00,365,374,166,2,1,2
5,AEROGEAR-I01::W06,AEROGEAR-I01,6,2018-02-28 05:30:01+00:00,396,428,168,2,1,2
6,AEROGEAR-I01::W07,AEROGEAR-I01,7,2018-03-07 05:30:01+00:00,538,676,177,17,8,14
7,AEROGEAR-I01::W08,AEROGEAR-I01,8,2018-03-14 05:30:01+00:00,670,928,180,20,10,17
8,AEROGEAR-I01::W09,AEROGEAR-I01,9,2018-03-21 05:30:01+00:00,711,1010,180,20,10,17
9,AEROGEAR-I01::W10,AEROGEAR-I01,10,2018-03-28 05:30:01+00:00,720,1028,180,20,10,17



Summary:


,kg_nodes,kg_edges,historical_issues,active_dependency_links,blocked_issues,upstream_issues
count,987.000000,987.000000,987.000000,987.000000,987.000000,987.000000
mean,1790.075988,2885.753799,231.261398,28.713273,17.888551,18.886525
std,1612.221813,3115.384624,47.059784,98.992736,65.684720,59.932072
min,88.000000,0.000000,46.000000,0.000000,0.000000,0.000000
25%,693.500000,807.000000,199.000000,0.000000,0.000000,0.000000
50%,1230.000000,1752.000000,244.000000,0.000000,0.000000,0.000000
75%,2312.500000,3914.000000,268.000000,7.000000,6.000000,6.000000
max,9185.000000,17472.000000,299.000000,587.000000,415.000000,337.000000


In [ ]:
# ============================================================
# STEP 3.5.11 — FINAL SNAPSHOT MANIFEST
# ============================================================

snapshot_manifest_final = snapshot_manifest.merge(
    snapshot_statistics.drop(
        columns=[
            "instance_id",
            "week_number",
            "checkpoint_time"
        ]
    ),
    on="snapshot_id",
    how="left",
    validate="one_to_one"
)

print("Final snapshot records:",
      len(snapshot_manifest_final))

print("\nProjects represented:",
      snapshot_manifest_final["instance_id"].nunique())

display(snapshot_manifest_final.head(10))

Final snapshot records: 987

Projects represented: 30


,snapshot_id,instance_id,week_number,checkpoint_time,horizon_end,kg_nodes,kg_edges,historical_issues,active_dependency_links,blocked_issues,upstream_issues
0,AEROGEAR-I01::W01,AEROGEAR-I01,1,2018-01-24 05:30:01+00:00,2018-02-07 05:30:01+00:00,239,142,162,0,0,0
1,AEROGEAR-I01::W02,AEROGEAR-I01,2,2018-01-31 05:30:01+00:00,2018-02-14 05:30:01+00:00,271,202,162,0,0,0
2,AEROGEAR-I01::W03,AEROGEAR-I01,3,2018-02-07 05:30:01+00:00,2018-02-21 05:30:01+00:00,284,228,162,0,0,0
3,AEROGEAR-I01::W04,AEROGEAR-I01,4,2018-02-14 05:30:01+00:00,2018-02-28 05:30:01+00:00,328,308,164,0,0,0
4,AEROGEAR-I01::W05,AEROGEAR-I01,5,2018-02-21 05:30:01+00:00,2018-03-07 05:30:01+00:00,365,374,166,2,1,2
5,AEROGEAR-I01::W06,AEROGEAR-I01,6,2018-02-28 05:30:01+00:00,2018-03-14 05:30:01+00:00,396,428,168,2,1,2
6,AEROGEAR-I01::W07,AEROGEAR-I01,7,2018-03-07 05:30:01+00:00,2018-03-21 05:30:01+00:00,538,676,177,17,8,14
7,AEROGEAR-I01::W08,AEROGEAR-I01,8,2018-03-14 05:30:01+00:00,2018-03-28 05:30:01+00:00,670,928,180,20,10,17
8,AEROGEAR-I01::W09,AEROGEAR-I01,9,2018-03-21 05:30:01+00:00,2018-04-04 05:30:01+00:00,711,1010,180,20,10,17
9,AEROGEAR-I01::W10,AEROGEAR-I01,10,2018-03-28 05:30:01+00:00,2018-04-11 05:30:01+00:00,720,1028,180,20,10,17


In [ ]:
# ============================================================
# STEP 3.5 — CONSOLIDATED SNAPSHOT VALIDATION
# ============================================================

checks = []

def check(name, condition, details):
    checks.append({
        "validation": name,
        "status": "PASS" if bool(condition) else "FAIL",
        "details": details
    })

check(
    "987 snapshots generated",
    len(snapshot_manifest_final) == 987,
    len(snapshot_manifest_final)
)

check(
    "30 projects represented",
    snapshot_manifest_final["instance_id"].nunique() == 30,
    snapshot_manifest_final["instance_id"].nunique()
)

check(
    "Unique snapshot identifiers",
    snapshot_manifest_final["snapshot_id"].is_unique,
    "Checked"
)

check(
    "No duplicate project-week records",
    not snapshot_manifest_final.duplicated(
        ["instance_id", "week_number"]
    ).any(),
    "Checked"
)

check(
    "14-day prediction horizon",
    (
        snapshot_manifest_final["horizon_end"]
        - snapshot_manifest_final["checkpoint_time"]
        == pd.Timedelta(days=14)
    ).all(),
    "Checked"
)

check(
    "All KG nodes timestamped",
    kg_node_registry["available_at"].notna().all(),
    "Checked"
)

check(
    "All KG edges timestamped",
    kg_edge_registry["available_at"].notna().all(),
    "Checked"
)

node_ids = set(kg_node_registry["node_uid"])

check(
    "All edge sources exist",
    kg_edge_registry["from_uid"].isin(node_ids).all(),
    "Checked"
)

check(
    "All edge targets exist",
    kg_edge_registry["to_uid"].isin(node_ids).all(),
    "Checked"
)

node_time_lookup = kg_node_registry.set_index(
    "node_uid"
)["available_at"]

check(
    "Edges do not precede source nodes",
    (
        kg_edge_registry["available_at"]
        >= kg_edge_registry["from_uid"].map(node_time_lookup)
    ).all(),
    "Checked"
)

check(
    "Edges do not precede target nodes",
    (
        kg_edge_registry["available_at"]
        >= kg_edge_registry["to_uid"].map(node_time_lookup)
    ).all(),
    "Checked"
)

check(
    "No Jira states before issue creation",
    (
        jira_states["issue_key"].map(
            jira_history_df.set_index("issue_key")["created_dt"]
        )
        <= jira_states["checkpoint_time"]
    ).all(),
    "Checked"
)

check(
    "No dependencies before known start",
    (
        dependency_states["active_from"]
        <= dependency_states["checkpoint_time"]
    ).all(),
    "Checked"
)

check(
    "No future dependency removal fields",
    not any(
        c in dependency_states.columns
        for c in ["active_to", "end_event_uid"]
    ),
    "Checked"
)

count_columns = [
    "kg_nodes",
    "kg_edges",
    "historical_issues",
    "active_dependency_links",
    "blocked_issues",
    "upstream_issues"
]

check(
    "No missing snapshot counts",
    snapshot_manifest_final[count_columns].notna().all().all(),
    "Checked"
)

check(
    "No negative snapshot counts",
    (
        snapshot_manifest_final[count_columns] >= 0
    ).all().all(),
    "Checked"
)

# Full point-in-time audit across all snapshots
future_nodes = 0
future_edges = 0
invalid_edge_endpoints = 0

for _, cp in snapshot_manifest.iterrows():

    snap = get_temporal_snapshot(
        cp["instance_id"],
        cp["week_number"]
    )

    t = snap["checkpoint_time"]
    nodes = snap["nodes"]
    edges = snap["edges"]

    future_nodes += int(
        (nodes["available_at"] > t).sum()
    )

    future_edges += int(
        (edges["available_at"] > t).sum()
    )

    visible_ids = set(nodes["node_uid"])

    invalid_edge_endpoints += int(
        (
            ~edges["from_uid"].isin(visible_ids)
            |
            ~edges["to_uid"].isin(visible_ids)
        ).sum()
    )

check(
    "No future KG nodes in snapshots",
    future_nodes == 0,
    future_nodes
)

check(
    "No future KG edges in snapshots",
    future_edges == 0,
    future_edges
)

check(
    "All snapshot edge endpoints exist",
    invalid_edge_endpoints == 0,
    invalid_edge_endpoints
)

validation_df = pd.DataFrame(checks)

display(validation_df)

failed = validation_df[
    validation_df["status"] == "FAIL"
]

print("\n" + "=" * 60)

if failed.empty:
    print("ALL WEEKLY SNAPSHOT STRUCTURAL VALIDATIONS PASSED")
else:
    print(f"{len(failed)} VALIDATIONS FAILED")

print("=" * 60)

,validation,status,details
0,987 snapshots generated,PASS,987
1,30 projects represented,PASS,30
2,Unique snapshot identifiers,PASS,Checked
3,No duplicate project-week records,PASS,Checked
4,14-day prediction horizon,PASS,Checked
5,All KG nodes timestamped,PASS,Checked
6,All KG edges timestamped,PASS,Checked
7,All edge sources exist,PASS,Checked
8,All edge targets exist,PASS,Checked
9,Edges do not precede source nodes,PASS,Checked



ALL WEEKLY SNAPSHOT STRUCTURAL VALIDATIONS PASSED


In [ ]:
# ============================================================
# STEP 3.5 — EXPORT FINAL SNAPSHOT MANIFEST
# ============================================================

assert (validation_df["status"] == "PASS").all(), \
    "Snapshot validation failed."

snapshot_manifest_final.to_csv(
    OUTPUT_DIR / "weekly_snapshot_manifest.csv",
    index=False
)

snapshot_statistics.to_parquet(
    OUTPUT_DIR / "weekly_snapshot_statistics.parquet",
    index=False
)

print("Weekly snapshot construction completed.")

print("\nFiles saved:")
print("- weekly_snapshot_manifest.csv")
print("- weekly_snapshot_statistics.parquet")

print("\nOutput directory:", OUTPUT_DIR)

Weekly snapshot construction completed.

Files saved:
- weekly_snapshot_manifest.csv
- weekly_snapshot_statistics.parquet

Output directory: /content/drive/MyDrive/GC1/Temporal_snapshots


### 5.6 Predictive dependency corrections

In [ ]:
# ============================================================
# STEP 3.5.12 — MATCH DEPENDENCY EXCLUSION FLAGS
# ============================================================

import pandas as pd
import numpy as np

history = final_dependency_history.copy()
intervals = corrected_intervals_df.copy()

flags = [
    "suspected_bulk_update",
    "suspected_jbide_migration",
    "exclude_from_escalation"
]

for col in flags:
    history[col] = history[col].fillna(False).astype(bool)

print("DEPENDENCY HISTORY CLASSIFICATION")

display(
    history.groupby(
        ["action"] + flags,
        dropna=False
    ).size().reset_index(name="count")
)

# Only ADD events can establish a known-start interval
add_events = history[
    history["action"] == "ADD"
].copy()

assert add_events["event_uid"].is_unique, (
    "Duplicate ADD event UIDs found"
)

# Match start_event_uid with original event_uid
intervals = intervals.merge(
    add_events[
        ["event_uid"] + flags
    ],
    left_on="start_event_uid",
    right_on="event_uid",
    how="left",
    validate="many_to_one",
    indicator=True
)

intervals["matched_start_event"] = (
    intervals["_merge"] == "both"
)

# Unmatched records are not automatically considered safe
intervals["predictive_eligible"] = (
    intervals["start_known"].eq(True)
    & intervals["matched_start_event"]
    & ~intervals[flags].fillna(False).any(axis=1)
)

print("\nINTERVAL MATCHING")
display(intervals["_merge"].value_counts())

print("\nPREDICTIVE ELIGIBILITY")
display(intervals["predictive_eligible"].value_counts())

print("\nEXCLUDED INTERVALS BY PROJECT")
display(
    intervals.loc[
        ~intervals["predictive_eligible"]
    ].groupby("instance_id").size()
    .sort_values(ascending=False)
)

DEPENDENCY HISTORY CLASSIFICATION


,action,suspected_bulk_update,suspected_jbide_migration,exclude_from_escalation,count
0,ADD,False,False,False,816
1,ADD,False,True,True,15
2,ADD,True,False,True,828
3,MIGRATION,False,False,False,808
4,REMOVE,False,False,False,912



INTERVAL MATCHING


/tmp/ipykernel_4111/3807651592.py:58: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  & ~intervals[flags].fillna(False).any(axis=1)


,count
_merge,
both,1657
left_only,808
right_only,0



PREDICTIVE ELIGIBILITY


,count
predictive_eligible,
False,1650
True,815



EXCLUDED INTERVALS BY PROJECT


,0
instance_id,
JBIDE-I01,1144
ENTESB-I01,464
BXMSDOC-I01,42


In [ ]:
# ============================================================
# STEP 3.5.13 — RECONSTRUCT PREDICTIVE DEPENDENCIES
# ============================================================

safe_intervals = intervals[
    intervals["predictive_eligible"]
].copy()

for col in ["active_from", "active_to"]:
    safe_intervals[col] = pd.to_datetime(
        safe_intervals[col],
        utc=True,
        errors="coerce",
        format="mixed"
    )

records = []

for _, cp in snapshot_checkpoints.iterrows():

    project = cp["instance_id"]
    t = cp["checkpoint_time"]

    active = safe_intervals[
        (safe_intervals["instance_id"] == project)
        & (safe_intervals["active_from"] <= t)
        & (
            safe_intervals["active_to"].isna()
            | (safe_intervals["active_to"] > t)
        )
    ]

    for _, link in active.iterrows():
        records.append({
            "instance_id": project,
            "week_number": cp["week_number"],
            "checkpoint_time": t,
            "upstream_issue": link["upstream_issue"],
            "downstream_issue": link["downstream_issue"],
            "active_from": link["active_from"],
            "start_event_uid": link["start_event_uid"]
        })

predictive_dependencies_df = pd.DataFrame(
    records,
    columns=[
        "instance_id",
        "week_number",
        "checkpoint_time",
        "upstream_issue",
        "downstream_issue",
        "active_from",
        "start_event_uid"
    ]
)

predictive_dependencies_df = (
    predictive_dependencies_df
    .drop_duplicates(
        [
            "instance_id",
            "week_number",
            "upstream_issue",
            "downstream_issue"
        ]
    )
    .reset_index(drop=True)
)

print("Predictive dependency records:",
      len(predictive_dependencies_df))

print("Projects represented:",
      predictive_dependencies_df["instance_id"].nunique())

Predictive dependency records: 12684
Projects represented: 16


In [ ]:
# ============================================================
# STEP 3.5.14 — UPDATE SNAPSHOT DEPENDENCY STATISTICS
# ============================================================

keys = [
    "instance_id",
    "week_number",
    "checkpoint_time"
]

dependency_counts = (
    predictive_dependencies_df.groupby(keys)
    .agg(
        predictive_dependency_links=(
            "upstream_issue", "count"
        ),
        predictive_blocked_issues=(
            "downstream_issue", "nunique"
        ),
        predictive_upstream_issues=(
            "upstream_issue", "nunique"
        )
    )
    .reset_index()
)

snapshot_manifest_predictive = (
    snapshot_manifest_final.merge(
        dependency_counts,
        on=keys,
        how="left",
        validate="one_to_one"
    )
)

new_cols = [
    "predictive_dependency_links",
    "predictive_blocked_issues",
    "predictive_upstream_issues"
]

snapshot_manifest_predictive[new_cols] = (
    snapshot_manifest_predictive[new_cols]
    .fillna(0)
    .astype(int)
)

print("Updated checkpoints:",
      len(snapshot_manifest_predictive))

display(
    snapshot_manifest_predictive[new_cols].describe()
)

print("\nPROJECT-LEVEL RESULTS")

display(
    snapshot_manifest_predictive.groupby("instance_id")
    .agg(
        original_peak=("active_dependency_links", "max"),
        predictive_peak=("predictive_dependency_links", "max")
    )
    .reset_index()
    .sort_values("original_peak", ascending=False)
)

Updated checkpoints: 987


,predictive_dependency_links,predictive_blocked_issues,predictive_upstream_issues
count,987.000000,987.000000,987.000000
mean,12.851064,6.579534,9.579534
std,45.544988,19.250366,32.119113
min,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000
75%,6.000000,5.000000,5.000000
max,353.000000,135.000000,256.000000



PROJECT-LEVEL RESULTS


,instance_id,original_peak,predictive_peak
13,JBIDE-I01,587,0
12,JBEAP-I01,353,353
6,ENTESB-I01,258,32
4,DAOS-I01,142,142
0,AEROGEAR-I01,39,39
9,HBASE-I01,28,28
2,BXMSDOC-I01,21,2
27,SPARK-I01,19,19
11,JBDS-I01,17,17
17,MC-I01,13,13


In [ ]:
# ============================================================
# STEP 3.5.15 — VALIDATE AND EXPORT
# ============================================================

checks = {}

checks["987 checkpoints"] = (
    len(snapshot_manifest_predictive) == 987
)

checks["30 projects"] = (
    snapshot_manifest_predictive["instance_id"].nunique() == 30
)

checks["No duplicate dependency pairs"] = (
    not predictive_dependencies_df.duplicated(
        [
            "instance_id",
            "week_number",
            "upstream_issue",
            "downstream_issue"
        ]
    ).any()
)

checks["No future dependency starts"] = (
    predictive_dependencies_df["active_from"]
    <= predictive_dependencies_df["checkpoint_time"]
).all()

checks["No excluded start events retained"] = (
    not predictive_dependencies_df["start_event_uid"].isin(
        intervals.loc[
            ~intervals["predictive_eligible"],
            "start_event_uid"
        ].dropna()
    ).any()
)

checks["Predictive counts do not exceed original"] = (
    snapshot_manifest_predictive["predictive_dependency_links"]
    <= snapshot_manifest_predictive["active_dependency_links"]
).all()

checks["No negative counts"] = (
    snapshot_manifest_predictive[new_cols] >= 0
).all().all()

validation = pd.DataFrame([
    {
        "validation": name,
        "status": "PASS" if passed else "FAIL"
    }
    for name, passed in checks.items()
])

display(validation)

assert all(checks.values()), "Validation failed"

from pathlib import Path

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/GC1/Temporal_snapshots"
)

predictive_dependencies_df.to_parquet(
    OUTPUT_DIR / "predictive_active_dependencies.parquet",
    index=False
)

snapshot_manifest_predictive.to_csv(
    OUTPUT_DIR / "weekly_snapshot_manifest_predictive.csv",
    index=False
)

print("ALL PREDICTIVE DEPENDENCY VALIDATIONS PASSED")
print("Filtered dependency snapshots exported.")

,validation,status
0,987 checkpoints,PASS
1,30 projects,PASS
2,No duplicate dependency pairs,PASS
3,No future dependency starts,PASS
4,No excluded start events retained,PASS
5,Predictive counts do not exceed original,PASS
6,No negative counts,PASS


ALL PREDICTIVE DEPENDENCY VALIDATIONS PASSED
Filtered dependency snapshots exported.


## Temporal Leakage & Integrity Checks

Validate timestamp ordering, endpoint consistency, checkpoint alignment and exclusion of future outcome fields.

In [ ]:
# ============================================================
# STEP 3.6.1 — LOAD TEMPORAL INPUTS (CORRECTED)
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/GC1")
OUTPUT_DIR = ROOT / "Temporal_snapshots"

assert OUTPUT_DIR.is_dir(), "Temporal_snapshots folder missing"

# ------------------------------------------------------------
# 1. LOAD TEMPORAL KG FILES
# ------------------------------------------------------------

nodes = pd.read_parquet(
    OUTPUT_DIR / "temporal_node_registry.parquet"
)

edges = pd.read_parquet(
    OUTPUT_DIR / "temporal_edge_registry.parquet"
)

manifest = pd.read_csv(
    OUTPUT_DIR / "weekly_snapshot_manifest_predictive.csv"
)

dependencies = pd.read_parquet(
    OUTPUT_DIR / "predictive_active_dependencies.parquet"
)

# ------------------------------------------------------------
# 2. LOAD HISTORICAL JIRA STATES
# ------------------------------------------------------------

jira_file = OUTPUT_DIR / "jira_checkpoint_issue_states_final.csv"

if jira_file.exists():

    jira_states = pd.read_csv(jira_file)
    print("Loaded Jira states from Temporal_snapshots.")

elif (
    "checkpoint_issue_states_final" in globals()
    and isinstance(
        globals()["checkpoint_issue_states_final"],
        pd.DataFrame
    )
):

    jira_states = checkpoint_issue_states_final.copy()
    print("Loaded Jira states from Colab memory.")

else:

    # Search existing GC1 directories
    matches = list(
        ROOT.rglob("jira_checkpoint_issue_states_final.csv")
    )

    if matches:
        jira_states = pd.read_csv(matches[0])
        print("Loaded Jira states from:", matches[0])

    else:
        raise FileNotFoundError(
            "Historical Jira states not found in Drive or memory. "
            "Rerun Step 3.3 to reconstruct them."
        )

# ------------------------------------------------------------
# 3. NORMALIZE TIMESTAMPS
# ------------------------------------------------------------

for df, columns in [
    (nodes, ["available_at"]),
    (edges, ["available_at"]),
    (manifest, ["checkpoint_time", "horizon_end"]),
    (dependencies, ["checkpoint_time", "active_from"]),
    (jira_states, ["checkpoint_time"])
]:

    for col in columns:

        df[col] = pd.to_datetime(
            df[col],
            utc=True,
            errors="coerce",
            format="mixed"
        )

KEYS = [
    "instance_id",
    "week_number",
    "checkpoint_time"
]

# ------------------------------------------------------------
# 4. VALIDATE INPUT COUNTS
# ------------------------------------------------------------

assert len(nodes) == 96487, "Unexpected KG node count"
assert len(edges) == 164101, "Unexpected KG edge count"
assert len(manifest) == 987, "Unexpected checkpoint count"

assert len(jira_states) == 228255, (
    f"Expected 228255 Jira states, found {len(jira_states)}"
)

assert jira_states["checkpoint_time"].notna().all()

# ------------------------------------------------------------
# 5. SAVE JIRA STATES FOR FUTURE USE
# ------------------------------------------------------------

if not jira_file.exists():

    jira_states.to_csv(
        jira_file,
        index=False
    )

    print("\nHistorical Jira states saved to:")
    print(jira_file)

# ------------------------------------------------------------
# 6. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("STEP 3.6 — INPUT LOADING COMPLETE")
print("=" * 60)

print("Temporal nodes:", len(nodes))
print("Temporal edges:", len(edges))
print("Checkpoints:", len(manifest))
print("Predictive dependencies:", len(dependencies))
print("Historical Jira states:", len(jira_states))
print("Projects:", manifest["instance_id"].nunique())

print("\nALL INPUT VALIDATIONS PASSED")

Loaded Jira states from Colab memory.

Historical Jira states saved to:
/content/drive/MyDrive/GC1/Temporal_snapshots/jira_checkpoint_issue_states_final.csv

STEP 3.6 — INPUT LOADING COMPLETE
Temporal nodes: 96487
Temporal edges: 164101
Checkpoints: 987
Predictive dependencies: 12684
Historical Jira states: 228255
Projects: 30

ALL INPUT VALIDATIONS PASSED


In [ ]:
# ============================================================
# STEP 3.6.2 — NODE AND EDGE TEMPORAL VALIDATION
# ============================================================

node_lookup = nodes.set_index("node_uid")

edges_audit = edges.copy()

edges_audit["source_available_at"] = (
    edges_audit["from_uid"].map(
        node_lookup["available_at"]
    )
)

edges_audit["target_available_at"] = (
    edges_audit["to_uid"].map(
        node_lookup["available_at"]
    )
)

edges_audit["source_instance"] = (
    edges_audit["from_uid"].map(
        node_lookup["instance_id"]
    )
)

edges_audit["target_instance"] = (
    edges_audit["to_uid"].map(
        node_lookup["instance_id"]
    )
)

edges_audit["invalid_time"] = (
    (edges_audit["available_at"] <
     edges_audit["source_available_at"])
    |
    (edges_audit["available_at"] <
     edges_audit["target_available_at"])
)

edges_audit["invalid_project"] = (
    (edges_audit["instance_id"] !=
     edges_audit["source_instance"])
    |
    (edges_audit["instance_id"] !=
     edges_audit["target_instance"])
)

print("EDGE TEMPORAL AUDIT")

print("Invalid temporal relationships:",
      edges_audit["invalid_time"].sum())

print("Cross-project inconsistencies:",
      edges_audit["invalid_project"].sum())

print("Missing endpoint timestamps:",
      edges_audit[
          ["source_available_at", "target_available_at"]
      ].isna().any(axis=1).sum())

display(
    edges_audit[
        edges_audit["invalid_time"]
        | edges_audit["invalid_project"]
    ].head(20)
)

EDGE TEMPORAL AUDIT
Invalid temporal relationships: 0
Cross-project inconsistencies: 0
Missing endpoint timestamps: 0


,edge_uid,instance_id,from_uid,to_uid,type,available_at,source_available_at,target_available_at,source_instance,target_instance,invalid_time,invalid_project


In [ ]:
# ============================================================
# STEP 3.6.3 — HISTORICAL JIRA STATUS LEAKAGE AUDIT
# ============================================================

assert "jira_history_df" in globals(), (
    "jira_history_df is required for status-history auditing. "
    "Run Step 3.3 preparation first."
)

assert "status_transitions" in globals(), (
    "status_transitions is required. "
    "Run Step 3.3 transition extraction first."
)

# Parse transitions
transition_audit = status_transitions.copy()

transition_audit["changed_dt"] = pd.to_datetime(
    transition_audit["changed_dt"],
    utc=True,
    errors="coerce",
    format="mixed"
)

# Use the last known TO status at or before checkpoint
transition_audit = transition_audit.sort_values(
    ["instance_id", "issue_key", "changed_dt",
     "transition_index"]
)

state_audit = jira_states.copy()

state_audit["_row_id"] = np.arange(len(state_audit))

# Merge transitions to matching issues
audit_join = state_audit[
    ["_row_id", "instance_id", "issue_key", "checkpoint_time"]
].merge(
    transition_audit[
        ["instance_id", "issue_key",
         "changed_dt", "to_status"]
    ],
    on=["instance_id", "issue_key"],
    how="left"
)

# Retain transitions observable at the checkpoint
audit_join = audit_join[
    audit_join["changed_dt"].isna()
    |
    (
        audit_join["changed_dt"]
        <= audit_join["checkpoint_time"]
    )
].copy()

latest = (
    audit_join.sort_values(
        ["_row_id", "changed_dt"]
    )
    .drop_duplicates("_row_id", keep="last")
)

latest = latest[
    ["_row_id", "to_status", "changed_dt"]
].rename(columns={
    "to_status": "latest_observed_status",
    "changed_dt": "latest_status_change"
})

state_audit = state_audit.merge(
    latest,
    on="_row_id",
    how="left",
    validate="one_to_one"
)

# A missing latest status means no prior transition
state_audit["strict_status"] = (
    state_audit["latest_observed_status"]
    .fillna("UNKNOWN")
)

state_audit["status_disagreement"] = (
    state_audit["historical_status"]
    .astype(str).str.lower()
    !=
    state_audit["strict_status"]
    .astype(str).str.lower()
)

print("JIRA STATUS LEAKAGE AUDIT")

print("Checkpoint states:", len(state_audit))

print(
    "States with no prior observable transition:",
    state_audit["latest_observed_status"].isna().sum()
)

print(
    "Disagreements with strict reconstruction:",
    state_audit["status_disagreement"].sum()
)

print("\nDisagreements by project:")

display(
    state_audit[
        state_audit["status_disagreement"]
    ].groupby("instance_id")
    .size()
    .reset_index(name="disagreements")
    .sort_values("disagreements", ascending=False)
)

display(
    state_audit[
        state_audit["status_disagreement"]
    ][
        [
            "instance_id",
            "issue_key",
            "checkpoint_time",
            "historical_status",
            "strict_status",
            "latest_status_change"
        ]
    ].head(20)
)

JIRA STATUS LEAKAGE AUDIT
Checkpoint states: 228255
States with no prior observable transition: 12727
Disagreements with strict reconstruction: 8439

Disagreements by project:


,instance_id,disagreements
14,JRACLOUD-I01,919
16,JSDCLOUD-I01,738
3,CONFSERVER-I01,706
23,QTCREATORBUG-I01,507
27,SPARK-I01,477
12,JBEAP-I01,477
4,DAOS-I01,471
9,HBASE-I01,468
25,SAK-I01,442
5,DOCS-I01,432


,instance_id,issue_key,checkpoint_time,historical_status,strict_status,latest_status_change
75,AEROGEAR-I01,AEROGEAR-5672,2018-01-24 05:30:01+00:00,Open,UNKNOWN,NaT
76,AEROGEAR-I01,AEROGEAR-5668,2018-01-24 05:30:01+00:00,Open,UNKNOWN,NaT
97,AEROGEAR-I01,AEROGEAR-5020,2018-01-24 05:30:01+00:00,Open,UNKNOWN,NaT
110,AEROGEAR-I01,AEROGEAR-4347,2018-01-24 05:30:01+00:00,Open,UNKNOWN,NaT
112,AEROGEAR-I01,AEROGEAR-4321,2018-01-24 05:30:01+00:00,Open,UNKNOWN,NaT
122,AEROGEAR-I01,AEROGEAR-3240,2018-01-24 05:30:01+00:00,Open,UNKNOWN,NaT
123,AEROGEAR-I01,AEROGEAR-3151,2018-01-24 05:30:01+00:00,Open,UNKNOWN,NaT
124,AEROGEAR-I01,AEROGEAR-1968,2018-01-24 05:30:01+00:00,Open,UNKNOWN,NaT
125,AEROGEAR-I01,AEROGEAR-1932,2018-01-24 05:30:01+00:00,Open,UNKNOWN,NaT
126,AEROGEAR-I01,AEROGEAR-1905,2018-01-24 05:30:01+00:00,Open,UNKNOWN,NaT


In [ ]:
# ============================================================
# STEP 3.6.4 — DEPENDENCY LEAKAGE AUDIT
# ============================================================

print("PREDICTIVE DEPENDENCY AUDIT")

# No future starts
future_dependency_starts = (
    dependencies["active_from"]
    >
    dependencies["checkpoint_time"]
).sum()

print("Future dependency starts:",
      future_dependency_starts)

# No duplicate pairs within checkpoints
duplicate_pairs = dependencies.duplicated(
    [
        "instance_id",
        "week_number",
        "upstream_issue",
        "downstream_issue"
    ]
).sum()

print("Duplicate active pairs:", duplicate_pairs)

# Check against original excluded events
if "intervals" in globals() and "predictive_eligible" in intervals.columns:

    excluded_start_ids = set(
        intervals.loc[
            ~intervals["predictive_eligible"],
            "start_event_uid"
        ].dropna()
    )

    retained_excluded = (
        dependencies["start_event_uid"]
        .isin(excluded_start_ids)
    ).sum()

    print("Excluded start events retained:",
          retained_excluded)

else:
    retained_excluded = None
    print(
        "Original classification unavailable: "
        "exclusion validation requires loading "
        "the classification table."
    )

# Project-level dependency coverage
display(
    dependencies.groupby("instance_id")
    .agg(
        checkpoint_records=("week_number", "count"),
        unique_upstream=("upstream_issue", "nunique"),
        unique_downstream=("downstream_issue", "nunique")
    )
    .reset_index()
    .sort_values("checkpoint_records", ascending=False)
)

PREDICTIVE DEPENDENCY AUDIT
Future dependency starts: 0
Duplicate active pairs: 0
Excluded start events retained: 0


,instance_id,checkpoint_records,unique_upstream,unique_downstream
9,JBEAP-I01,6955,265,138
3,DAOS-I01,2526,115,55
0,AEROGEAR-I01,818,34,20
4,ENTESB-I01,614,45,39
15,SPARK-I01,349,12,16
6,HBASE-I01,296,28,7
13,MC-I01,267,10,13
8,JBDS-I01,227,13,12
1,AMBARI-I01,193,7,6
7,INFRA-I01,175,6,11


In [ ]:
# ============================================================
# STEP 3.6.5 — LABEL LEAKAGE AUDIT
# ============================================================

FORBIDDEN_COLUMNS = {
    "schedule_risk_label",
    "scope_risk_label",
    "dependency_risk_label",
    "future_schedule_events",
    "future_affected_issues",
    "future_scope_events",
    "future_new_links",
    "future_affected_downstream",
    "future_affected_issues"
}

tables = {
    "nodes": nodes,
    "edges": edges,
    "manifest": manifest,
    "dependencies": dependencies,
    "jira_states": jira_states
}

leakage_records = []

for name, df in tables.items():

    leaked = sorted(
        set(df.columns) & FORBIDDEN_COLUMNS
    )

    leakage_records.append({
        "table": name,
        "forbidden_columns_found": leaked,
        "passed": len(leaked) == 0
    })

label_leakage_report = pd.DataFrame(leakage_records)

display(label_leakage_report)

,table,forbidden_columns_found,passed
0,nodes,[],True
1,edges,[],True
2,manifest,[],True
3,dependencies,[],True
4,jira_states,[],True


In [ ]:
# ============================================================
# STEP 3.6.6 — SNAPSHOT CONSISTENCY AUDIT
# ============================================================

# Recalculate dependency counts
recalculated = (
    dependencies.groupby(KEYS)
    .agg(
        recalculated_links=("upstream_issue", "count"),
        recalculated_blocked=("downstream_issue", "nunique"),
        recalculated_upstream=("upstream_issue", "nunique")
    )
    .reset_index()
)

manifest_audit = manifest.merge(
    recalculated,
    on=KEYS,
    how="left",
    validate="one_to_one"
)

for col in [
    "recalculated_links",
    "recalculated_blocked",
    "recalculated_upstream"
]:
    manifest_audit[col] = (
        manifest_audit[col].fillna(0).astype(int)
    )

count_mismatches = (
    (
        manifest_audit["predictive_dependency_links"]
        != manifest_audit["recalculated_links"]
    )
    |
    (
        manifest_audit["predictive_blocked_issues"]
        != manifest_audit["recalculated_blocked"]
    )
    |
    (
        manifest_audit["predictive_upstream_issues"]
        != manifest_audit["recalculated_upstream"]
    )
).sum()

print("SNAPSHOT COUNT AUDIT")

print("Manifest records:", len(manifest_audit))
print("Dependency count mismatches:", count_mismatches)

print("\nDependency count statistics:")
display(
    manifest_audit[
        [
            "predictive_dependency_links",
            "predictive_blocked_issues",
            "predictive_upstream_issues"
        ]
    ].describe()
)

SNAPSHOT COUNT AUDIT
Manifest records: 987
Dependency count mismatches: 0

Dependency count statistics:


,predictive_dependency_links,predictive_blocked_issues,predictive_upstream_issues
count,987.000000,987.000000,987.000000
mean,12.851064,6.579534,9.579534
std,45.544988,19.250366,32.119113
min,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000
75%,6.000000,5.000000,5.000000
max,353.000000,135.000000,256.000000


In [ ]:
# ============================================================
# STEP 3.6 — CONSOLIDATED VALIDATION REPORT
# ============================================================

checks = []

def check(name, passed, details):
    checks.append({
        "validation": name,
        "status": "PASS" if bool(passed) else "FAIL",
        "details": details
    })

check(
    "987 checkpoints",
    len(manifest) == 987,
    len(manifest)
)

check(
    "30 projects",
    manifest["instance_id"].nunique() == 30,
    manifest["instance_id"].nunique()
)

check(
    "Unique snapshot IDs",
    manifest["snapshot_id"].is_unique,
    "Checked"
)

check(
    "No duplicate checkpoints",
    not manifest.duplicated(KEYS).any(),
    "Checked"
)

check(
    "14-day prediction horizons",
    (
        manifest["horizon_end"]
        - manifest["checkpoint_time"]
        == pd.Timedelta(days=14)
    ).all(),
    "Checked"
)

check(
    "No invalid edge timestamps",
    edges_audit["invalid_time"].sum() == 0,
    int(edges_audit["invalid_time"].sum())
)

check(
    "No cross-project edge inconsistencies",
    edges_audit["invalid_project"].sum() == 0,
    int(edges_audit["invalid_project"].sum())
)

check(
    "No future dependency starts",
    future_dependency_starts == 0,
    int(future_dependency_starts)
)

check(
    "No duplicate dependency pairs",
    duplicate_pairs == 0,
    int(duplicate_pairs)
)

check(
    "No direct label leakage",
    label_leakage_report["passed"].all(),
    "Checked"
)

check(
    "Dependency manifest counts match",
    count_mismatches == 0,
    int(count_mismatches)
)

check(
    "All Jira states have checkpoint timestamps",
    jira_states["checkpoint_time"].notna().all(),
    "Checked"
)

check(
    "No future status transitions used in strict reconstruction",
    (
        state_audit["latest_status_change"].isna()
        |
        (
            state_audit["latest_status_change"]
            <= state_audit["checkpoint_time"]
        )
    ).all(),
    "Checked"
)

check(
    "No excluded dependency events retained",
    retained_excluded == 0 if retained_excluded is not None else False,
    retained_excluded if retained_excluded is not None
    else "NOT VERIFIED"
)

validation_report = pd.DataFrame(checks)

display(validation_report)

print("\n" + "=" * 65)

if (validation_report["status"] == "PASS").all():
    print("ALL STRUCTURAL AND TEMPORAL AUDITS PASSED")
else:
    print("SOME VALIDATIONS FAILED OR REQUIRE REVIEW")

print("=" * 65)

print("\nJira status disagreements requiring review:",
      int(state_audit["status_disagreement"].sum()))

,validation,status,details
0,987 checkpoints,PASS,987
1,30 projects,PASS,30
2,Unique snapshot IDs,PASS,Checked
3,No duplicate checkpoints,PASS,Checked
4,14-day prediction horizons,PASS,Checked
5,No invalid edge timestamps,PASS,0
6,No cross-project edge inconsistencies,PASS,0
7,No future dependency starts,PASS,0
8,No duplicate dependency pairs,PASS,0
9,No direct label leakage,PASS,Checked



ALL STRUCTURAL AND TEMPORAL AUDITS PASSED

Jira status disagreements requiring review: 8439


## 7. Final Snapshot Dataset Export

In [ ]:
# ============================================================
# STEP 3.7.1 — PREPARE FINAL SNAPSHOT EXPORT
# ============================================================

import pandas as pd
import numpy as np
import json
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/GC1")
OUTPUT_DIR = ROOT / "Temporal_snapshots"
FINAL_DIR = OUTPUT_DIR / "final_snapshots"

FINAL_DIR.mkdir(parents=True, exist_ok=True)

required_variables = [
    "nodes",
    "edges",
    "manifest",
    "jira_states",
    "dependencies",
    "validation_report"
]

missing = [
    name for name in required_variables
    if name not in globals()
]

assert not missing, (
    f"Missing variables: {missing}. "
    "Run Step 3.6 first."
)

# Do not alter the original DataFrames
final_nodes = nodes.copy()
final_edges = edges.copy()
final_manifest = manifest.copy()
final_jira = jira_states.copy()
final_dependencies = dependencies.copy()

# Normalize timestamps
for df, columns in [
    (final_nodes, ["available_at"]),
    (final_edges, ["available_at"]),
    (final_manifest, ["checkpoint_time", "horizon_end"]),
    (final_jira, ["checkpoint_time"]),
    (final_dependencies, ["checkpoint_time", "active_from"])
]:
    for col in columns:
        df[col] = pd.to_datetime(
            df[col],
            utc=True,
            errors="coerce",
            format="mixed"
        )

print("FINAL EXPORT INPUTS")
print("=" * 55)

print("Nodes:", len(final_nodes))
print("Edges:", len(final_edges))
print("Checkpoints:", len(final_manifest))
print("Jira states:", len(final_jira))
print("Predictive dependencies:", len(final_dependencies))

FINAL EXPORT INPUTS
Nodes: 96487
Edges: 164101
Checkpoints: 987
Jira states: 228255
Predictive dependencies: 12684


In [ ]:
# ============================================================
# STEP 3.7.2 — PREPARE LEAKAGE-SAFE TABLES
# ============================================================

KEYS = [
    "instance_id",
    "week_number",
    "checkpoint_time"
]

# ------------------------------------------------------------
# 1. Strict historical Jira status
# ------------------------------------------------------------

assert "state_audit" in globals(), (
    "state_audit not found. Run Step 3.6 Cell 3."
)

strict_jira = state_audit[
    KEYS + [
        "issue_key",
        "strict_status",
        "latest_status_change"
    ]
].copy()

strict_jira = strict_jira.rename(
    columns={"strict_status": "historical_status"}
)

assert len(strict_jira) == len(final_jira)

assert not strict_jira.duplicated(
    KEYS + ["issue_key"]
).any()

# ------------------------------------------------------------
# 2. Safe manifest
# ------------------------------------------------------------

manifest_columns = [
    "snapshot_id",
    "instance_id",
    "week_number",
    "checkpoint_time",
    "kg_nodes",
    "kg_edges",
    "historical_issues",
    "predictive_dependency_links",
    "predictive_blocked_issues",
    "predictive_upstream_issues"
]

missing_manifest = [
    c for c in manifest_columns
    if c not in final_manifest.columns
]

assert not missing_manifest, (
    f"Missing manifest columns: {missing_manifest}"
)

safe_manifest = final_manifest[
    manifest_columns
].copy()

# ------------------------------------------------------------
# 3. Safe dependency table
# ------------------------------------------------------------

dependency_columns = [
    "instance_id",
    "week_number",
    "checkpoint_time",
    "upstream_issue",
    "downstream_issue",
    "active_from",
    "start_event_uid"
]

safe_dependencies = final_dependencies[
    dependency_columns
].copy()

# ------------------------------------------------------------
# 4. Safe node and edge registries
# ------------------------------------------------------------

safe_nodes = final_nodes[
    [
        "node_uid",
        "instance_id",
        "label",
        "available_at",
        "source_id",
        "source_type",
        "temporal_basis",
        "temporal_status"
    ]
].copy()

safe_edges = final_edges[
    [
        "edge_uid",
        "instance_id",
        "from_uid",
        "to_uid",
        "type",
        "available_at"
    ]
].copy()

print("Leakage-safe tables prepared.")

print("Jira states:", len(strict_jira))
print("Dependencies:", len(safe_dependencies))
print("Nodes:", len(safe_nodes))
print("Edges:", len(safe_edges))
print("Manifest:", len(safe_manifest))

Leakage-safe tables prepared.
Jira states: 228255
Dependencies: 12684
Nodes: 96487
Edges: 164101
Manifest: 987


In [ ]:
# ============================================================
# STEP 3.7.3 — FINAL EXPORT VALIDATIONS
# ============================================================

checks = []

def check(name, passed, detail):
    checks.append({
        "validation": name,
        "status": "PASS" if bool(passed) else "FAIL",
        "details": detail
    })

# Previous validation
check(
    "Step 3.6 validations passed",
    (validation_report["status"] == "PASS").all(),
    "Previous validation report"
)

# Snapshot integrity
check(
    "987 checkpoints",
    len(safe_manifest) == 987,
    len(safe_manifest)
)

check(
    "30 projects",
    safe_manifest["instance_id"].nunique() == 30,
    safe_manifest["instance_id"].nunique()
)

check(
    "Unique snapshot IDs",
    safe_manifest["snapshot_id"].is_unique,
    "Checked"
)

check(
    "No duplicate project-weeks",
    not safe_manifest.duplicated(KEYS).any(),
    "Checked"
)

# Node and edge integrity
node_ids = set(safe_nodes["node_uid"])

check(
    "Unique node IDs",
    safe_nodes["node_uid"].is_unique,
    "Checked"
)

check(
    "Unique edge IDs",
    safe_edges["edge_uid"].is_unique,
    "Checked"
)

check(
    "All edge endpoints exist",
    (
        safe_edges["from_uid"].isin(node_ids)
        &
        safe_edges["to_uid"].isin(node_ids)
    ).all(),
    "Checked"
)

check(
    "All nodes timestamped",
    safe_nodes["available_at"].notna().all(),
    "Checked"
)

check(
    "All edges timestamped",
    safe_edges["available_at"].notna().all(),
    "Checked"
)

# Historical Jira integrity
check(
    "Jira state count preserved",
    len(strict_jira) == 228255,
    len(strict_jira)
)

check(
    "No duplicate Jira checkpoint states",
    not strict_jira.duplicated(
        KEYS + ["issue_key"]
    ).any(),
    "Checked"
)

check(
    "No future Jira status transitions",
    (
        strict_jira["latest_status_change"].isna()
        |
        (
            strict_jira["latest_status_change"]
            <= strict_jira["checkpoint_time"]
        )
    ).all(),
    "Checked"
)

# Dependency integrity
check(
    "No future dependency starts",
    (
        safe_dependencies["active_from"]
        <= safe_dependencies["checkpoint_time"]
    ).all(),
    "Checked"
)

check(
    "No duplicate active dependency pairs",
    not safe_dependencies.duplicated(
        KEYS + ["upstream_issue", "downstream_issue"]
    ).any(),
    "Checked"
)

# Checkpoint alignment
valid_keys = set(
    map(
        tuple,
        safe_manifest[KEYS].itertuples(
            index=False, name=None
        )
    )
)

for name, df in [
    ("Jira", strict_jira),
    ("Dependency", safe_dependencies)
]:
    table_keys = set(
        map(
            tuple,
            df[KEYS].itertuples(
                index=False, name=None
            )
        )
    )

    check(
        f"{name} checkpoints align",
        table_keys.issubset(valid_keys),
        len(table_keys)
    )

# Forbidden columns
FORBIDDEN = {
    "schedule_risk_label",
    "scope_risk_label",
    "dependency_risk_label",
    "future_new_links",
    "future_scope_events",
    "future_schedule_events",
    "future_affected_issues",
    "future_affected_downstream",
    "status_uncertain",
    "recorded_historical_status",
    "feature_safe_status",
    "active_to",
    "end_event_uid"
}

for name, df in [
    ("Nodes", safe_nodes),
    ("Edges", safe_edges),
    ("Manifest", safe_manifest),
    ("Jira", strict_jira),
    ("Dependencies", safe_dependencies)
]:
    found = sorted(set(df.columns) & FORBIDDEN)

    check(
        f"No forbidden columns in {name}",
        len(found) == 0,
        found
    )

# ------------------------------------------------------------
# REPORT
# ------------------------------------------------------------

final_validation_report = pd.DataFrame(checks)

display(final_validation_report)

failed = final_validation_report[
    final_validation_report["status"] != "PASS"
]

print("\n" + "=" * 65)

if failed.empty:
    print("ALL FINAL SNAPSHOT EXPORT VALIDATIONS PASSED")
else:
    print(f"{len(failed)} FINAL VALIDATIONS FAILED")

print("=" * 65)

,validation,status,details
0,Step 3.6 validations passed,PASS,Previous validation report
1,987 checkpoints,PASS,987
2,30 projects,PASS,30
3,Unique snapshot IDs,PASS,Checked
4,No duplicate project-weeks,PASS,Checked
5,Unique node IDs,PASS,Checked
6,Unique edge IDs,PASS,Checked
7,All edge endpoints exist,PASS,Checked
8,All nodes timestamped,PASS,Checked
9,All edges timestamped,PASS,Checked



ALL FINAL SNAPSHOT EXPORT VALIDATIONS PASSED


In [ ]:
# ============================================================
# STEP 3.7.4 — EXPORT FINAL TEMPORAL SNAPSHOT DATASET
# ============================================================

assert (
    final_validation_report["status"] == "PASS"
).all(), "Final validation failed."

# Save data tables
safe_nodes.to_parquet(
    FINAL_DIR / "nodes.parquet",
    index=False
)

safe_edges.to_parquet(
    FINAL_DIR / "edges.parquet",
    index=False
)

strict_jira.to_parquet(
    FINAL_DIR / "jira_states.parquet",
    index=False
)

safe_dependencies.to_parquet(
    FINAL_DIR / "active_dependencies.parquet",
    index=False
)

safe_manifest.to_parquet(
    FINAL_DIR / "snapshot_manifest.parquet",
    index=False
)

# Save validation report
final_validation_report.to_csv(
    FINAL_DIR / "validation_report.csv",
    index=False
)

# Save dataset metadata
metadata = {
    "dataset_name": "GC1 Temporal KG Snapshots",
    "phase": "Phase 2",
    "notebook": "Notebook 1",
    "snapshot_count": int(len(safe_manifest)),
    "project_count": int(
        safe_manifest["instance_id"].nunique()
    ),
    "node_count": int(len(safe_nodes)),
    "edge_count": int(len(safe_edges)),
    "jira_checkpoint_states": int(len(strict_jira)),
    "dependency_checkpoint_records": int(
        len(safe_dependencies)
    ),
    "prediction_horizon_days": 14,
    "snapshot_frequency": "weekly",
    "decision_risk_included": False,
    "risk_types": [
        "Schedule Risk",
        "Scope Risk",
        "Dependency Risk"
    ],
    "temporal_policy": (
        "Point-in-time event availability with "
        "strict historical Jira status reconstruction"
    ),
    "limitations": [
        "Generated meeting and PRD events excluded",
        "Unknown-start dependency intervals excluded",
        "Suspected administrative dependency additions excluded",
        "Historical status UNKNOWN where no prior transition exists",
        "Canonical entity identity is retrospectively resolved",
        "Full Fact-Evidence-SourceRecord graph not reconstructed"
    ]
}

with open(
    FINAL_DIR / "dataset_metadata.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(metadata, f, indent=2)

print("=" * 65)
print("FINAL TEMPORAL SNAPSHOT EXPORT COMPLETE")
print("=" * 65)

print("\nOutput directory:")
print(FINAL_DIR)

print("\nFiles exported:")

for file in sorted(FINAL_DIR.iterdir()):
    print(f"- {file.name}")

print("\nSnapshots:", len(safe_manifest))
print("Projects:", safe_manifest["instance_id"].nunique())
print("Nodes:", len(safe_nodes))
print("Edges:", len(safe_edges))
print("Jira states:", len(strict_jira))
print("Dependencies:", len(safe_dependencies))

FINAL TEMPORAL SNAPSHOT EXPORT COMPLETE

Output directory:
/content/drive/MyDrive/GC1/Temporal_snapshots/final_snapshots

Files exported:
- active_dependencies.parquet
- dataset_metadata.json
- edges.parquet
- jira_states.parquet
- nodes.parquet
- snapshot_manifest.parquet
- validation_report.csv

Snapshots: 987
Projects: 30
Nodes: 96487
Edges: 164101
Jira states: 228255
Dependencies: 12684


In [ ]:
# ============================================================
# STEP 3.7.5 — VERIFY EXPORTED DATASET
# ============================================================

saved_nodes = pd.read_parquet(
    FINAL_DIR / "nodes.parquet"
)

saved_edges = pd.read_parquet(
    FINAL_DIR / "edges.parquet"
)

saved_jira = pd.read_parquet(
    FINAL_DIR / "jira_states.parquet"
)

saved_dependencies = pd.read_parquet(
    FINAL_DIR / "active_dependencies.parquet"
)

saved_manifest = pd.read_parquet(
    FINAL_DIR / "snapshot_manifest.parquet"
)

verification = {
    "Node count": len(saved_nodes) == len(safe_nodes),
    "Edge count": len(saved_edges) == len(safe_edges),
    "Jira state count": len(saved_jira) == len(strict_jira),
    "Dependency count": (
        len(saved_dependencies) == len(safe_dependencies)
    ),
    "Manifest count": len(saved_manifest) == 987,
    "Project coverage": (
        saved_manifest["instance_id"].nunique() == 30
    ),
    "Unique snapshot IDs": (
        saved_manifest["snapshot_id"].is_unique
    )
}

verification_df = pd.DataFrame([
    {
        "check": name,
        "status": "PASS" if result else "FAIL"
    }
    for name, result in verification.items()
])

display(verification_df)

assert all(verification.values()), (
    "Saved dataset verification failed."
)

print("\nFINAL DATASET VERIFIED SUCCESSFULLY")

,check,status
0,Node count,PASS
1,Edge count,PASS
2,Jira state count,PASS
3,Dependency count,PASS
4,Manifest count,PASS
5,Project coverage,PASS
6,Unique snapshot IDs,PASS



FINAL DATASET VERIFIED SUCCESSFULLY


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/GC1/Temporal_snapshots")

keywords = [
    "dependency",
    "blocking",
    "link_event",
    "escalation",
    "eligible"
]

print("POTENTIAL DEPENDENCY EVENT FILES\n")

for path in sorted(BASE.rglob("*")):
    if path.is_file() and any(
        word in path.name.lower() for word in keywords
    ):
        print(path)

Mounted at /content/drive
POTENTIAL DEPENDENCY EVENT FILES

/content/drive/MyDrive/GC1/Temporal_snapshots/Labels/dependency_escalation_labels.csv
/content/drive/MyDrive/GC1/Temporal_snapshots/catboost_experiment/cv_predictions/enhanced_41__dependency_risk_label__fold1.parquet
/content/drive/MyDrive/GC1/Temporal_snapshots/catboost_experiment/cv_predictions/enhanced_41__dependency_risk_label__fold2.parquet
/content/drive/MyDrive/GC1/Temporal_snapshots/catboost_experiment/cv_predictions/enhanced_41__dependency_risk_label__fold3.parquet
/content/drive/MyDrive/GC1/Temporal_snapshots/catboost_experiment/cv_predictions/enhanced_41__dependency_risk_label__fold4.parquet
/content/drive/MyDrive/GC1/Temporal_snapshots/catboost_experiment/cv_predictions/original_34__dependency_risk_label__fold1.parquet
/content/drive/MyDrive/GC1/Temporal_snapshots/catboost_experiment/cv_predictions/original_34__dependency_risk_label__fold2.parquet
/content/drive/MyDrive/GC1/Temporal_snapshots/catboost_experiment/cv

In [ ]:
import pandas as pd
from pathlib import Path

BASE = Path("/content/drive/MyDrive/GC1/Temporal_snapshots")

candidate_files = [
    "dependency_outcome_candidates.jsonl",
    "dependency_reviewed_outcomes.jsonl",
    "dependency_episode_outcomes.jsonl",
    "dependency_evidence_review.csv",
    "dependency_unknown_start_audit.csv",
    "dependency_checkpoint_manifest.csv",
]

for filename in candidate_files:

    path = BASE / filename

    print("\n" + "=" * 75)
    print("FILE:", filename)
    print("=" * 75)

    if not path.exists():
        print("File not found.")
        continue

    try:
        if path.suffix == ".jsonl":
            df = pd.read_json(path, lines=True)
        else:
            df = pd.read_csv(path)

        print("Records:", len(df))
        print("Columns:", df.columns.tolist())

        print("\nSample records:")
        display(df.head(3))

    except Exception as e:
        print("Error:", str(e))


FILE: dependency_outcome_candidates.jsonl
Records: 91
Columns: ['event_uid', 'instance_id', 'downstream_issue', 'outcome_time', 'disruption_type', 'upstream_issue_ids', 'upstream_statuses', 'blocking_evidence_ids', 'outcome_status', 'risk_type', 'label_basis', 'evidence_review_status']

Sample records:


,event_uid,instance_id,downstream_issue,outcome_time,disruption_type,upstream_issue_ids,upstream_statuses,blocking_evidence_ids,outcome_status,risk_type,label_basis,evidence_review_status
0,T_c61b17b353e492806f08b590,JBEAP-I01,JBEAP-3969,2016-05-31 12:02:09+00:00,REOPENED_WORK,[JBEAP-4646],[],[T_097cba8729cf966a92a2e495],INSUFFICIENT_UPSTREAM_EVIDENCE,DEPENDENCY,TEMPORAL_ASSOCIATION_PROXY,PENDING
1,T_645c755ca868ae7196cd7ed3,JBEAP-I01,JBEAP-3154,2016-02-23 09:34:12+00:00,REOPENED_WORK,[WFCORE-1356],[],[T_9a1010efba88ef5c001a9cbf],INSUFFICIENT_UPSTREAM_EVIDENCE,DEPENDENCY,TEMPORAL_ASSOCIATION_PROXY,PENDING
2,T_e203021d0ec6ca3296aa71fe,JBEAP-I01,JBEAP-3154,2016-02-25 06:26:39+00:00,REOPENED_WORK,[WFCORE-1356],[],[T_9a1010efba88ef5c001a9cbf],INSUFFICIENT_UPSTREAM_EVIDENCE,DEPENDENCY,TEMPORAL_ASSOCIATION_PROXY,PENDING



FILE: dependency_reviewed_outcomes.jsonl
Records: 29
Columns: ['event_uid', 'instance_id', 'downstream_issue', 'outcome_time', 'disruption_type', 'upstream_issue_ids', 'upstream_statuses', 'blocking_evidence_ids', 'outcome_status', 'risk_type', 'label_basis', 'evidence_review_status', 'evidence_class', 'review_notes', 'comment_id', 'evidence_time', 'evidence_issue', 'evidence_text', 'previous_event_time', 'days_since_previous', 'new_episode', 'episode_number', 'episode_id']

Sample records:


,event_uid,instance_id,downstream_issue,outcome_time,disruption_type,upstream_issue_ids,upstream_statuses,blocking_evidence_ids,outcome_status,risk_type,...,review_notes,comment_id,evidence_time,evidence_issue,evidence_text,previous_event_time,days_since_previous,new_episode,episode_number,episode_id
0,T_c5a3c203511123019f4cf53b,DAOS-I01,DAOS-4823,2021-01-20 19:26:13+00:00,BLOCKED_STATUS,"[DAOS-6462, DAOS-6510]",[open],"[T_3f783ecba111b6dbc2d64eb5, T_4da17a18310f853...",TEMPORALLY_SUPPORTED,DEPENDENCY,...,,NaN,NaT,None,None,NaT,NaN,True,1,DAOS-I01::DAOS-4823::E1
1,T_5c8facaa5312b00dea3049fe,DAOS-I01,DAOS-4836,2021-01-20 19:26:00+00:00,BLOCKED_STATUS,[DAOS-6510],[open],[T_551bbd2f36a04d1e27386929],TEMPORALLY_SUPPORTED,DEPENDENCY,...,Test failure explicitly attributed to DAOS-6510.,34146.0,2021-01-15 01:01:24+00:00,DAOS-4836,tart testing on daos container redundant facto...,NaT,NaN,True,1,DAOS-I01::DAOS-4836::E1
2,T_659d8724f9ff8b7644d9066a,DAOS-I01,DAOS-4837,2021-01-20 19:26:22+00:00,BLOCKED_STATUS,[DAOS-6510],[open],[T_d44f7f92f512953299c27aa1],TEMPORALLY_SUPPORTED,DEPENDENCY,...,Test failure explicitly attributed to DAOS-6510.,34145.0,2021-01-15 01:00:18+00:00,DAOS-4837,Start testing on daos container redundant fact...,NaT,NaN,True,1,DAOS-I01::DAOS-4837::E1



FILE: dependency_episode_outcomes.jsonl
Records: 22
Columns: ['episode_id', 'instance_id', 'downstream_issue', 'outcome_time', 'event_uids', 'upstream_issue_ids', 'comment_ids', 'evidence_class', 'event_count', 'risk_type', 'review_status']

Sample records:


...

,event_uid,instance_id,upstream_issue,downstream_issue,disruption_time,disruption_type,upstream_status,has_direct_mention,direct_mentions,review_decision,review_notes,evidence_source_ids
0,T_7ad42d7799a9b2bdcd0d137f,JBEAP-I01,JBEAP-2499,JBEAP-2544,2016-01-12 07:26:52+00:00,REOPENED_WORK,new,False,[],PENDING,NaN,NaN
1,T_3fa7a9ef832fd67b02be4d60,JBEAP-I01,JBEAP-2499,JBEAP-2544,2016-01-19 02:20:43+00:00,REOPENED_WORK,new,False,[],PENDING,NaN,NaN
2,T_26bce03fe181b51967caede5,JBEAP-I01,JBEAP-2499,JBEAP-2544,2016-01-19 02:23:10+00:00,REOPENED_WORK,new,False,[],PENDING,NaN,NaN



FILE: dependency_unknown_start_audit.csv
Records: 808
Columns: ['instance_id', 'upstream_issue', 'downstream_issue', 'active_from', 'first_confirmed_at', 'active_to', 'start_known', 'start_event_uid', 'end_event_uid', 'temporal_class']

Sample records:


,instance_id,upstream_issue,downstream_issue,active_from,first_confirmed_at,active_to,start_known,start_event_uid,end_event_uid,temporal_class
0,BXMSDOC-I01,BXMSDOC-1119,BXMSDOC-407,NaN,2020-09-14 05:41:51+00:00,NaN,False,T_ae91c5df906c390e6caaaced,NaN,UNKNOWN_START
1,BXMSDOC-I01,BXMSDOC-1429,BXMSDOC-1317,NaN,2020-09-14 05:41:47+00:00,NaN,False,T_3347d0641ef035f0821b112b,NaN,UNKNOWN_START
2,BXMSDOC-I01,BXMSDOC-1430,BXMSDOC-1317,NaN,2020-09-14 05:41:47+00:00,NaN,False,T_b61cec9756490f6a2498359c,NaN,UNKNOWN_START



FILE: dependency_checkpoint_manifest.csv
Records: 987
Columns: ['instance_id', 'week_number', 'checkpoint_time', 'active_dependency_links', 'blocked_downstream_issues', 'unique_upstream_issues']

Sample records:


,instance_id,week_number,checkpoint_time,active_dependency_links,blocked_downstream_issues,unique_upstream_issues
0,AEROGEAR-I01,1,2018-01-24 05:30:01+00:00,0,0,0
1,AEROGEAR-I01,2,2018-01-31 05:30:01+00:00,0,0,0
2,AEROGEAR-I01,3,2018-02-07 05:30:01+00:00,0,0,0
